# From Risk Scores to Reasons: reproducible external-score stress test

This notebook reproduces the empirical results used in the credit-explanation workbench.

The policy question is specific: if a consumer-facing reason depends on an external score, how much can a lender support from the model it can inspect, and what still requires records from the score provider? The notebook measures the research systems used for that test. It does **not** reconstruct FICO's proprietary score formula, a named lender's production model, or an actual adverse-action notice.

The raw HELOC CSV is not redistributed. Supply an authorized copy of `heloc_dataset_v1.csv` when prompted.


In [ ]:
from pathlib import Path
import json, sys, subprocess, os
ROOT=Path('from-risk-scores-to-reasons').resolve()
ROOT.mkdir(exist_ok=True)
FILES={'README.md': '# From Risk Scores to Reasons\n\n**A research workbench for reviewing credit explanations when a lender model depends on an external score.**\n\nA person denied credit may receive only a few stated reasons. Those reasons can sit at the end of several separate systems: the lender\'s model, an external score, an explanation method, and a mapping from model evidence to notice language. This project asks a narrower question than "is the model explainable?": **what evidence is enough to support the reason given to the applicant?**\n\nThe empirical test focuses on an external score because the FICO HELOC data provide the score value but not the formula or provider reasoning behind it. I reconstruct that score in two different ways and ask whether close downstream predictions also produce the same explanations.\n\nThis is research software. It is not a lending system, legal opinion, compliance certificate, or reconstruction of a named lender\'s production model.\n\n## Main result\n\nThe downstream model is a Random Forest that uses eight credit attributes plus the observed `ExternalRiskEstimate`. I estimate the external score with standardized Ridge regression and histogram gradient boosting, then substitute each estimate into the **same frozen Random Forest**.\n\nAmong **12 selected records** where all three repayment-risk predictions were within **1 percentage point**, **4/12** had different leading individual-factor sets. **0/12** had different leading broader credit-topic sets.\n\nIn other words, the reconstructions can agree on the prediction and on the broad topic while disagreeing about the detailed factors. That does not tell us which factors the unavailable upstream score provider actually used. The counts are exploratory; they are not population estimates or legal thresholds.\n\n## What the workbench is for\n\nThe site is organized around review questions rather than around SHAP or LIME. It includes:\n\n- a **Rules → tests** table that maps governance sources to claims, protocol tests and missing records;\n- record-level inspection of the supplied score\'s contribution to the research model;\n- side-by-side explanations from the Ridge and boosted-score reconstructions;\n- a **five-test validation protocol** covering decision fidelity, reason consistency, mapping integrity, provenance and auditability;\n- study-wide sensitivity bands for prediction/explanation agreement; and\n- an exportable review note that separates measured findings from evidence gaps.\n\nThe workbench permits outcomes such as **MEASURED**, **PARTIAL**, **NOT ESTABLISHED**, and **EXTERNAL EVIDENCE REQUIRED**. It does not force every review question into a numerical explanation score.\n\n## Policy implication\n\nIf a consumer-facing reason depends on a material external score, the creditor should be able to distinguish what its own model establishes from what it can support about the score provider. A reconstructed score can be useful for testing sensitivity, but it should not stand in for provider-linked reasons or validation records.\n\nThat leads to four practical recommendations:\n\n1. retain the score/model versions, explanation settings, reason mapping and notice output used for the decision under review;\n2. label reconstructed upstream factors as inference unless provider records support them;\n3. validate the detailed factors behind a broad notice category, not only the stability of the category label; and\n4. allow "not established from available evidence" as a review result when the record is incomplete.\n\nSee [`docs/POLICY_RECOMMENDATIONS.md`](docs/POLICY_RECOMMENDATIONS.md) for the policy, consumer-advocate and supervisory discussion.\n\n## Explore the site\n\nThe site is static. It has no backend, API key, visitor-data upload or browser-side model execution.\n\n```bash\npython -m http.server 8000 --directory site\n```\n\nThen open `http://localhost:8000`.\n\nThe repository also includes a self-contained single-file version, `From_Risk_Scores_to_Reasons_Workbench.html`.\n\n## Reproduce in Colab / Jupyter\n\nOpen [`notebooks/03_reproduce_workbench.ipynb`](notebooks/03_reproduce_workbench.ipynb). The notebook stages the project code, asks for an authorized copy of `heloc_dataset_v1.csv`, installs pinned dependencies, runs the empirical pipeline, rebuilds the site and checks the expected headline result.\n\nThe raw HELOC data are **not redistributed**. Follow the original AIX360/FICO data instructions and verify the applicable data terms.\n\nExpected headline check after reproduction:\n\n```text\n1.0 pp agreement band: 12 records\ndifferent detailed-factor sets: 4\ndifferent grouped-topic sets: 0\n```\n\n## Local reproduction\n\nTested target: Python 3.12.\n\n```bash\npython -m pip install -r requirements-reference.txt\npython src/audit.py --data data/heloc_dataset_v1.csv --out results/reference --cases 40 --repeats 5\npython src/notice_benchmark.py\npython src/reference_experiment.py\npython src/challenge_examples.py\npython src/external_score_experiment.py\npython src/grouping_analysis.py\npython src/decision_reconstruction.py\nOPENBLAS_NUM_THREADS=2 OMP_NUM_THREADS=2 python src/score_reconstruction.py\npython src/build_workbench.py\npython -m unittest discover -s tests -p \'test_*.py\'\nnode tests/test_site_smoke.cjs .\n```\n\n## Methods\n\nThe research model is a nine-input Random Forest predicting the historical HELOC `Bad` repayment category. Ridge regression and histogram gradient boosting estimate the observed external score from the other eight credit attributes. Each estimate is substituted into the frozen Random Forest. Exact empirical interventional Shapley decompositions compare the resulting systems against the same 24 reference-record identities. A custom LIME-style weighted local regression is repeated near each record as a local-fidelity diagnostic. The displayed calculations are custom implementations; the site does not claim that the official SHAP or LIME packages produced them.\n\n## Limits\n\nThe project can measure how the supplied score contributes to the research Random Forest, how well the two reconstructions predict the observed score, how their downstream predictions compare, and how their own feature attributions differ. It cannot establish the original score provider\'s internal factors, validate an actual lender notice, determine a legal violation, or estimate population prevalence from the 40 selected detailed cases. It also does not evaluate protected-group fairness, consumer comprehension or reviewer effectiveness.\n\n## Repository map\n\n```text\nsite/                 static workbench and precomputed results\nsrc/                  experiment and site-build code\nnotebooks/            Colab/Jupyter reproduction\ndocs/                 methods, governance, legal sources and limits\nresults/              small checked-in protocol/benchmark artifacts\ntests/                Python + Node tests\nposter/               poster source/build materials\n```\n\nUseful entry points:\n\n- `site/index.html` — interactive workbench\n- `site/app.js` — client-side review logic and evidence export\n- `src/score_reconstruction.py` — external-score reconstruction experiment\n- `src/build_workbench.py` — rebuilds `site/data.js` and the standalone preview\n- `docs/POLICY_RECOMMENDATIONS.md` — policy, consumer-advocate and supervisory discussion\n- `docs/EXTERNAL_SCORE_RECONSTRUCTION.md` — research protocol and technical interpretation\n\n## Citation / authorship\n\nMaitreyee Singh · MA Applied Ethics & Technology Policy · Duke University, 2026.\n\nIf you reuse the research framing or software, please cite the repository and note that the analysis is exploratory.\n', 'requirements-reference.txt': 'pandas==2.2.3\nnumpy==2.3.5\nscikit-learn==1.8.0\njoblib==1.5.3\nmatplotlib==3.10.8\nreportlab==4.4.9\n', 'requirements.txt': '-r requirements-reference.txt\nshap>=0.47,<1\nlime==0.2.0.1\n', 'COLAB.md': '# Colab reproduction\n\nUse `notebooks/03_reproduce_workbench.ipynb`. The notebook is intentionally portable: it stages the repository files inside the Colab runtime, then asks you to upload `heloc_dataset_v1.csv`.\n\nThe notebook does **not** download or redistribute the raw HELOC dataset. Verify that you are authorized to use the file and that its schema matches the project expectations.\n\nThe final verification cell checks the project’s headline 1.0 percentage-point agreement result: 12 close-prediction records, 4 with different leading detailed-factor sets, and 0 with different grouped-topic sets.\n', 'CITATION.cff': 'cff-version: 1.2.0\nmessage: "If you use this software or research framing, please cite it."\ntitle: "From Risk Scores to Reasons"\ntype: software\nauthors:\n  - family-names: Singh\n    given-names: Maitreyee\nyear: 2026\nabstract: "An explanation-assurance workbench for credit decisions that depend on external scores."\n', '.gitignore': '# data / generated artifacts\ndata/*.csv\nresults/reference/\nresults/official/\nresults/external_score/\nresults/score_reconstruction/\nresults/reference_sensitivity/\n__pycache__/\n*.pyc\n.ipynb_checkpoints/\n.DS_Store\n', 'src/audit.py': '"""Evidence-linked explanation audit. Official library mode or disclosed reference pilot.\nRaw CSV is never redistributed. Counterfactual or notice legality is not inferred.\n"""\nfrom pathlib import Path\nimport argparse, hashlib, itertools, json, math, platform, time\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.linear_model import Ridge\nfrom sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.calibration import calibration_curve\n\nFEATURES=[\'MSinceOldestTradeOpen\',\'AverageMInFile\',\'NumSatisfactoryTrades\',\'PercentTradesNeverDelq\',\n          \'MSinceMostRecentDelq\',\'NumInqLast6M\',\'NetFractionRevolvingBurden\',\'NetFractionInstallBurden\']\nGROUPS={\'MSinceOldestTradeOpen\':\'Credit history\',\'AverageMInFile\':\'Credit history\',\n        \'NumSatisfactoryTrades\':\'Repayment history\',\'PercentTradesNeverDelq\':\'Repayment history\',\n        \'MSinceMostRecentDelq\':\'Repayment history\',\'NumInqLast6M\':\'Recent inquiries\',\n        \'NetFractionRevolvingBurden\':\'Revolving balances\',\'NetFractionInstallBurden\':\'Installment balances\',\n        \'ExternalRiskEstimate\':\'External risk score\'}\nLABELS={\'MSinceOldestTradeOpen\':\'Oldest account age\',\'AverageMInFile\':\'Average account age\',\n        \'NumSatisfactoryTrades\':\'Satisfactory accounts\',\'PercentTradesNeverDelq\':\'Never-delinquent share\',\n        \'MSinceMostRecentDelq\':\'Time since delinquency\',\'NumInqLast6M\':\'Recent inquiries\',\n        \'NetFractionRevolvingBurden\':\'Revolving balance ratio\',\'NetFractionInstallBurden\':\'Installment balance ratio\',\n        \'ExternalRiskEstimate\':\'External risk estimate\'}\nSEED=2026\n\nclass Encoding:\n    """Numeric value and separate -7/-8/-9 indicators. No ordinal interpretation of sentinels."""\n    def fit(self, a):\n        self.medians=np.array([np.median(c[c>=0]) for c in np.asarray(a).T]);return self\n    def transform(self,a):\n        a=np.asarray(a,dtype=float)\n        numeric=np.where(a<0,self.medians,a)\n        return np.column_stack([numeric,a==-7,a==-8,a==-9]).astype(float)\n\nclass Predictor:\n    def __init__(self,model,enc):self.model,self.enc=model,enc\n    def __call__(self,a):return self.model.predict_proba(self.enc.transform(a))[:,1]\n    def probabilities(self,a):\n        p=self(a);return np.column_stack([1-p,p])\n\n\ndef exact_shapley(x,background,predict):\n    """Exact empirical interventional Shapley; coalition replacement may break dependence.\n    Each term uses the identical fixed empirical background. No TreeSHAP approximation.\n    """\n    d=len(x);m=len(background);n=1<<d\n    masks=((np.arange(n)[:,None]>>np.arange(d))&1).astype(bool)\n    rows=np.where(masks[:,None,:],np.asarray(x)[None,None,:],background[None,:,:])\n    values=predict(rows.reshape(-1,d)).reshape(n,m).mean(axis=1)\n    phi=np.zeros(d)\n    for j in range(d):\n        for s in range(n):\n            if not (s>>j)&1:\n                k=s.bit_count();weight=1/(d*math.comb(d-1,k))\n                phi[j]+=weight*(values[s|(1<<j)]-values[s])\n    return phi,{\'base\':float(values[0]),\'residual\':float(values[-1]-values[0]-phi.sum())}\n\n\ndef local_surrogate(x,train,predict,seed,n=1200):\n    """LIME-style pilot: empirical marginal perturbations and mismatch kernel.\n    Not the LIME package; coefficients measure local sensitivity, not point attribution.\n    """\n    rng=np.random.default_rng(seed);d=len(x)\n    z=np.column_stack([rng.choice(train[:,j],n) for j in range(d)])\n    same=rng.random((n,d))<.55;z=np.where(same,x,z);z[0]=x\n    scale=np.std(np.where(train<0,np.nan,train),axis=0) # replaced below with nan-aware\n    scale=np.nanstd(np.where(train<0,np.nan,train),axis=0);scale=np.maximum(scale,1)\n    delta=(z-x)/scale\n    # Special states are categorical differences, not numerical distances.\n    delta=np.where((z<0)|(x<0),np.where(z==x,0,1),delta)\n    weights=np.exp(-np.mean(delta**2,axis=1)/(.75**2))\n    # Interpretable representation: membership in the same empirical quartile/status as x.\n    q=np.array([np.quantile(train[:,j][train[:,j]>=0],[.25,.5,.75]) for j in range(d)])\n    def bins(a):\n        return np.column_stack([np.where(a[:,j]<0,a[:,j],np.searchsorted(q[j],a[:,j])) for j in range(d)])\n    reps=(bins(z)==bins(np.asarray(x)[None,:])).astype(float)\n    y=predict(z);fit=Ridge(alpha=1).fit(reps,y,sample_weight=weights)\n    return fit.coef_,{\'local_fit_r2\':float(fit.score(reps,y,sample_weight=weights)),\n                     \'local_prediction\':float(fit.predict(np.ones((1,d)))[0])}\n\n\ndef reason_list(phi,features,grouped=False,k=4):\n    # Signed sums before ranking prevent cherry-picking positive members within a group.\n    scores={}\n    for f,v in zip(features,phi):\n        key=GROUPS[f] if grouped else f;scores[key]=scores.get(key,0)+float(v)\n    return [{\'id\':f,\'label\':f if grouped else LABELS[f],\'weight\':v}\n            for f,v in sorted(scores.items(),key=lambda x:(-x[1],x[0])) if v>1e-10][:k]\n\n\ndef compare(a,b):\n    A=[x[\'id\'] for x in a];B=[x[\'id\'] for x in b];sa,sb=set(A),set(B)\n    if not sa or not sb:return {\'retention\':None,\'jaccard\':None,\'turnover\':None,\'first_changed\':None,\'empty\':True}\n    return {\'retention\':len(sa&sb)/len(sa),\'jaccard\':len(sa&sb)/len(sa|sb),\n            \'turnover\':sa!=sb,\'first_changed\':A[0]!=B[0],\'empty\':False}\n\n\ndef summarize(comparisons):\n    out=[];df=pd.DataFrame(comparisons)\n    for key,g in df.groupby([\'model\',\'method\',\'test\',\'mapping\'],sort=True):\n        # First aggregate all comparisons within applicants, then across applicants.\n        valid=g[g.retention.notna()].copy()\n        if not len(valid):continue\n        per=valid.groupby(\'case_id\')[[\'retention\',\'jaccard\',\'turnover\',\'first_changed\']].mean()\n        rng=np.random.default_rng(SEED);a=per.turnover.to_numpy(dtype=float)\n        boots=np.mean(rng.choice(a,(1000,len(a)),replace=True),axis=1)\n        out.append(dict(zip([\'model\',\'method\',\'test\',\'mapping\'],key),applicants=len(per),\n                        comparisons=len(valid),empty_comparisons=int(g.retention.isna().sum()),\n                        retention=float(per.retention.mean()),jaccard=float(per.jaccard.mean()),\n                        turnover=float(a.mean()),first_changed=float(per.first_changed.mean()),\n                        turnover_ci=list(map(float,np.quantile(boots,[.025,.975])))))\n    return out\n\n\ndef run(path,out,ncases=40,repeats=5,official=False):\n    start=time.time();out.mkdir(parents=True,exist_ok=True)\n    raw=pd.read_csv(path);Xraw=raw.drop(columns=\'RiskPerformance\')\n    assert set(raw.RiskPerformance)=={\'Good\',\'Bad\'}\n    quarantine=Xraw.eq(-9).all(axis=1)\n    df=raw.loc[~quarantine].copy()\n    # Predictor-identical records stay together even if labels differ.\n    groups=pd.util.hash_pandas_object(df.drop(columns=\'RiskPerformance\'),index=False)\n    unique=groups.drop_duplicates().to_numpy()\n    train_groups,temp=train_test_split(unique,test_size=.4,random_state=SEED)\n    val_groups,test_groups=train_test_split(temp,test_size=.5,random_state=SEED)\n    splits={name:df.index[groups.isin(g)].to_numpy() for name,g in [(\'train\',train_groups),(\'validation\',val_groups),(\'test\',test_groups)]}\n    assert not set(train_groups)&set(test_groups)\n    report={\'rows\':len(raw),\'quarantined_no_bureau\':int(quarantine.sum()),\'exact_duplicate_rows\':int(raw.duplicated().sum()),\n            \'remaining_rows\':len(df),\'remaining_predictor_duplicates\':int(df.drop(columns=\'RiskPerformance\').duplicated().sum()),\n            \'sha256\':hashlib.sha256(path.read_bytes()).hexdigest(),\'features_original\':len(Xraw.columns),\n            \'split_rows\':{k:len(v) for k,v in splits.items()},\'label_counts\':raw.RiskPerformance.value_counts().to_dict(),\n            \'codes\':{c:{str(k):int(v) for k,v in Xraw[c][Xraw[c]<0].value_counts().items()} for c in Xraw}}\n    cases=[];comparisons=[];models=[];attempts=[];baseline={}\n    for variant in [\'without_score\',\'with_score\']:\n        feats=FEATURES+([\'ExternalRiskEstimate\'] if variant==\'with_score\' else [])\n        frames={k:df.loc[v,feats].to_numpy(dtype=float) for k,v in splits.items()}\n        labels={k:(df.loc[v,\'RiskPerformance\']==\'Bad\').astype(int).to_numpy() for k,v in splits.items()}\n        enc=Encoding().fit(frames[\'train\'])\n        model=RandomForestClassifier(n_estimators=100,max_depth=7,min_samples_leaf=15,max_features=.8,random_state=SEED,n_jobs=2)\n        model.fit(enc.transform(frames[\'train\']),labels[\'train\']);predict=Predictor(model,enc)\n        metric={}\n        for k in [\'validation\',\'test\']:\n            p=predict(frames[k]);frac,mean=calibration_curve(labels[k],p,n_bins=8)\n            metric[k]={\'auc\':float(roc_auc_score(labels[k],p)),\'average_precision\':float(average_precision_score(labels[k],p)),\n                       \'brier\':float(brier_score_loss(labels[k],p)),\'calibration\':{\'observed\':frac.tolist(),\'predicted\':mean.tolist()}}\n        rng=np.random.default_rng(SEED);bg=frames[\'train\'][rng.choice(len(frames[\'train\']),24,replace=False)]\n        scores=predict(frames[\'test\']);eligible=np.flatnonzero(scores>=.5)\n        ordered=eligible[np.argsort(scores[eligible])]\n        if variant==\'without_score\':\n            n=min(ncases//2,len(ordered)//2)\n            positions=np.r_[ordered[:n],ordered[-n:]];ids=splits[\'test\'][positions];primary_ids=set(map(int,ids))\n        else:\n            positions=np.array([j for j,i in enumerate(splits[\'test\']) if int(i) in primary_ids and scores[j]>=.5])\n        models.append({\'id\':variant,\'features\':feats,\'metrics\':metric,\'audit_cases\':len(positions),\'model\':\'RandomForestClassifier\',\n                       \'settings\':{\'n_estimators\':100,\'max_depth\':7,\'min_samples_leaf\':15,\'max_features\':.8},\n                       \'background_rows\':24})\n        if official:\n            import shap\n            from lime.lime_tabular import LimeTabularExplainer\n            # KernelSHAP on RAW feature blocks through the encoder: signed feature sum includes status effects.\n            sh=shap.KernelExplainer(predict,bg)\n            def explain_shap(x):\n                np.random.seed(SEED)\n                v=np.asarray(sh.shap_values(x,nsamples=1024,l1_reg=0))\n                return v,{\'base\':float(sh.expected_value),\'residual\':float(predict(x[None,:])[0]-sh.expected_value-v.sum())}\n            def explain_lime(x,seed):\n                le=LimeTabularExplainer(frames[\'train\'],feature_names=feats,class_names=[\'Good\',\'Bad\'],\n                                       mode=\'classification\',random_state=seed,discretize_continuous=True)\n                ex=le.explain_instance(x,predict.probabilities,labels=(1,),num_features=len(feats),num_samples=2000)\n                v=np.zeros(len(feats))\n                for j,w in ex.local_exp[1]:v[j]=w\n                return v,{\'local_fit_r2\':float(ex.score),\'local_prediction\':float(ex.local_pred[0]),\n                          \'condition_weights\':[[str(label),float(weight)] for label,weight in ex.as_list(label=1)]}\n            methods=[\'KernelSHAP\',\'LIME\']\n        else:\n            explain_shap=lambda x:exact_shapley(x,bg,predict)\n            explain_lime=lambda x,seed:local_surrogate(x,frames[\'train\'],predict,seed)\n            methods=[\'Exact interventional Shapley\',\'Custom LIME-style surrogate\']\n        for ii,pos in enumerate(positions):\n            idx=int(splits[\'test\'][pos]);x=frames[\'test\'][pos];p=float(scores[pos]);stratum=\'near\' if p<.6 else \'far\'\n            rec={\'case_id\':idx,\'model\':variant,\'score\':p,\'stratum\':stratum,\'features\':{f:float(v) for f,v in zip(feats,x)},\'explanations\':{},\'perturbations\':[]}\n            for method in methods:\n                runs=[]\n                for seed in range(repeats if \'surrogate\' in method or method==\'LIME\' else 1):\n                    phi,meta=explain_lime(x,seed) if (\'surrogate\' in method or method==\'LIME\') else explain_shap(x)\n                    entry={\'seed\':seed,\'values\':phi.tolist(),\'individual\':reason_list(phi,feats),\n                           \'grouped\':reason_list(phi,feats,True),\'diagnostics\':meta}\n                    runs.append(entry)\n                rec[\'explanations\'][method]=runs\n                for a,b in itertools.combinations(runs,2):\n                    for mapping in [\'individual\',\'grouped\']:\n                        comparisons.append({\'case_id\':idx,\'model\':variant,\'method\':method,\'test\':\'repeatability\',\'mapping\':mapping,**compare(a[mapping],b[mapping])})\n            # Fixed-model measurement sensitivity only; not actionable recourse or longitudinal transitions.\n            for f in [\'NetFractionRevolvingBurden\',\'NetFractionInstallBurden\']:\n                j=feats.index(f)\n                for delta in [-1,1]:\n                    attempt={\'case_id\':idx,\'model\':variant,\'feature\':f,\'delta\':delta}\n                    if x[j]<0:attempts.append(dict(attempt,status=\'special state frozen\'));continue\n                    z=x.copy();z[j]+=delta\n                    if z[j]<0 or z[j]>np.max(frames[\'train\'][:,j]):attempts.append(dict(attempt,status=\'outside observed training range\'));continue\n                    pp=float(predict(z[None,:])[0])\n                    if pp<.5:attempts.append(dict(attempt,status=\'decision changed\',score_after=pp));continue\n                    if abs(pp-p)>.02:attempts.append(dict(attempt,status=\'score tolerance exceeded\',score_after=pp));continue\n                    attempts.append(dict(attempt,status=\'accepted\',score_after=pp))\n                    perturb={\'feature\':f,\'delta\':delta,\'before\':float(x[j]),\'after\':float(z[j]),\'score_after\':pp,\'explanations\':{}}\n                    for method in methods:\n                        phi,meta=explain_lime(z,0) if (\'surrogate\' in method or method==\'LIME\') else explain_shap(z)\n                        ent={\'values\':phi.tolist(),\'individual\':reason_list(phi,feats),\'grouped\':reason_list(phi,feats,True),\'diagnostics\':meta}\n                        perturb[\'explanations\'][method]=ent\n                        original=rec[\'explanations\'][method][0]\n                        for mapping in [\'individual\',\'grouped\']:\n                            comparisons.append({\'case_id\':idx,\'model\':variant,\'method\':method,\'test\':\'measurement sensitivity\',\'mapping\':mapping,**compare(original[mapping],ent[mapping])})\n                    rec[\'perturbations\'].append(perturb)\n            for mapping in [\'individual\',\'grouped\']:\n                comparisons.append({\'case_id\':idx,\'model\':variant,\'method\':\'Between methods\',\'test\':\'disagreement\',\'mapping\':mapping,\n                                    **compare(rec[\'explanations\'][methods[0]][0][mapping],rec[\'explanations\'][methods[1]][0][mapping])})\n            cases.append(rec)\n            print(variant,ii+1,\'/\',len(positions),\'case\',idx,flush=True)\n    summary=summarize(comparisons)\n    result={\'status\':\'official-library experiment\' if official else \'REFERENCE PILOT: exact Shapley + custom LIME-style implementation\',\n            \'date\':\'2026-10-14\',\'data\':report,\'models\':models,\'cases\':cases,\'summary\':summary,\'attempts\':attempts,\n            \'config\':{\'seed\':SEED,\'threshold\':.5,\'score_tolerance\':.02,\'repeats\':repeats,\'n_requested\':ncases,\'official\':official},\n            \'limitations\':[\'No legally validated notices. Group labels are illustrative audit categories, not sufficient principal reasons.\',\n                          \'Grouped sums can mask opposite effects and lose specificity; compare raw and mapped evidence.\',\n                          \'Coalition replacements and local perturbations may violate feature dependence.\',\n                          \'Ratio +/-1 tests measurement sensitivity, not plausible borrower action; clinical/financial realism not established.\',\n                          \'No protected-attribute or North Carolina representativeness claims.\',\n                          \'With/without score models are separately trained; common-case comparison does not isolate a causal score effect.\',\n                          \'Official FICO dictionary and redistribution terms remain unverified.\'],\n            \'runtime_seconds\':time.time()-start}\n    (out/\'audit_results.json\').write_text(json.dumps(result,indent=2))\n    pd.DataFrame(comparisons).to_csv(out/\'comparisons.csv\',index=False)\n    pd.DataFrame(attempts).to_csv(out/\'perturbation_log.csv\',index=False)\n    (out/\'split.json\').write_text(json.dumps({k:list(map(int,v)) for k,v in splits.items()},indent=2))\n    (out/\'run_manifest.json\').write_text(json.dumps({\'python\':platform.python_version(),\'dataset_sha256\':report[\'sha256\'],\n                        \'seed\':SEED,\'mode\':result[\'status\'],\'elapsed_seconds\':result[\'runtime_seconds\']},indent=2))\n    print(\'Complete\',round(time.time()-start,1),\'seconds\',flush=True)\n    return result\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',type=Path,required=True);p.add_argument(\'--out\',type=Path,default=Path(\'results\'))\n    p.add_argument(\'--cases\',type=int,default=40);p.add_argument(\'--repeats\',type=int,default=5);p.add_argument(\'--official\',action=\'store_true\')\n    a=p.parse_args();run(a.data,a.out,a.cases,a.repeats,a.official)\n', 'src/benchmark_models.py': '"""Compare predeclared models on the existing split; select by validation AUC."""\nfrom pathlib import Path\nimport json, sys\nimport pandas as pd\nfrom sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import roc_auc_score, brier_score_loss\nfrom audit import Encoding\n\nR=Path(__file__).resolve().parents[1]\ndf=pd.read_csv(R/\'data/heloc_dataset_v1.csv\')\nsplit=json.loads((R/\'results/reference/split.json\').read_text())\nrows=[]\nfor scope in [\'all_23\',\'all_22_without_external_score\']:\n    features=[c for c in df if c!=\'RiskPerformance\' and not (scope.endswith(\'without_external_score\') and c==\'ExternalRiskEstimate\')]\n    enc=Encoding().fit(df.loc[split[\'train\'],features].to_numpy())\n    X={s:enc.transform(df.loc[ids,features].to_numpy()) for s,ids in split.items()}\n    y={s:(df.loc[ids,\'RiskPerformance\']==\'Bad\').astype(int) for s,ids in split.items()}\n    candidates={\n        \'Logistic regression\':make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,C=1)),\n        \'Random Forest\':RandomForestClassifier(n_estimators=300,max_depth=8,min_samples_leaf=15,max_features=.8,random_state=2026,n_jobs=2),\n        \'Histogram gradient boosting\':HistGradientBoostingClassifier(max_iter=150,max_leaf_nodes=15,min_samples_leaf=30,l2_regularization=10,random_state=2026),\n    }\n    for name,model in candidates.items():\n        model.fit(X[\'train\'],y[\'train\'])\n        p=model.predict_proba(X[\'validation\'])[:,1]\n        rows.append({\'scope\':scope,\'model\':name,\'features\':len(features),\'validation_auc\':float(roc_auc_score(y[\'validation\'],p)),\'validation_brier\':float(brier_score_loss(y[\'validation\'],p)),\'estimator\':model,\'X\':X,\'y\':y})\n        print(scope,name,rows[-1][\'validation_auc\'],flush=True)\n# Evaluate only the validation-selected candidate, leaving other test results uninspected.\nbest=max(rows,key=lambda r:r[\'validation_auc\'])\np=best[\'estimator\'].predict_proba(best[\'X\'][\'test\'])[:,1]\nselected={k:v for k,v in best.items() if k not in [\'estimator\',\'X\',\'y\']}\nselected.update(test_auc=float(roc_auc_score(best[\'y\'][\'test\'],p)),test_brier=float(brier_score_loss(best[\'y\'][\'test\'],p)))\nreport={\'selection\':\'Highest validation AUC among six predeclared candidates. Only selected candidate evaluated on test. Existing split reused; original eight-feature test score was already known.\',\'candidates\':[{k:v for k,v in r.items() if k not in [\'estimator\',\'X\',\'y\']} for r in rows],\'selected\':selected,\'audit_status\':\'Benchmark only. Existing explanation audit still describes the eight/nine-feature Random Forests; explanations have not been rerun for this candidate.\'}\n(R/\'results/model_benchmark.json\').write_text(json.dumps(report,indent=2))\nprint(json.dumps(selected,indent=2))\n', 'src/build_workbench.py': '"""Build the measured external-score investigation and offline preview."""\nfrom pathlib import Path\nimport json,re\nfrom audit import LABELS,GROUPS\nR=Path(__file__).resolve().parents[1]\nexternal=json.loads((R/\'results/external_score/experiment.json\').read_text())\nstudy=json.loads((R/\'results/score_reconstruction/experiment.json\').read_text())\nlabels={**LABELS,\'MSinceOldestTradeOpen\':\'Oldest account age\',\'AverageMInFile\':\'Average account age\',\'NumSatisfactoryTrades\':\'Accounts with on-time payments\',\'PercentTradesNeverDelq\':\'Accounts never delinquent\',\'MSinceMostRecentDelq\':\'Time since most recent delinquency\',\'NumInqLast6M\':\'Recent credit inquiries\',\'NetFractionRevolvingBurden\':\'Revolving balance / credit limit\',\'NetFractionInstallBurden\':\'Installment balance / original amount\',\'ExternalRiskEstimate\':\'Externally supplied score\'}\nlocal={str(c[\'case_id\']):[{k:r[k] for k in [\'seed\',\'heldout_r2\',\'heldout_mae\',\'point_error\']} for r in c[\'surrogate\']] for c in external[\'records\'] if c[\'model\']==\'with_score\'}\ndata={\'study\':study,\'labels\':labels,\'groups\':GROUPS,\'local\':local,\'external_summary\':external[\'summary\']}\njs=\'const PROJECT=\'+json.dumps(data,separators=(\',\',\':\'))+\';\'\n(R/\'site/data.js\').write_text(js)\nhtml=(R/\'src/workbench.html\').read_text().replace(\'<!-- LEGAL_APPENDIX -->\',(R/\'src/requirements-appendix.html\').read_text()).replace(\'<!-- EVIDENCE_APPENDIX -->\',(R/\'src/evidence-appendix.html\').read_text());(R/\'site/index.html\').write_text(html)\npreview=html.replace(\'<script src="data.js"></script>\',\'<script>\'+js+\'</script>\').replace(\'<script src="app.js"></script>\',\'<script>\'+(R/\'site/app.js\').read_text()+\'</script>\')\n(R.parent/\'heloc_workbench_preview.html\').write_text(preview)\n# The inline rendering uses the same application and measured data, root-scoped.\ncss=re.search(r\'<style>(.*?)</style>\',preview,re.S).group(1)\nbody=re.search(r\'<body>(.*?)</body>\',preview,re.S).group(1)\ncss=css.replace(\'body{\',\'#score-investigation-render{\').replace(\'header{\',\'#score-investigation-render header{\').replace(\'main{\',\'#score-investigation-render main{\').replace(\'footer{\',\'#score-investigation-render footer{\')\ncss="@import url(\'https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Fraunces:opsz,wght@9..144,400;9..144,500&family=IBM+Plex+Mono:wght@400;500&display=swap\');\\n"+css\nfragment=\'<style>\'+css+\'</style><div id="score-investigation-render">\'+body+\'</div>\'\nfragment=fragment.replace("window.scrollTo({top:0,behavior:\'instant\'});",\'\').replace(\'</script><script>\',\'\\n\')\nif Path(\'/workspace\').is_dir():\n    Path(\'/workspace/credit-score-investigation.html\').write_text(fragment)\nassert len(fragment.encode())<1024*1024\nprint(\'Built site, offline preview and inline render:\',len(fragment.encode()),\'bytes\')\n', 'src/challenge_examples.py': "from pathlib import Path\nfrom copy import deepcopy\nimport json\nfrom evidence_review import review\nR=Path(__file__).resolve().parents[1]\ninputs=json.loads((R/'results/notice_audit/inputs.json').read_text())\ntruth={x['id']:x for x in json.loads((R/'results/notice_audit/truth.json').read_text())}\nrows=[]\nfor s in inputs:\n    t=truth[s['id']]\n    if t['partition']!='evaluation' or t['scenario']!='correct':continue\n    b=deepcopy(s['bundle']);b['notice']['reasons'][0]['text']='Credit-file factor considered: '+b['notice']['reasons'][0]['text'].lower()\n    rows.append({'case_id':s['case_id'],'challenge':'accurate_unlisted_paraphrase','stipulated_truth':'correct','bundle':b,'review':review(b)})\n    b=deepcopy(s['bundle']);r=b['notice']['reasons'][0];r['text']='Income insufficient for the requested amount';b['catalog'][r['code']]['approved_wording']=[r['text']]\n    rows.append({'case_id':s['case_id'],'challenge':'corrupted_trusted_catalog','stipulated_truth':'wrong','bundle':b,'review':review(b)})\nsummary=[]\nfor kind in ['accurate_unlisted_paraphrase','corrupted_trusted_catalog']:\n    subset=[r for r in rows if r['challenge']==kind]\n    summary.append({'challenge':kind,'cases':len(subset),'flagged':sum(r['review']['disposition']=='Investigate' for r in subset),'interpretation':'False-alarm challenge: valid wording falls outside catalog.' if kind.startswith('accurate') else 'Missed-defect challenge: source catalog and notice share the error.'})\n(R/'results/notice_audit/challenges.json').write_text(json.dumps({'status':'Post-primary illustrative challenge set; no checker changes','summary':summary,'cases':rows},indent=2))\nprint(json.dumps(summary,indent=2))\n", 'src/decision_reconstruction.py': '"""Replay actual model outputs under controlled, explicitly constructed policies.\n\nNever load a model supplied by a site visitor: the joblib artifact here is the\ntrusted project artifact only. The browser displays these precomputed executions.\n"""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport copy\nimport hashlib\nimport json\nimport joblib\nimport numpy as np\nfrom audit import Predictor\n\nR = Path(__file__).resolve().parents[1]\nMODES = {\n    \'chart_only\': \'Explanation report only\',\n    \'original\': \'Original records recovered\',\n    \'missing_policy\': \'Original lending policy missing\',\n    \'new_policy\': \'Newer lending policy substituted\',\n    \'wrong_model\': \'Model identity mismatch\',\n    \'changed_input\': \'Input snapshot altered\',\n    \'changed_explanation\': \'Explanation report altered\',\n}\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode()).hexdigest()\n\ndef decide(score, policy):\n    return \'Declined\' if score >= policy[\'decline_at_or_above\'] else \'Not declined by this rule\'\n\ndef assess(package, predict, artifact_digest, feature_names):\n    event = package[\'event\']\n    checks = {}\n    for key, expected in [(\'inputs\', \'inputs_sha256\'), (\'policy\', \'policy_sha256\')]:\n        obj = package.get(key)\n        checks[key] = \'missing\' if obj is None else (\'verified\' if digest(obj) == event[expected] else \'mismatch\')\n    supplied_model = package.get(\'model_sha256\')\n    checks[\'model\'] = \'missing\' if supplied_model is None else (\'verified\' if supplied_model == event[\'model_sha256\'] == artifact_digest else \'mismatch\')\n    explanation = package.get(\'explanation\')\n    residual = None if explanation is None else float(explanation[\'base\'] + sum(explanation[\'values\']) - event[\'prediction\'])\n    explanation_matches = residual is not None and abs(residual) < 1e-8\n    can_replay = checks[\'model\'] == checks[\'inputs\'] == \'verified\'\n    replayed = None\n    if can_replay:\n        row = np.array([[package[\'inputs\'][f] for f in feature_names]], dtype=float)\n        replayed = float(predict(row)[0])\n    score_matches = replayed is not None and abs(replayed - event[\'prediction\']) < 1e-8\n    decision = decide(replayed, package[\'policy\']) if score_matches and checks[\'policy\'] == \'verified\' else None\n    return {\n        \'record_checks\': checks,\n        \'explanation_sum_matches\': explanation_matches,\n        \'explanation_residual\': residual,\n        \'prediction_reproduced\': score_matches,\n        \'replayed_prediction\': replayed,\n        \'original_decision_reconstructed\': decision is not None and decision == event[\'decision\'],\n        \'replayed_decision\': decision,\n        \'inconsistencies\': [k for k, v in checks.items() if v == \'mismatch\'] + ([] if explanation_matches else [\'explanation\']),\n        \'missing_records\': [k for k, v in checks.items() if v == \'missing\'],\n        \'upstream_score_explained\': False,\n    }\n\ndef run():\n    out = R / \'results/decision_reconstruction\'\n    out.mkdir(exist_ok=True)\n    model_path = R / \'results/reference_sensitivity/rf8_model.joblib\'\n    model_sha = hashlib.sha256(model_path.read_bytes()).hexdigest()\n    artifact = joblib.load(model_path)\n    predict = Predictor(artifact[\'model\'], artifact[\'encoding\'])\n    names = artifact[\'features\']\n    reference = json.loads((R / \'results/reference/audit_results.json\').read_text())\n    cases = [c for c in reference[\'cases\'] if c[\'model\'] == \'without_score\']\n    original_policy = {\'id\': \'illustrative-policy-v1\', \'decline_at_or_above\': .50, \'role\': \'constructed research rule, not observed lender policy\'}\n    later_policy = {**original_policy, \'id\': \'illustrative-policy-v2\', \'decline_at_or_above\': .65}\n    records, evaluations = [], []\n    for case in cases:\n        phi = case[\'explanations\'][\'Exact interventional Shapley\'][0]\n        event = {\'case_id\': case[\'case_id\'], \'timestamp\': \'2026-09-01T12:00:00Z\',\n                 \'prediction\': case[\'score\'], \'decision\': decide(case[\'score\'], original_policy),\n                 \'inputs_sha256\': digest(case[\'features\']), \'policy_sha256\': digest(original_policy), \'model_sha256\': model_sha}\n        full = {\'event\': event, \'inputs\': case[\'features\'], \'model_sha256\': model_sha,\n                \'policy\': original_policy, \'explanation\': {\'base\': phi[\'diagnostics\'][\'base\'], \'values\': phi[\'values\']}}\n        packages = {}\n        for mode in MODES:\n            p = copy.deepcopy(full)\n            if mode == \'chart_only\':\n                for key in [\'inputs\', \'model_sha256\', \'policy\']: p.pop(key)\n            elif mode == \'missing_policy\': p.pop(\'policy\')\n            elif mode == \'new_policy\': p[\'policy\'] = copy.deepcopy(later_policy)\n            elif mode == \'wrong_model\': p[\'model_sha256\'] = hashlib.sha256(b\'constructed substitute artifact\').hexdigest()\n            elif mode == \'changed_input\': p[\'inputs\'][names[0]] += 24\n            elif mode == \'changed_explanation\': p[\'explanation\'][\'values\'][0] += .10\n            result = assess(p, predict, model_sha, names)\n            evaluations.append({\'case_id\': case[\'case_id\'], \'package\': mode, **result})\n            packages[mode] = {\'records\': p, \'assessment\': result}\n        records.append({\'case_id\': case[\'case_id\'], \'score\': case[\'score\'], \'packages\': packages,\n                        \'later_policy_decision\': decide(case[\'score\'], later_policy)})\n    summary = []\n    for mode, label in MODES.items():\n        rows = [e for e in evaluations if e[\'package\'] == mode]\n        summary.append({\'package\': mode, \'label\': label, \'records\': len(rows),\n                        \'explanations_add_up\': sum(e[\'explanation_sum_matches\'] for e in rows),\n                        \'predictions_reproduced\': sum(e[\'prediction_reproduced\'] for e in rows),\n                        \'decisions_reconstructed\': sum(e[\'original_decision_reconstructed\'] for e in rows),\n                        \'inconsistent_packages\': sum(bool(e[\'inconsistencies\']) for e in rows)})\n    payload = {\'generated_at\': datetime.now(timezone.utc).isoformat(), \'design\': \'Controlled evidence-removal and inconsistency demonstration; constructed lending policies around measured predictions.\',\n               \'protocol_sha256\': hashlib.sha256((R / \'docs/DECISION_RECONSTRUCTION_PROTOCOL.md\').read_bytes()).hexdigest(),\n               \'model_sha256\': model_sha, \'features\': names, \'policies\': [original_policy, later_policy],\n               \'case_count\': len(cases), \'package_count\': len(evaluations), \'summary\': summary, \'cases\': records,\n               \'changed_outcomes_under_later_policy\': sum(c[\'later_policy_decision\'] != c[\'packages\'][\'original\'][\'records\'][\'event\'][\'decision\'] for c in records),\n               \'trust_assumption\': \'Logged identities are retained independently and trusted. Matching hashes do not authenticate real-world data or establish regulatory adequacy.\'}\n    (out / \'experiment.json\').write_text(json.dumps(payload, indent=2))\n    print(json.dumps({\'summary\': summary, \'changed_outcomes_under_later_policy\': payload[\'changed_outcomes_under_later_policy\']}, indent=2))\n\nif __name__ == \'__main__\':\n    run()\n', 'src/evidence-appendix.html': '<p class="intro">Select a review requirement to see the records it needs, or an explanation method to see the question it can answer. The study measures model behavior; several review questions still require lender or score-provider records.</p>\n<section class="policy-box"><p class="eyebrow">Core claim</p><h2>A downstream explanation can establish what happened in a model you can inspect. It cannot, by itself, authenticate an unavailable upstream scoring mechanism.</h2><p>For an external score, the review should end in one of four states: <span class="status-tag status-measured">MEASURED</span> <span class="status-tag status-limited">SUPPORTED WITH LIMITATIONS</span> <span class="status-tag status-gap">NOT ESTABLISHED</span> or <span class="status-tag status-gap">PROVIDER / LENDER RECORDS REQUIRED</span>.</p></section>\n<table class="claim-matrix"><thead><tr><th>Claim a reviewer may want to make</th><th>Evidence in this project</th><th>Status</th></tr></thead><tbody><tr><td>The supplied external score affected the lender-model prediction.</td><td>Exact Shapley contribution in the research Random Forest.</td><td><span class="status-tag status-measured">MEASURED</span></td></tr><tr><td>A reconstructed score can reproduce a similar downstream prediction.</td><td>Ridge and boosted-tree substitution tests.</td><td><span class="status-tag status-measured">MEASURED</span></td></tr><tr><td>The inferred detailed factors were the original provider’s reasons.</td><td>No authenticated upstream formula or provider reasons are present.</td><td><span class="status-tag status-gap">NOT ESTABLISHED</span></td></tr><tr><td>A real consumer notice accurately reflects the production decision path.</td><td>No actual lender decision, mapping version or delivered notice is in the dataset.</td><td><span class="status-tag status-gap">RECORDS REQUIRED</span></td></tr></tbody></table>\n<div class="sliding-options evidence-options" id="evidence-switch" data-mode="requirements" role="group" aria-label="Choose evidence reference"><button id="requirements-view" aria-pressed="true">Requirements &amp; records</button><button id="other-methods-view" aria-pressed="false">Explanation methods</button></div>\n<div id="requirements-panel" class="evidence-panel">\n<details open><summary>Specific principal reasons for the adverse action</summary><div><p><strong>Records needed.</strong> The actual decision path, scored factors, policy overrides, reason-selection method and notice wording.</p><p><strong>Role of XAI.</strong> Signed Shapley/SHAP contributions explain a prediction relative to a reference. A reviewer must validate the reference and how selected factors become the stated reasons. An actual scorecard or rule-based model may expose its contributions directly. Estimating a supplied score does not identify its original factors.</p><p class="small"><a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/9/" target="_blank" rel="noopener">§1002.9(b)(2)</a> · <a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/" target="_blank" rel="noopener">Comments 9(b)(2)-2, -4 and -5</a></p></div></details>\n<details><summary>Choice of comparison population</summary><div><p><strong>Records needed.</strong> The reference-group definition, selection process, sample sizes and results under alternative choices.</p><p><strong>Role of XAI.</strong> A reference-sensitivity experiment can show how Shapley reason selection changes when the comparison group changes. The official interpretation describes scoring comparisons with applicants near a passing score or with all applicants. A generic SHAP background is not automatically equivalent. This external-score study uses a fixed set of 24 reference records.</p><a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/" target="_blank" rel="noopener">Comment 9(b)(2)-5</a></div></details>\n<details><summary>Translation from detailed factors to notice categories</summary><div><p><strong>Records needed.</strong> Versioned mapping rules, the factors in each category, signed effects and paired examples before and after grouping.</p><p><strong>Role of XAI.</strong> Compare the retention and turnover of leading factors with the retention and turnover of grouped categories. The site provides both views. Independent evaluation of whether the wording accurately describes those factors remains additional work; grouping alone does not validate a notice.</p><a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/" target="_blank" rel="noopener">Comments 9(b)(2)-2 through -4</a></div></details>\n<details><summary>Notification, request procedure and retained records</summary><div><p><strong>Records needed.</strong> Application-completion and action dates, delivery records, creditor and agency identity, the ECOA statement, and reasons or the applicable request procedure.</p><p><strong>Role of XAI.</strong> Attribution methods do not produce these records. Workflow and notice-template checks need actual operational data. Consumer-application records are generally retained for 25 months, with longer periods in specified proceedings. Versioned models and mappings can help reconstruct the explanation used at the time.</p><a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/9/" target="_blank" rel="noopener">§1002.9(a)</a> · <a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/12/" target="_blank" rel="noopener">§1002.12</a></div></details>\n<details><summary>Consumer-report and credit-score disclosures</summary><div><p><strong>Records needed.</strong> Whether a consumer report or qualifying credit score was used, the supplier, actual score and applicable disclosure fields.</p><p><strong>Role of XAI.</strong> Track creditor-decision reasons separately from the score’s adverse factors. FCRA disclosures do not automatically satisfy Regulation B’s specific-reasons duty. The dataset’s external-score field does not identify the original supplier or applicable notice contents.</p><a href="https://www.ftc.gov/business-guidance/resources/using-consumer-reports-credit-decisions-what-know-about-adverse-action-risk-based-pricing-notices" target="_blank" rel="noopener">FTC notice guidance</a> · <a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/" target="_blank" rel="noopener">Comment 9(b)(2)-9</a></div></details>\n<details><summary>Group-level discrimination and the applicable legal framework</summary><div><p><strong>Records needed.</strong> Protected-group information, lending outcomes, the challenged practice, appropriate comparisons and evidence about alternatives.</p><p><strong>Role of XAI.</strong> Subgroup evaluation can locate differences, and explanations can help inspect features. Neither supplies a complete discrimination analysis. For housing-related lending, the Fair Housing Act’s discriminatory-effects framework requires separate analysis. This study does not evaluate protected-group outcomes.</p><a href="https://www.ecfr.gov/current/title-24/subtitle-B/chapter-I/part-100/subpart-G/section-100.500" target="_blank" rel="noopener">24 CFR 100.500</a> · <a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/6/" target="_blank" rel="noopener">Current Regulation B evaluation rules</a></div></details>\n<details><summary>North Carolina product requirements and third-party governance</summary><div><p><strong>Records needed.</strong> Lender charter, collateral, lien position, transaction terms, consumer-report use and state/federal applicability analysis. For an outside score, obtain the applicable version, input documentation, provider reasons and relevant validation evidence.</p><p><strong>Role of XAI.</strong> Trace a displayed explanation to its evidence to investigate its accuracy. The explanation cannot establish the loan’s terms or legal scope. Current interagency model-risk guidance discusses proprietary vendor products; it does not prescribe a single explanation method.</p><a href="https://www.ncleg.gov/EnactedLegislation/Statutes/HTML/BySection/Chapter_75/GS_75-1.1.html" target="_blank" rel="noopener">NC G.S. 75-1.1</a> · <a href="https://www.ncleg.gov/EnactedLegislation/Statutes/HTML/ByChapter/Chapter_24.html" target="_blank" rel="noopener">NC Chapter 24</a> · <a href="https://www.federalreserve.gov/supervisionreg/srletters/SR2602a1.pdf" target="_blank" rel="noopener">Model-risk guidance, §VII</a></div></details>\n</div>\n<div id="other-methods-panel" class="evidence-panel" hidden>\n<p class="method-status">Shapley and custom LIME-style calculations were run in this study. The other methods below are possible additions, not reported experiments.</p>\n<details open><summary>Shapley / SHAP: contributions relative to a comparison</summary><div><p>Allocates a prediction’s difference from a reference across features. Reference sensitivity and individual-versus-grouped contributions can investigate reason-selection choices.</p><p><strong>Evaluate:</strong> reference justification, feature dependence, signed aggregation and the connection to actual scored factors. Attribution of a supplied score leaves that score’s original calculation unexplained.</p><a href="https://arxiv.org/abs/2204.12365" target="_blank" rel="noopener">Nair and colleagues: adverse-action Shapley decomposition</a></div></details>\n<details><summary>LIME: a simple approximation near one record</summary><div><p>Fits an interpretable local model around a prediction. Separate generated records and repeated random seeds can assess local fidelity and repeatability.</p><p><strong>Evaluate:</strong> approximation error at the record, held-out local fit, sampling variability and whether generated combinations respect credit-feature relationships. A good local fit cannot establish the original reasons of an unknown upstream scorer.</p><a href="https://arxiv.org/abs/1602.04938" target="_blank" rel="noopener">Ribeiro, Singh and Guestrin: LIME</a></div></details>\n<details><summary>Interpretable scorecards, additive models and decision rules</summary><div><p>Expose the contributions or rules of the model actually used. A separately evaluated interpretable alternative could establish whether comparable predictive performance is achievable with a more inspectable design.</p><p><strong>Evaluate:</strong> predictive quality, actual rule execution and the wording of the resulting reasons. A transparent substitute explains the substitute, not the original complex model.</p><a href="https://docs.interpretable.ai/stable/examples/fico/" target="_blank" rel="noopener">HELOC interpretable-model example</a></div></details>\n<details><summary>Counterfactual explanations / DiCE: changes that reverse a decision</summary><div><p>Search for alternative inputs that change the specified model’s output. They can support a separate investigation of feasible recourse.</p><p><strong>Evaluate:</strong> decision reversal, immutable fields, costs, feasible changes, stability and the full lending policy. A path to approval answers a different question from the principal reasons for the original refusal.</p><a href="https://interpret.ml/DiCE/readme.html" target="_blank" rel="noopener">DiCE documentation</a></div></details>\n<details><summary>Anchors: local conditions that preserve a prediction</summary><div><p>Finds rule-style conditions under which a prediction remains the same with high precision under a defined sampling distribution. It can supplement ranked feature explanations.</p><p><strong>Evaluate:</strong> precision, coverage and the relevance of the perturbation distribution. An anchor is a local behavioral rule; it does not recover an unknown provider’s internal mechanism.</p><a href="https://ojs.aaai.org/index.php/AAAI/article/view/11491" target="_blank" rel="noopener">Ribeiro and colleagues: Anchors</a></div></details>\n</div>\n<p class="note">There is no general legal pass mark for AUC, explanation fit or reason retention. The conclusion must address a specific claim and the records supporting it.</p>\n', 'src/evidence_export.py': '"""Export model-agnostic probability-scale Shapley evidence for the browser workbench.\nUse all feature attributions, not only the top reasons. See docs/IMPORT_FORMAT.md.\n"""\nfrom pathlib import Path\nimport json\n\ndef export_explanations(path, *, model_id, outcome, features, applications, title=\'Explanation evidence\'):\n    """features: [{id,label,group?}]; applications: [{id,score,shapley:[{id,reference,baseline,values:{feature:phi}}]}].\n    Optional local_approximations carry coefficients and explicitly labeled fidelity evidence.\n    This function packages evidence; it does not validate the model or authenticate its provenance.\n    """\n    pack={\'schema_version\':\'credit-explanation-evidence/v1\',\'title\':title,\n          \'model\':{\'id\':model_id,\'output\':outcome,\'score_scale\':\'probability\'},\n          \'features\':features,\'applications\':applications}\n    Path(path).write_text(json.dumps(pack,indent=2,allow_nan=False))\n    return pack\n', 'src/evidence_review.py': '"""Contract-based review of supplied decision/notice evidence. No scenario labels accepted.\n\nThis checks consistency against retained records, not authenticity, legal sufficiency,\nor semantic equivalence of arbitrary natural-language paraphrases.\n"""\ndef review(bundle):\n    decision=bundle.get(\'decision\');catalog=bundle.get(\'catalog\');notice=bundle.get(\'notice\',{})\n    if not decision or not catalog:\n        return {\'disposition\':\'Insufficient evidence\',\'findings\':[{\'code\':\'missing_evidence\',\'title\':\'Decision trace or mapping catalog is unavailable\',\'evidence\':\'decision / catalog\',\'request\':\'Obtain the versioned trace and reason catalog before assessing the notice.\'}]}\n    findings=[]\n    def add(code,title,path,request):\n        findings.append(dict(code=code,title=title,evidence=path,request=request))\n    if notice.get(\'decision_id\')!=decision.get(\'id\'):\n        add(\'identity\',\'Notice refers to a different decision\',\'notice.decision_id / decision.id\',\'Recover the notice for the same decision identifier.\')\n    for field,expected in decision[\'artifacts\'].items():\n        if notice.get(\'artifacts\',{}).get(field)!=expected:\n            add(\'version\',f\'{field} does not match the decision record\',f\'notice.artifacts.{field} / decision.artifacts.{field}\',\'Regenerate from the retained decision-time artifacts; preserve the original notice.\')\n    reasons=notice.get(\'reasons\',[])\n    for i,r in enumerate(reasons):\n        spec=catalog.get(r.get(\'code\'))\n        if not spec:\n            add(\'mapping\',\'Reason code is absent from the supplied catalog\',f\'notice.reasons[{i}].code\',\'Provide the applicable catalog version and validate the description.\');continue\n        if r.get(\'text\') not in spec[\'approved_wording\']:\n            add(\'wording\',\'Reason wording has no support in the supplied mapping catalog\',f\'notice.reasons[{i}].text / catalog.{r["code"]}\',\'Review the description and direction against feature evidence; an unlisted paraphrase needs human review.\')\n        if sorted(r.get(\'source_features\',[]))!=sorted(spec[\'source_features\']):\n            add(\'mapping\',\'Stated reason and cited source features do not match\',f\'notice.reasons[{i}].source_features / catalog.{r["code"]}\',\'Correct the source-to-reason mapping and examine affected notices.\')\n        if decision[\'basis\']==\'model\':\n            if not all(decision[\'attribution\'].get(f,0)>0 for f in r.get(\'source_features\',[])):\n                add(\'support\',\'Model reason cites a feature without a positive risk attribution\',f\'notice.reasons[{i}] / decision.attribution\',\'Verify sign, output class and reference before selecting this feature as an adverse contributor.\')\n    observed={r.get(\'code\') for r in reasons}\n    if decision[\'basis\']==\'eligibility_gate\':\n        required=set(decision[\'required_reasons\'])\n        if not required.issubset(observed):\n            add(\'decision_coverage\',\'The terminating eligibility rule is missing from the reasons\',\'decision.rule_trace / notice.reasons\',\'Explain the recorded rule that stopped the application; do not substitute an unused risk-score explanation.\')\n        if observed-required:\n            add(\'unused_score\',\'Notice cites model factors after the policy gate stopped evaluation\',\'decision.basis / notice.reasons\',\'Separate diagnostic model outputs from the factors actually used in this decision.\')\n    return {\'disposition\':\'Investigate\' if findings else \'No discrepancy detected\',\'findings\':findings}\n', 'src/external_score_experiment.py': '"""Paired upstream-score dependence experiment; see docs/EXTERNAL_SCORE_ADDENDUM.md."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport hashlib,json,warnings\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import roc_auc_score,r2_score,mean_absolute_error\nfrom sklearn.linear_model import Ridge\nfrom audit import FEATURES,Encoding,Predictor,exact_shapley,reason_list,SEED\nfrom reference_experiment import neighborhood\nR=Path(__file__).resolve().parents[1]\nwarnings.filterwarnings(\'ignore\',message=\'.*sklearn.utils.parallel.delayed.*\',category=UserWarning)\n\ndef run():\n    out=R/\'results/external_score\';out.mkdir(parents=True,exist_ok=True)\n    df=pd.read_csv(R/\'data/heloc_dataset_v1.csv\')\n    split=json.loads((R/\'results/reference/split.json\').read_text())\n    old=json.loads((R/\'results/reference/audit_results.json\').read_text())\n    primary=[c for c in old[\'cases\'] if c[\'model\']==\'without_score\']\n    prev=json.loads((R/\'results/reference_sensitivity/experiment.json\').read_text())\n    y=(df.loc[split[\'test\'],\'RiskPerformance\']==\'Bad\').astype(int).to_numpy()\n    bgpos=np.random.default_rng(SEED).choice(len(split[\'train\']),24,replace=False)\n    models={};predicts={};rows=[];importance=[]\n    for variant in [\'without_score\',\'with_score\']:\n        features=FEATURES+([\'ExternalRiskEstimate\'] if variant==\'with_score\' else [])\n        train=df.loc[split[\'train\'],features].to_numpy(float);test=df.loc[split[\'test\'],features].to_numpy(float)\n        enc=Encoding().fit(train)\n        model=RandomForestClassifier(n_estimators=100,max_depth=7,min_samples_leaf=15,max_features=.8,random_state=SEED,n_jobs=2)\n        model.fit(enc.transform(train),(df.loc[split[\'train\'],\'RiskPerformance\']==\'Bad\').astype(int))\n        predict=Predictor(model,enc);pred=predict(test);predicts[variant]=pred\n        auc=float(roc_auc_score(y,pred));models[variant]={\'features\':features,\'test_auc\':auc,\'test_cases\':len(y)}\n        # Raw input block moves with its special-state representation through Encoding.\n        for j,feature in enumerate(features):\n            drops=[]\n            for seed in range(10):\n                perm=test.copy();perm[:,j]=np.random.default_rng(20261016+seed).permutation(test[:,j]);drops.append(auc-float(roc_auc_score(y,predict(perm))))\n            importance.append({\'model\':variant,\'feature\':feature,\'auc_drop_mean\':float(np.mean(drops)),\'auc_drop_sd\':float(np.std(drops,ddof=1)),\'drops\':drops})\n        for i,c in enumerate(primary):\n            x=df.loc[c[\'case_id\'],features].to_numpy(float);score=float(predict(x[None,:])[0])\n            phi,diagnostics=exact_shapley(x,train[bgpos],predict)\n            if variant==\'without_score\':\n                assert abs(score-c[\'score\'])<1e-10\n                np.testing.assert_allclose(phi,c[\'explanations\'][\'Exact interventional Shapley\'][0][\'values\'],atol=1e-10)\n            fits=[]\n            for seed in range(5):\n                if variant==\'without_score\':\n                    fit=dict(next(f for f in prev[\'fidelity\'] if f[\'case_id\']==c[\'case_id\'] and f[\'seed\']==seed))\n                    fit[\'values\']=c[\'explanations\'][\'Custom LIME-style surrogate\'][seed][\'values\'];fit[\'individual\']=c[\'explanations\'][\'Custom LIME-style surrogate\'][seed][\'individual\']\n                else:\n                    z,reps,w=neighborhood(x,train,seed)\n                    f=Ridge(alpha=1).fit(reps,predict(z),sample_weight=w)\n                    z2,reps2,w2=neighborhood(x,train,seed+10000,include_anchor=False)\n                    actual=predict(z2);estimated=f.predict(reps2)\n                    fit={\'case_id\':c[\'case_id\'],\'seed\':seed,\'values\':f.coef_.tolist(),\'individual\':reason_list(f.coef_,features),\'fit_r2\':float(f.score(reps,predict(z),sample_weight=w)),\'heldout_r2\':float(r2_score(actual,estimated,sample_weight=w2)),\'heldout_mae\':float(mean_absolute_error(actual,estimated,sample_weight=w2)),\'point_error\':abs(float(f.predict(np.ones((1,len(x))))[0])-score)}\n                fits.append(fit)\n            rows.append({\'case_id\':c[\'case_id\'],\'model\':variant,\'score\':score,\'research_outcome\':\'deny\' if score>=.5 else \'accept\',\'features\':{f:float(v) for f,v in zip(features,x)},\'shapley\':{\'values\':phi.tolist(),\'individual\':reason_list(phi,features),\'grouped\':reason_list(phi,features,True),\'diagnostics\':diagnostics},\'surrogate\':fits})\n            if (i+1)%10==0:print(variant,i+1,\'/\',len(primary),flush=True)\n    pairs=[]\n    for c in primary:\n        a=next(r for r in rows if r[\'case_id\']==c[\'case_id\'] and r[\'model\']==\'without_score\');b=next(r for r in rows if r[\'case_id\']==c[\'case_id\'] and r[\'model\']==\'with_score\')\n        v=b[\'shapley\'][\'values\'][-1]\n        pairs.append({\'case_id\':c[\'case_id\'],\'score_without\':a[\'score\'],\'score_with\':b[\'score\'],\'score_change\':b[\'score\']-a[\'score\'],\'outcome_changed\':a[\'research_outcome\']!=b[\'research_outcome\'],\'external_score_value\':b[\'features\'][\'ExternalRiskEstimate\'],\'external_shapley\':v,\'external_in_top_adverse\':any(r[\'id\']==\'ExternalRiskEstimate\' for r in b[\'shapley\'][\'individual\']),\'external_largest_absolute\':int(np.argmax(np.abs(b[\'shapley\'][\'values\'])))==len(FEATURES)})\n    rng=np.random.default_rng(20261016);diffs=[]\n    for _ in range(1000):\n        ix=rng.choice(len(y),len(y),replace=True)\n        if len(np.unique(y[ix]))<2:continue\n        diffs.append(roc_auc_score(y[ix],predicts[\'with_score\'][ix])-roc_auc_score(y[ix],predicts[\'without_score\'][ix]))\n    summary={\'paired_applications\':len(pairs),\'outcomes_changed\':sum(r[\'outcome_changed\'] for r in pairs),\'external_in_top_adverse\':sum(r[\'external_in_top_adverse\'] for r in pairs),\'external_largest_absolute\':sum(r[\'external_largest_absolute\'] for r in pairs),\'external_positive\':sum(r[\'external_shapley\']>0 for r in pairs),\'test_outcomes_disagree\':int(np.sum((predicts[\'with_score\']>=.5)!=(predicts[\'without_score\']>=.5))),\'test_applications\':len(y),\'test_auc_difference\':models[\'with_score\'][\'test_auc\']-models[\'without_score\'][\'test_auc\'],\'test_auc_difference_ci\':list(map(float,np.quantile(diffs,[.025,.975]))),\'median_absolute_external_contribution\':float(np.median([abs(r[\'external_shapley\']) for r in pairs]))}\n    result={\'generated_at\':datetime.now(timezone.utc).isoformat(),\'design\':\'Exploratory paired external-score extension; earlier overlapping results were known.\',\'addendum_sha256\':hashlib.sha256((R/\'docs/EXTERNAL_SCORE_ADDENDUM.md\').read_bytes()).hexdigest(),\'data_sha256\':hashlib.sha256((R/\'data/heloc_dataset_v1.csv\').read_bytes()).hexdigest(),\'models\':models,\'summary\':summary,\'pairs\':pairs,\'records\':rows,\'permutation_importance\':importance,\'background_record_ids\':[int(split[\'train\'][i]) for i in bgpos],\'upstream_evidence\':{\'score_values\':\'Available in the supplied dataset\',\'score_calculation\':\'Not provided\',\'upstream_model\':\'Not provided\',\'upstream_reasons\':\'Not provided\',\'inference_warning\':\'Correlated credit-history inputs do not authenticate the upstream calculation.\'}}\n    (out/\'experiment.json\').write_text(json.dumps(result,indent=2));pd.DataFrame(pairs).to_csv(out/\'paired_results.csv\',index=False)\n    pd.DataFrame([{k:v for k,v in r.items() if k!=\'drops\'} for r in importance]).to_csv(out/\'permutation_importance.csv\',index=False)\n    print(json.dumps(summary,indent=2))\nif __name__==\'__main__\':run()\n', 'src/grouping_analysis.py': '"""Descriptive extension: changes hidden when detailed factors become broad categories.\nComputed from existing fixed-model reference experiments; not a new preregistered test.\n"""\nfrom pathlib import Path\nimport json\nfrom audit import compare\nfrom notice_benchmark import cluster_interval\nR=Path(__file__).resolve().parents[1]\n\ndef run():\n    ref=json.loads((R/\'results/reference_sensitivity/experiment.json\').read_text())\n    rows=[]\n    for cid in sorted({r[\'case_id\'] for r in ref[\'records\']}):\n        for seed in [101,202,303]:\n            a=next(r for r in ref[\'records\'] if r[\'case_id\']==cid and r[\'reference\']==\'population\' and r[\'seed\']==seed)\n            b=next(r for r in ref[\'records\'] if r[\'case_id\']==cid and r[\'reference\']==\'near_cutoff\' and r[\'seed\']==seed)\n            detailed=compare(a[\'individual\'],b[\'individual\']);grouped=compare(a[\'grouped\'],b[\'grouped\'])\n            rows.append({\'case_id\':cid,\'seed\':seed,\'detailed\':detailed,\'grouped\':grouped,\n                         \'hidden_change\':bool(detailed[\'turnover\'] and grouped[\'turnover\'] is False)})\n    valid=[r for r in rows if not r[\'detailed\'][\'empty\'] and not r[\'grouped\'][\'empty\']]\n    summary={field:cluster_interval([r[\'case_id\'] for r in valid],[float(r[field][\'turnover\']) for r in valid]) for field in [\'detailed\',\'grouped\']}\n    summary[\'hidden_change\']=cluster_interval([r[\'case_id\'] for r in valid],[float(r[\'hidden_change\']) for r in valid])\n    result={\'design\':\'Exploratory descriptive summary of existing fixed-model reference results. Grouping is a research mapping, not a lender-approved taxonomy.\',\n            \'applications\':len({r[\'case_id\'] for r in valid}),\'paired_comparisons\':len(valid),\'excluded_empty_comparisons\':len(rows)-len(valid),\n            \'hidden_change_comparisons\':sum(r[\'hidden_change\'] for r in valid),\n            \'applications_with_hidden_change\':len({r[\'case_id\'] for r in valid if r[\'hidden_change\']}),\n            \'summary\':summary,\'records\':rows}\n    (R/\'results/reference_sensitivity/grouping_analysis.json\').write_text(json.dumps(result,indent=2))\n    print(json.dumps({k:v for k,v in result.items() if k!=\'records\'},indent=2))\nif __name__==\'__main__\':run()\n', 'src/notice_benchmark.py': '"""Generate controlled notice cases, with truth kept separate from review inputs."""\nfrom pathlib import Path\nfrom copy import deepcopy\nimport hashlib,json,itertools\nfrom datetime import datetime,timezone\nimport numpy as np\nimport pandas as pd\nif __package__:\n    from .audit import LABELS, SEED\n    from .evidence_review import review\nelse:\n    from audit import LABELS, SEED\n    from evidence_review import review\nR=Path(__file__).resolve().parents[1]\n\ndef cluster_interval(records,values):\n    per=pd.DataFrame({\'case\':records,\'v\':values}).groupby(\'case\').v.mean().to_numpy()\n    rng=np.random.default_rng(SEED)\n    boot=rng.choice(per,(1000,len(per)),replace=True).mean(axis=1)\n    return {\'rate\':float(per.mean()),\'ci\':list(map(float,np.quantile(boot,[.025,.975]))),\'records\':len(per),\'scenarios\':len(values)}\n\ndef run():\n    data=json.loads((R/\'results/reference/audit_results.json\').read_text())\n    cases=[c for c in data[\'cases\'] if c[\'model\']==\'without_score\']\n    rng=np.random.default_rng(20261016);ids=rng.permutation(sorted(c[\'case_id\'] for c in cases))\n    split={str(int(i)):\'development\' if j<20 else \'evaluation\' for j,i in enumerate(ids)}\n    out=R/\'results/notice_audit\';out.mkdir(exist_ok=True)\n    (out/\'partition.json\').write_text(json.dumps(split,indent=2))\n    catalog={f:{\'source_features\':[f],\'approved_wording\':[LABELS[f], \'Recorded \'+LABELS[f].lower()]} for f in LABELS}\n    catalog[\'POLICY_ELIGIBILITY\']={\'source_features\':[\'research_eligibility_flag\'],\'approved_wording\':[\'Application does not meet the research eligibility rule\']}\n    scenarios=[];truth=[];outputs=[]\n    for c in cases:\n        cid=c[\'case_id\'];part=split[str(cid)];features=list(c[\'features\']);shap=c[\'explanations\'][\'Exact interventional Shapley\'][0]\n        atoms=shap[\'individual\'];assert atoms\n        artifacts={\'model\':\'rf8-seed2026\',\'preprocessing\':\'special-status-v1\',\'explainer\':\'exact-interventional-v1\',\'reference\':\'training-24-seed2026\',\'mapping\':\'feature-reasons-v1\'}\n        dec={\'id\':f\'HELOC-{cid}-research\',\'case_id\':cid,\'basis\':\'model\',\'outcome\':\'research_reject\',\'score\':c[\'score\'],\'cutoff\':.5,\'artifacts\':artifacts,\'required_reasons\':[], \'attribution\':dict(zip(features,shap[\'values\'])),\'rule_trace\':[\'Research eligibility gate: pass\',\'Risk model evaluated\',f\'Bad-class score {c["score"]:.6f} >= research cutoff 0.50\']}\n        reasons=[{\'code\':r[\'id\'],\'text\':LABELS[r[\'id\']],\'source_features\':[r[\'id\']]} for r in atoms]\n        clean={\'decision\':dec,\'catalog\':catalog,\'notice\':{\'decision_id\':dec[\'id\'],\'artifacts\':deepcopy(artifacts),\'reasons\':reasons}}\n        entries=[]\n        entries.append((\'correct\',\'correct\',\'Correct model-driven notice\',deepcopy(clean)))\n        p=deepcopy(clean)\n        for r in p[\'notice\'][\'reasons\']:r[\'text\']=\'Recorded \'+r[\'text\'].lower()\n        entries.append((\'paraphrase\',\'correct\',\'Catalog-approved wording variation\',p))\n        p=deepcopy(clean);p[\'decision\'][\'artifacts\'][\'mapping\']=\'feature-reasons-v0\';p[\'notice\'][\'artifacts\'][\'mapping\']=\'feature-reasons-v0\';p[\'newest_available_mapping\']=\'feature-reasons-v1\'\n        entries.append((\'historical\',\'correct\',\'Historical notice with matching historical mapping\',p))\n        gate=deepcopy(clean);gate[\'decision\'].update(basis=\'eligibility_gate\',required_reasons=[\'POLICY_ELIGIBILITY\'],rule_trace=[\'Constructed research eligibility flag: fail\',\'Decision stopped at eligibility gate\',\'Risk score retained for diagnostics only; not used in this decision\'])\n        gate[\'notice\'][\'reasons\']=[{\'code\':\'POLICY_ELIGIBILITY\',\'text\':catalog[\'POLICY_ELIGIBILITY\'][\'approved_wording\'][0],\'source_features\':[\'research_eligibility_flag\']}]\n        entries.append((\'policy_correct\',\'correct\',\'Notice identifies the terminating research policy rule\',deepcopy(gate)))\n        wrong=deepcopy(clean)\n        # Different replacement concepts in development/evaluation; preserve otherwise identical evidence.\n        wrong[\'notice\'][\'reasons\'][0][\'code\']=\'NumInqLast6M\' if atoms[0][\'id\']!=\'NumInqLast6M\' else \'AverageMInFile\'\n        if part==\'evaluation\':wrong[\'notice\'][\'reasons\'][0][\'code\']=\'NetFractionInstallBurden\' if atoms[0][\'id\']!=\'NetFractionInstallBurden\' else \'MSinceMostRecentDelq\'\n        wrong[\'notice\'][\'reasons\'][0][\'text\']=LABELS[wrong[\'notice\'][\'reasons\'][0][\'code\']]\n        entries.append((\'wrong_mapping\',\'mapping\',\'A reason cites evidence for a different concept\',wrong))\n        wrong=deepcopy(clean);wrong[\'notice\'][\'reasons\'][0][\'text\']=(\'Excessively high \' if part==\'development\' else \'Unacceptably low \')+LABELS[atoms[0][\'id\']].lower()\n        entries.append((\'unsupported_wording\',\'mapping\',\'Directional wording lacks supporting catalog evidence\',wrong))\n        wrong=deepcopy(gate);wrong[\'notice\'][\'reasons\']=deepcopy(reasons)\n        entries.append((\'omitted_policy\',\'coverage\',\'Notice substitutes diagnostic score reasons for the terminating rule\',wrong))\n        wrong=deepcopy(clean);wrong[\'notice\'][\'artifacts\'][\'model\' if part==\'development\' else \'reference\']=\'different-retained-artifact\'\n        entries.append((\'stale\',\'provenance\',\'Explanation artifact differs from the decision-time artifact\',wrong))\n        wrong=deepcopy(clean);wrong[\'decision\']=None\n        entries.append((\'missing\',\'insufficient\',\'Decision trace withheld: correctness cannot be established\',wrong))\n        runs=c[\'explanations\'][\'Custom LIME-style surrogate\'];sets=[{r[\'id\'] for r in e[\'individual\']} for e in runs]\n        instability=float(np.mean([a!=b for a,b in itertools.combinations(sets,2)]))\n        for kind,label,description,bundle in entries:\n            sid=f\'{cid}-{kind}\';scenarios.append({\'id\':sid,\'case_id\':cid,\'bundle\':bundle,\'seed_turnover\':instability})\n            truth.append({\'id\':sid,\'partition\':part,\'family\':label,\'scenario\':kind,\'description\':description})\n            # Review function sees only the supplied bundle, never label/partition/scenario metadata.\n            finding=review(bundle)\n            outputs.append({\'id\':sid,\'case_id\':cid,\'partition\':part,\'family\':label,\'scenario\':kind,\'stability_flag\':instability>.20,\'stability_turnover\':instability,\'evidence_flag\':finding[\'disposition\']==\'Investigate\',**finding})\n    evals=[x for x in outputs if x[\'partition\']==\'evaluation\'];summary=[]\n    for method,field in [(\'Repeatability screen\',\'stability_flag\'),(\'Structured evidence review\',\'evidence_flag\')]:\n        for family in [\'mapping\',\'coverage\',\'provenance\',\'all_defects\',\'correct\']:\n            selected=[x for x in evals if (x[\'family\'] in [\'mapping\',\'coverage\',\'provenance\'] if family==\'all_defects\' else x[\'family\']==family)]\n            summary.append({\'method\':method,\'family\':family,**cluster_interval([x[\'case_id\'] for x in selected],[x[field] for x in selected]),\'flagged\':sum(x[field] for x in selected)})\n    missing=[x for x in evals if x[\'family\']==\'insufficient\']\n    sweep=[]\n    for threshold in np.arange(0,1.001,.05):\n        sweep.append({\'threshold\':round(float(threshold),2),\'defect_detection\':float(np.mean([x[\'stability_turnover\']>threshold for x in evals if x[\'family\'] in [\'mapping\',\'coverage\',\'provenance\']])),\'false_alarm\':float(np.mean([x[\'stability_turnover\']>threshold for x in evals if x[\'family\']==\'correct\']))})\n    lock=json.loads((R/\'results/protocol_v2_lock.json\').read_text());actual=hashlib.sha256((R/\'docs/PROTOCOL_v2.md\').read_bytes()).hexdigest();assert actual==lock[\'protocol_sha256\']\n    result={\'title\':\'From Risk Scores to Reasons\',\'generated_at\':datetime.now(timezone.utc).isoformat(),\'protocol_sha256\':actual,\'scope\':\'Controlled contract-consistency benchmark; all policies, notices and defects are constructed.\',\'counts\':{\'development_records\':20,\'evaluation_records\':20,\'total_scenarios\':len(outputs),\'evaluation_scenarios\':len(evals),\'evaluation_defects\':80,\'evaluation_correct\':80,\'evaluation_missing_evidence\':20},\'summary\':summary,\'missing_evidence\':{\'cases\':len(missing),\'correct_abstentions\':sum(x[\'disposition\']==\'Insufficient evidence\' for x in missing)},\'threshold_sweep\':sweep,\'findings\':outputs}\n    for name,obj in [(\'inputs.json\',scenarios),(\'truth.json\',truth),(\'evaluation.json\',result)]: (out/name).write_text(json.dumps(obj,indent=2))\n    print(json.dumps({\'counts\':result[\'counts\'],\'summary\':summary,\'missing\':result[\'missing_evidence\']},indent=2))\n\nif __name__==\'__main__\':run()\n', 'src/notice_review_legacy.js': "/* Browser implementation of the evidence contract checker. Inputs contain no truth labels. */\nfunction reviewEvidence(bundle){\n const d=bundle.decision,c=bundle.catalog,n=bundle.notice||{},findings=[];\n if(!d||!c)return {disposition:'Insufficient evidence',findings:[{code:'missing_evidence',title:'Decision trace or mapping catalog is unavailable',evidence:'decision / catalog',request:'Obtain the versioned trace and reason catalog before assessing the notice.'}]};\n function add(code,title,evidence,request){findings.push({code,title,evidence,request})}\n if(n.decision_id!==d.id)add('identity','Notice refers to a different decision','notice.decision_id / decision.id','Recover the notice for the same decision identifier.');\n for(const [field,expected] of Object.entries(d.artifacts))if((n.artifacts||{})[field]!==expected)add('version',field+' does not match the decision record','notice.artifacts.'+field+' / decision.artifacts.'+field,'Regenerate from the retained decision-time artifacts; preserve the original notice.');\n (n.reasons||[]).forEach((r,i)=>{const spec=c[r.code];if(!spec){add('mapping','Reason code is absent from the supplied catalog',`notice.reasons[${i}].code`,'Provide the applicable catalog version and validate the description.');return}\n if(!spec.approved_wording.includes(r.text))add('wording','Reason wording has no support in the supplied mapping catalog',`notice.reasons[${i}].text / catalog.${r.code}`,'Review the description and direction against feature evidence; an unlisted paraphrase needs human review.');\n if(JSON.stringify([...(r.source_features||[])].sort())!==JSON.stringify([...spec.source_features].sort()))add('mapping','Stated reason and cited source features do not match',`notice.reasons[${i}].source_features / catalog.${r.code}`,'Correct the source-to-reason mapping and examine affected notices.');\n if(d.basis==='model'&&!(r.source_features||[]).every(f=>(d.attribution[f]||0)>0))add('support','Model reason cites a feature without a positive risk attribution',`notice.reasons[${i}] / decision.attribution`,'Verify sign, output class and reference before selecting this feature as an adverse contributor.');\n });\n if(d.basis==='eligibility_gate'){let observed=new Set((n.reasons||[]).map(r=>r.code)),required=new Set(d.required_reasons);\n if(![...required].every(r=>observed.has(r)))add('decision_coverage','The terminating eligibility rule is missing from the reasons','decision.rule_trace / notice.reasons','Explain the recorded rule that stopped the application; do not substitute an unused risk-score explanation.');\n if([...observed].some(r=>!required.has(r)))add('unused_score','Notice cites model factors after the policy gate stopped evaluation','decision.basis / notice.reasons','Separate diagnostic model outputs from the factors actually used in this decision.');}\n return {disposition:findings.length?'Investigate':'No discrepancy detected',findings};\n}\nfunction replayForest(forest,features){\n const raw=forest.features.map(f=>Number(features[f]));\n const x=raw.map((v,i)=>v<0?forest.medians[i]:v).concat(raw.map(v=>v===-7?1:0),raw.map(v=>v===-8?1:0),raw.map(v=>v===-9?1:0));\n let total=0;\n for(const t of forest.trees){let n=0;while(t.left[n]!==-1)n=x[t.feature[n]]<=t.threshold[n]?t.left[n]:t.right[n];let v=t.value[n];total+=v[1]/(v[0]+v[1]);}\n return total/forest.trees.length;\n}\nif(typeof module!=='undefined')module.exports={reviewEvidence,replayForest};\n", 'src/reference_experiment.py': '"""Frozen-model reference sensitivity and independent-neighborhood surrogate evaluation."""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport hashlib,itertools,json,warnings\nimport numpy as np\nimport pandas as pd\nimport joblib\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.linear_model import Ridge\nfrom sklearn.metrics import r2_score,mean_absolute_error\nif __package__:\n    from .audit import FEATURES,Encoding,Predictor,exact_shapley,reason_list,compare,SEED\n    from .notice_benchmark import cluster_interval\nelse:\n    from audit import FEATURES,Encoding,Predictor,exact_shapley,reason_list,compare,SEED\n    from notice_benchmark import cluster_interval\nR=Path(__file__).resolve().parents[1]\nwarnings.filterwarnings(\'ignore\',message=\'.*sklearn.utils.parallel.delayed.*\',category=UserWarning)\n\ndef neighborhood(x,train,seed,n=1200,include_anchor=True):\n    rng=np.random.default_rng(seed);d=len(x)\n    z=np.column_stack([rng.choice(train[:,j],n) for j in range(d)])\n    z=np.where(rng.random((n,d))<.55,x,z)\n    if include_anchor:z[0]=x\n    scale=np.maximum(np.nanstd(np.where(train<0,np.nan,train),axis=0),1)\n    delta=(z-x)/scale;delta=np.where((z<0)|(x<0),np.where(z==x,0,1),delta)\n    weights=np.exp(-np.mean(delta**2,axis=1)/(.75**2))\n    q=np.array([np.quantile(train[:,j][train[:,j]>=0],[.25,.5,.75]) for j in range(d)])\n    def bins(a):return np.column_stack([np.where(a[:,j]<0,a[:,j],np.searchsorted(q[j],a[:,j])) for j in range(d)])\n    reps=(bins(z)==bins(np.asarray(x)[None,:])).astype(float)\n    return z,reps,weights\n\ndef run():\n    lock=json.loads((R/\'results/protocol_v2_lock.json\').read_text());assert hashlib.sha256((R/\'docs/PROTOCOL_v2.md\').read_bytes()).hexdigest()==lock[\'protocol_sha256\']\n    df=pd.read_csv(R/\'data/heloc_dataset_v1.csv\');split=json.loads((R/\'results/reference/split.json\').read_text())\n    old=json.loads((R/\'results/reference/audit_results.json\').read_text());cases=[c for c in old[\'cases\'] if c[\'model\']==\'without_score\']\n    train=df.loc[split[\'train\'],FEATURES].to_numpy(float);y=(df.loc[split[\'train\'],\'RiskPerformance\']==\'Bad\').astype(int)\n    enc=Encoding().fit(train);model=RandomForestClassifier(n_estimators=100,max_depth=7,min_samples_leaf=15,max_features=.8,random_state=SEED,n_jobs=2)\n    model.fit(enc.transform(train),y);predict=Predictor(model,enc)\n    for c in cases:assert abs(predict(np.array(list(c[\'features\'].values()))[None,:])[0]-c[\'score\'])<1e-10\n    scores=predict(train);near=train[(scores>=.45)&(scores<.50)];assert len(near)>=24\n    refs={}\n    for scope,pool in [(\'population\',train),(\'near_cutoff\',near)]:\n        for seed in [101,202,303]:refs[(scope,seed)]=pool[np.random.default_rng(seed).choice(len(pool),24,replace=False)]\n    out=R/\'results/reference_sensitivity\';out.mkdir(exist_ok=True)\n    artifact=out/\'rf8_model.joblib\';joblib.dump({\'model\':model,\'encoding\':enc,\'features\':FEATURES,\'split\':split},artifact)\n    manifests=[{\'reference\':scope,\'seed\':seed,\'values\':bg.tolist(),\'sha256\':hashlib.sha256(bg.tobytes()).hexdigest()} for (scope,seed),bg in refs.items()]\n    records=[];comparisons=[];fidelity=[]\n    for i,c in enumerate(cases):\n        cid=c[\'case_id\'];x=df.loc[cid,FEATURES].to_numpy(float);by={}\n        for (scope,seed),bg in refs.items():\n            v,diagnostics=exact_shapley(x,bg,predict);reason=reason_list(v,FEATURES)\n            entry={\'case_id\':cid,\'reference\':scope,\'seed\':seed,\'score\':c[\'score\'],\'values\':v.tolist(),\'individual\':reason,\'grouped\':reason_list(v,FEATURES,True),\'diagnostics\':diagnostics};records.append(entry);by[(scope,seed)]=entry\n        for seed in [101,202,303]:\n            comparisons.append({\'case_id\':cid,\'test\':\'reference_definition\',\'reference\':\'population_vs_near_cutoff\',\'seed\':seed,**compare(by[(\'population\',seed)][\'individual\'],by[(\'near_cutoff\',seed)][\'individual\'])})\n        for scope in [\'population\',\'near_cutoff\']:\n            for a,b in itertools.combinations([101,202,303],2):comparisons.append({\'case_id\':cid,\'test\':\'background_resampling\',\'reference\':scope,\'seed\':f\'{a}:{b}\',**compare(by[(scope,a)][\'individual\'],by[(scope,b)][\'individual\'])})\n        for seed in range(5):\n            z,reps,w=neighborhood(x,train,seed);f=Ridge(alpha=1).fit(reps,predict(z),sample_weight=w)\n            z2,reps2,w2=neighborhood(x,train,seed+10000,include_anchor=False);actual=predict(z2);estimated=f.predict(reps2)\n            # Exact agreement with earlier local-surrogate weights validates the reconstruction.\n            np.testing.assert_allclose(f.coef_,c[\'explanations\'][\'Custom LIME-style surrogate\'][seed][\'values\'],rtol=1e-10,atol=1e-10)\n            point_prediction=float(f.predict(np.ones((1,len(x))))[0])\n            fidelity.append({\'case_id\':cid,\'seed\':seed,\'fit_r2\':float(f.score(reps,predict(z),sample_weight=w)),\'heldout_r2\':float(r2_score(actual,estimated,sample_weight=w2)),\'heldout_mae\':float(mean_absolute_error(actual,estimated,sample_weight=w2)),\'point_error\':abs(point_prediction-c[\'score\'])})\n        if (i+1)%10==0:print(f\'Explained {i+1}/{len(cases)} records\',flush=True)\n    summary=[]\n    for test,scope in [(\'reference_definition\',\'population_vs_near_cutoff\'),(\'background_resampling\',\'population\'),(\'background_resampling\',\'near_cutoff\')]:\n        rows=[r for r in comparisons if r[\'test\']==test and r[\'reference\']==scope and r[\'retention\'] is not None]\n        metrics={}\n        for field in [\'turnover\',\'retention\',\'first_changed\']:metrics[field]=cluster_interval([r[\'case_id\'] for r in rows],[float(r[field]) for r in rows])\n        summary.append({\'test\':test,\'reference\':scope,**metrics})\n    result={\'generated_at\':datetime.now(timezone.utc).isoformat(),\'protocol_sha256\':lock[\'protocol_sha256\'],\'model_sha256\':hashlib.sha256(artifact.read_bytes()).hexdigest(),\'method\':\'Exact empirical interventional Shapley; custom LIME-style surrogate\',\'reference_pool_counts\':{\'population\':len(train),\'near_cutoff\':len(near)},\'model_test_auc\':old[\'models\'][0][\'metrics\'][\'test\'][\'auc\'],\'summary\':summary,\'fidelity_summary\':{k:{\'median\':float(np.median([r[k] for r in fidelity])),\'min\':float(min(r[k] for r in fidelity)),\'max\':float(max(r[k] for r in fidelity))} for k in [\'fit_r2\',\'heldout_r2\',\'heldout_mae\',\'point_error\']},\'records\':records,\'fidelity\':fidelity,\'comparisons\':comparisons,\'reference_manifests\':manifests}\n    (out/\'experiment.json\').write_text(json.dumps(result,indent=2));pd.DataFrame(comparisons).to_csv(out/\'comparisons.csv\',index=False);pd.DataFrame(fidelity).to_csv(out/\'fidelity.csv\',index=False)\n    print(json.dumps({k:result[k] for k in [\'reference_pool_counts\',\'summary\',\'fidelity_summary\']},indent=2))\n\nif __name__==\'__main__\':run()\n', 'src/requirements-appendix.html': '<p class="legal-standard">Under ECOA and Regulation B, creditors must give the principal reasons for a covered adverse action or explain how applicants can request them. The reasons must accurately describe factors actually considered or scored. Saying that an applicant failed to reach a qualifying score is insufficient.</p>\n<p class="small"><a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/9/" target="_blank" rel="noopener">Regulation B §1002.9</a> · <a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/" target="_blank" rel="noopener">Official interpretation</a></p>\n\n<section class="policy-box"><p class="eyebrow">Recommendation</p><h2>External-score use should trigger explanation-provenance controls</h2><p>If a lender materially relies on an external score, it should preserve the score/version used, distinguish its own model evidence from inferred upstream explanations, retain the mapping that produced any consumer-facing reason, and obtain provider-linked reason or validation evidence when the notice depends on claims about the upstream score.</p><p><strong>Reconstruction is diagnostic evidence, not a substitute for provider-linked reasons.</strong></p></section>\n<div class="consumer-takeaway"><p class="eyebrow">Consumer-advocate takeaway</p><h2>Vendor opacity should not be passed downstream to the applicant.</h2><p>A specific-sounding reason should be challengeable when the creditor cannot show how that reason connects to the decision system actually used. The project does not establish that any named lender failed this standard; it identifies the evidence gap a reviewer or advocate should probe.</p></div>\n<section class="case-entry"><p class="eyebrow">CFPB · Auto lending · 2025</p><h2>Lenders had not validated the accuracy of their denial reasons</h2><p>CFPB examiners reviewed AI/ML credit-scoring models, some with over 1,000 variables. They found that institutions had not validated whether their reason-selection processes produced accurate results and directed lenders to test those methods.</p><p>The finding concerns the same distinction examined in this project: a model can produce a prediction and an explanation without establishing that the selected denial reasons are accurate. The report does not identify SHAP or LIME as the methods involved.</p><a href="https://business.cch.com/BFLD/CFPB-Supervisory-Highlights-Advanced-Technologies-Special-Edition-Issue-38-Winter-2025-01172025.pdf" target="_blank" rel="noopener">Supervisory Highlights, Winter 2025, §2.1.3, pp. 7–8 · preserved report</a></section>\n<section class="case-entry"><p class="eyebrow">NYDFS · Apple Card · 2021</p><h2>Customers had little insight into decisions the bank could explain to examiners</h2><p>Goldman could explain individual credit decisions to NYDFS, but customers experienced poor transparency and frustrating appeals over credit terms. The investigation did not establish unlawful discrimination. The case distinguishes a regulator’s access to decision evidence from a customer’s ability to understand and challenge the outcome. This study provides descriptive labels, but has not tested applicant comprehension.</p><a href="https://www.dfs.ny.gov/system/files/documents/2021/03/rpt_202103_apple_card_investigation.pdf" target="_blank" rel="noopener">NYDFS investigation report, pp. 7–8</a></section>\n<details><summary>Further examples: incorrect notices and an external credit assessment</summary><div><p><strong>CFPB, 2020:</strong> payday lenders sent incorrect principal reasons, including numerous notices caused by a coding error. The notices violated Regulation B; lenders corrected them and changed their systems. No AI method was identified. Controlled notice-mapping errors would be a separate evaluation of the translation into notice wording. <a href="https://files.consumerfinance.gov/f/documents/cfpb_supervisory-highlights_issue-21_2020-02.pdf" target="_blank" rel="noopener">Official finding, §2.3.5</a></p><p><strong>Dun &amp; Bradstreet Austria, 2025:</strong> following a mobile-contract refusal based on an automated credit assessment, the CJEU required an intelligible account of how the person’s data were actually used. This EU case concerns an outside scoring provider under GDPR, a different legal framework from US notice duties. <a href="https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:62022CJ0203" target="_blank" rel="noopener">C-203/22 judgment</a></p><p><strong>CFPB Adverse Action Tech Sprint, 2020:</strong> participants explored digital notices and interactive explanations. This was a design event, not an enforcement finding. It establishes prior work on the interface; this project’s contribution must rest on the evaluated explanation problem. <a href="https://www.consumerfinance.gov/rules-policy/competition-innovation/cfpb-tech-sprints/electronic-disclosures-tech-sprint/" target="_blank" rel="noopener">Official account</a></p></div></details>\n<p class="small muted">The CFPB withdrew Circulars 2022-03 and 2023-03 in May 2025. The notice standard described here rests on Regulation B and its official interpretation. <a href="https://www.consumerfinance.gov/compliance/guidance/withdrawn-guidance/" target="_blank" rel="noopener">Source status</a></p>\n<div class="actions next-page"><button class="primary" data-page="investigate">Inspect the research model</button></div>\n', 'src/score_reconstruction.py': '"""Exploratory external-score approximations; no original upstream ground truth."""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport json, hashlib, warnings\nimport numpy as np,pandas as pd,joblib\nfrom sklearn.ensemble import HistGradientBoostingRegressor,RandomForestClassifier\nfrom sklearn.linear_model import Ridge\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import mean_absolute_error,r2_score\nfrom audit import FEATURES,Encoding,Predictor,SEED,exact_shapley,reason_list,compare\nR=Path(__file__).resolve().parents[1]\nwarnings.filterwarnings(\'ignore\',message=\'.*sklearn.utils.parallel.delayed.*\')\n\nclass Combined:\n    def __init__(self,approx,encoding,downstream,bounds):self.approx,self.encoding,self.downstream,self.bounds=approx,encoding,downstream,bounds\n    def score(self,x):return np.clip(self.approx.predict(self.encoding.transform(x)),*self.bounds)\n    def __call__(self,x):return self.downstream(np.column_stack([x,self.score(x)]))\n\ndef run():\n    out=R/\'results/score_reconstruction\';out.mkdir(exist_ok=True)\n    raw=pd.read_csv(R/\'data/heloc_dataset_v1.csv\');split=json.loads((R/\'results/reference/split.json\').read_text())\n    previous=json.loads((R/\'results/external_score/experiment.json\').read_text())\n    eligible={k:[i for i in split[k] if raw.loc[i,\'ExternalRiskEstimate\']>=0] for k in [\'train\',\'validation\',\'test\']}\n    tr=raw.loc[eligible[\'train\']];train=tr[FEATURES].to_numpy(float)\n    enc=Encoding().fit(train);bounds=[float(tr.ExternalRiskEstimate.min()),float(tr.ExternalRiskEstimate.max())]\n    alltrain=raw.loc[split[\'train\'],FEATURES+[\'ExternalRiskEstimate\']].to_numpy(float)\n    downenc=Encoding().fit(alltrain)\n    rf=RandomForestClassifier(n_estimators=100,max_depth=7,min_samples_leaf=15,max_features=.8,random_state=SEED,n_jobs=2)\n    rf.fit(downenc.transform(alltrain),(raw.loc[split[\'train\'],\'RiskPerformance\']==\'Bad\').astype(int))\n    original=Predictor(rf,downenc)\n    approximations={\'additive\':make_pipeline(StandardScaler(),Ridge(alpha=10)),\n                    \'nonlinear\':HistGradientBoostingRegressor(max_iter=150,max_leaf_nodes=15,l2_regularization=5,random_state=SEED)}\n    labels={\'additive\':\'Additive score approximation\',\'nonlinear\':\'Nonlinear score approximation\'}\n    combined={};metrics={};test=raw.loc[eligible[\'test\']];x_test=test[FEATURES].to_numpy(float)\n    true_prob=original(test[FEATURES+[\'ExternalRiskEstimate\']].to_numpy(float));test_predictions={}\n    for key,m in approximations.items():\n        m.fit(enc.transform(train),tr.ExternalRiskEstimate)\n        fn=Combined(m,enc,original,bounds);combined[key]=fn;sp=fn.score(x_test);pr=fn(x_test);test_predictions[key]=pr\n        metrics[key]={\'label\':labels[key],\'test_records\':len(test),\'external_score_mae\':float(mean_absolute_error(test.ExternalRiskEstimate,sp)),\n                      \'external_score_r2\':float(r2_score(test.ExternalRiskEstimate,sp)),\n                      \'probability_mae\':float(np.mean(abs(pr-true_prob))),\n                      \'probability_error_quantiles\':list(map(float,np.quantile(abs(pr-true_prob),[.5,.9,.95,.99]))),\n                      \'unclipped_out_of_training_range\':int(np.sum((m.predict(enc.transform(x_test))<bounds[0])|(m.predict(enc.transform(x_test))>bounds[1])))}\n        print(key,json.dumps(metrics[key]),flush=True)\n    ref_positions=np.random.default_rng(SEED).choice(len(train),24,replace=False);background=train[ref_positions]\n    original_background=raw.loc[[eligible[\'train\'][i] for i in ref_positions],FEATURES+[\'ExternalRiskEstimate\']].to_numpy(float)\n    audit_ids=[r[\'case_id\'] for r in previous[\'records\'] if r[\'model\']==\'with_score\' and r[\'features\'][\'ExternalRiskEstimate\']>=0]\n    cases=[]\n    for idx,cid in enumerate(audit_ids):\n        x=raw.loc[cid,FEATURES].to_numpy(float);orig=next(r for r in previous[\'records\'] if r[\'model\']==\'with_score\' and r[\'case_id\']==cid)\n        actual=float(original(raw.loc[cid,FEATURES+[\'ExternalRiskEstimate\']].to_numpy(float)[None,:])[0])\n        assert abs(actual-orig[\'score\'])<1e-10\n        original_values,original_diagnostics=exact_shapley(raw.loc[cid,FEATURES+[\'ExternalRiskEstimate\']].to_numpy(float),original_background,original)\n        original_explanation={\'values\':original_values.tolist(),\'diagnostics\':original_diagnostics,\'individual\':reason_list(original_values,FEATURES+[\'ExternalRiskEstimate\']),\'grouped\':reason_list(original_values,FEATURES+[\'ExternalRiskEstimate\'],True)}\n        item={\'case_id\':cid,\'inputs\':dict(zip(FEATURES,map(float,x))),\'observed_external_score\':float(raw.loc[cid,\'ExternalRiskEstimate\']),\n              \'original_probability\':actual,\'original_shapley\':original_explanation,\'approximations\':{}}\n        for key,fn in combined.items():\n            v,d=exact_shapley(x,background,fn);p=float(fn(x[None,:])[0])\n            item[\'approximations\'][key]={\'estimated_external_score\':float(fn.score(x[None,:])[0]),\'probability\':p,\'probability_error\':abs(p-actual),\n                                         \'values\':list(map(float,v)),\'diagnostics\':d,\'individual\':reason_list(v,FEATURES),\'grouped\':reason_list(v,FEATURES,True)}\n        a,b=[item[\'approximations\'][k] for k in [\'additive\',\'nonlinear\']]\n        ca=compare(a[\'individual\'],b[\'individual\']);cg=compare(a[\'grouped\'],b[\'grouped\'])\n        item[\'comparison\']={\'probability_difference\':abs(a[\'probability\']-b[\'probability\']),\n                            \'maximum_original_probability_error\':max(a[\'probability_error\'],b[\'probability_error\']),\n                            \'individual\':ca,\'grouped\':cg,\'same_order\':[r[\'id\'] for r in a[\'individual\']]==[r[\'id\'] for r in b[\'individual\']]}\n        cases.append(item)\n        if (idx+1)%10==0:print(\'Explained\',idx+1,\'/\',len(audit_ids),flush=True)\n    bands=[]\n    for t in [.005,.01,.02,.05]:\n        close=[c for c in cases if c[\'comparison\'][\'probability_difference\']<=t]\n        both=[c for c in close if c[\'comparison\'][\'maximum_original_probability_error\']<=t]\n        bands.append({\'probability_band\':t,\'approximate_outputs_close\':len(close),\n                      \'both_close_to_original\':len(both),\n                      \'different_factor_sets_when_outputs_close\':sum(bool(c[\'comparison\'][\'individual\'][\'turnover\']) for c in close),\n                      \'different_factor_sets_when_both_close_to_original\':sum(bool(c[\'comparison\'][\'individual\'][\'turnover\']) for c in both),\n                      \'different_concept_sets_when_both_close_to_original\':sum(bool(c[\'comparison\'][\'grouped\'][\'turnover\']) for c in both)})\n    ranked=sorted(cases,key=lambda c:(not bool(c[\'comparison\'][\'individual\'][\'turnover\']),c[\'comparison\'][\'maximum_original_probability_error\'],c[\'case_id\']))\n    # Pick smallest maximum original-prediction error among changed factor sets; disclose selection.\n    featured=8193 if any(c[\'case_id\']==8193 for c in cases) else ranked[0][\'case_id\']\n    payload={\'generated_at\':datetime.now(timezone.utc).isoformat(),\'design\':\'Exploratory; same-test feasibility metrics were already known.\',\n             \'protocol_sha256\':hashlib.sha256((R/\'docs/EXTERNAL_SCORE_RECONSTRUCTION.md\').read_bytes()).hexdigest(),\n             \'features\':FEATURES,\'counts\':{k:len(v) for k,v in eligible.items()},\'score_clipping_bounds\':bounds,\'metrics\':metrics,\'cases\':cases,\'bands\':bands,\n             \'summary\':{\'audit_records\':len(cases),\'different_factor_sets\':sum(bool(c[\'comparison\'][\'individual\'][\'turnover\']) for c in cases),\n                        \'different_concept_sets\':sum(bool(c[\'comparison\'][\'grouped\'][\'turnover\']) for c in cases),\n                        \'test_probability_difference_mean\':float(np.mean(abs(test_predictions[\'additive\']-test_predictions[\'nonlinear\'])))},\n             \'featured_case\':featured,\'featured_selection\':\'Record 8193 selected after inspection to illustrate detailed-factor changes with close original-prediction agreement and unchanged broader topics. See all forty records; the example is not representative.\',\n             \'background_record_ids\':[eligible[\'train\'][i] for i in ref_positions],\n             \'limitations\':[\'Original external-score mechanism is unavailable.\',\'Both explanations describe reconstructed systems, not authenticated original upstream factors.\',\'References and feature dependence affect attributions.\',\'Earlier test results and feasibility results were known; evaluation is exploratory.\',\'Forty audited records are deliberately selected, not representative of lender decisions.\']}\n    (out/\'experiment.json\').write_text(json.dumps(payload,indent=2))\n    joblib.dump({\'approximations\':approximations,\'encoding\':enc,\'original_model\':rf,\'original_encoding\':downenc,\'score_clipping_bounds\':bounds,\'features\':FEATURES},out/\'models.joblib\')\n    testrows=pd.DataFrame({\'case_id\':eligible[\'test\'],\'observed_external_score\':test.ExternalRiskEstimate.to_numpy(),\'original_probability\':true_prob,\n                          \'additive_probability\':test_predictions[\'additive\'],\'nonlinear_probability\':test_predictions[\'nonlinear\']})\n    testrows.to_csv(out/\'test_predictions.csv\',index=False)\n    print(json.dumps({\'summary\':payload[\'summary\'],\'bands\':bands,\'featured_case\':featured},indent=2))\nif __name__==\'__main__\':run()\n', 'src/workbench.html': '<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Credit explanation review · External scores</title>\n<link rel="preconnect" href="https://fonts.googleapis.com"><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin><link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Fraunces:opsz,wght@9..144,400;9..144,500&family=IBM+Plex+Mono:wght@400;500&display=swap" rel="stylesheet"><style>\n:root{--paper:#f7f5f0;--ink:#182c2d;--sub:#506362;--line:#d8ddd7;--white:#fffefa;--teal:#14655d;--pale:#e5eee8;--rust:#a44125;--warm:#f8ebe2;--blue:#345fa2}*{box-sizing:border-box}body{margin:0;background:var(--paper);color:var(--ink);font:16px/1.65 system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif}button,select,textarea{font:inherit}button,a,select,summary{touch-action:manipulation}button{cursor:pointer}a{color:var(--teal);text-underline-offset:4px}button:focus-visible,a:focus-visible,select:focus-visible,summary:focus-visible{outline:3px solid var(--blue);outline-offset:4px}header{border-bottom:1px solid var(--line);padding:20px max(24px,calc((100vw - 1160px)/2));display:flex;align-items:center;justify-content:space-between;gap:24px}.brand{font-size:14px;font-weight:650;white-space:nowrap}.brand span{font-weight:400;color:var(--sub)}nav{display:flex;gap:4px;flex-wrap:wrap}nav button{background:none;border:0;padding:10px 12px;color:var(--sub);border-radius:4px}nav button[aria-current=page]{color:var(--ink);background:var(--pale);font-weight:650}main{max-width:1160px;margin:0 auto;padding:48px 28px 70px}h1,h2,h3,p{margin-top:0}h1,h2{font-family:Georgia,"Times New Roman",serif;font-weight:400;line-height:1.13;letter-spacing:-.03em}h1{font-size:clamp(38px,5vw,62px);max-width:870px;margin-bottom:24px}h2{font-size:30px;margin-bottom:18px}h3{font-size:18px;line-height:1.4;margin-bottom:12px}p{margin-bottom:20px}.intro{max-width:790px;font-size:19px;line-height:1.7;color:var(--sub)}.eyebrow{font-size:12px;letter-spacing:.08em;text-transform:uppercase;color:var(--teal);font-weight:650;margin-bottom:16px}.hero{max-width:850px;padding:16px 0 35px}.actions{display:flex;flex-wrap:wrap;gap:12px;margin:22px 0}.primary,.secondary{display:inline-block;border-radius:4px;border:1px solid var(--teal);padding:10px 18px;font-weight:600;text-decoration:none}.primary{background:var(--teal);color:var(--white)}.secondary{background:transparent;color:var(--teal)}.text-button{border:0;background:none;color:var(--teal);padding:6px 0;text-decoration:underline;text-underline-offset:4px;font-size:14px}.page[hidden],[hidden]{display:none!important}.reading{max-width:820px}.reading>section{margin:32px 0}.context{max-width:820px;padding:25px 0;border-top:1px solid var(--line)}.small{font-size:13px;line-height:1.6}.muted{color:var(--sub)}.grid{display:grid;grid-template-columns:1fr 1fr;gap:30px}.card{background:var(--white);border:1px solid var(--line);padding:24px;border-radius:5px;margin:20px 0}.metric{font:36px/1.2 Georgia,serif;margin-bottom:8px}.metric-row{display:grid;grid-template-columns:1fr 1.2fr;gap:35px;padding:22px 0;border-top:1px solid var(--line);border-bottom:1px solid var(--line);margin-bottom:28px}.metric-row p{margin:0}.metric-label{font-size:14px;font-weight:600;display:block;margin-bottom:8px}.case-toolbar{display:flex;justify-content:space-between;align-items:center;gap:20px;margin-bottom:28px;padding-bottom:20px;border-bottom:1px solid var(--line)}.case-toolbar p{margin:0;font-size:13px;color:var(--sub)}.field{font-size:13px;font-weight:600;display:flex;gap:12px;align-items:center;flex-wrap:wrap}.field select{max-width:100%;min-height:44px;padding:7px 30px 7px 10px;border:1px solid var(--line);background:var(--white);color:var(--ink);border-radius:4px}details{border-top:1px solid var(--line);padding:16px 0;margin-top:12px}summary{cursor:pointer;font-weight:600}details>div{padding-top:16px}.table-scroll{overflow:auto}table{width:100%;border-collapse:collapse;font-size:14px;text-align:left}th,td{padding:13px 10px;border-bottom:1px solid var(--line);vertical-align:top}th{font-weight:600;background:var(--pale)}caption{text-align:left;margin-bottom:12px;font-weight:600}.bar-chart{margin:20px 0}.bar-axis,.bar-row{display:grid;grid-template-columns:minmax(150px,1fr) minmax(90px,1.2fr) 72px;gap:10px;align-items:center}.bar-axis{font-size:12px;color:var(--sub);margin-bottom:9px}.bar-row{min-height:38px;font-size:13px;padding:5px 0}.bar-track{height:20px;position:relative;background:linear-gradient(to right,transparent 49.5%,var(--line) 49.5%,var(--line) 50.5%,transparent 50.5%)}.bar-fill{position:absolute;height:14px;top:3px}.bar-val{text-align:right;font-variant-numeric:tabular-nums;white-space:nowrap}.highlight-row{background:var(--pale);border-radius:3px}.chart-summary{display:flex;justify-content:space-between;gap:15px;flex-wrap:wrap;font-size:14px;border-top:1px solid var(--line);padding-top:14px}.compare-head{display:flex;justify-content:space-between;gap:18px;align-items:center;flex-wrap:wrap;margin:25px 0 10px}.compare-head h2{margin:0;font-size:25px}.segmented{display:flex;gap:4px;flex-wrap:wrap}.segmented button{border:1px solid var(--line);background:transparent;padding:8px 12px;font-size:13px;color:var(--sub);border-radius:3px}.segmented button[aria-pressed=true]{background:var(--teal);color:var(--white);border-color:var(--teal)}.stat-line{display:flex;justify-content:space-between;gap:15px;border-bottom:1px solid var(--line);padding:8px 0;font-size:14px}.stat-line strong{font-variant-numeric:tabular-nums;white-space:nowrap}.conclusion{max-width:820px;border-left:3px solid var(--teal);padding:4px 0 4px 20px;margin:24px 0}.conclusion p{margin:0}.download-links{display:flex;gap:18px;flex-wrap:wrap;margin:16px 0}#report-preview{width:100%;min-height:190px;background:var(--paper);border:1px solid var(--line);border-radius:4px;padding:14px;color:var(--ink)}.report-state{font-size:13px;color:var(--sub)}code{font-size:12px;overflow-wrap:anywhere}.legal-row{padding:22px 0;border-top:1px solid var(--line)}.legal-row h3{margin:8px 0}.pill{font-size:12px;font-weight:600;color:var(--teal)}.note{border-left:2px solid var(--line);padding-left:16px;color:var(--sub);font-size:14px}.result-lead{font-size:20px;line-height:1.65}.export{margin-top:35px;padding-top:22px;border-top:1px solid var(--line)}footer{max-width:1160px;margin:auto;border-top:1px solid var(--line);padding:20px 28px;display:flex;justify-content:space-between;gap:18px;flex-wrap:wrap;font-size:12px;color:var(--sub)}footer div{display:flex;gap:22px;flex-wrap:wrap}footer button{font-size:12px!important}ul{padding-left:20px}@media(max-width:750px){header{display:block;padding:18px 20px}nav{margin-top:15px;gap:4px}nav button{font-size:13px;padding:9px 10px;min-height:44px}main{padding:28px 20px 50px}.intro{font-size:17px}.grid{grid-template-columns:1fr;gap:0}.metric-row{gap:20px;grid-template-columns:1fr 1fr}.card{padding:18px}.case-toolbar{align-items:flex-start;flex-direction:column;gap:8px}.bar-axis,.bar-row{grid-template-columns:minmax(95px,1fr) minmax(70px,1.1fr) 62px;gap:6px}.bar-row{font-size:12px;min-height:44px}.bar-axis{font-size:11px}.bar-val{font-size:11px}.segmented button{min-height:44px}.reading table{min-width:460px}.result-lead{font-size:18px}.stat-line{font-size:13px}h2{font-size:27px}.metric{font-size:31px}}@media(max-width:420px){.metric-row{grid-template-columns:1fr;gap:18px}.brand span{display:none}}@media print{header,footer,.case-toolbar,.actions,.export,.segmented{display:none}.page{display:none!important}#research{display:block!important}body{background:white}main{padding:0}.card{break-inside:avoid}}\n\n/* Document excerpts provide context without adding dashboard controls. */\nheader{display:block;padding-top:22px;padding-bottom:0}.brand{margin-bottom:14px}nav{gap:12px}nav button{border-radius:0;padding:11px 0;margin-right:10px;border-bottom:2px solid transparent;font-size:14px}nav button[aria-current=page]{background:transparent;border-bottom-color:var(--teal)}.opening-layout{display:grid;grid-template-columns:minmax(0,1.55fr) minmax(260px,.85fr);gap:65px;align-items:start}.hero h1{font-size:clamp(38px,4.5vw,58px)}.hero .intro{font-size:19px}.hero p{max-width:640px}.document-example{padding:50px 5px 15px}.paper-notice{position:relative;transform:rotate(1deg);background:#fffdf6;color:#293333;padding:32px 28px 30px;box-shadow:0 5px 15px #182c2d15;border:1px solid #d9d4c7;border-radius:1px}.paper-notice:after{content:"";position:absolute;bottom:-6px;left:-1px;right:-1px;height:8px;background:#fffdf6;clip-path:polygon(0 0,100% 0,100% 55%,94% 80%,87% 55%,79% 95%,72% 63%,66% 80%,59% 45%,52% 85%,46% 60%,39% 95%,33% 65%,26% 80%,19% 40%,12% 80%,6% 55%,0 75%)}.document-label{font:12px/1.5 ui-monospace,monospace;letter-spacing:.03em;color:#6b695f;margin-bottom:26px}.paper-notice h2{font-size:29px;line-height:1.2;letter-spacing:-.025em}.document-rule{border-top:1px solid #ddd7c9;margin:20px 0}.notice-reason{font:17px/1.6 Georgia,serif;margin:18px 0;padding-left:15px;border-left:2px solid #bc8b38}.document-note{font-size:12px;line-height:1.6;color:#65655f}.document-source{font-size:12px;color:#315d59}.document-caption{font-size:12px;line-height:1.6;color:var(--sub);margin:20px 12px}.source-extract{max-width:820px;margin-top:8px;padding:24px 0 20px;border-top:1px solid var(--line)}.source-extract h2{font-size:27px;margin-bottom:12px}.source-extract h2 a{color:var(--ink);text-decoration:none}.source-extract h2 a:hover{text-decoration:underline}.source-extract p{max-width:760px}.source-date{font-size:12px;color:var(--sub);margin-bottom:12px}.case-quotation{font:23px/1.5 Georgia,serif;margin:16px 0 22px;padding-left:18px;border-left:2px solid #bc8b38}.case-quotation cite{display:block;font:12px/1.6 system-ui,sans-serif;color:var(--sub);margin-top:10px}@media(max-width:850px){.opening-layout{grid-template-columns:1fr;gap:0}.document-example{max-width:460px;padding-top:0;padding-bottom:20px}.paper-notice{transform:none}.hero{padding-bottom:15px}nav{gap:5px 14px}nav button{font-size:13px;margin-right:0;min-height:44px}}@media(prefers-reduced-motion:reduce){.paper-notice{transform:none}}\n\n.opening-layout{grid-template-columns:minmax(0,1.1fr) minmax(320px,1fr);gap:52px;align-items:start}.hero{padding-top:0}.hero h1{font-size:clamp(36px,4vw,53px)}.hero .intro{font-size:18px}.document-example{padding:0 0 20px}.paper-notice{transform:none;background:#fff;color:#20272b;padding:30px 32px;border:1px solid #ddd;box-shadow:0 3px 12px #182c2d10;border-radius:0;font:13px/1.65 Arial,Helvetica,sans-serif}.paper-notice:after{display:none}.paper-notice p{margin:0 0 17px}.document-label{font:12px/1.5 system-ui,sans-serif;margin:0 0 12px;color:var(--sub)}.letter-department{font-size:13px;font-weight:700;letter-spacing:.09em;padding-bottom:16px;border-bottom:1px solid #d7d7d7;margin-bottom:22px}.letter-date{font-size:12px;color:#50575a}.letter-subject{font-weight:700}.letter-reasons{padding-left:21px;margin:0 0 20px}.letter-reasons li{margin:9px 0}.letter-signature{margin-top:25px!important}.document-caption{margin:14px 2px 0}.landing-connection{max-width:none;display:grid;grid-template-columns:1fr 2fr;gap:10px 45px;padding-top:30px}.landing-connection h2{font-size:28px}.landing-connection p{max-width:710px}.landing-connection .small{grid-column:2}.case-entry{border-top:1px solid var(--line);padding:24px 0}.case-entry h2{font-size:26px}.case-entry p:last-child{margin-bottom:0}.evidence-table td:first-child{min-width:150px}.evidence-table td{line-height:1.65}.legal-standard{font-size:19px;line-height:1.7;padding-bottom:24px}.next-page{margin-top:28px}nav{column-gap:15px;row-gap:0}nav button{margin-right:0;font-size:13px}@media(max-width:850px){.opening-layout{grid-template-columns:1fr;gap:10px}.document-example{max-width:510px;margin-bottom:10px}.landing-connection{grid-template-columns:1fr;gap:0}.landing-connection .small{grid-column:auto}.paper-notice{padding:26px}.hero h1{font-size:40px}}@media(max-width:420px){.hero h1{font-size:35px}.paper-notice{padding:22px;font-size:12.5px}}\n\n/* Editorial palette, document textures and sliding controls. */\n:root{--paper:#f7f1e6;--ink:#32283d;--sub:#665b6e;--line:#ddd1d8;--white:#fffbf3;--teal:#6a315f;--pale:#eee1ef;--rust:#bf4935;--warm:#f7e0cd;--blue:#5f559d;--ochre:#e7b35a;--sea:#247972}body{font-family:\'DM Sans\',system-ui,sans-serif;background:radial-gradient(ellipse at 95% 5%,#e9dced55,transparent 45%),var(--paper)}h1,h2{font-family:\'Fraunces\',Georgia,serif;font-weight:400;letter-spacing:-.035em}header{background:#f7f1e6ee;border-top:5px solid var(--teal);border-bottom:1px solid #d8c8d5;padding-top:20px}.brand{font-family:\'IBM Plex Mono\',monospace;font-size:16px;letter-spacing:.09em;color:var(--teal)}.brand span{font:13px \'DM Sans\',system-ui,sans-serif;letter-spacing:0}.eyebrow{font-family:\'IBM Plex Mono\',monospace;font-size:11px;font-weight:500;letter-spacing:.06em}.hero .eyebrow{display:inline-block;background:var(--pale);padding:7px 10px;transform:rotate(-1deg)}.hero h1{color:var(--teal);font-size:clamp(39px,4.5vw,61px);line-height:1.07}.hero .intro{color:var(--ink);font-size:19px}.primary{background:var(--teal);border-color:var(--teal);box-shadow:3px 3px 0 #d8b2c5}.primary:hover{background:#502348}.secondary{border-color:#bba1b6;background:#f8f0f5}.primary,.secondary{border-radius:22px;font-size:14px;padding:11px 18px;transition:background .18s,box-shadow .18s}.primary:active{box-shadow:none}nav button{color:var(--sub)}nav button[aria-current=page]{color:var(--teal);border-bottom:3px solid var(--teal)}.metric-row{background:linear-gradient(110deg,#eee1ef,#fbedd9);padding:23px;border:0;border-radius:4px}.metric{font-family:\'Fraunces\',Georgia,serif;color:var(--teal);font-size:40px}.metric-label{font-weight:600}.card{border-color:#dfd0dd;background:#fffaf5;box-shadow:0 3px 0 #eadde7;border-radius:8px}.card:nth-child(2){background:#fff8ec}.card h3{color:var(--teal)}.highlight-row{background:#eee1ef}.bar-fill{border-radius:2px;transition:width .2s}.bar-row:not(.highlight-row) .bar-fill[style*="var(--teal)"]{background:var(--sea)!important}.chart-summary{font-family:\'IBM Plex Mono\',monospace;font-size:12px}.conclusion{border-color:var(--rust);background:#fae6d8;padding:16px 20px;border-radius:0 7px 7px 0}.case-entry{border-top:2px solid #dfbdc8}.case-entry .eyebrow{color:var(--rust)}.case-entry h2{color:var(--teal)}details{border-top-color:#d7c6d4}summary{padding-left:4px;color:var(--ink)}details[open]>summary{color:var(--teal)}th{background:var(--pale)}.note{border-left:3px solid var(--ochre);padding:13px 18px;background:#f5ead5}.paper-notice{border:0;background:linear-gradient(175deg,#fffef8 0%,#fffdf5 53%,#f2ead9 54%,#fffdf5 54.7%,#fffaf0 100%);padding:38px 34px 42px;transform:rotate(1.1deg);clip-path:polygon(.5% .5%,8% .2%,15% .7%,24% 0,36% .5%,48% .1%,60% .6%,73% 0,84% .5%,96% .1%,99.6% .6%,100% 98.2%,97% 98.9%,94% 98.3%,91% 99.5%,88% 98.8%,85% 99.3%,82% 98.4%,78% 99.5%,75% 98.6%,71% 99.3%,68% 98.3%,65% 99.7%,61% 98.9%,58% 99.4%,55% 98.5%,51% 99.6%,47% 98.8%,43% 99.5%,39% 98.2%,36% 99.4%,32% 98.9%,28% 99.6%,24% 98.3%,20% 99.4%,16% 98.8%,12% 99.6%,9% 98.5%,5% 99.3%,1% 98.5%,0 83%,.5% 66%,0 47%,.6% 28%);box-shadow:none}.document-example{filter:drop-shadow(3px 7px 5px #39273521);padding-top:10px}.paper-notice:after{display:none}.document-label{font:11px \'IBM Plex Mono\',monospace;color:var(--teal);margin-bottom:16px;letter-spacing:.02em}.letter-department{color:#453a49;border-bottom-color:#cebdbe;font-size:12px;letter-spacing:.12em;padding-right:70px}.letter-stamp{position:absolute;right:27px;top:27px;color:#9f5144;border:1px solid #bc8475;padding:4px 6px;font:10px \'IBM Plex Mono\',monospace;transform:rotate(-9deg)}.letter-reasons{background:linear-gradient(transparent 0,#f7e4b9 0);padding:8px 12px 8px 30px;margin-left:-8px;margin-right:-6px}.letter-reasons li{margin:6px 0}.document-caption{font-size:12px;padding-left:10px}.landing-connection{border-top:2px solid #d6b6c9}.landing-connection h2{color:var(--teal)}.sliding-options{position:relative;isolation:isolate;display:grid;grid-template-columns:1fr 1fr;background:#e9dbe9;border:1px solid #dac4d9;border-radius:26px;padding:4px;max-width:100%;min-height:43px}.sliding-options:before{content:"";position:absolute;z-index:-1;top:4px;bottom:4px;left:4px;width:calc((100% - 8px)/2);background:var(--teal);border-radius:22px;box-shadow:0 2px 3px #38213d22;transition:transform .22s ease}.sliding-options[data-mode="topics"]:before,.sliding-options[data-mode="methods"]:before{transform:translateX(100%)}.sliding-options button{border:0;background:none;position:relative;z-index:1;padding:8px 14px;color:#624e67;font-size:13px;font-weight:500;border-radius:22px;white-space:normal}.sliding-options button[aria-pressed="true"]{color:#fffaf4}.evidence-options{max-width:440px;margin:25px 0 30px}.evidence-panel details{padding:17px 0}.evidence-panel details[open]{border-left:3px solid #c7a0bb;padding-left:17px}.evidence-panel p{max-width:730px}.method-status{font-size:13px;background:#f1e4d2;padding:12px 15px;border-radius:4px}.threshold-control{padding:20px 24px 14px;background:#eee1ef;border-radius:9px;margin:25px 0 28px}.threshold-control label{display:block;font-size:14px;font-weight:500}.threshold-control output{display:block;font:25px \'Fraunces\',Georgia,serif;color:var(--teal);margin-top:8px}.threshold-control input{width:100%;accent-color:var(--teal);cursor:pointer;margin:20px 0 8px;min-height:24px}.range-labels{display:flex;justify-content:space-between;gap:12px;font-family:\'IBM Plex Mono\',monospace;font-size:11px;color:#6d546f}.threshold-control .small{margin:15px 0 0}.result-lead{color:var(--teal)}.reading>h1{color:var(--teal)}footer{background:#eee3e6;font-family:\'DM Sans\',system-ui,sans-serif;border-top:1px solid #d5c0d0}@media(max-width:850px){.hero h1{font-size:44px}.sliding-options{width:100%;max-width:420px}.opening-layout{gap:12px}.paper-notice{transform:rotate(.5deg)}.document-example{padding-right:7px}}@media(max-width:420px){.hero h1{font-size:38px}.paper-notice{padding:32px 25px 38px}.threshold-control{padding:18px}.sliding-options button{font-size:12px;padding:9px}.range-labels{font-size:11px;gap:5px}.letter-stamp{right:20px;top:24px}}@media(prefers-reduced-motion:reduce){.sliding-options:before,.bar-fill,.primary,.secondary{transition:none}.paper-notice{transform:none}}\n\n/* Retain aubergine; use warm stone for the palest supporting surfaces. */\nfooter{background:#eae7de;border-top-color:#d2cbbd}.highlight-row{background:#eae7de}.sliding-options{background:#eae7de;border-color:#d3cbbd}.secondary{background:#f6f2e8}.note{background:#f1eadb}.method-status{background:#f1eadb}\n/* Explicit chart text colors prevent light host-theme text on light cards. */\n.card{color:var(--ink)}#approximation-charts .card{color:#32283d}#approximation-charts .bar-row,#approximation-charts .bar-row span,#approximation-charts .bar-row strong,#approximation-charts .bar-val,#approximation-charts .stat-line,#approximation-charts .stat-line span,#approximation-charts .stat-line strong{color:#32283d}#approximation-charts .bar-axis,#approximation-charts .bar-axis span,#approximation-charts .small{color:#554c5b}#approximation-charts .card h3{color:#6a315f}.bar-row{color:var(--ink)}.bar-row{font-size:14px}.bar-val{font-size:13px}@media(max-width:750px){.bar-row{font-size:12px}.bar-val{font-size:12px}}\n\n/* Table surfaces and text are explicit so embedded dark themes cannot override contrast. */\n:root{--table-ink:#32283d;--table-sub:#554c5b;--table-head:#f6e8c8;--table-cell:#fffaf0;--table-rule:#d9cfb9}\n#score-investigation-render table,table{color:var(--table-ink);background:var(--table-cell)}\n#score-investigation-render table thead,#score-investigation-render table th,table thead,table th{background:var(--table-head);color:var(--table-ink)}\n#score-investigation-render table tbody,#score-investigation-render table td,table tbody,table td{background:var(--table-cell);color:var(--table-ink)}\n#score-investigation-render table :is(caption,th,td,p,span,strong,code),table :is(caption,th,td,p,span,strong,code){color:var(--table-ink)}\n#score-investigation-render table caption,table caption{background:transparent;color:var(--table-ink)}\n#score-investigation-render table :is(th,td),table :is(th,td){border-bottom-color:var(--table-rule)}\n#score-investigation-render table a,table a{color:#6a315f}\n.threshold-control{background:#f7edd6;color:#32283d;border:1px solid #e7d7b6}.threshold-control label,.threshold-control p,.threshold-control .small,.threshold-control .muted,.threshold-control .range-labels,.threshold-control .range-labels span{color:#554c5b}.threshold-control output{color:#6a315f}.highlight-row{background:#f7edd6}.sliding-options{background:#f5eddd;border-color:#d9ccb4}footer{background:#f2ecdf;border-top-color:#ded3bd}\n\n/* Measured findings: direct labels and symbols preserve meaning without color. */\n.cohort-figure{margin:28px 0 34px;padding:28px;background:#fffaf0;border:1px solid #e3d8c2;border-radius:12px;color:#32283d}.cohort-figure h2{font-size:28px}.cohort-row{padding:22px 0;border-top:1px solid #ded3bd;display:grid;grid-template-columns:1fr 1fr;gap:20px;align-items:center}.cohort-row p{margin:6px 0 0;font-size:15px;color:#554c5b}.cohort-marks{display:grid;grid-template-columns:repeat(6,30px);gap:7px;justify-content:end}.cohort-dot{height:30px;width:30px;display:grid;place-items:center;border-radius:5px;font:600 19px/1 system-ui}.cohort-dot.changed{background:#6a315f;color:#fffaf0;border:1px solid #6a315f}.cohort-dot.unchanged{background:#f7edd6;color:#493747;border:1px solid #bdac94}.cohort-key{margin:12px 0;color:#554c5b}.evidence-outcomes{margin:22px 0}.evidence-outcomes>div{padding:17px 0;border-top:1px solid var(--line)}.evidence-outcomes dt{font-weight:700;color:#6a315f;margin-bottom:5px}.evidence-outcomes dd{margin:0;color:#32283d}.review-implication{margin-top:30px}@media(max-width:600px){.cohort-figure{padding:20px}.cohort-row{grid-template-columns:1fr;gap:12px}.cohort-marks{justify-content:start}}\n\n/* Final policy-facing framing */\n.policy-chain{display:grid;grid-template-columns:repeat(5,minmax(0,1fr));gap:8px;align-items:stretch;margin:24px 0 10px}.policy-chain .node{background:#fffaf0;border:1px solid #d9ccb4;border-radius:8px;padding:14px 12px;text-align:center;font-size:13px;line-height:1.4;position:relative}.policy-chain .node:not(:last-child):after{content:"→";position:absolute;right:-10px;top:50%;transform:translateY(-50%);z-index:2;color:#6a315f;font-weight:700}.handoff-line{font:12px \'IBM Plex Mono\',monospace;color:#6a315f;margin:8px 0 24px}.audience-strip{display:flex;gap:8px;flex-wrap:wrap;margin:14px 0 24px}.audience-strip span{border:1px solid #d9ccb4;background:#fffaf0;border-radius:18px;padding:7px 11px;font-size:12px;font-weight:600}.policy-box{background:#f7edd6;border-left:4px solid #bf4935;padding:22px 24px;margin:28px 0}.policy-box h2{margin-bottom:10px}.policy-box p:last-child{margin-bottom:0}.finding-pair{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin:22px 0}.finding-pair>div{background:#fffaf0;border:1px solid #ded3bd;border-radius:10px;padding:20px}.finding-number{font:42px/1 \'Fraunces\',Georgia,serif;color:#6a315f;display:block;margin-bottom:7px}.claim-matrix{width:100%;border-collapse:separate;border-spacing:0;margin:20px 0}.claim-matrix th,.claim-matrix td{padding:12px;border-bottom:1px solid #ded3bd;vertical-align:top}.status-tag{display:inline-block;border-radius:14px;padding:3px 8px;font:11px \'IBM Plex Mono\',monospace;font-weight:600}.status-measured{background:#e1efe9;color:#1d5e51}.status-limited{background:#f7edd6;color:#744f1a}.status-gap{background:#f6dfda;color:#8a3427}.policy-actions{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:14px;margin:20px 0}.policy-actions>div{background:#fffaf0;border:1px solid #ded3bd;border-radius:8px;padding:18px}.consumer-takeaway{border:1px solid #d7c6d4;background:#eee1ef;padding:20px 22px;border-radius:8px;margin:24px 0}@media(max-width:760px){.policy-chain{grid-template-columns:1fr}.policy-chain .node:not(:last-child):after{content:"↓";right:50%;top:auto;bottom:-15px;transform:translateX(50%)}.finding-pair,.policy-actions{grid-template-columns:1fr}}\n</style></head><body>\n<header><div class="brand">REASON <span>/ Explanation assurance for credit</span></div><nav aria-label="Main navigation"><button data-page="about" aria-current="page">Why it matters</button><button data-page="legal">Policy problem</button><button data-page="investigate">Review evidence</button><button data-page="compare">Compare reconstructions</button><button data-page="research">Findings</button><button data-page="evidence">Policy &amp; evidence</button><button data-page="methods">Methods</button></nav></header>\n<main>\n<div id="case-toolbar" class="case-toolbar" hidden><label class="field">Historical credit record<select id="case-select" aria-label="Choose a historical credit record"></select></label><p>The same record is used on both comparison pages.</p></div>\n<section class="page" id="about">\n<div class="opening-layout"><div class="hero"><p class="eyebrow">Explanation assurance · Consumer credit</p><h1>Can you stand behind the reason you give an applicant?</h1><p class="intro">A consumer may see one short reason for a credit refusal. Behind it can sit a lender model, an external score supplied by another company, an explanation method, and a mapping from technical evidence into notice language. The policy problem is whether the institution can substantiate that consumer-facing claim from the decision process that actually ran.</p><div class="audience-strip" aria-label="Intended users"><span>Consumer compliance</span><span>Model validation</span><span>Model risk</span><span>Counsel</span><span>Supervisory review</span></div><div class="actions"><button class="primary" data-page="legal">See the policy problem</button><button class="secondary" data-page="investigate">Review a record</button></div></div>\n<aside class="document-example" aria-label="Illustrative credit denial letter excerpt"><p class="document-label">Illustrative letter · Decision and reasons excerpt</p><div class="paper-notice"><span class="letter-stamp">EXCERPT</span><div class="letter-department">APPLICATION SERVICES</div><p class="letter-date">18 October 2026</p><p class="letter-subject">Re: Home equity line of credit application</p><p>Dear Applicant,</p><p>Following our review, we are unable to approve the home equity line of credit you requested.</p><p>The principal reasons for this decision are:</p><ul class="letter-reasons"><li>Limited credit experience</li><li>Number of recent inquiries on credit bureau report</li></ul><p>Your credit report was part of this review. The reporting agency did not make the credit decision.</p><p class="letter-signature">Sincerely,<br>Credit Application Team</p></div><p class="document-caption">Fictional illustration using reason wording from Regulation B sample forms. These reasons have not been assigned to a study record.</p></aside></div>\n<div class="context"><p class="eyebrow">The evidence chain</p><div class="policy-chain" role="img" aria-label="Consumer-facing reason traces through notice mapping, lender model, external score, and score provider"><div class="node"><strong>Consumer-facing reason</strong><br>What the applicant is told</div><div class="node"><strong>Reason mapping</strong><br>How evidence becomes language</div><div class="node"><strong>Lender model</strong><br>Decision or risk model</div><div class="node"><strong>External score</strong><br>Supplied input</div><div class="node"><strong>Provider model</strong><br>Potentially unavailable mechanism</div></div><p class="handoff-line">Every arrow is an evidence handoff.</p><p>The experiment stress-tests the weakest handoff: an external score whose observed value is available but whose original formula and provider reasoning are not.</p></div>\n<div class="policy-box"><p class="eyebrow">Policy implication</p><h2>Do not validate only the explanation output. Validate the evidence chain.</h2><p>When an external score materially affects an adverse action, a downstream XAI explanation or reconstructed score should not be treated as sufficient evidence of the score provider’s actual reasons. The review should preserve which claims are measured, which are inferred, and which require provider-linked evidence.</p></div>\n<div class="context landing-connection"><h2>Why the external-score problem matters</h2><div><div class="finding-pair"><div><span class="finding-number">4 / 12</span><strong>close-prediction cases changed leading detailed factor sets</strong></div><div><span class="finding-number">0 / 12</span><strong>changed their leading broader credit-topic sets</strong></div></div><p>Stable predictions — and even stable broad labels — can coexist with uncertainty about the mechanism underneath. <strong>Prediction agreement is not explanation provenance.</strong></p><p class="small muted">The 12 records are an exploratory subset of 40 selected historical HELOC records. The counts are descriptive, not population estimates or legal thresholds.</p><button class="text-button" data-page="research">See the measured finding →</button></div></div>\n<div class="context"><h2>What should change in practice?</h2><div class="policy-actions"><div><strong>1 · Preserve provenance</strong><p>Record the external score and version, lender model/version, explanation settings, reason-mapping version and notice output used for the reviewed decision.</p></div><div><strong>2 · Separate measured from inferred</strong><p>Label downstream model contributions as measured evidence. Label reconstructed upstream factors as inference unless authenticated by provider-linked records.</p></div><div><strong>3 · Validate both category and basis</strong><p>Do not treat a stable broad reason label as enough. Test whether the underlying factors supporting that category remain accurate and traceable.</p></div><div><strong>4 · Allow “not established”</strong><p>If the chain cannot be completed, the responsible output is an evidence gap and a request for provider/lender records — not a more confident explanation.</p></div></div></div>\n</section>\n<section class="page" id="investigate" hidden><p class="eyebrow">The original research model</p><h1>A repayment prediction using an externally supplied score</h1><p class="intro">The Random Forest uses account history, balances and inquiries alongside the score recorded as <code>"ExternalRiskEstimate"</code>. It predicts the dataset’s serious-late-payment outcome: at least 90 days overdue, or worse, within 24 months of opening. The data contain repayment outcomes rather than actual lending decisions.</p><div id="case-content" aria-live="polite"></div><details><summary>Check the LIME-style approximation of this prediction</summary><div id="local-fit"></div></details><details><summary>View the credit information for this record</summary><div class="table-scroll" id="record-inputs"></div></details><div class="actions"><button class="primary" data-page="compare">Compare the two score estimates</button></div></section>\n<section class="page" id="compare" hidden><p class="eyebrow">The external-score experiment</p><h1>Explanations depend on the model used to estimate the score</h1><p class="intro">I trained two models to estimate the supplied score from eight other credit attributes: Ridge regression adds weighted inputs, while gradient-boosted trees can learn interactions between them. Each estimate replaces the score in the unchanged Random Forest. The charts explain the resulting repayment predictions, including the contribution through the estimated score. Different models can legitimately have different contributions; the question is how those differences relate to prediction agreement and to the broader topics a notice might use.</p><div class="compare-head"><h2>Credit factors in each explanation</h2><div class="sliding-options" id="explanation-mode" data-mode="factors" role="group" aria-label="Choose explanation detail"><button id="factors-view" aria-pressed="true">Individual factors</button><button id="topics-view" aria-pressed="false">Grouped factors</button></div></div><div id="approximation-charts" aria-live="polite"></div><div id="comparison-result" class="conclusion" aria-live="polite"></div><p class="small muted">Grouping adds the signed contributions of related measures, such as account ages, into broader credit topics. Lenders sometimes group features when producing notice reasons; these research groupings allow the comparison to show whether detailed changes also affect the broader topics.</p><div class="actions"><button class="primary" data-page="research">Read the findings</button></div></section>\n<section class="page reading" id="research" hidden><p class="eyebrow">Findings and review conclusion</p><h1>How closely do predictions and explanations agree?</h1><p class="intro">The experiment compares two estimated-score systems with the original research model. Its contribution is the measured relationship between prediction agreement, detailed-factor changes and broader-topic agreement.</p><div class="threshold-control"><label for="agreement-band">Maximum difference between the three repayment predictions <output id="agreement-band-label" for="agreement-band">1.0 percentage points</output></label><input id="agreement-band" type="range" min="0" max="3" step="1" value="1" aria-valuetext="1.0 percentage points"><div class="range-labels" aria-hidden="true"><span>0.5</span><span>1.0</span><span>2.0</span><span>5.0</span></div><p class="small muted">Choose how close predictions must be to enter the comparison. This changes the records counted below; it is not a legal acceptance threshold.</p></div><div id="findings-summary"></div><div class="actions"><button class="primary" data-page="evidence">Review the remaining evidence needs</button></div><section class="export"><h2>Save the evidence for a record</h2><p class="small">This note describes the record selected on the prediction and comparison pages, separately from the study-wide counts above.</p><button class="text-button" data-page="compare">Inspect or change the selected record →</button><p class="small muted" id="report-selection"></p><div class="actions"><button class="secondary" id="prepare-report">Prepare review note</button></div><div id="report-output" hidden><div class="download-links"><a id="open-report" target="_blank" rel="noopener">Open note</a><a id="download-report" download="score-review.html">Download note (.html)</a><a id="download-json" download="score-review.json">Download measurements (.json)</a></div><p class="report-state" id="report-state" role="status"></p><details><summary>Copy the note text</summary><div><textarea id="report-preview" readonly aria-label="Review note text"></textarea></div></details></div></section></section>\n<section class="page reading" id="methods" hidden><p class="eyebrow">Research documentation</p><h1>Methods and numerical results</h1><section><h2>Dataset and models</h2><p>The public FICO HELOC data contain credit attributes and an account repayment outcome. After cleaning, the study fitted a nine-input Random Forest to predict the “Bad” repayment category. Eight inputs describe credit history; the ninth is the externally supplied score. The data do not identify the score’s original formula or provider.</p><p>Ridge regression and histogram gradient boosting were trained to estimate that score using the eight credit attributes. Training used only the original training partition, excluding negative external-score status codes. Each estimate was clipped to the observed training range before substitution into the frozen Random Forest.</p><div class="table-scroll" id="performance-table"></div></section><section><h2>Shapley and LIME-style calculations</h2><p>The displayed Shapley values were calculated by exact empirical interventional enumeration against 24 training records. Each value allocates part of the difference between the reference average and the prediction. All three systems use the same reference-record identities, with estimated scores substituted in the reconstructed systems. Correlated-input replacements can create implausible combinations.</p><p>A custom LIME-style weighted regression was fitted five times near each record. Separate generated records evaluated its approximation error and weighted R². Generated records mix the applicant’s values with training values and may violate credit-feature dependencies. The displayed results use these custom implementations; the official SHAP and LIME packages were not executed.</p><p>For each explanation, the leading list contains up to four positive contributions. Grouped comparisons first add signed contributions within research-defined credit topics. Neither list is treated as a validated consumer notice.</p></section><section><h2>Sample and comparison limits</h2><p>Prediction accuracy was evaluated on 1,973 test records with observed external scores. Forty previously selected records received detailed explanations. They were originally selected to contrast predictions near 0.50 and high predictions in an earlier eight-input model; they are not a representative sample of credit refusals. The default example, record 8193, was selected after inspection.</p><p>The table counts records where both reconstructed probabilities are within the stated distance of the original probability and of each other. These are research comparison limits, not regulatory thresholds. Test results had already been inspected during earlier work and a feasibility run, so this study is exploratory.</p><div class="table-scroll" id="agreement-table"></div><p class="small muted">The study does not evaluate actual notices, protected-group fairness, consumer understanding or reviewer effectiveness.</p></section><section><h2>Data and method references</h2><p><a href="https://github.com/Trusted-AI/AIX360/blob/master/aix360/datasets/heloc_dataset.py" target="_blank" rel="noopener">HELOC dataset documentation</a> · <a href="https://arxiv.org/abs/1705.07874" target="_blank" rel="noopener">SHAP</a> · <a href="https://arxiv.org/abs/1602.04938" target="_blank" rel="noopener">LIME</a> · <a href="https://arxiv.org/abs/1908.08474" target="_blank" rel="noopener">Alternative Shapley definitions</a></p><p>FinRegLab’s consumer-lending research evaluates explanation fidelity and consistency across model families. Its tests offer a basis for evaluating explanations, without establishing the adequacy of actual notices. <a href="https://finreglab.org/wp-content/uploads/2023/12/FinRegLab_2023-07-13_Empirical-White-Paper_Explainability-and-Fairness_Insights-from-Consumer-Lending.pdf" target="_blank" rel="noopener">Empirical report</a></p></section></section>\n<section class="page reading" id="legal" hidden><p class="eyebrow">The policy problem</p><h1>External scores create an explanation-provenance obligation</h1><p class="intro">The central recommendation is not to ban external scores or require one XAI method. It is to require a traceable evidence record when a consumer-facing reason depends on a model component supplied by another party.</p><!-- LEGAL_APPENDIX --></section>\n<section class="page reading" id="evidence" hidden><p class="eyebrow">Policy &amp; evidence</p><h1>What can the institution actually stand behind?</h1><p class="intro">The workbench treats explanation as a claim-evidence problem. A technically valid attribution is useful evidence about the model it explains; it is not automatically evidence about an unavailable upstream scorer.</p><!-- EVIDENCE_APPENDIX --></section>\n</main><footer><span>Independent project · ms@duke.edu · Public FICO HELOC data</span><span>Dataset, experiment and sources documented in the tabs above</span></footer>\n<script src="data.js"></script><script src="app.js"></script>\n</body></html>\n', 'site/app.js': '\'use strict\';\nconst APP_ROOT=document.getElementById(\'score-investigation-render\')||document;\nconst $=id=>APP_ROOT.querySelector(\'#\'+id),ALL=s=>APP_ROOT.querySelectorAll(s);\nconst S=PROJECT.study,KEYS=[\'additive\',\'nonlinear\'];\nlet caseId=S.featured_case,detail=\'factors\',bandIndex=1,reportUrls=[];\nconst esc=x=>String(x).replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\nconst pct=x=>(x*100).toFixed(1)+\'%\',points=x=>(x*100).toFixed(2),signed=x=>(x>=0?\'+\':\'\')+points(x);\nconst sources={regulation_b:\'https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/\',finreglab:\'https://finreglab.org/research/explainability-and-fairness-in-machine-learning-for-credit-underwriting/\',cfpb_case:\'https://business.cch.com/BFLD/CFPB-Supervisory-Highlights-Advanced-Technologies-Special-Edition-Issue-38-Winter-2025-01172025.pdf\'};\nfunction current(){return S.cases.find(c=>c.case_id===caseId)}\nfunction page(id){ALL(\'.page\').forEach(e=>e.hidden=e.id!==id);ALL(\'nav [data-page]\').forEach(e=>{if(e.dataset.page===id)e.setAttribute(\'aria-current\',\'page\');else e.removeAttribute(\'aria-current\')});$(\'case-toolbar\').hidden=![\'investigate\',\'compare\',\'research\'].includes(id);window.scrollTo({top:0,behavior:\'instant\'});}\nfunction grouped(values,features){const out={};features.forEach((f,i)=>{const g=PROJECT.groups[f];out[g]=(out[g]||0)+values[i]});return Object.entries(out).map(([id,value])=>({id,label:id,value}));}\nfunction entries(values,features,topics=false){return topics?grouped(values,features):features.map((f,i)=>({id:f,label:PROJECT.labels[f],value:values[i]}));}\nfunction bars(items,max){const ordered=[...items].sort((a,b)=>b.value-a.value);const leaders=new Set(ordered.filter(x=>x.value>1e-10).slice(0,4).map(x=>x.id));return `<div class="bar-chart" role="group" aria-label="Shapley contributions in percentage points. Positive values raise predicted risk; negative values lower it."><div class="bar-axis"><span>Credit information</span><span>Lower ← risk → Higher</span><span class="bar-val">Percentage<br>points</span></div>${ordered.map(r=>`<div class="bar-row ${r.id===\'ExternalRiskEstimate\'?\'highlight-row\':\'\'}"><span>${leaders.has(r.id)?\'<strong>\':\'\'}${esc(r.label)}${leaders.has(r.id)?\'</strong>\':\'\'}</span><div class="bar-track"><span class="bar-fill" style="${r.value>=0?\'left:50%\':\'right:50%\'};width:${Math.min(49,Math.abs(r.value)/max*47)}%;background:${r.value>=0?\'var(--rust)\':\'var(--teal)\'}"></span></div><span class="bar-val">${signed(r.value)}</span></div>`).join(\'\')}</div>`;}\nfunction comparisonText(c){const d=c.comparison;return `The Random Forest predicts ${pct(c.original_probability)} serious-late-payment risk with the supplied score, ${pct(c.approximations.additive.probability)} with the Ridge estimate and ${pct(c.approximations.nonlinear.probability)} with the gradient-boosting estimate. ${d.individual.turnover?\'The leading individual-factor sets differ between the two estimates.\':\'The leading individual-factor sets match between the two estimates.\'} ${d.grouped.turnover?\'The grouped-factor sets also differ.\':\'The grouped-factor sets match.\'}`;}\nfunction render(){const c=current(),original=c.original_shapley,oi=entries(original.values,[...S.features,\'ExternalRiskEstimate\']);\n clearReport();\n const external=original.values.at(-1),val=external>=0?\'raises\':\'lowers\';\n $(\'case-content\').innerHTML=`<div class="metric-row"><div><span class="metric-label">Score supplied in the credit record</span><div class="metric">${c.observed_external_score.toFixed(0)}</div><p class="small muted">“ExternalRiskEstimate”; the formula and provider are absent from the data.</p></div><div><span class="metric-label">Random Forest’s repayment-risk estimate</span><div class="metric">${pct(c.original_probability)}</div><p class="small muted">Predicted probability of serious late payment under the dataset’s outcome definition.</p></div></div><h2>Shapley contributions to the repayment prediction</h2><p>The external score ${val} the predicted probability by <strong>${points(Math.abs(external))} percentage points</strong> relative to the model’s average for 24 reference records. The calculation shows how the score affects this Random Forest; it leaves the score’s own calculation unexplained.</p>${bars(oi,Math.max(.01,...oi.map(x=>Math.abs(x.value))))}<div class="chart-summary"><span>Reference average <strong>${pct(original.diagnostics.base)}</strong></span><span>Record ${caseId} <strong>${pct(c.original_probability)}</strong></span></div><p class="small muted" style="margin-top:12px">Positive contributions raise predicted risk and negative contributions lower it. Contributions are in percentage points, not credit-score points.</p>`;\n $(\'record-inputs\').innerHTML=`<table><thead><tr><th>Credit information</th><th>Dataset field</th><th>Value</th></tr></thead><tbody>${S.features.map(f=>`<tr><td>${esc(PROJECT.labels[f])}</td><td><code>"${esc(f)}"</code></td><td>${esc(inputValue(f,c.inputs[f]))}</td></tr>`).join(\'\')}</tbody></table>`;\n $(\'report-selection\').textContent=`The note will contain record ${caseId}, its numerical comparisons and the evidence needed to investigate the original score’s reasons.`;\n renderApprox();renderLocal();renderProtocol();}\nfunction setDetail(v){detail=v;$(\'explanation-mode\').dataset.mode=v;$(\'factors-view\').setAttribute(\'aria-pressed\',String(v===\'factors\'));$(\'topics-view\').setAttribute(\'aria-pressed\',String(v===\'topics\'));renderApprox();clearReport();}\nfunction renderApprox(){const c=current(),items=KEYS.map(k=>entries(c.approximations[k].values,S.features,detail===\'topics\'));const scale=Math.max(.01,...items.flat().map(x=>Math.abs(x.value)));\n $(\'approximation-charts\').innerHTML=`<div class="grid">${KEYS.map((k,i)=>{const v=c.approximations[k];return `<div class="card"><h3>${k===\'additive\'?\'Ridge regression\':\'Gradient-boosted trees\'}</h3><div class="stat-line"><span>Estimated external score</span><strong>${v.estimated_external_score.toFixed(2)}</strong></div><div class="stat-line"><span>Resulting repayment-risk prediction</span><strong>${pct(v.probability)}</strong></div>${bars(items[i],scale)}<p class="small muted">Reference average: ${pct(v.diagnostics.base)}. Bold labels identify up to four leading positive contributors.</p></div>`}).join(\'\')}</div>`;\n $(\'comparison-result\').innerHTML=`<p>${esc(comparisonText(c))}</p>`;}\nfunction inputValue(f,v){if(v===-7)return \'Condition not met (code -7)\';if(v===-8)return \'No usable accounts or inquiries (code -8)\';if(v<0)return `Missing or unavailable (code ${v})`;if([\'MSinceOldestTradeOpen\',\'AverageMInFile\',\'MSinceMostRecentDelq\'].includes(f))return v+\' months\';if([\'NetFractionRevolvingBurden\',\'NetFractionInstallBurden\',\'PercentTradesNeverDelq\'].includes(f))return v+\'%\';return String(v)}\nfunction median(a){const b=[...a].sort((a,b)=>a-b);return b[Math.floor(b.length/2)]}\nfunction renderLocal(){const fits=PROJECT.local[String(caseId)],mae=median(fits.map(f=>f.heldout_mae)),r2=median(fits.map(f=>f.heldout_r2));$(\'local-fit\').innerHTML=`<p><strong>Supporting question: can a simpler local explanation represent this Random Forest reliably?</strong></p><p>A custom LIME-style weighted regression approximates the Random Forest around this record. Across five runs, median average error on separate generated records was <strong>${points(mae)} percentage points</strong>; median weighted R² was <strong>${r2.toFixed(2)}</strong>.</p><p>R² measures approximation fit, with 1 indicating a perfect fit and 0 indicating no improvement over a constant prediction. A low fit or large error weakens reliance on that simpler explanation. Even a good fit only concerns the Random Forest. The generated records mix training values with this record’s attributes. These results evaluate the local approximation of the Random Forest, without recovering the external score’s formula.</p>`;}\nfunction renderProtocol(){\n const tests=$(\'protocol-tests\'),matrix=$(\'protocol-case-matrix\');if(!tests||!matrix)return;\n const c=current(),fits=PROJECT.local[String(caseId)],mae=median(fits.map(f=>f.heldout_mae)),r2=median(fits.map(f=>f.heldout_r2)),res=Math.abs(c.original_shapley.diagnostics.residual);\n const individual=c.comparison.individual.turnover?\'differ\':\'match\',groupedSet=c.comparison.grouped.turnover?\'differ\':\'match\';\n tests.innerHTML=`\n <article class="protocol-card"><span class="status-tag status-measured">MEASURED</span><h3>1 · Decision fidelity</h3><p>The exact Shapley decomposition closes with residual <strong>${res.toExponential(1)}</strong>. Across five local-surrogate runs, median held-out error is <strong>${points(mae)} pp</strong> and median weighted R² is <strong>${r2.toFixed(2)}</strong>.</p><p class="reviewer-action"><strong>Reviewer action.</strong> Use these results to judge the explanation of this research Random Forest. They do not validate the unknown score provider.</p></article>\n <article class="protocol-card"><span class="status-tag status-measured">MEASURED</span><h3>2 · Reason consistency</h3><p>For record ${caseId}, the Ridge and boosted-score reconstructions\' leading individual-factor sets <strong>${individual}</strong>; their broader-topic sets <strong>${groupedSet}</strong>.</p><p class="reviewer-action"><strong>Reviewer action.</strong> Treat agreement or disagreement as evidence about the two reconstructions, not as proof of the original provider\'s reasons.</p></article>\n <article class="protocol-card"><span class="status-tag status-gap">EXTERNAL EVIDENCE REQUIRED</span><h3>3 · Mapping integrity</h3><p>The HELOC data do not include an actual adverse-action notice or a production feature-to-reason mapping.</p><p class="reviewer-action"><strong>Reviewer action.</strong> Obtain the notice wording, mapping/version, factor membership and directionality checks before validating a consumer-facing reason.</p></article>\n <article class="protocol-card"><span class="status-tag status-gap">EVIDENCE GAP</span><h3>4 · Provenance</h3><p>The supplied external score is observed, but its provider and original calculation are absent. A substitute model can explain its own estimate; it cannot authenticate the unavailable upstream mechanism.</p><p class="reviewer-action"><strong>Reviewer action.</strong> Request provider reason codes, input definitions, model/version information and relevant validation records.</p></article>\n <article class="protocol-card wide"><span class="status-tag status-partial">PARTIAL</span><h3>5 · Auditability</h3><p>This research artifact retains the record ID, inputs, reference population, model outputs, explanation settings and comparison results.</p><p class="reviewer-action"><strong>Reviewer action.</strong> A production review would also need the actual decision rules, overrides, notice delivery record, mapping version and provider records.</p></article>`;\n matrix.innerHTML=`<h2>Claim-to-evidence matrix for record ${caseId}</h2><table class="claim-matrix"><thead><tr><th>Claim</th><th>Evidence in this project</th><th>Status</th></tr></thead><tbody><tr><td>The external score affected this Random Forest prediction.</td><td>The exact decomposition measures the score\'s signed contribution relative to the 24-record reference set.</td><td><span class="status-tag status-measured">SUPPORTED HERE</span></td></tr><tr><td>The reconstructed systems produce similar downstream predictions.</td><td>Original ${pct(c.original_probability)} · Ridge ${pct(c.approximations.additive.probability)} · boosted ${pct(c.approximations.nonlinear.probability)}.</td><td><span class="status-tag status-measured">MEASURED</span></td></tr><tr><td>The inferred detailed factors were used inside the original external score.</td><td>The reconstruction explains the substitute mechanism, not the unavailable provider mechanism.</td><td><span class="status-tag status-gap">NOT ESTABLISHED</span></td></tr><tr><td>A proposed adverse-action reason accurately describes this historical record.</td><td>No actual lending decision, notice, policy override or production reason mapping is present in HELOC.</td><td><span class="status-tag status-gap">NOT TESTABLE HERE</span></td></tr></tbody></table>`;\n}\nfunction research(){const one=S.bands[bandIndex];$(\'agreement-band-label\').textContent=(one.probability_band*100).toFixed(1)+\' percentage points\';$(\'agreement-band\').setAttribute(\'aria-valuetext\',(one.probability_band*100).toFixed(1)+\' percentage points\');$(\'findings-summary\').innerHTML=cohortFigure(one)+`<section class="review-implication"><h2>Protocol conclusion</h2><p>Prediction agreement and explanation agreement answer different questions. The study can measure fidelity and consistency inside the research systems. It cannot authenticate the original score provider\'s internal reasons or validate notice wording that is absent from the data.</p><dl class="evidence-outcomes"><div><dt>Evidence available</dt><dd>Prediction errors and Shapley contributions for the research systems, with the inputs and calculation settings used to produce them.</dd></div><div><dt>Question still unresolved</dt><dd>Whether a proposed explanation describes the factors used by the original score provider.</dd></div><div><dt>Next evidence to obtain</dt><dd>Provider reasons linked to the score under review, input definitions, and validation records connecting those reasons to that score. Reviewing an actual refusal would also require the lender\'s decision rules, mapping and notice.</dd></div></dl><p class="small muted">The study has not tested actual notices or whether the workflow improves professional review.</p></section>`;\n $(\'performance-table\').innerHTML=`<table><caption>Performance on 1,973 test records</caption><thead><tr><th>Score estimator</th><th>Average external-score error</th><th>Average change in repayment prediction</th></tr></thead><tbody>${KEYS.map(k=>`<tr><td>${k===\'additive\'?\'Ridge regression\':\'Gradient-boosted trees\'}</td><td>${S.metrics[k].external_score_mae.toFixed(2)} score points</td><td>${points(S.metrics[k].probability_mae)} percentage points</td></tr>`).join(\'\')}</tbody></table>`;\n $(\'agreement-table\').innerHTML=`<table><thead><tr><th>Maximum prediction difference</th><th>Records</th><th>Different individual-factor sets</th><th>Different grouped-factor sets</th></tr></thead><tbody>${S.bands.map(b=>`<tr><td>${(b.probability_band*100).toFixed(1)} percentage points</td><td>${b.both_close_to_original}</td><td>${b.different_factor_sets_when_both_close_to_original}</td><td>${b.different_concept_sets_when_both_close_to_original}</td></tr>`).join(\'\')}</tbody></table>`;}\nfunction cohortFigure(b){const matching=S.cases.filter(c=>Math.max(Math.abs(c.original_probability-c.approximations.additive.probability),Math.abs(c.original_probability-c.approximations.nonlinear.probability),Math.abs(c.approximations.additive.probability-c.approximations.nonlinear.probability))<=b.probability_band+1e-12);return `<figure class="cohort-figure"><figcaption><h2>Prediction agreement and explanation agreement</h2><p>${b.both_close_to_original} of ${S.summary.audit_records} examined records have all three predictions within ${(b.probability_band*100).toFixed(1)} percentage points of each other. The comparisons below concern the two estimated-score systems.</p></figcaption>${[[\'individual\',\'Individual credit factors\',b.different_factor_sets_when_both_close_to_original],[\'grouped\',\'Broader credit topics\',b.different_concept_sets_when_both_close_to_original]].map(([key,label,count])=>`<div class="cohort-row"><div><strong>${label}</strong><p>${count} of ${b.both_close_to_original} have different leading sets</p></div><div class="cohort-marks" role="img" aria-label="${count} of ${b.both_close_to_original} records have different leading ${key===\'individual\'?\'individual-factor\':\'credit-topic\'} sets">${matching.map(c=>`<span class="cohort-dot ${c.comparison[key].turnover?\'changed\':\'unchanged\'}" title="Record ${c.case_id}: ${c.comparison[key].turnover?\'different\':\'same\'} set" aria-hidden="true">${c.comparison[key].turnover?\'×\':\'=\'}</span>`).join(\'\')}</div></div>`).join(\'\')}<p class="small cohort-key">Each square is one historical record. <strong>×</strong> Different set · <strong>=</strong> Same set. Only membership is compared; order and contribution sizes can still change.</p><p class="small muted">Leading sets contain up to four positive contributors. Credit topics are research-defined groupings, not validated notice reasons. The 40 records were selected for earlier analysis; these counts are descriptive.</p></figure>`;}\nfunction record(){const c=current();return {title:\'Reason-integrity review for an estimated external credit score\',case_id:caseId,explanation_view:detail,measurements:c,study_metrics:S.metrics,local_approximation:PROJECT.local[String(caseId)],evidence_status:{downstream_model_contribution:\'measured\',reconstruction_behavior:\'measured\',original_provider_reason:\'not established\',production_notice_accuracy:\'provider / lender records required\'},findings:{comparison:comparisonText(c),supported_original_claim:\'The external score contributes to the study’s Random Forest prediction relative to 24 reference records.\',supported_approximation_claim:\'The reconstructed-system explanations describe the Random Forest using a score estimated by Ridge regression or gradient-boosted trees.\',unestablished_claim:\'The inferred credit factors were used to calculate the original external credit-risk score.\'},policy_recommendation:\'If a consumer-facing reason depends on a material external score, separate what the lender model establishes from what can be supported about the score provider. A reconstruction is useful for testing, but provider-linked reasons or validation records are still needed for claims about the upstream score.\',consumer_advocate_takeaway:\'An external provider’s opacity should not leave the applicant with a specific reason that the creditor cannot connect to the decision path actually used.\',request:\'Obtain applicable provider reason codes, score/model version, input documentation and validation evidence linking proposed reasons to the original score. For a refusal, obtain the lender’s actual decision rules, reason-mapping version, human assessment and delivered notice.\',limits:[\'The original score mechanism and authenticated provider reasons are absent.\',\'The study uses historical repayment data and does not evaluate actual notices.\',\'The analysis is exploratory and the forty records are not representative of lending decisions.\',\'Interventional replacements can violate credit-feature relationships.\'],sources};}\nfunction textNote(d){return `${d.title}\\nHistorical record: ${d.case_id}\\n\\nPREDICTIONS AND EXPLANATIONS\\n${d.findings.comparison}\\n\\nEVIDENCE FROM THE RANDOM FOREST\\n${d.findings.supported_original_claim}\\n\\nEVIDENCE FROM THE SCORE ESTIMATORS\\n${d.findings.supported_approximation_claim}\\n\\nWHAT REMAINS UNESTABLISHED\\n${d.findings.unestablished_claim}\\n\\nPOLICY RECOMMENDATION\\n${d.policy_recommendation}\\n\\nCONSUMER-ADVOCATE TAKEAWAY\\n${d.consumer_advocate_takeaway}\\n\\nRECORDS NEEDED FOR FURTHER REVIEW\\n${d.request}\\n\\nSTUDY LIMITATIONS\\n${d.limits.join(\'\\n\')}\\n\\nSOURCES\\n${Object.values(d.sources).join(\'\\n\')}`;}\nfunction clearReport(){reportUrls.forEach(u=>URL.revokeObjectURL(u));reportUrls=[];$(\'report-output\').hidden=true;$(\'report-preview\').value=\'\';}\nfunction prepareReport(){const d=record(),t=textNote(d),h=`<!doctype html><html lang="en"><meta charset="utf-8"><title>Credit explanation review ${d.case_id}</title><style>body{font:16px/1.65 system-ui;max-width:850px;margin:40px auto;padding:20px;color:#182c2d}pre{white-space:pre-wrap;font:inherit}</style><h1>Review of estimated credit-score explanations</h1><pre>${esc(t)}</pre></html>`;clearReport();const hu=URL.createObjectURL(new Blob([h],{type:\'text/html;charset=utf-8\'})),ju=URL.createObjectURL(new Blob([JSON.stringify(d,null,2)],{type:\'application/json\'}));reportUrls=[hu,ju];$(\'open-report\').href=hu;$(\'download-report\').href=hu;$(\'download-report\').download=`external-score-review-${caseId}.html`;$(\'download-json\').href=ju;$(\'download-json\').download=`external-score-review-${caseId}.json`;$(\'report-preview\').value=t;$(\'report-output\').hidden=false;$(\'report-state\').textContent=\'The note is ready to open, download or copy.\';}\n$(\'case-select\').innerHTML=S.cases.map(c=>`<option value="${c.case_id}">Record ${c.case_id}${c.case_id===S.featured_case?\' (default example)\':\'\'}</option>`).join(\'\');\n$(\'case-select\').value=String(caseId);$(\'case-select\').addEventListener(\'change\',e=>{caseId=Number(e.target.value);render()});\nALL(\'[data-page]\').forEach(b=>b.addEventListener(\'click\',()=>page(b.dataset.page)));\n$(\'factors-view\').addEventListener(\'click\',()=>setDetail(\'factors\'));$(\'topics-view\').addEventListener(\'click\',()=>setDetail(\'topics\'));$(\'prepare-report\').addEventListener(\'click\',prepareReport);\nfunction evidenceView(mode){$(\'evidence-switch\').dataset.mode=mode;$(\'requirements-panel\').hidden=mode!==\'requirements\';$(\'other-methods-panel\').hidden=mode!==\'methods\';$(\'requirements-view\').setAttribute(\'aria-pressed\',String(mode===\'requirements\'));$(\'other-methods-view\').setAttribute(\'aria-pressed\',String(mode===\'methods\'));}\n$(\'requirements-view\').addEventListener(\'click\',()=>evidenceView(\'requirements\'));$(\'other-methods-view\').addEventListener(\'click\',()=>evidenceView(\'methods\'));\n$(\'agreement-band\').addEventListener(\'input\',e=>{bandIndex=Number(e.target.value);research()});\nresearch();render();\n', 'site/data.js': 'const PROJECT={"study":{"generated_at":"2026-10-18T05:55:27.518296+00:00","design":"Exploratory; same-test feasibility metrics were already known.","protocol_sha256":"51b58dd298def9429eca160bcaf52fbb306b140d6bd4df1fa0ec3740d028ebcc","features":["MSinceOldestTradeOpen","AverageMInFile","NumSatisfactoryTrades","PercentTradesNeverDelq","MSinceMostRecentDelq","NumInqLast6M","NetFractionRevolvingBurden","NetFractionInstallBurden"],"counts":{"train":5915,"validation":1973,"test":1973},"score_clipping_bounds":[36.0,94.0],"metrics":{"additive":{"label":"Additive score approximation","test_records":1973,"external_score_mae":3.848025717689414,"external_score_r2":0.7645333613478831,"probability_mae":0.050146878486452696,"probability_error_quantiles":[0.0277760686194648,0.14276865478543,0.18532556609494108,0.2505555535612225],"unclipped_out_of_training_range":4},"nonlinear":{"label":"Nonlinear score approximation","test_records":1973,"external_score_mae":3.243338784287528,"external_score_r2":0.8183807334210851,"probability_mae":0.043288371094502384,"probability_error_quantiles":[0.021155569730752333,0.12128815208297046,0.16849930628847445,0.2569754378784586],"unclipped_out_of_training_range":0}},"cases":[{"case_id":7240,"inputs":{"MSinceOldestTradeOpen":96.0,"AverageMInFile":57.0,"NumSatisfactoryTrades":10.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":0.0,"NetFractionRevolvingBurden":-8.0,"NetFractionInstallBurden":-8.0},"observed_external_score":80.0,"original_probability":0.36410963086077675,"original_shapley":{"values":[0.035687176857836914,0.0353465735548168,0.04345954349641532,-0.02412404579025107,-0.001053047791736741,-0.02008426709063755,-0.02403711898213757,-0.00017846038970968593,-0.1936786876598619],"diagnostics":{"base":0.5127719646560421,"residual":8.326672684688674e-17},"individual":[{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.04345954349641532},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.035687176857836914},{"id":"AverageMInFile","label":"Average account age","weight":0.0353465735548168}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.07103375041265372},{"id":"Repayment history","label":"Repayment history","weight":0.018282449914427512}]},"approximations":{"additive":{"estimated_external_score":74.70413572249285,"probability":0.44900912066429804,"probability_error":0.0848994898035213,"values":[0.0400541259510703,0.07088512139404432,0.023593287112741966,-0.06432129327051614,-0.08656505916775643,-0.04820473405125418,0.005148145229341507,-0.019547363293995403],"diagnostics":{"base":0.5279668907606222,"residual":-5.551115123125783e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.07088512139404432},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.0400541259510703},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.023593287112741966},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.005148145229341507}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.11093924734511462},{"id":"Revolving balances","label":"Revolving balances","weight":0.005148145229341507}]},"nonlinear":{"estimated_external_score":79.51997049372363,"probability":0.36410963086077675,"probability_error":0.0,"values":[0.03239315534301555,0.05774602880344353,0.0030420280427521026,-0.07253896745241431,-0.0735013273952227,-0.04681519049347819,-0.04400659346371828,-0.004545453407253087],"diagnostics":{"base":0.5123359508836521,"residual":-2.7755575615628914e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.05774602880344353},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.03239315534301555},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.0030420280427521026}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.09013918414645908}]}},"comparison":{"probability_difference":0.0848994898035213,"maximum_original_probability_error":0.0848994898035213,"individual":{"retention":0.75,"jaccard":0.75,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":0.5,"jaccard":0.5,"turnover":true,"first_changed":false,"empty":false},"same_order":false}},{"case_id":9178,"inputs":{"MSinceOldestTradeOpen":241.0,"AverageMInFile":80.0,"NumSatisfactoryTrades":25.0,"PercentTradesNeverDelq":96.0,"MSinceMostRecentDelq":3.0,"NumInqLast6M":2.0,"NetFractionRevolvingBurden":37.0,"NetFractionInstallBurden":-8.0},"observed_external_score":65.0,"original_probability":0.5450777894350789,"original_shapley":{"values":[-0.004207608571271194,-0.0140852812811701,-0.02549657880131228,-0.012332602565833333,0.01405790978163055,0.002482902940666679,-0.024867567595999302,0.006045454199295286,0.09070919667303043],"diagnostics":{"base":0.5127719646560421,"residual":2.0816681711721685e-17},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.09070919667303043},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.01405790978163055},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.006045454199295286},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.002482902940666679}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.09070919667303043},{"id":"Installment balances","label":"Installment balances","weight":0.006045454199295286},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.002482902940666679}]},"approximations":{"additive":{"estimated_external_score":67.0251410278565,"probability":0.46333499077618084,"probability_error":0.08174279865889805,"values":[-0.005823269940456728,-0.016979860126690585,-0.032391262616984,-0.03307236445736507,0.07238579755921096,0.0009196310497972308,-0.04743845899358419,-0.002232112458368971],"diagnostics":{"base":0.5279668907606222,"residual":-2.7755575615628914e-17},"individual":[{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.07238579755921096},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.0009196310497972308}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.006922170484861878},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.0009196310497972308}]},"nonlinear":{"estimated_external_score":64.81258475122975,"probability":0.5450777894350789,"probability_error":0.0,"values":[-0.01477134092458453,-0.023541940724662674,-0.0302704509539723,-0.023284016734178906,0.1079920152902114,0.016404902346472513,-0.008323253847686441,0.00853592409982773],"diagnostics":{"base":0.5123359508836521,"residual":-2.7755575615628914e-17},"individual":[{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.1079920152902114},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.016404902346472513},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.00853592409982773}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.05443754760206018},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.016404902346472513},{"id":"Installment balances","label":"Installment balances","weight":0.00853592409982773}]}},"comparison":{"probability_difference":0.08174279865889805,"maximum_original_probability_error":0.08174279865889805,"individual":{"retention":1.0,"jaccard":0.6666666666666666,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":0.6666666666666666,"turnover":true,"first_changed":false,"empty":false},"same_order":false}},{"case_id":654,"inputs":{"MSinceOldestTradeOpen":132.0,"AverageMInFile":62.0,"NumSatisfactoryTrades":22.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":0.0,"NetFractionRevolvingBurden":56.0,"NetFractionInstallBurden":-8.0},"observed_external_score":79.0,"original_probability":0.4057656962513249,"original_shapley":{"values":[0.0032761364501984343,0.012404276392794124,-0.01180068140629744,-0.02625423918398385,-0.0041103136435535655,-0.020464584565383335,0.060058316723143365,0.005380834409827036,-0.12549601358146198],"diagnostics":{"base":0.5127719646560421,"residual":-1.3877787807814457e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.060058316723143365},{"id":"AverageMInFile","label":"Average account age","weight":0.012404276392794124},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.005380834409827036},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.0032761364501984343}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.060058316723143365},{"id":"Credit history","label":"Credit history","weight":0.01568041284299256},{"id":"Installment balances","label":"Installment balances","weight":0.005380834409827036}]},"approximations":{"additive":{"estimated_external_score":73.2636762054669,"probability":0.5127528112798673,"probability_error":0.10698711502854236,"values":[0.014834386023296767,0.04602393918811599,-0.013766340831881514,-0.05412715729174483,-0.07549472529323008,-0.020682427706770538,0.09145454880151635,-0.0034563023700570285],"diagnostics":{"base":0.5279668907606222,"residual":-4.163336342344337e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.09145454880151635},{"id":"AverageMInFile","label":"Average account age","weight":0.04602393918811599},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.014834386023296767}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.09145454880151635},{"id":"Credit history","label":"Credit history","weight":0.06085832521141275}]},"nonlinear":{"estimated_external_score":73.88653843631243,"probability":0.5102081977440647,"probability_error":0.10444250149273976,"values":[0.013717167566119666,0.0301795333744492,-0.0049130777094411665,-0.060329133338451024,-0.059730263632540095,-0.02280133440249451,0.10072180182666701,0.0010275531761034463],"diagnostics":{"base":0.5123359508836521,"residual":-6.938893903907228e-18},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.10072180182666701},{"id":"AverageMInFile","label":"Average account age","weight":0.0301795333744492},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.013717167566119666},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.0010275531761034463}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.10072180182666701},{"id":"Credit history","label":"Credit history","weight":0.04389670094056886},{"id":"Installment balances","label":"Installment balances","weight":0.0010275531761034463}]}},"comparison":{"probability_difference":0.002544613535802598,"maximum_original_probability_error":0.10698711502854236,"individual":{"retention":1.0,"jaccard":0.75,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":0.6666666666666666,"turnover":true,"first_changed":false,"empty":false},"same_order":false}},{"case_id":8745,"inputs":{"MSinceOldestTradeOpen":124.0,"AverageMInFile":72.0,"NumSatisfactoryTrades":21.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":0.0,"NetFractionRevolvingBurden":64.0,"NetFractionInstallBurden":76.0},"observed_external_score":75.0,"original_probability":0.45728226466306215,"original_shapley":{"values":[0.006402121232853847,0.0029457337514476817,-0.0021580504887744483,-0.02383310332576463,-0.005718541806590357,-0.023213022720257423,0.08355714648439945,0.009493905696267728,-0.10296588881656188],"diagnostics":{"base":0.5127719646560421,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.08355714648439945},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.009493905696267728},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.006402121232853847},{"id":"AverageMInFile","label":"Average account age","weight":0.0029457337514476817}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.08355714648439945},{"id":"Installment balances","label":"Installment balances","weight":0.009493905696267728},{"id":"Credit history","label":"Credit history","weight":0.009347854984301528}]},"approximations":{"additive":{"estimated_external_score":72.02141245891725,"probability":0.5508936766057859,"probability_error":0.09361141194272377,"values":[0.016328823429742977,0.011997174030329964,-0.009421324850580337,-0.04963684819020801,-0.08344411726231914,-0.021172162641412213,0.1461985050631129,0.012076736266497617],"diagnostics":{"base":0.5279668907606222,"residual":-3.8163916471489756e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.1461985050631129},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.016328823429742977},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.012076736266497617},{"id":"AverageMInFile","label":"Average account age","weight":0.011997174030329964}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.1461985050631129},{"id":"Credit history","label":"Credit history","weight":0.02832599746007294},{"id":"Installment balances","label":"Installment balances","weight":0.012076736266497617}]},"nonlinear":{"estimated_external_score":73.65644264128679,"probability":0.5548022998527196,"probability_error":0.0975200351896574,"values":[0.012180417096134832,-0.006731175484240496,0.0011753663089600218,-0.054523162766157075,-0.06178425993601198,-0.020530574046725102,0.1648213018581814,0.007858435938925705],"diagnostics":{"base":0.5123359508836522,"residual":-1.0408340855860843e-16},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.1648213018581814},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.012180417096134832},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.007858435938925705},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.0011753663089600218}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.1648213018581814},{"id":"Installment balances","label":"Installment balances","weight":0.007858435938925705},{"id":"Credit history","label":"Credit history","weight":0.005449241611894336}]}},"comparison":{"probability_difference":0.00390862324693364,"maximum_original_probability_error":0.0975200351896574,"individual":{"retention":0.75,"jaccard":0.6,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":false}},{"case_id":7202,"inputs":{"MSinceOldestTradeOpen":187.0,"AverageMInFile":62.0,"NumSatisfactoryTrades":51.0,"PercentTradesNeverDelq":96.0,"MSinceMostRecentDelq":10.0,"NumInqLast6M":1.0,"NetFractionRevolvingBurden":32.0,"NetFractionInstallBurden":42.0},"observed_external_score":72.0,"original_probability":0.4357337037940341,"original_shapley":{"values":[0.0037505965793792873,0.0113667171954379,-0.055877368469172434,-0.011210921526354996,0.007097693204740039,-0.007001089659805224,-0.023321415902806533,-0.006837695098089629,0.00499522281466353],"diagnostics":{"base":0.5127719646560421,"residual":2.7755575615628914e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.0113667171954379},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.007097693204740039},{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.00499522281466353},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.0037505965793792873}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.015117313774817188},{"id":"External risk score","label":"External risk score","weight":0.00499522281466353}]},"approximations":{"additive":{"estimated_external_score":64.89753044943107,"probability":0.5665318750647472,"probability_error":0.13079817127071314,"values":[0.005005636924175745,0.04575404714809692,-5.197768324801603e-05,-0.022308863896428898,0.07336956473582916,-0.014211180023914083,-0.02670809141413096,-0.022284151486254816],"diagnostics":{"base":0.5279668907606222,"residual":-2.7755575615628914e-17},"individual":[{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.07336956473582916},{"id":"AverageMInFile","label":"Average account age","weight":0.04575404714809692},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.005005636924175745}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.05100872315615225},{"id":"Credit history","label":"Credit history","weight":0.050759684072272666}]},"nonlinear":{"estimated_external_score":63.606516316293195,"probability":0.6085785783962983,"probability_error":0.17284487460226416,"values":[0.007784489833690232,0.018430678350623994,0.019555751802773937,-0.007819962614744878,0.08039605401915635,-0.00990337175822947,0.010183585329297359,-0.02238459744992144],"diagnostics":{"base":0.5123359508836521,"residual":2.7755575615628914e-17},"individual":[{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.08039605401915635},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.019555751802773937},{"id":"AverageMInFile","label":"Average account age","weight":0.018430678350623994},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.010183585329297359}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.09213184320718541},{"id":"Credit history","label":"Credit history","weight":0.026215168184314228},{"id":"Revolving balances","label":"Revolving balances","weight":0.010183585329297359}]}},"comparison":{"probability_difference":0.04204670333155103,"maximum_original_probability_error":0.17284487460226416,"individual":{"retention":0.6666666666666666,"jaccard":0.4,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":0.6666666666666666,"turnover":true,"first_changed":false,"empty":false},"same_order":false}},{"case_id":4869,"inputs":{"MSinceOldestTradeOpen":204.0,"AverageMInFile":79.0,"NumSatisfactoryTrades":15.0,"PercentTradesNeverDelq":87.0,"MSinceMostRecentDelq":9.0,"NumInqLast6M":0.0,"NetFractionRevolvingBurden":0.0,"NetFractionInstallBurden":53.0},"observed_external_score":76.0,"original_probability":0.4912707993978067,"original_shapley":{"values":[0.0036516599541250206,-0.0073720508676030074,0.019693831799650348,0.06364104430315082,0.0520603705000623,-0.018634207967396777,-0.046021048213285856,-0.013201853389345269,-0.07531891137759307],"diagnostics":{"base":0.5127719646560421,"residual":9.020562075079397e-17},"individual":[{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.06364104430315082},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.0520603705000623},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.019693831799650348},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.0036516599541250206}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.13539524660286345}]},"approximations":{"additive":{"estimated_external_score":73.23543387191914,"probability":0.5099664483138473,"probability_error":0.018695648916040586,"values":[-0.00034845908165943374,-0.007007763376649979,0.011786217926030948,0.0615733707201191,0.09131004493905027,-0.024134150616713913,-0.14294657415340783,-0.008233128803544107],"diagnostics":{"base":0.5279668907606222,"residual":2.7755575615628914e-17},"individual":[{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.09131004493905027},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.0615733707201191},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.011786217926030948}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.16466963358520031}]},"nonlinear":{"estimated_external_score":74.92218931008809,"probability":0.5077115984381024,"probability_error":0.016440799040295717,"values":[-0.0004938523499215574,-0.01799931107325597,0.012333193629366721,0.07375900780346326,0.08433533656413639,-0.025743457405589178,-0.12075704938376956,-0.010058220229979788],"diagnostics":{"base":0.5123359508836521,"residual":-2.7755575615628914e-17},"individual":[{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.08433533656413639},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.07375900780346326},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.012333193629366721}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.17042753799696636}]}},"comparison":{"probability_difference":0.0022548498757448687,"maximum_original_probability_error":0.018695648916040586,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":5103,"inputs":{"MSinceOldestTradeOpen":132.0,"AverageMInFile":65.0,"NumSatisfactoryTrades":34.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":0.0,"NetFractionRevolvingBurden":50.0,"NetFractionInstallBurden":88.0},"observed_external_score":68.0,"original_probability":0.5195656112660718,"original_shapley":{"values":[0.025969151718289604,0.0013931770871143632,-0.038645701730909136,-0.021280785266336998,-0.0022233912002691857,-0.012684691895344553,0.026054793350373546,0.004180851959000326,0.024030242588111716],"diagnostics":{"base":0.5127719646560421,"residual":-6.938893903907228e-18},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.026054793350373546},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.025969151718289604},{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.024030242588111716},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.004180851959000326}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.027362328805403967},{"id":"Revolving balances","label":"Revolving balances","weight":0.026054793350373546},{"id":"External risk score","label":"External risk score","weight":0.024030242588111716},{"id":"Installment balances","label":"Installment balances","weight":0.004180851959000326}]},"approximations":{"additive":{"estimated_external_score":72.40397553623978,"probability":0.4459335388239953,"probability_error":0.07363207244207653,"values":[0.01722324095370715,0.018505708803022932,-0.025960113660273006,-0.05415001018532806,-0.08439427425437533,-0.021478032700852297,0.05539696749711613,0.012823161610355635],"diagnostics":{"base":0.5279668907606222,"residual":-1.3877787807814457e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.05539696749711613},{"id":"AverageMInFile","label":"Average account age","weight":0.018505708803022932},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.01722324095370715},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.012823161610355635}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.05539696749711613},{"id":"Credit history","label":"Credit history","weight":0.03572894975673008},{"id":"Installment balances","label":"Installment balances","weight":0.012823161610355635}]},"nonlinear":{"estimated_external_score":71.02122343473694,"probability":0.4462541878485269,"probability_error":0.07331142341754493,"values":[0.014391052425338554,3.349851105826154e-05,-0.010949058860705997,-0.06421327210152397,-0.0654561114776383,-0.023402785435260386,0.07646939434499467,0.00704551955861182],"diagnostics":{"base":0.5123359508836522,"residual":-1.3877787807814457e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.07646939434499467},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.014391052425338554},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.00704551955861182},{"id":"AverageMInFile","label":"Average account age","weight":3.349851105826154e-05}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.07646939434499467},{"id":"Credit history","label":"Credit history","weight":0.014424550936396816},{"id":"Installment balances","label":"Installment balances","weight":0.00704551955861182}]}},"comparison":{"probability_difference":0.00032064902453160515,"maximum_original_probability_error":0.07363207244207653,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":false}},{"case_id":5211,"inputs":{"MSinceOldestTradeOpen":137.0,"AverageMInFile":56.0,"NumSatisfactoryTrades":36.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":3.0,"NetFractionRevolvingBurden":32.0,"NetFractionInstallBurden":92.0},"observed_external_score":63.0,"original_probability":0.732400004577804,"original_shapley":{"values":[0.022018242445248744,0.04717535390519408,-0.024963965692401377,-0.02470313697118987,-3.6003646842036044e-05,0.006294850867601784,-0.007843444552085316,0.025441511993069384,0.17624463157316628],"diagnostics":{"base":0.5127719646560421,"residual":1.6653345369377348e-16},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.17624463157316628},{"id":"AverageMInFile","label":"Average account age","weight":0.04717535390519408},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.025441511993069384},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.022018242445248744}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.17624463157316628},{"id":"Credit history","label":"Credit history","weight":0.06919359635044282},{"id":"Installment balances","label":"Installment balances","weight":0.025441511993069384},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.006294850867601784}]},"approximations":{"additive":{"estimated_external_score":73.05550910422306,"probability":0.5100801371049242,"probability_error":0.2223198674728798,"values":[0.010228454551037158,0.10306371972787637,-0.020672342434485312,-0.05524941386962545,-0.08159556414967473,0.022803401229498206,-0.02559660273739521,0.029131594027070933],"diagnostics":{"base":0.5279668907606222,"residual":1.3877787807814457e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.10306371972787637},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.029131594027070933},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.022803401229498206},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.010228454551037158}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.11329217427891353},{"id":"Installment balances","label":"Installment balances","weight":0.029131594027070933},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.022803401229498206}]},"nonlinear":{"estimated_external_score":70.6433542272467,"probability":0.5195278486382274,"probability_error":0.21287215593957654,"values":[-0.001035043365387324,0.09426849962851082,-0.006297431298325885,-0.06318253611891,-0.057728059786123304,0.019520693571422915,0.00738518561350391,0.014260589509884162],"diagnostics":{"base":0.5123359508836521,"residual":1.3877787807814457e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.09426849962851082},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.019520693571422915},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.014260589509884162},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.00738518561350391}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.09323345626312349},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.019520693571422915},{"id":"Installment balances","label":"Installment balances","weight":0.014260589509884162},{"id":"Revolving balances","label":"Revolving balances","weight":0.00738518561350391}]}},"comparison":{"probability_difference":0.00944771153330326,"maximum_original_probability_error":0.2223198674728798,"individual":{"retention":0.75,"jaccard":0.6,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":0.75,"turnover":true,"first_changed":false,"empty":false},"same_order":false}},{"case_id":5318,"inputs":{"MSinceOldestTradeOpen":409.0,"AverageMInFile":127.0,"NumSatisfactoryTrades":16.0,"PercentTradesNeverDelq":94.0,"MSinceMostRecentDelq":53.0,"NumInqLast6M":0.0,"NetFractionRevolvingBurden":49.0,"NetFractionInstallBurden":-8.0},"observed_external_score":79.0,"original_probability":0.372308960532593,"original_shapley":{"values":[-0.016580585873674964,-0.024544082578073424,0.009104693675044337,0.016152338213588104,-0.030349746476881295,-0.01696014401619072,0.03346991857999139,0.0027851338390467322,-0.1135405294862992],"diagnostics":{"base":0.5127719646560421,"residual":-5.551115123125783e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.03346991857999139},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.016152338213588104},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.009104693675044337},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.0027851338390467322}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.03346991857999139},{"id":"Installment balances","label":"Installment balances","weight":0.0027851338390467322}]},"approximations":{"additive":{"estimated_external_score":76.33203845748292,"probability":0.37814594118150835,"probability_error":0.0058369806489153575,"values":[-0.021161643004572475,-0.07777048488106693,0.0006389702865910764,0.004138606648936626,-0.07118637873791317,-0.027499883909398625,0.046597601628970815,-0.003577737610661111],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.046597601628970815},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.004138606648936626},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.0006389702865910764}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.046597601628970815}]},"nonlinear":{"estimated_external_score":72.84529415967734,"probability":0.47245351113025813,"probability_error":0.10014455059766514,"values":[-0.020597588771791533,-0.05138508566810418,0.006280157656478826,0.01807522815735475,-0.04967279754328438,-0.025722326239408434,0.08683202585253236,-0.0036920531971714084],"diagnostics":{"base":0.5123359508836521,"residual":-1.3877787807814457e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.08683202585253236},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.01807522815735475},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.006280157656478826}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.08683202585253236}]}},"comparison":{"probability_difference":0.09430756994874978,"maximum_original_probability_error":0.10014455059766514,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":10419,"inputs":{"MSinceOldestTradeOpen":118.0,"AverageMInFile":45.0,"NumSatisfactoryTrades":23.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":1.0,"NetFractionRevolvingBurden":24.0,"NetFractionInstallBurden":86.0},"observed_external_score":73.0,"original_probability":0.5501561751813262,"original_shapley":{"values":[0.014721397077731225,0.10691614160916624,-0.018724287114295815,-0.029085289320705934,-0.00964586077923256,-0.00920666651640714,-0.034188863306033455,0.013344045074632353,0.003253593800429175],"diagnostics":{"base":0.5127719646560421,"residual":-1.3877787807814457e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.10691614160916624},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.014721397077731225},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.013344045074632353},{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.003253593800429175}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.12163753868689746},{"id":"Installment balances","label":"Installment balances","weight":0.013344045074632353},{"id":"External risk score","label":"External risk score","weight":0.003253593800429175}]},"approximations":{"additive":{"estimated_external_score":75.87863275141449,"probability":0.4680958761507532,"probability_error":0.08206029903057299,"values":[0.008493870898196202,0.14613205266344748,-0.021443332853891457,-0.05551479262659611,-0.08436374975829265,-0.015259326951316632,-0.07086322001489184,0.03294748403347599],"diagnostics":{"base":0.5279668907606222,"residual":4.163336342344337e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.14613205266344748},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.03294748403347599},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.008493870898196202}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.15462592356164367},{"id":"Installment balances","label":"Installment balances","weight":0.03294748403347599}]},"nonlinear":{"estimated_external_score":72.27055107468782,"probability":0.5522210205299544,"probability_error":0.0020648453486281637,"values":[0.013074775506427841,0.1850755617145477,-0.011404206477605677,-0.057865740167866886,-0.056830006304118115,-0.013543245213562673,-0.03861421602451095,0.01999214661299091],"diagnostics":{"base":0.5123359508836522,"residual":-2.7755575615628914e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.1850755617145477},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.01999214661299091},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.013074775506427841}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.19815033722097553},{"id":"Installment balances","label":"Installment balances","weight":0.01999214661299091}]}},"comparison":{"probability_difference":0.08412514437920116,"maximum_original_probability_error":0.08206029903057299,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":3886,"inputs":{"MSinceOldestTradeOpen":-8.0,"AverageMInFile":51.0,"NumSatisfactoryTrades":10.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":0.0,"NetFractionRevolvingBurden":39.0,"NetFractionInstallBurden":22.0},"observed_external_score":76.0,"original_probability":0.3813148648364903,"original_shapley":{"values":[-0.0018063429176334936,0.07481477815188695,0.03122429307151829,-0.02134373495590687,-0.008145453934978179,-0.025221137316604837,-0.019669255981991256,-0.02688015729248909,-0.13443008864335326],"diagnostics":{"base":0.5127719646560421,"residual":-5.551115123125783e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.07481477815188695},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.03122429307151829}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.07300843523425345},{"id":"Repayment history","label":"Repayment history","weight":0.0017351041806332422}]},"approximations":{"additive":{"estimated_external_score":76.05277653810602,"probability":0.3813148648364903,"probability_error":0.0,"values":[0.0039023350977151258,0.11693412257105805,-0.005873807509374994,-0.05552190685842341,-0.1006011185254346,-0.04260453914770128,-0.03606037215607359,-0.026826739395897147],"diagnostics":{"base":0.5279668907606222,"residual":-5.551115123125783e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.11693412257105805},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.0039023350977151258}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.12083645766877317}]},"nonlinear":{"estimated_external_score":76.11940872257256,"probability":0.3813148648364903,"probability_error":0.0,"values":[-0.0010323538057095709,0.1267148033798065,-0.030398261438543482,-0.07180092719717535,-0.08147329057329206,-0.04459303528128082,-0.004810907740864477,-0.023627113390102643],"diagnostics":{"base":0.5123359508836521,"residual":5.551115123125783e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.1267148033798065}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.12568244957409694}]}},"comparison":{"probability_difference":0.0,"maximum_original_probability_error":0.0,"individual":{"retention":0.5,"jaccard":0.5,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":false}},{"case_id":9412,"inputs":{"MSinceOldestTradeOpen":210.0,"AverageMInFile":63.0,"NumSatisfactoryTrades":11.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":0.0,"NetFractionRevolvingBurden":65.0,"NetFractionInstallBurden":78.0},"observed_external_score":76.0,"original_probability":0.5052256405444726,"original_shapley":{"values":[-0.0005487675317374036,0.016315618036586314,0.03322177563374923,-0.021169702663863123,-0.0035133134472199834,-0.025069682805810813,0.08645968947807872,0.014892823681819019,-0.10813476449317153],"diagnostics":{"base":0.5127719646560421,"residual":2.7755575615628914e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.08645968947807872},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.03322177563374923},{"id":"AverageMInFile","label":"Average account age","weight":0.016315618036586314},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.014892823681819019}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.08645968947807872},{"id":"Credit history","label":"Credit history","weight":0.01576685050484891},{"id":"Installment balances","label":"Installment balances","weight":0.014892823681819019},{"id":"Repayment history","label":"Repayment history","weight":0.008538759522666122}]},"approximations":{"additive":{"estimated_external_score":72.36089325381151,"probability":0.6725655606153343,"probability_error":0.16733992007086174,"values":[-0.008241068225933658,0.060886010651355654,0.041545462521863384,-0.040187215501349946,-0.0664894462545784,-0.024798262427798312,0.17212186321201084,0.009761325879142583],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.17212186321201084},{"id":"AverageMInFile","label":"Average account age","weight":0.060886010651355654},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.041545462521863384},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.009761325879142583}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.17212186321201084},{"id":"Credit history","label":"Credit history","weight":0.052644942425421996},{"id":"Installment balances","label":"Installment balances","weight":0.009761325879142583}]},"nonlinear":{"estimated_external_score":74.08096278071719,"probability":0.6634608598131136,"probability_error":0.15823521926864104,"values":[-0.009621139196510621,0.04883625238874485,0.04583974417249826,-0.04966754472092885,-0.053940847948415416,-0.0291935391427191,0.19129350304659537,0.007578480330196595],"diagnostics":{"base":0.5123359508836522,"residual":1.1102230246251565e-16},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.19129350304659537},{"id":"AverageMInFile","label":"Average account age","weight":0.04883625238874485},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.04583974417249826},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.007578480330196595}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.19129350304659537},{"id":"Credit history","label":"Credit history","weight":0.03921511319223423},{"id":"Installment balances","label":"Installment balances","weight":0.007578480330196595}]}},"comparison":{"probability_difference":0.0091047008022207,"maximum_original_probability_error":0.16733992007086174,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":8792,"inputs":{"MSinceOldestTradeOpen":198.0,"AverageMInFile":55.0,"NumSatisfactoryTrades":30.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":1.0,"NetFractionRevolvingBurden":37.0,"NetFractionInstallBurden":87.0},"observed_external_score":72.0,"original_probability":0.5525779837761894,"original_shapley":{"values":[0.0030998365933362477,0.09491745216610827,-0.03633491208753055,-0.02373072268093618,-0.007749416305332462,-0.007679401763289704,-0.006361528908265548,0.010969374080958263,0.01267533802509894],"diagnostics":{"base":0.5127719646560421,"residual":-2.7755575615628914e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.09491745216610827},{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.01267533802509894},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.010969374080958263},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.0030998365933362477}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.09801728875944452},{"id":"External risk score","label":"External risk score","weight":0.01267533802509894},{"id":"Installment balances","label":"Installment balances","weight":0.010969374080958263}]},"approximations":{"additive":{"estimated_external_score":73.85948272759217,"probability":0.49740960300571513,"probability_error":0.05516838077047426,"values":[0.001098272470243672,0.12075819073057711,-0.019086418578372046,-0.05786549866859349,-0.0825744663965602,-0.01692143451390635,-0.007128426012355754,0.031162493214060002],"diagnostics":{"base":0.5279668907606222,"residual":-1.3877787807814457e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.12075819073057711},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.031162493214060002},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.001098272470243672}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.12185646320082079},{"id":"Installment balances","label":"Installment balances","weight":0.031162493214060002}]},"nonlinear":{"estimated_external_score":72.53520080337312,"probability":0.5463698044216686,"probability_error":0.006208179354520782,"values":[-0.00186176363671307,0.1273320965724938,0.0031583893881011087,-0.05914865939247445,-0.060762090853506416,-0.012563555034454789,0.02117884553358323,0.016700590960987013],"diagnostics":{"base":0.5123359508836521,"residual":1.3877787807814457e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.1273320965724938},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.02117884553358323},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.016700590960987013},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.0031583893881011087}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.12547033293578075},{"id":"Revolving balances","label":"Revolving balances","weight":0.02117884553358323},{"id":"Installment balances","label":"Installment balances","weight":0.016700590960987013}]}},"comparison":{"probability_difference":0.048960201415953475,"maximum_original_probability_error":0.05516838077047426,"individual":{"retention":0.6666666666666666,"jaccard":0.4,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":0.6666666666666666,"turnover":true,"first_changed":false,"empty":false},"same_order":false}},{"case_id":1568,"inputs":{"MSinceOldestTradeOpen":110.0,"AverageMInFile":52.0,"NumSatisfactoryTrades":33.0,"PercentTradesNeverDelq":97.0,"MSinceMostRecentDelq":28.0,"NumInqLast6M":6.0,"NetFractionRevolvingBurden":12.0,"NetFractionInstallBurden":99.0},"observed_external_score":72.0,"original_probability":0.5784945792911391,"original_shapley":{"values":[0.020341112839581503,0.07471448952398713,-0.04172692628237554,-0.0222391978357351,-0.0009761911017790571,0.046212176453441875,-0.060358499614340415,0.018333518632810315,0.03142213201950628],"diagnostics":{"base":0.5127719646560421,"residual":0.0},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.07471448952398713},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.046212176453441875},{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.03142213201950628},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.020341112839581503}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.09505560236356864},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.046212176453441875},{"id":"External risk score","label":"External risk score","weight":0.03142213201950628},{"id":"Installment balances","label":"Installment balances","weight":0.018333518632810315}]},"approximations":{"additive":{"estimated_external_score":68.77080216116599,"probability":0.5988429951010203,"probability_error":0.020348415809881226,"values":[0.020878974230509335,0.11262343602372106,-0.028778530448158562,-0.03451181208361585,0.020614916895961437,0.07205203955477259,-0.1168887196994488,0.02488579986665694],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.11262343602372106},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.07205203955477259},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.02488579986665694},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.020878974230509335}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.1335024102542304},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.07205203955477259},{"id":"Installment balances","label":"Installment balances","weight":0.02488579986665694}]},"nonlinear":{"estimated_external_score":72.17529171748157,"probability":0.5784945792911391,"probability_error":0.0,"values":[0.018436224164970146,0.12797546800817827,-0.02057673446711341,-0.0399225140765089,-0.011243761114425857,0.07409226569562802,-0.10369901520630768,0.021096695403066378],"diagnostics":{"base":0.5123359508836521,"residual":4.163336342344337e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.12797546800817827},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.07409226569562802},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.021096695403066378},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.018436224164970146}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.14641169217314842},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.07409226569562802},{"id":"Installment balances","label":"Installment balances","weight":0.021096695403066378}]}},"comparison":{"probability_difference":0.020348415809881226,"maximum_original_probability_error":0.020348415809881226,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":1707,"inputs":{"MSinceOldestTradeOpen":295.0,"AverageMInFile":84.0,"NumSatisfactoryTrades":18.0,"PercentTradesNeverDelq":95.0,"MSinceMostRecentDelq":8.0,"NumInqLast6M":0.0,"NetFractionRevolvingBurden":24.0,"NetFractionInstallBurden":99.0},"observed_external_score":72.0,"original_probability":0.5637596568127216,"original_shapley":{"values":[-0.01037947838163043,-0.01537923049530876,0.021570325067252972,0.003533417840903169,0.02449087378887779,-0.010807277880875751,-0.025189670510348757,0.030737542342811723,0.03241119038499748],"diagnostics":{"base":0.5127719646560421,"residual":-1.3877787807814457e-17},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.03241119038499748},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.030737542342811723},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.02449087378887779},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.021570325067252972}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.04959461669703393},{"id":"External risk score","label":"External risk score","weight":0.03241119038499748},{"id":"Installment balances","label":"Installment balances","weight":0.030737542342811723}]},"approximations":{"additive":{"estimated_external_score":70.62066045729439,"probability":0.5515066242812736,"probability_error":0.012253032531447938,"values":[-0.014601850593035329,-0.013479446196335409,0.012761581909782146,-0.010485861523229948,0.09798940702159559,-0.025475191831460343,-0.05755475775164537,0.03438585248498008],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.09798940702159559},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.03438585248498008},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.012761581909782146}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.10026512740814779},{"id":"Installment balances","label":"Installment balances","weight":0.03438585248498008}]},"nonlinear":{"estimated_external_score":73.39708501013033,"probability":0.5578662356135491,"probability_error":0.005893421199172466,"values":[-0.010914400718327276,-0.02331476777184172,0.013459036851167287,-0.0034297202935006715,0.10415452583414492,-0.02646748379186564,-0.03154788981551936,0.023590984435639402],"diagnostics":{"base":0.5123359508836521,"residual":1.3877787807814457e-17},"individual":[{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.10415452583414492},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.023590984435639402},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.013459036851167287}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.11418384239181154},{"id":"Installment balances","label":"Installment balances","weight":0.023590984435639402}]}},"comparison":{"probability_difference":0.006359611332275472,"maximum_original_probability_error":0.012253032531447938,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":7189,"inputs":{"MSinceOldestTradeOpen":241.0,"AverageMInFile":104.0,"NumSatisfactoryTrades":23.0,"PercentTradesNeverDelq":70.0,"MSinceMostRecentDelq":26.0,"NumInqLast6M":1.0,"NetFractionRevolvingBurden":43.0,"NetFractionInstallBurden":53.0},"observed_external_score":70.0,"original_probability":0.5038900845196684,"original_shapley":{"values":[-0.006116931431752833,-0.029371094199538508,-0.01343710803869601,0.058136333109852306,-0.008663820226906089,-0.007470843317590047,-0.014798297694082113,-0.006895854497245318,0.019735736159584873],"diagnostics":{"base":0.5127719646560421,"residual":3.8163916471489756e-17},"individual":[{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.058136333109852306},{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.019735736159584873}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.03603540484425021},{"id":"External risk score","label":"External risk score","weight":0.019735736159584873}]},"approximations":{"additive":{"estimated_external_score":65.45505682022473,"probability":0.5432729415702705,"probability_error":0.03938285705060207,"values":[-0.010242370046366456,-0.047181641200191,-0.014590276604746837,0.11057010051567451,0.008470901466521987,-0.01191395514847911,-0.004400839826933763,-0.015405868345831035],"diagnostics":{"base":0.5279668907606222,"residual":6.938893903907228e-18},"individual":[{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.11057010051567451},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.008470901466521987}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.10445072537744966}]},"nonlinear":{"estimated_external_score":66.63380656035801,"probability":0.5351765433741782,"probability_error":0.03128645885450976,"values":[-0.014066008094990659,-0.046116706699151126,-0.01366064592718647,0.14284206194481386,-0.02522423667535299,-0.009595321707904513,0.002297019395810767,-0.01363556974551278],"diagnostics":{"base":0.5123359508836521,"residual":-4.163336342344337e-17},"individual":[{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.14284206194481386},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.002297019395810767}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.1039571793422744},{"id":"Revolving balances","label":"Revolving balances","weight":0.002297019395810767}]}},"comparison":{"probability_difference":0.008096398196092314,"maximum_original_probability_error":0.03938285705060207,"individual":{"retention":0.5,"jaccard":0.3333333333333333,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":0.5,"turnover":true,"first_changed":false,"empty":false},"same_order":false}},{"case_id":10034,"inputs":{"MSinceOldestTradeOpen":256.0,"AverageMInFile":168.0,"NumSatisfactoryTrades":35.0,"PercentTradesNeverDelq":86.0,"MSinceMostRecentDelq":43.0,"NumInqLast6M":0.0,"NetFractionRevolvingBurden":53.0,"NetFractionInstallBurden":49.0},"observed_external_score":70.0,"original_probability":0.5106631848832416,"original_shapley":{"values":[-0.004376647925418815,-0.030421905720340543,-0.031156881535426898,0.04561732299045813,-0.016147919490358136,-0.011111963263804094,0.027992833500363714,-0.0064948866533871726,0.023991268325113394],"diagnostics":{"base":0.5127719646560421,"residual":-1.3877787807814457e-17},"individual":[{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.04561732299045813},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.027992833500363714},{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.023991268325113394}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.027992833500363714},{"id":"External risk score","label":"External risk score","weight":0.023991268325113394}]},"approximations":{"additive":{"estimated_external_score":72.93990401725287,"probability":0.4978110785760089,"probability_error":0.012852106307232658,"values":[-0.006660677093902157,-0.07982427579865786,-0.009197577871303023,0.050534541064677335,-0.03128362951068523,-0.02258781700101596,0.07927353324954697,-0.010409909223273324],"diagnostics":{"base":0.5279668907606222,"residual":2.7755575615628914e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.07927353324954697},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.050534541064677335}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.07927353324954697},{"id":"Repayment history","label":"Repayment history","weight":0.010053333682689083}]},"nonlinear":{"estimated_external_score":69.47049603603304,"probability":0.5116566707703086,"probability_error":0.000993485887067047,"values":[-0.007383741249347599,-0.046998967555613426,-0.015261018103522402,0.0550786212173473,-0.042585854441236956,-0.020450737205082233,0.0855135797870745,-0.008591162562962738],"diagnostics":{"base":0.5123359508836521,"residual":6.245004513516506e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.0855135797870745},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.0550786212173473}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.0855135797870745}]}},"comparison":{"probability_difference":0.013845592194299705,"maximum_original_probability_error":0.012852106307232658,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":0.5,"jaccard":0.5,"turnover":true,"first_changed":false,"empty":false},"same_order":true}},{"case_id":5420,"inputs":{"MSinceOldestTradeOpen":243.0,"AverageMInFile":80.0,"NumSatisfactoryTrades":57.0,"PercentTradesNeverDelq":98.0,"MSinceMostRecentDelq":-8.0,"NumInqLast6M":2.0,"NetFractionRevolvingBurden":48.0,"NetFractionInstallBurden":87.0},"observed_external_score":70.0,"original_probability":0.4486906658813239,"original_shapley":{"values":[0.0005511258844613309,-0.015570444197972261,-0.03782483287262142,-0.02242266308078426,0.002813270150481456,-0.001740549493207494,0.005326013260213152,0.006635763144935228,-0.0018489815702240559],"diagnostics":{"base":0.5127719646560421,"residual":2.7755575615628914e-17},"individual":[{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.006635763144935228},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.005326013260213152},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.002813270150481456},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.0005511258844613309}],"grouped":[{"id":"Installment balances","label":"Installment balances","weight":0.006635763144935228},{"id":"Revolving balances","label":"Revolving balances","weight":0.005326013260213152}]},"approximations":{"additive":{"estimated_external_score":68.52564054871867,"probability":0.47150578057300024,"probability_error":0.022815114691676353,"values":[-0.0050052598145347345,-0.009679918223364278,0.013216858294099302,-0.04898918492455707,-0.0599024198055384,0.004097426828095769,0.03527741747858922,0.014523969979588252],"diagnostics":{"base":0.5279668907606222,"residual":-2.0816681711721685e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.03527741747858922},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.014523969979588252},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.013216858294099302},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.004097426828095769}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.03527741747858922},{"id":"Installment balances","label":"Installment balances","weight":0.014523969979588252},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.004097426828095769}]},"nonlinear":{"estimated_external_score":67.62636273648562,"probability":0.49282428229427344,"probability_error":0.04413361641294955,"values":[-0.012394177869215187,-0.019123956447712295,0.007217873027809154,-0.05005555956123524,-0.024048698116057497,0.01157527170979445,0.0503207777415969,0.016996800925640924],"diagnostics":{"base":0.5123359508836522,"residual":-2.7755575615628914e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.0503207777415969},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.016996800925640924},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.01157527170979445},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.007217873027809154}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.0503207777415969},{"id":"Installment balances","label":"Installment balances","weight":0.016996800925640924},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.01157527170979445}]}},"comparison":{"probability_difference":0.021318501721273198,"maximum_original_probability_error":0.04413361641294955,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":false}},{"case_id":8371,"inputs":{"MSinceOldestTradeOpen":133.0,"AverageMInFile":58.0,"NumSatisfactoryTrades":34.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":4.0,"NetFractionRevolvingBurden":36.0,"NetFractionInstallBurden":77.0},"observed_external_score":69.0,"original_probability":0.5382339030629459,"original_shapley":{"values":[0.015523273175954085,0.0530061774288692,-0.04472189656925778,-0.025959386664297625,-0.005149148757166119,0.009384646806159438,-0.014611746564152837,0.004790818531210341,0.03319920101958508],"diagnostics":{"base":0.5127719646560421,"residual":2.7755575615628914e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.0530061774288692},{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.03319920101958508},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.015523273175954085},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.009384646806159438}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.06852945060482328},{"id":"External risk score","label":"External risk score","weight":0.03319920101958508},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.009384646806159438},{"id":"Installment balances","label":"Installment balances","weight":0.004790818531210341}]},"approximations":{"additive":{"estimated_external_score":72.40703668508041,"probability":0.4961802100513437,"probability_error":0.04205369301160222,"values":[0.01440243635452818,0.08551128100583888,-0.029065808371566734,-0.0545300521200986,-0.07911905829445673,0.0363269686355446,-0.01737801856571792,0.012065570646649863],"diagnostics":{"base":0.5279668907606222,"residual":-6.245004513516506e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.08551128100583888},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.0363269686355446},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.01440243635452818},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.012065570646649863}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.09991371736036705},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.0363269686355446},{"id":"Installment balances","label":"Installment balances","weight":0.012065570646649863}]},"nonlinear":{"estimated_external_score":69.74367252008722,"probability":0.5162597241777771,"probability_error":0.021974178885168838,"values":[0.009303704979448437,0.07829342862478647,-0.02152710083274932,-0.059458057650266544,-0.05871727958792796,0.04359893346032498,0.004876999688013557,0.00755314461249534],"diagnostics":{"base":0.5123359508836521,"residual":1.0408340855860843e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.07829342862478647},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.04359893346032498},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.009303704979448437},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.00755314461249534}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.0875971336042349},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.04359893346032498},{"id":"Installment balances","label":"Installment balances","weight":0.00755314461249534},{"id":"Revolving balances","label":"Revolving balances","weight":0.004876999688013557}]}},"comparison":{"probability_difference":0.020079514126433384,"maximum_original_probability_error":0.04205369301160222,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":0.75,"turnover":true,"first_changed":false,"empty":false},"same_order":true}},{"case_id":7928,"inputs":{"MSinceOldestTradeOpen":208.0,"AverageMInFile":83.0,"NumSatisfactoryTrades":25.0,"PercentTradesNeverDelq":85.0,"MSinceMostRecentDelq":13.0,"NumInqLast6M":1.0,"NetFractionRevolvingBurden":10.0,"NetFractionInstallBurden":65.0},"observed_external_score":71.0,"original_probability":0.5039242677916991,"original_shapley":{"values":[0.0023387891325657396,-0.016908918675755474,-0.01944270965513895,0.06947888260654715,0.013005659283750267,-0.006888999339202559,-0.05408763481905781,0.0020464760486685577,0.0016107585532799803],"diagnostics":{"base":0.5127719646560421,"residual":2.7755575615628914e-17},"individual":[{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.06947888260654715},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.013005659283750267},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.0023387891325657396},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.0020464760486685577}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.06304183223515847},{"id":"Installment balances","label":"Installment balances","weight":0.0020464760486685577},{"id":"External risk score","label":"External risk score","weight":0.0016107585532799803}]},"approximations":{"additive":{"estimated_external_score":70.41509319423437,"probability":0.518445083623105,"probability_error":0.014520815831405964,"values":[0.0030424006134649067,-0.014467631438135726,-0.016121947097268465,0.0784592791135052,0.07254738986394269,-0.010958936323581064,-0.12483016219602011,0.0028078003265753757],"diagnostics":{"base":0.5279668907606222,"residual":4.85722573273506e-17},"individual":[{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.0784592791135052},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.07254738986394269},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.0030424006134649067},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.0028078003265753757}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.13488472188017941},{"id":"Installment balances","label":"Installment balances","weight":0.0028078003265753757}]},"nonlinear":{"estimated_external_score":71.0074885563354,"probability":0.5039242677916991,"probability_error":0.0,"values":[-0.000849075056368328,-0.026828989477559437,-0.01816411346706703,0.09958894763247782,0.052825702524807235,-0.007917076545228566,-0.11151108221162732,0.004444003508612614],"diagnostics":{"base":0.5123359508836521,"residual":-3.469446951953614e-17},"individual":[{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.09958894763247782},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.052825702524807235},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.004444003508612614}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.13425053669021803},{"id":"Installment balances","label":"Installment balances","weight":0.004444003508612614}]}},"comparison":{"probability_difference":0.014520815831405964,"maximum_original_probability_error":0.014520815831405964,"individual":{"retention":0.75,"jaccard":0.75,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":false}},{"case_id":5460,"inputs":{"MSinceOldestTradeOpen":110.0,"AverageMInFile":63.0,"NumSatisfactoryTrades":9.0,"PercentTradesNeverDelq":62.0,"MSinceMostRecentDelq":11.0,"NumInqLast6M":1.0,"NetFractionRevolvingBurden":92.0,"NetFractionInstallBurden":89.0},"observed_external_score":55.0,"original_probability":0.9406777557891253,"original_shapley":{"values":[0.02623032309649096,0.028424145325153506,0.045232953377115304,0.033853473611293025,0.012949516748367243,-0.0026072186855005145,0.088517258885761,0.020506631545648153,0.1747987072287543],"diagnostics":{"base":0.5127719646560421,"residual":5.551115123125783e-17},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.1747987072287543},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.088517258885761},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.045232953377115304},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.033853473611293025}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.1747987072287543},{"id":"Repayment history","label":"Repayment history","weight":0.09203594373677557},{"id":"Revolving balances","label":"Revolving balances","weight":0.088517258885761},{"id":"Credit history","label":"Credit history","weight":0.05465446842164447}]},"approximations":{"additive":{"estimated_external_score":52.11501548025508,"probability":0.9406777557891253,"probability_error":0.0,"values":[0.02825247781594599,0.04499793536505031,0.03348605919635956,0.07951612000248111,0.04704039405091815,-0.0059780361190553185,0.16001789932605548,0.025378015390747546],"diagnostics":{"base":0.5279668907606222,"residual":1.1102230246251565e-16},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.16001789932605548},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.07951612000248111},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.04704039405091815},{"id":"AverageMInFile","label":"Average account age","weight":0.04499793536505031}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.16004257324975882},{"id":"Revolving balances","label":"Revolving balances","weight":0.16001789932605548},{"id":"Credit history","label":"Credit history","weight":0.0732504131809963},{"id":"Installment balances","label":"Installment balances","weight":0.025378015390747546}]},"nonlinear":{"estimated_external_score":57.389951712317654,"probability":0.9376282032602709,"probability_error":0.0030495525288544156,"values":[0.02670221032372649,0.03825753478996378,0.03262444296523913,0.10797423346997341,0.04693913524456176,-0.0037051784915620232,0.15351254422226573,0.022987329852450434],"diagnostics":{"base":0.5123359508836521,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.15351254422226573},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.10797423346997341},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.04693913524456176},{"id":"AverageMInFile","label":"Average account age","weight":0.03825753478996378}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.1875378116797743},{"id":"Revolving balances","label":"Revolving balances","weight":0.15351254422226573},{"id":"Credit history","label":"Credit history","weight":0.06495974511369026},{"id":"Installment balances","label":"Installment balances","weight":0.022987329852450434}]}},"comparison":{"probability_difference":0.0030495525288544156,"maximum_original_probability_error":0.0030495525288544156,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":6592,"inputs":{"MSinceOldestTradeOpen":126.0,"AverageMInFile":41.0,"NumSatisfactoryTrades":5.0,"PercentTradesNeverDelq":83.0,"MSinceMostRecentDelq":8.0,"NumInqLast6M":1.0,"NetFractionRevolvingBurden":90.0,"NetFractionInstallBurden":86.0},"observed_external_score":63.0,"original_probability":0.92174793266144,"original_shapley":{"values":[0.02204172092907354,0.0628867882214298,0.05748820252212096,0.02712478215656779,0.014318395704979715,-0.00246311071001973,0.08315759175676844,0.023775506734248886,0.12064609069022843],"diagnostics":{"base":0.5127719646560421,"residual":0.0},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.12064609069022843},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.08315759175676844},{"id":"AverageMInFile","label":"Average account age","weight":0.0628867882214298},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.05748820252212096}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.12064609069022843},{"id":"Repayment history","label":"Repayment history","weight":0.09893138038366846},{"id":"Credit history","label":"Credit history","weight":0.08492850915050335},{"id":"Revolving balances","label":"Revolving balances","weight":0.08315759175676844}]},"approximations":{"additive":{"estimated_external_score":55.32881136257803,"probability":0.9512565465505688,"probability_error":0.029508613889128843,"values":[0.021180630259003033,0.08997939351035905,0.0414433103113483,0.037157402225784335,0.049233132107729304,-0.0047894268586588535,0.1608751782045266,0.02821003602985481],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.1608751782045266},{"id":"AverageMInFile","label":"Average account age","weight":0.08997939351035905},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.049233132107729304},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.0414433103113483}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.1608751782045266},{"id":"Repayment history","label":"Repayment history","weight":0.12783384464486194},{"id":"Credit history","label":"Credit history","weight":0.11116002376936208},{"id":"Installment balances","label":"Installment balances","weight":0.02821003602985481}]},"nonlinear":{"estimated_external_score":57.90736139681362,"probability":0.9493078712146967,"probability_error":0.02755993855325678,"values":[0.02151672749808457,0.09577906308386373,0.04064126462609595,0.056646427691804294,0.04171545789134307,-0.0032076356433426703,0.15693760203849003,0.026943013144705506],"diagnostics":{"base":0.5123359508836521,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.15693760203849003},{"id":"AverageMInFile","label":"Average account age","weight":0.09577906308386373},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.056646427691804294},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.04171545789134307}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.15693760203849003},{"id":"Repayment history","label":"Repayment history","weight":0.1390031502092433},{"id":"Credit history","label":"Credit history","weight":0.1172957905819483},{"id":"Installment balances","label":"Installment balances","weight":0.026943013144705506}]}},"comparison":{"probability_difference":0.0019486753358720632,"maximum_original_probability_error":0.029508613889128843,"individual":{"retention":0.75,"jaccard":0.6,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":false}},{"case_id":1661,"inputs":{"MSinceOldestTradeOpen":77.0,"AverageMInFile":25.0,"NumSatisfactoryTrades":8.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":1.0,"NetFractionRevolvingBurden":93.0,"NetFractionInstallBurden":72.0},"observed_external_score":66.0,"original_probability":0.8798809611060042,"original_shapley":{"values":[0.02105301272982388,0.09362115788879352,0.06466226455169705,-0.00805512011568908,0.0032545804304659077,-0.003914127574467484,0.12393466045010339,0.005388963929250811,0.06716360415998447],"diagnostics":{"base":0.5127719646560421,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.12393466045010339},{"id":"AverageMInFile","label":"Average account age","weight":0.09362115788879352},{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.06716360415998447},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.06466226455169705}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.12393466045010339},{"id":"Credit history","label":"Credit history","weight":0.1146741706186174},{"id":"External risk score","label":"External risk score","weight":0.06716360415998447},{"id":"Repayment history","label":"Repayment history","weight":0.05986172486647387}]},"approximations":{"additive":{"estimated_external_score":64.97866523001633,"probability":0.874939943910522,"probability_error":0.0049410171954822335,"values":[0.032038556170184175,0.1397192327325591,0.038978260922634705,-0.024650449114652268,-0.04136382756463318,-0.00735283367194192,0.1982541788727593,0.011349934802989872],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.1982541788727593},{"id":"AverageMInFile","label":"Average account age","weight":0.1397192327325591},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.038978260922634705},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.032038556170184175}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.1982541788727593},{"id":"Credit history","label":"Credit history","weight":0.17175778890274326},{"id":"Installment balances","label":"Installment balances","weight":0.011349934802989872}]},"nonlinear":{"estimated_external_score":65.4733476424476,"probability":0.8749399439105219,"probability_error":0.0049410171954823445,"values":[0.030080772010641806,0.16853672911971204,0.025617622317970774,-0.030395317266144372,-0.025249544684283556,-0.0049474457749991456,0.18870480510749818,0.01025637219647416],"diagnostics":{"base":0.5123359508836521,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.18870480510749818},{"id":"AverageMInFile","label":"Average account age","weight":0.16853672911971204},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.030080772010641806},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.025617622317970774}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.19861750113035384},{"id":"Revolving balances","label":"Revolving balances","weight":0.18870480510749818},{"id":"Installment balances","label":"Installment balances","weight":0.01025637219647416}]}},"comparison":{"probability_difference":1.1102230246251565e-16,"maximum_original_probability_error":0.0049410171954823445,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":true,"empty":false},"same_order":false}},{"case_id":8561,"inputs":{"MSinceOldestTradeOpen":61.0,"AverageMInFile":27.0,"NumSatisfactoryTrades":7.0,"PercentTradesNeverDelq":75.0,"MSinceMostRecentDelq":11.0,"NumInqLast6M":4.0,"NetFractionRevolvingBurden":41.0,"NetFractionInstallBurden":96.0},"observed_external_score":61.0,"original_probability":0.9326786947372203,"original_shapley":{"values":[0.048697722505161974,0.0783140647195536,0.05335645283327017,0.037553225718573176,0.01441909545721944,0.011555310245901375,0.019137455583748925,0.025242068660854756,0.13163133435689486],"diagnostics":{"base":0.5127719646560421,"residual":-1.1102230246251565e-16},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.13163133435689486},{"id":"AverageMInFile","label":"Average account age","weight":0.0783140647195536},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.05335645283327017},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.048697722505161974}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.13163133435689486},{"id":"Credit history","label":"Credit history","weight":0.12701178722471557},{"id":"Repayment history","label":"Repayment history","weight":0.10532877400906278},{"id":"Installment balances","label":"Installment balances","weight":0.025242068660854756}]},"approximations":{"additive":{"estimated_external_score":59.23007692304913,"probability":0.9496421390169455,"probability_error":0.01696344427972518,"values":[0.048432361523499674,0.12192152278240767,0.040295351445585444,0.0731220929499772,0.0574226023872469,0.02638060118512074,0.024469693701647404,0.029631022280838127],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.12192152278240767},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.0731220929499772},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.0574226023872469},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.048432361523499674}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.17084004678280953},{"id":"Credit history","label":"Credit history","weight":0.17035388430590734},{"id":"Installment balances","label":"Installment balances","weight":0.029631022280838127},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.02638060118512074}]},"nonlinear":{"estimated_external_score":62.47510225012036,"probability":0.9303657576742833,"probability_error":0.002312937062937004,"values":[0.04606995654228163,0.12284919989876751,0.029985629404446146,0.0903430237094234,0.047056365128539175,0.027422379193401684,0.028352100031769172,0.025951152882002494],"diagnostics":{"base":0.5123359508836521,"residual":-5.551115123125783e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.12284919989876751},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.0903430237094234},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.047056365128539175},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.04606995654228163}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.16891915644104916},{"id":"Repayment history","label":"Repayment history","weight":0.16738501824240873},{"id":"Revolving balances","label":"Revolving balances","weight":0.028352100031769172},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.027422379193401684}]}},"comparison":{"probability_difference":0.019276381342662186,"maximum_original_probability_error":0.01696344427972518,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":0.75,"jaccard":0.6,"turnover":true,"first_changed":true,"empty":false},"same_order":true}},{"case_id":5473,"inputs":{"MSinceOldestTradeOpen":54.0,"AverageMInFile":25.0,"NumSatisfactoryTrades":26.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":8.0,"NetFractionRevolvingBurden":73.0,"NetFractionInstallBurden":97.0},"observed_external_score":58.0,"original_probability":0.949115894776675,"original_shapley":{"values":[0.037284718678228156,0.0901491496512824,-0.007743023794721815,-0.010696256272278137,0.0030846440768933,0.021904230073087463,0.09741122407802938,0.022947547153861596,0.18200169647625053],"diagnostics":{"base":0.5127719646560421,"residual":0.0},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.18200169647625053},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.09741122407802938},{"id":"AverageMInFile","label":"Average account age","weight":0.0901491496512824},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.037284718678228156}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.18200169647625053},{"id":"Credit history","label":"Credit history","weight":0.12743386832951056},{"id":"Revolving balances","label":"Revolving balances","weight":0.09741122407802938},{"id":"Installment balances","label":"Installment balances","weight":0.022947547153861596}]},"approximations":{"additive":{"estimated_external_score":62.83594905344809,"probability":0.9290005263188145,"probability_error":0.02011536845786055,"values":[0.03537153530058595,0.1683494113777338,-0.00761550869896831,-0.027368170512856924,-0.044114368300663793,0.07983555274629515,0.16449104863455452,0.03208413501151199],"diagnostics":{"base":0.5279668907606222,"residual":-1.1102230246251565e-16},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.1683494113777338},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.16449104863455452},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.07983555274629515},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.03537153530058595}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.20372094667831975},{"id":"Revolving balances","label":"Revolving balances","weight":0.16449104863455452},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.07983555274629515},{"id":"Installment balances","label":"Installment balances","weight":0.03208413501151199}]},"nonlinear":{"estimated_external_score":60.675005711768094,"probability":0.9293349276391141,"probability_error":0.019780967137560945,"values":[0.03377509076658142,0.2082569285783773,-0.0023653644328668335,-0.024989892442911155,-0.024251293823140844,0.040579798148697235,0.1601144349661586,0.025879274994566194],"diagnostics":{"base":0.5123359508836521,"residual":1.6653345369377348e-16},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.2082569285783773},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.1601144349661586},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.040579798148697235},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.03377509076658142}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.24203201934495872},{"id":"Revolving balances","label":"Revolving balances","weight":0.1601144349661586},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.040579798148697235},{"id":"Installment balances","label":"Installment balances","weight":0.025879274994566194}]}},"comparison":{"probability_difference":0.00033440132029960434,"maximum_original_probability_error":0.02011536845786055,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":8193,"inputs":{"MSinceOldestTradeOpen":38.0,"AverageMInFile":19.0,"NumSatisfactoryTrades":13.0,"PercentTradesNeverDelq":93.0,"MSinceMostRecentDelq":15.0,"NumInqLast6M":1.0,"NetFractionRevolvingBurden":82.0,"NetFractionInstallBurden":91.0},"observed_external_score":58.0,"original_probability":0.954655663685682,"original_shapley":{"values":[0.031571921996015447,0.07864328841349487,0.027876107074549978,0.012975808319390084,0.004843597668739871,-0.0017070179420093953,0.09387935960364248,0.019514408570879743,0.1742862253249367],"diagnostics":{"base":0.5127719646560421,"residual":-5.551115123125783e-17},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.1742862253249367},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.09387935960364248},{"id":"AverageMInFile","label":"Average account age","weight":0.07864328841349487},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.031571921996015447}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.1742862253249367},{"id":"Credit history","label":"Credit history","weight":0.11021521040951032},{"id":"Revolving balances","label":"Revolving balances","weight":0.09387935960364248},{"id":"Repayment history","label":"Repayment history","weight":0.04569551306267994}]},"approximations":{"additive":{"estimated_external_score":57.107905160592196,"probability":0.9534454484312339,"probability_error":0.0012102152544480882,"values":[0.03605035345522786,0.1308715750198432,0.0168554892484255,0.008992587714686744,0.041209010782488315,-0.005051137284274818,0.17181186411655266,0.024738814617662235],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.17181186411655266},{"id":"AverageMInFile","label":"Average account age","weight":0.1308715750198432},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.041209010782488315},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.03605035345522786}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.17181186411655266},{"id":"Credit history","label":"Credit history","weight":0.16692192847507106},{"id":"Repayment history","label":"Repayment history","weight":0.06705708774560055},{"id":"Installment balances","label":"Installment balances","weight":0.024738814617662235}]},"nonlinear":{"estimated_external_score":58.853467052010565,"probability":0.954655663685682,"probability_error":0.0,"values":[0.03233236198628696,0.18124818970672513,0.017438485722681696,0.012074297417847955,-0.001323912531416938,-0.002409453726910494,0.18349796344173458,0.01946178078508059],"diagnostics":{"base":0.5123359508836521,"residual":2.220446049250313e-16},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.18349796344173458},{"id":"AverageMInFile","label":"Average account age","weight":0.18124818970672513},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.03233236198628696},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.01946178078508059}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.2135805516930121},{"id":"Revolving balances","label":"Revolving balances","weight":0.18349796344173458},{"id":"Repayment history","label":"Repayment history","weight":0.028188870609112713},{"id":"Installment balances","label":"Installment balances","weight":0.01946178078508059}]}},"comparison":{"probability_difference":0.0012102152544480882,"maximum_original_probability_error":0.0012102152544480882,"individual":{"retention":0.75,"jaccard":0.6,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":true,"empty":false},"same_order":false}},{"case_id":5738,"inputs":{"MSinceOldestTradeOpen":70.0,"AverageMInFile":39.0,"NumSatisfactoryTrades":10.0,"PercentTradesNeverDelq":83.0,"MSinceMostRecentDelq":19.0,"NumInqLast6M":1.0,"NetFractionRevolvingBurden":89.0,"NetFractionInstallBurden":93.0},"observed_external_score":60.0,"original_probability":0.9443670033866977,"original_shapley":{"values":[0.040404539887936144,0.06946155233308819,0.03757662351166823,0.02636537520918493,0.0037241410205791593,-0.002754788822171164,0.08548860802602307,0.020042291850139893,0.15128669571420697],"diagnostics":{"base":0.5127719646560421,"residual":-1.1102230246251565e-16},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.15128669571420697},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.08548860802602307},{"id":"AverageMInFile","label":"Average account age","weight":0.06946155233308819},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.040404539887936144}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.15128669571420697},{"id":"Credit history","label":"Credit history","weight":0.10986609222102434},{"id":"Revolving balances","label":"Revolving balances","weight":0.08548860802602307},{"id":"Repayment history","label":"Repayment history","weight":0.06766613974143232}]},"approximations":{"additive":{"estimated_external_score":56.1167789689703,"probability":0.9624203722456712,"probability_error":0.018053368858973573,"values":[0.042853319615191475,0.10346452213219934,0.02267917137164038,0.04136158256848287,0.03298018884527511,-0.005043694162430507,0.17016870953016966,0.025989681584520558],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.17016870953016966},{"id":"AverageMInFile","label":"Average account age","weight":0.10346452213219934},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.042853319615191475},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.04136158256848287}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.17016870953016966},{"id":"Credit history","label":"Credit history","weight":0.14631784174739082},{"id":"Repayment history","label":"Repayment history","weight":0.09702094278539836},{"id":"Installment balances","label":"Installment balances","weight":0.025989681584520558}]},"nonlinear":{"estimated_external_score":60.39616018511112,"probability":0.9443670033866975,"probability_error":1.1102230246251565e-16,"values":[0.03758395572575934,0.12345670703401833,0.014157260652096432,0.06584539894588512,0.0011954271417915292,-0.003644844015020692,0.1726549429860741,0.02078220403244131],"diagnostics":{"base":0.5123359508836521,"residual":-1.1102230246251565e-16},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.1726549429860741},{"id":"AverageMInFile","label":"Average account age","weight":0.12345670703401833},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.06584539894588512},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.03758395572575934}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.1726549429860741},{"id":"Credit history","label":"Credit history","weight":0.16104066275977766},{"id":"Repayment history","label":"Repayment history","weight":0.08119808673977309},{"id":"Installment balances","label":"Installment balances","weight":0.02078220403244131}]}},"comparison":{"probability_difference":0.018053368858973684,"maximum_original_probability_error":0.018053368858973573,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":false}},{"case_id":7225,"inputs":{"MSinceOldestTradeOpen":40.0,"AverageMInFile":23.0,"NumSatisfactoryTrades":7.0,"PercentTradesNeverDelq":86.0,"MSinceMostRecentDelq":14.0,"NumInqLast6M":2.0,"NetFractionRevolvingBurden":89.0,"NetFractionInstallBurden":74.0},"observed_external_score":54.0,"original_probability":0.9577355446161281,"original_shapley":{"values":[0.03200150376623601,0.0727365505530128,0.045990095687910446,0.020489437885330967,0.007567882158327943,0.0049740571094289785,0.08409669655801678,0.022489907867019234,0.15461744837480276],"diagnostics":{"base":0.5127719646560421,"residual":0.0},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.15461744837480276},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.08409669655801678},{"id":"AverageMInFile","label":"Average account age","weight":0.0727365505530128},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.045990095687910446}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.15461744837480276},{"id":"Credit history","label":"Credit history","weight":0.10473805431924882},{"id":"Revolving balances","label":"Revolving balances","weight":0.08409669655801678},{"id":"Repayment history","label":"Repayment history","weight":0.07404741573156935}]},"approximations":{"additive":{"estimated_external_score":54.97520004750845,"probability":0.9577355446161281,"probability_error":0.0,"values":[0.03637775890242668,0.10686514949091232,0.03434103542058564,0.026535181622934303,0.03590329763743843,0.006753894370996176,0.15723529160093366,0.025757044809278696],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.15723529160093366},{"id":"AverageMInFile","label":"Average account age","weight":0.10686514949091232},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.03637775890242668},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.03590329763743843}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.15723529160093366},{"id":"Credit history","label":"Credit history","weight":0.143242908393339},{"id":"Repayment history","label":"Repayment history","weight":0.09677951468095838},{"id":"Installment balances","label":"Installment balances","weight":0.025757044809278696}]},"nonlinear":{"estimated_external_score":58.88406248321801,"probability":0.9563805197943887,"probability_error":0.0013550248217394367,"values":[0.03846668455084926,0.12611036110437862,0.029579446332319344,0.03140479351990561,0.01613449208126367,0.009486002975147952,0.16865826862742236,0.024204519719449574],"diagnostics":{"base":0.5123359508836521,"residual":1.1102230246251565e-16},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.16865826862742236},{"id":"AverageMInFile","label":"Average account age","weight":0.12611036110437862},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.03846668455084926},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.03140479351990561}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.16865826862742236},{"id":"Credit history","label":"Credit history","weight":0.16457704565522788},{"id":"Repayment history","label":"Repayment history","weight":0.07711873193348862},{"id":"Installment balances","label":"Installment balances","weight":0.024204519719449574}]}},"comparison":{"probability_difference":0.0013550248217394367,"maximum_original_probability_error":0.0013550248217394367,"individual":{"retention":0.75,"jaccard":0.6,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":false}},{"case_id":7308,"inputs":{"MSinceOldestTradeOpen":51.0,"AverageMInFile":25.0,"NumSatisfactoryTrades":18.0,"PercentTradesNeverDelq":96.0,"MSinceMostRecentDelq":10.0,"NumInqLast6M":16.0,"NetFractionRevolvingBurden":90.0,"NetFractionInstallBurden":82.0},"observed_external_score":60.0,"original_probability":0.9350909796729034,"original_shapley":{"values":[0.03380471505734972,0.0838431359466281,0.01769436791580194,-0.005098678204342019,0.010037759979831267,0.017777568642838523,0.09183444241737897,0.022270261056899016,0.15015544220447583],"diagnostics":{"base":0.5127719646560421,"residual":0.0},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.15015544220447583},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.09183444241737897},{"id":"AverageMInFile","label":"Average account age","weight":0.0838431359466281},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.03380471505734972}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.15015544220447583},{"id":"Credit history","label":"Credit history","weight":0.11764785100397782},{"id":"Revolving balances","label":"Revolving balances","weight":0.09183444241737897},{"id":"Repayment history","label":"Repayment history","weight":0.02263344969129119}]},"approximations":{"additive":{"estimated_external_score":48.72801270939796,"probability":0.9488440895335889,"probability_error":0.013753109860685453,"values":[0.03769236483266328,0.10369717846467039,0.012426355900328288,-0.007831045857101024,0.038364604309147836,0.05905793226059519,0.14903525606800383,0.02843455279465888],"diagnostics":{"base":0.5279668907606222,"residual":5.551115123125783e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.14903525606800383},{"id":"AverageMInFile","label":"Average account age","weight":0.10369717846467039},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.05905793226059519},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.038364604309147836}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.14903525606800383},{"id":"Credit history","label":"Credit history","weight":0.14138954329733366},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.05905793226059519},{"id":"Repayment history","label":"Repayment history","weight":0.0429599143523751}]},"nonlinear":{"estimated_external_score":56.342563921474095,"probability":0.9491014990300506,"probability_error":0.014010519357147144,"values":[0.03324763750655722,0.1341271811284213,0.010569320596010463,-0.0055732101984928666,0.04001279266184791,0.035160111723713874,0.1632076421492517,0.026014072579088835],"diagnostics":{"base":0.5123359508836521,"residual":2.220446049250313e-16},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.1632076421492517},{"id":"AverageMInFile","label":"Average account age","weight":0.1341271811284213},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.04001279266184791},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.035160111723713874}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.16737481863497855},{"id":"Revolving balances","label":"Revolving balances","weight":0.1632076421492517},{"id":"Repayment history","label":"Repayment history","weight":0.0450089030593655},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.035160111723713874}]}},"comparison":{"probability_difference":0.0002574094964616913,"maximum_original_probability_error":0.014010519357147144,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":true,"empty":false},"same_order":false}},{"case_id":8891,"inputs":{"MSinceOldestTradeOpen":61.0,"AverageMInFile":30.0,"NumSatisfactoryTrades":9.0,"PercentTradesNeverDelq":89.0,"MSinceMostRecentDelq":23.0,"NumInqLast6M":2.0,"NetFractionRevolvingBurden":98.0,"NetFractionInstallBurden":89.0},"observed_external_score":57.0,"original_probability":0.9550746488827793,"original_shapley":{"values":[0.035782045123319416,0.07349737180466857,0.04034412904203533,0.01669388276186781,-8.088379727632115e-05,0.00502112243101545,0.0866899478592192,0.023416988836842064,0.16093808016504552],"diagnostics":{"base":0.5127719646560421,"residual":1.1102230246251565e-16},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.16093808016504552},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.0866899478592192},{"id":"AverageMInFile","label":"Average account age","weight":0.07349737180466857},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.04034412904203533}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.16093808016504552},{"id":"Credit history","label":"Credit history","weight":0.10927941692798798},{"id":"Revolving balances","label":"Revolving balances","weight":0.0866899478592192},{"id":"Repayment history","label":"Repayment history","weight":0.05695712800662682}]},"approximations":{"additive":{"estimated_external_score":55.454001545288435,"probability":0.9573621111811889,"probability_error":0.0022874622984095927,"values":[0.03934733374692979,0.10999446635135869,0.029214389366767082,0.01836641276739737,0.0179573796713907,0.006027377180641075,0.17935248342959195,0.029135377906489988],"diagnostics":{"base":0.5279668907606222,"residual":1.1102230246251565e-16},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.17935248342959195},{"id":"AverageMInFile","label":"Average account age","weight":0.10999446635135869},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.03934733374692979},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.029214389366767082}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.17935248342959195},{"id":"Credit history","label":"Credit history","weight":0.14934180009828849},{"id":"Repayment history","label":"Repayment history","weight":0.06553818180555515},{"id":"Installment balances","label":"Installment balances","weight":0.029135377906489988}]},"nonlinear":{"estimated_external_score":60.76608605531169,"probability":0.9415453183651862,"probability_error":0.013529330517593086,"values":[0.035435884425419936,0.140312864402573,0.02266926671754471,0.03019158584807722,-0.003355376836473375,0.008332872891983091,0.171272756571973,0.024349513460436328],"diagnostics":{"base":0.5123359508836521,"residual":1.1102230246251565e-16},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.171272756571973},{"id":"AverageMInFile","label":"Average account age","weight":0.140312864402573},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.035435884425419936},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.03019158584807722}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.17574874882799296},{"id":"Revolving balances","label":"Revolving balances","weight":0.171272756571973},{"id":"Repayment history","label":"Repayment history","weight":0.049505475729148554},{"id":"Installment balances","label":"Installment balances","weight":0.024349513460436328}]}},"comparison":{"probability_difference":0.01581679281600268,"maximum_original_probability_error":0.013529330517593086,"individual":{"retention":0.75,"jaccard":0.6,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":true,"empty":false},"same_order":false}},{"case_id":9811,"inputs":{"MSinceOldestTradeOpen":45.0,"AverageMInFile":33.0,"NumSatisfactoryTrades":5.0,"PercentTradesNeverDelq":80.0,"MSinceMostRecentDelq":9.0,"NumInqLast6M":4.0,"NetFractionRevolvingBurden":83.0,"NetFractionInstallBurden":-8.0},"observed_external_score":57.0,"original_probability":0.9544816852166501,"original_shapley":{"values":[0.031897746513552136,0.06648298027379605,0.053252751028425686,0.030174865475692605,0.010787099362985449,0.008615219856843775,0.08058306215373481,0.016145251757529802,0.14377074413804744],"diagnostics":{"base":0.5127719646560421,"residual":1.1102230246251565e-16},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.14377074413804744},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.08058306215373481},{"id":"AverageMInFile","label":"Average account age","weight":0.06648298027379605},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.053252751028425686}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.14377074413804744},{"id":"Credit history","label":"Credit history","weight":0.09838072678734819},{"id":"Repayment history","label":"Repayment history","weight":0.09421471586710373},{"id":"Revolving balances","label":"Revolving balances","weight":0.08058306215373481}]},"approximations":{"additive":{"estimated_external_score":54.70865994025356,"probability":0.9571858141817267,"probability_error":0.0027041289650765465,"values":[0.03528592432083859,0.09052962150061139,0.03860194647750449,0.04664316322705649,0.04356025554579717,0.017387179221689274,0.14134488960643254,0.01586594352117459],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.14134488960643254},{"id":"AverageMInFile","label":"Average account age","weight":0.09052962150061139},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.04664316322705649},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.04356025554579717}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.14134488960643254},{"id":"Repayment history","label":"Repayment history","weight":0.12880536525035816},{"id":"Credit history","label":"Credit history","weight":0.12581554582145},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.017387179221689274}]},"nonlinear":{"estimated_external_score":58.602702485384555,"probability":0.9566009943668864,"probability_error":0.0021193091502362815,"values":[0.03202925603575857,0.09525564882228978,0.03717691571511148,0.05652024830907305,0.03663172941513628,0.019938776667806874,0.14734988580376956,0.01936258271428866],"diagnostics":{"base":0.5123359508836521,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.14734988580376956},{"id":"AverageMInFile","label":"Average account age","weight":0.09525564882228978},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.05652024830907305},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.03717691571511148}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.14734988580376956},{"id":"Repayment history","label":"Repayment history","weight":0.1303288934393208},{"id":"Credit history","label":"Credit history","weight":0.12728490485804833},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.019938776667806874}]}},"comparison":{"probability_difference":0.000584819814840265,"maximum_original_probability_error":0.0027041289650765465,"individual":{"retention":0.75,"jaccard":0.6,"turnover":true,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":false}},{"case_id":7112,"inputs":{"MSinceOldestTradeOpen":147.0,"AverageMInFile":43.0,"NumSatisfactoryTrades":15.0,"PercentTradesNeverDelq":71.0,"MSinceMostRecentDelq":6.0,"NumInqLast6M":0.0,"NetFractionRevolvingBurden":91.0,"NetFractionInstallBurden":76.0},"observed_external_score":47.0,"original_probability":0.9414989517895782,"original_shapley":{"values":[0.01331723398479624,0.061278856873449354,0.0216142026817965,0.0364015273738929,0.013869711305957986,-0.006800619992055104,0.08809089862152855,0.022192607541942685,0.17876256874222687],"diagnostics":{"base":0.5127719646560421,"residual":2.220446049250313e-16},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.17876256874222687},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.08809089862152855},{"id":"AverageMInFile","label":"Average account age","weight":0.061278856873449354},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.0364015273738929}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.17876256874222687},{"id":"Revolving balances","label":"Revolving balances","weight":0.08809089862152855},{"id":"Credit history","label":"Credit history","weight":0.07459609085824559},{"id":"Repayment history","label":"Repayment history","weight":0.07188544136164739}]},"approximations":{"additive":{"estimated_external_score":52.17584836431968,"probability":0.9414989517895782,"probability_error":0.0,"values":[0.01684372067959078,0.08930935985498552,0.014745394654068973,0.06856846715118685,0.05186827253439237,-0.013804312633469906,0.16019783916945532,0.025803319618746257],"diagnostics":{"base":0.5279668907606222,"residual":-5.551115123125783e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.16019783916945532},{"id":"AverageMInFile","label":"Average account age","weight":0.08930935985498552},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.06856846715118685},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.05186827253439237}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.16019783916945532},{"id":"Repayment history","label":"Repayment history","weight":0.1351821343396482},{"id":"Credit history","label":"Credit history","weight":0.1061530805345763},{"id":"Installment balances","label":"Installment balances","weight":0.025803319618746257}]},"nonlinear":{"estimated_external_score":56.340710289222116,"probability":0.9417563612860401,"probability_error":0.00025740949646191336,"values":[0.01735058115957331,0.09388412283547161,0.010738277982931697,0.08848684651184371,0.05302116263184329,-0.01423971428346734,0.15554601668305157,0.024633116881140126],"diagnostics":{"base":0.5123359508836521,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.15554601668305157},{"id":"AverageMInFile","label":"Average account age","weight":0.09388412283547161},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.08848684651184371},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.05302116263184329}],"grouped":[{"id":"Revolving balances","label":"Revolving balances","weight":0.15554601668305157},{"id":"Repayment history","label":"Repayment history","weight":0.1522462871266187},{"id":"Credit history","label":"Credit history","weight":0.11123470399504493},{"id":"Installment balances","label":"Installment balances","weight":0.024633116881140126}]}},"comparison":{"probability_difference":0.00025740949646191336,"maximum_original_probability_error":0.00025740949646191336,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":6611,"inputs":{"MSinceOldestTradeOpen":56.0,"AverageMInFile":35.0,"NumSatisfactoryTrades":4.0,"PercentTradesNeverDelq":75.0,"MSinceMostRecentDelq":2.0,"NumInqLast6M":4.0,"NetFractionRevolvingBurden":68.0,"NetFractionInstallBurden":69.0},"observed_external_score":45.0,"original_probability":0.9439581299382376,"original_shapley":{"values":[0.030604539787731644,0.06164665428886304,0.05357291035488469,0.03369495555074768,0.00840230636386732,0.010262565022329508,0.06725903065489867,0.01225376038375798,0.15348944287511507],"diagnostics":{"base":0.5127719646560421,"residual":-5.551115123125783e-17},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.15348944287511507},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.06725903065489867},{"id":"AverageMInFile","label":"Average account age","weight":0.06164665428886304},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.05357291035488469}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.15348944287511507},{"id":"Repayment history","label":"Repayment history","weight":0.0956701722694997},{"id":"Credit history","label":"Credit history","weight":0.09225119407659468},{"id":"Revolving balances","label":"Revolving balances","weight":0.06725903065489867}]},"approximations":{"additive":{"estimated_external_score":54.80635247138455,"probability":0.9439581299382376,"probability_error":0.0,"values":[0.03416445413129826,0.08843478002881333,0.037095932340679365,0.06194849506376501,0.05538742959739452,0.020354327116045064,0.10622136203726742,0.012384458862352524],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.10622136203726742},{"id":"AverageMInFile","label":"Average account age","weight":0.08843478002881333},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.06194849506376501},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.05538742959739452}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.1544318570018389},{"id":"Credit history","label":"Credit history","weight":0.12259923416011159},{"id":"Revolving balances","label":"Revolving balances","weight":0.10622136203726742},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.020354327116045064}]},"nonlinear":{"estimated_external_score":56.178157050268084,"probability":0.9442155394346995,"probability_error":0.00025740949646191336,"values":[0.03311162525836621,0.08703355636314665,0.03116151817559186,0.07908762753405682,0.06576632898510713,0.022589978733927384,0.09884140429729608,0.014287549203555249],"diagnostics":{"base":0.5123359508836521,"residual":-5.551115123125783e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.09884140429729608},{"id":"AverageMInFile","label":"Average account age","weight":0.08703355636314665},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.07908762753405682},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.06576632898510713}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.1760154746947558},{"id":"Credit history","label":"Credit history","weight":0.12014518162151286},{"id":"Revolving balances","label":"Revolving balances","weight":0.09884140429729608},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.022589978733927384}]}},"comparison":{"probability_difference":0.00025740949646191336,"maximum_original_probability_error":0.00025740949646191336,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":9808,"inputs":{"MSinceOldestTradeOpen":45.0,"AverageMInFile":19.0,"NumSatisfactoryTrades":10.0,"PercentTradesNeverDelq":100.0,"MSinceMostRecentDelq":-7.0,"NumInqLast6M":1.0,"NetFractionRevolvingBurden":83.0,"NetFractionInstallBurden":95.0},"observed_external_score":67.0,"original_probability":0.8957308907323605,"original_shapley":{"values":[0.0253567531869101,0.10351726621792424,0.05491758782835344,-0.007511893925144301,0.0037938440579836584,-0.0045032015238857005,0.12222140827485244,0.017298730338842324,0.0678684316204822],"diagnostics":{"base":0.5127719646560421,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.12222140827485244},{"id":"AverageMInFile","label":"Average account age","weight":0.10351726621792424},{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.0678684316204822},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.05491758782835344}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.12887401940483434},{"id":"Revolving balances","label":"Revolving balances","weight":0.12222140827485244},{"id":"External risk score","label":"External risk score","weight":0.0678684316204822},{"id":"Repayment history","label":"Repayment history","weight":0.051199537961192795}]},"approximations":{"additive":{"estimated_external_score":65.72856439001191,"probability":0.9050570204630816,"probability_error":0.009326129730721111,"values":[0.034697692769709056,0.16614885418404013,0.027164254017170472,-0.0245818031092725,-0.0426920184732795,-0.007552662223328069,0.1981081477565354,0.025797664780884445],"diagnostics":{"base":0.5279668907606222,"residual":-5.551115123125783e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.1981081477565354},{"id":"AverageMInFile","label":"Average account age","weight":0.16614885418404013},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.034697692769709056},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.027164254017170472}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.20084654695374918},{"id":"Revolving balances","label":"Revolving balances","weight":0.1981081477565354},{"id":"Installment balances","label":"Installment balances","weight":0.025797664780884445}]},"nonlinear":{"estimated_external_score":60.97998370158954,"probability":0.9465672394875481,"probability_error":0.05083634875518761,"values":[0.0333367477767557,0.21083988724950936,0.02176784919467008,-0.024474950373409823,-0.02419198222161769,-0.004725730485243813,0.1994952414788969,0.02218422598433501],"diagnostics":{"base":0.5123359508836522,"residual":5.551115123125783e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.21083988724950936},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.1994952414788969},{"id":"MSinceOldestTradeOpen","label":"Oldest account age","weight":0.0333367477767557},{"id":"NetFractionInstallBurden","label":"Installment balance ratio","weight":0.02218422598433501}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.24417663502626508},{"id":"Revolving balances","label":"Revolving balances","weight":0.1994952414788969},{"id":"Installment balances","label":"Installment balances","weight":0.02218422598433501}]}},"comparison":{"probability_difference":0.0415102190244665,"maximum_original_probability_error":0.05083634875518761,"individual":{"retention":0.75,"jaccard":0.6,"turnover":true,"first_changed":true,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":false}},{"case_id":9541,"inputs":{"MSinceOldestTradeOpen":48.0,"AverageMInFile":25.0,"NumSatisfactoryTrades":7.0,"PercentTradesNeverDelq":50.0,"MSinceMostRecentDelq":2.0,"NumInqLast6M":0.0,"NetFractionRevolvingBurden":75.0,"NetFractionInstallBurden":100.0},"observed_external_score":56.0,"original_probability":0.9455776435479892,"original_shapley":{"values":[0.032078746387825575,0.07209217937545066,0.0471404664735436,0.032500205398311166,0.00744631043446328,-0.008563026668585832,0.07232151076346437,0.01752477777069995,0.16026450895677435],"diagnostics":{"base":0.5127719646560421,"residual":0.0},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.16026450895677435},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.07232151076346437},{"id":"AverageMInFile","label":"Average account age","weight":0.07209217937545066},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.0471404664735436}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.16026450895677435},{"id":"Credit history","label":"Credit history","weight":0.10417092576327625},{"id":"Repayment history","label":"Repayment history","weight":0.08708698230631805},{"id":"Revolving balances","label":"Revolving balances","weight":0.07232151076346437}]},"approximations":{"additive":{"estimated_external_score":49.16382441540567,"probability":0.9453202340515271,"probability_error":0.0002574094964621354,"values":[0.03413069876759359,0.09678571403384867,0.039206123762962,0.08537472558222534,0.04099531357222239,-0.013521770059022217,0.11272289081767572,0.021659646813399585],"diagnostics":{"base":0.5279668907606222,"residual":5.551115123125783e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.11272289081767572},{"id":"AverageMInFile","label":"Average account age","weight":0.09678571403384867},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.08537472558222534},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.04099531357222239}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.16557616291740973},{"id":"Credit history","label":"Credit history","weight":0.13091641280144226},{"id":"Revolving balances","label":"Revolving balances","weight":0.11272289081767572},{"id":"Installment balances","label":"Installment balances","weight":0.021659646813399585}]},"nonlinear":{"estimated_external_score":55.21737178384444,"probability":0.9453202340515273,"probability_error":0.00025740949646191336,"values":[0.03269579941449803,0.10867527394685336,0.032316977674777064,0.08662789239813402,0.05801578573766426,-0.014065688929655595,0.11262957801177244,0.016088664913831492],"diagnostics":{"base":0.5123359508836521,"residual":5.551115123125783e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.11262957801177244},{"id":"AverageMInFile","label":"Average account age","weight":0.10867527394685336},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.08662789239813402},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.05801578573766426}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.17696065581057535},{"id":"Credit history","label":"Credit history","weight":0.1413710733613514},{"id":"Revolving balances","label":"Revolving balances","weight":0.11262957801177244},{"id":"Installment balances","label":"Installment balances","weight":0.016088664913831492}]}},"comparison":{"probability_difference":2.220446049250313e-16,"maximum_original_probability_error":0.0002574094964621354,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":1897,"inputs":{"MSinceOldestTradeOpen":44.0,"AverageMInFile":30.0,"NumSatisfactoryTrades":5.0,"PercentTradesNeverDelq":80.0,"MSinceMostRecentDelq":9.0,"NumInqLast6M":9.0,"NetFractionRevolvingBurden":68.0,"NetFractionInstallBurden":85.0},"observed_external_score":63.0,"original_probability":0.9423043721650649,"original_shapley":{"values":[0.03939323559159026,0.07266471690678802,0.05728954984182856,0.0294914662355191,0.013580415437279206,0.019972992152391317,0.05240530762575011,0.022635595458415196,0.12209912825946113],"diagnostics":{"base":0.5127719646560421,"residual":-1.1102230246251565e-16},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.12209912825946113},{"id":"AverageMInFile","label":"Average account age","weight":0.07266471690678802},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.05728954984182856},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.05240530762575011}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.12209912825946113},{"id":"Credit history","label":"Credit history","weight":0.11205795249837829},{"id":"Repayment history","label":"Repayment history","weight":0.10036143151462687},{"id":"Revolving balances","label":"Revolving balances","weight":0.05240530762575011}]},"approximations":{"additive":{"estimated_external_score":53.76900785568252,"probability":0.9642616811210024,"probability_error":0.021957308955937505,"values":[0.03880189415205402,0.0965819547766047,0.03899499960363292,0.045383920062067214,0.04684180054496003,0.045566970610385976,0.09842028204996507,0.025702968560710214],"diagnostics":{"base":0.5279668907606222,"residual":1.1102230246251565e-16},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.09842028204996507},{"id":"AverageMInFile","label":"Average account age","weight":0.0965819547766047},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.04684180054496003},{"id":"NumInqLast6M","label":"Recent inquiries","weight":0.045566970610385976}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.1353838489286587},{"id":"Repayment history","label":"Repayment history","weight":0.13122072021066017},{"id":"Revolving balances","label":"Revolving balances","weight":0.09842028204996507},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.045566970610385976}]},"nonlinear":{"estimated_external_score":59.09089799608805,"probability":0.9624899896325962,"probability_error":0.020185617467531336,"values":[0.03460402490126499,0.1084348119893568,0.03899426986357391,0.06080553102311695,0.042801091696423044,0.0359003676015806,0.10315635510675854,0.025457586566869316],"diagnostics":{"base":0.5123359508836521,"residual":5.551115123125783e-17},"individual":[{"id":"AverageMInFile","label":"Average account age","weight":0.1084348119893568},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.10315635510675854},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.06080553102311695},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.042801091696423044}],"grouped":[{"id":"Credit history","label":"Credit history","weight":0.1430388368906218},{"id":"Repayment history","label":"Repayment history","weight":0.1426008925831139},{"id":"Revolving balances","label":"Revolving balances","weight":0.10315635510675854},{"id":"Recent inquiries","label":"Recent inquiries","weight":0.0359003676015806}]}},"comparison":{"probability_difference":0.0017716914884061685,"maximum_original_probability_error":0.021957308955937505,"individual":{"retention":0.75,"jaccard":0.6,"turnover":true,"first_changed":true,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":false}},{"case_id":7124,"inputs":{"MSinceOldestTradeOpen":137.0,"AverageMInFile":46.0,"NumSatisfactoryTrades":5.0,"PercentTradesNeverDelq":55.0,"MSinceMostRecentDelq":3.0,"NumInqLast6M":3.0,"NetFractionRevolvingBurden":94.0,"NetFractionInstallBurden":88.0},"observed_external_score":43.0,"original_probability":0.9478865672312006,"original_shapley":{"values":[0.015493734834775046,0.05999149118895743,0.052434483590642955,0.03146240943579262,0.010297919443314336,0.006787638717339554,0.08250492635098322,0.02201851299129683,0.15412348602205658],"diagnostics":{"base":0.5127719646560421,"residual":-1.1102230246251565e-16},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.15412348602205658},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.08250492635098322},{"id":"AverageMInFile","label":"Average account age","weight":0.05999149118895743},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.052434483590642955}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.15412348602205658},{"id":"Repayment history","label":"Repayment history","weight":0.09419481246974991},{"id":"Revolving balances","label":"Revolving balances","weight":0.08250492635098322},{"id":"Credit history","label":"Credit history","weight":0.07548522602373248}]},"approximations":{"additive":{"estimated_external_score":47.69350527037258,"probability":0.9478865672312006,"probability_error":0.0,"values":[0.01857103345072975,0.07827987168119283,0.03725471344994243,0.07328448557880768,0.04066714227935699,0.011857171803560626,0.132351168484186,0.02765408974280209],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.132351168484186},{"id":"AverageMInFile","label":"Average account age","weight":0.07827987168119283},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.07328448557880768},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.04066714227935699}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.15120634130810712},{"id":"Revolving balances","label":"Revolving balances","weight":0.132351168484186},{"id":"Credit history","label":"Credit history","weight":0.09685090513192257},{"id":"Installment balances","label":"Installment balances","weight":0.02765408974280209}]},"nonlinear":{"estimated_external_score":54.82984138853462,"probability":0.9478865672312006,"probability_error":0.0,"values":[0.020003793582336706,0.08007828265659479,0.036221779813550785,0.07965338896306927,0.055455076027105726,0.00940496773059711,0.12909674137883032,0.02563658619546394],"diagnostics":{"base":0.5123359508836521,"residual":-1.1102230246251565e-16},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.12909674137883032},{"id":"AverageMInFile","label":"Average account age","weight":0.08007828265659479},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.07965338896306927},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.055455076027105726}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.17133024480372577},{"id":"Revolving balances","label":"Revolving balances","weight":0.12909674137883032},{"id":"Credit history","label":"Credit history","weight":0.1000820762389315},{"id":"Installment balances","label":"Installment balances","weight":0.02563658619546394}]}},"comparison":{"probability_difference":0.0,"maximum_original_probability_error":0.0,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":2040,"inputs":{"MSinceOldestTradeOpen":91.0,"AverageMInFile":27.0,"NumSatisfactoryTrades":7.0,"PercentTradesNeverDelq":75.0,"MSinceMostRecentDelq":3.0,"NumInqLast6M":4.0,"NetFractionRevolvingBurden":76.0,"NetFractionInstallBurden":89.0},"observed_external_score":51.0,"original_probability":0.9618851611845816,"original_shapley":{"values":[0.029595342754331008,0.06975791460728413,0.04796919696322558,0.0313157038673344,0.009438775364616889,0.010617982072394774,0.07665017206030966,0.020711116521969088,0.153056992317074],"diagnostics":{"base":0.5127719646560421,"residual":0.0},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.153056992317074},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.07665017206030966},{"id":"AverageMInFile","label":"Average account age","weight":0.06975791460728413},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.04796919696322558}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.153056992317074},{"id":"Credit history","label":"Credit history","weight":0.09935325736161514},{"id":"Repayment history","label":"Repayment history","weight":0.08872367619517686},{"id":"Revolving balances","label":"Revolving balances","weight":0.07665017206030966}]},"approximations":{"additive":{"estimated_external_score":52.66984850692366,"probability":0.9618851611845817,"probability_error":1.1102230246251565e-16,"values":[0.03210443160114057,0.09773464519143948,0.03585623315461439,0.05513740154509713,0.0450205763315748,0.018368452589397344,0.12370864645506859,0.025987883555626975],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.12370864645506859},{"id":"AverageMInFile","label":"Average account age","weight":0.09773464519143948},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.05513740154509713},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.0450205763315748}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.13601421103128633},{"id":"Credit history","label":"Credit history","weight":0.12983907679258005},{"id":"Revolving balances","label":"Revolving balances","weight":0.12370864645506859},{"id":"Installment balances","label":"Installment balances","weight":0.025987883555626975}]},"nonlinear":{"estimated_external_score":54.80857861543665,"probability":0.9618851611845817,"probability_error":1.1102230246251565e-16,"values":[0.032827091929477914,0.10131382417043014,0.03031598038941833,0.06917249170894342,0.055545167405290446,0.019224343373689372,0.11714064395945481,0.024009667364225037],"diagnostics":{"base":0.5123359508836522,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.11714064395945481},{"id":"AverageMInFile","label":"Average account age","weight":0.10131382417043014},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.06917249170894342},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.055545167405290446}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.1550336395036522},{"id":"Credit history","label":"Credit history","weight":0.13414091609990805},{"id":"Revolving balances","label":"Revolving balances","weight":0.11714064395945481},{"id":"Installment balances","label":"Installment balances","weight":0.024009667364225037}]}},"comparison":{"probability_difference":0.0,"maximum_original_probability_error":1.1102230246251565e-16,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}},{"case_id":10057,"inputs":{"MSinceOldestTradeOpen":62.0,"AverageMInFile":36.0,"NumSatisfactoryTrades":4.0,"PercentTradesNeverDelq":50.0,"MSinceMostRecentDelq":6.0,"NumInqLast6M":2.0,"NetFractionRevolvingBurden":86.0,"NetFractionInstallBurden":-8.0},"observed_external_score":60.0,"original_probability":0.9378055498157813,"original_shapley":{"values":[0.03116641453482142,0.06358006449785716,0.05434238573123098,0.033981884852090416,0.011679241909087907,0.00423976039403795,0.07183914705398212,0.018900257713146792,0.13530442847348437],"diagnostics":{"base":0.5127719646560421,"residual":1.1102230246251565e-16},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.13530442847348437},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.07183914705398212},{"id":"AverageMInFile","label":"Average account age","weight":0.06358006449785716},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.05434238573123098}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.13530442847348437},{"id":"Repayment history","label":"Repayment history","weight":0.10000351249240931},{"id":"Credit history","label":"Credit history","weight":0.09474647903267858},{"id":"Revolving balances","label":"Revolving balances","weight":0.07183914705398212}]},"approximations":{"additive":{"estimated_external_score":48.95808399898118,"probability":0.9566009485836996,"probability_error":0.018795398767918248,"values":[0.033067932760949534,0.0835843365580899,0.03664564378153137,0.08565422115031798,0.041722373837065486,0.0061087647765895306,0.12363959482373313,0.018211190134800197],"diagnostics":{"base":0.5279668907606222,"residual":1.6653345369377348e-16},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.12363959482373313},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.08565422115031798},{"id":"AverageMInFile","label":"Average account age","weight":0.0835843365580899},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.041722373837065486}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.16402223876891484},{"id":"Revolving balances","label":"Revolving balances","weight":0.12363959482373313},{"id":"Credit history","label":"Credit history","weight":0.11665226931903944},{"id":"Installment balances","label":"Installment balances","weight":0.018211190134800197}]},"nonlinear":{"estimated_external_score":57.828363519833765,"probability":0.9530887820829235,"probability_error":0.015283232267142122,"values":[0.03153636699398922,0.09104293047130745,0.02983954090915413,0.09095184080338027,0.044242408814912884,0.005627081725410358,0.1275884777191364,0.019924183761980626],"diagnostics":{"base":0.5123359508836521,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.1275884777191364},{"id":"AverageMInFile","label":"Average account age","weight":0.09104293047130745},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.09095184080338027},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.044242408814912884}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.16503379052744727},{"id":"Revolving balances","label":"Revolving balances","weight":0.1275884777191364},{"id":"Credit history","label":"Credit history","weight":0.12257929746529667},{"id":"Installment balances","label":"Installment balances","weight":0.019924183761980626}]}},"comparison":{"probability_difference":0.003512166500776126,"maximum_original_probability_error":0.018795398767918248,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":false}},{"case_id":8413,"inputs":{"MSinceOldestTradeOpen":51.0,"AverageMInFile":34.0,"NumSatisfactoryTrades":7.0,"PercentTradesNeverDelq":63.0,"MSinceMostRecentDelq":3.0,"NumInqLast6M":1.0,"NetFractionRevolvingBurden":86.0,"NetFractionInstallBurden":88.0},"observed_external_score":54.0,"original_probability":0.9590132457085171,"original_shapley":{"values":[0.030585356744858096,0.06876907861364986,0.046574924298393924,0.030830160702376264,0.0102953443209137,-0.0021533647289985703,0.07961831269981674,0.020800558057154174,0.16092091034431072],"diagnostics":{"base":0.5127719646560421,"residual":5.551115123125783e-17},"individual":[{"id":"ExternalRiskEstimate","label":"External risk estimate","weight":0.16092091034431072},{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.07961831269981674},{"id":"AverageMInFile","label":"Average account age","weight":0.06876907861364986},{"id":"NumSatisfactoryTrades","label":"Satisfactory accounts","weight":0.046574924298393924}],"grouped":[{"id":"External risk score","label":"External risk score","weight":0.16092091034431072},{"id":"Credit history","label":"Credit history","weight":0.09935443535850796},{"id":"Repayment history","label":"Repayment history","weight":0.08770042932168388},{"id":"Revolving balances","label":"Revolving balances","weight":0.07961831269981674}]},"approximations":{"additive":{"estimated_external_score":50.43429179245495,"probability":0.9590132457085171,"probability_error":0.0,"values":[0.03472821147181343,0.09120575875663736,0.03565834801854259,0.07014098408903509,0.043212995124394554,-0.004435104789302755,0.1339501474105354,0.026585014866238977],"diagnostics":{"base":0.5279668907606222,"residual":0.0},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.1339501474105354},{"id":"AverageMInFile","label":"Average account age","weight":0.09120575875663736},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.07014098408903509},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.043212995124394554}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.14901232723197222},{"id":"Revolving balances","label":"Revolving balances","weight":0.1339501474105354},{"id":"Credit history","label":"Credit history","weight":0.1259339702284508},{"id":"Installment balances","label":"Installment balances","weight":0.026585014866238977}]},"nonlinear":{"estimated_external_score":54.42689929805564,"probability":0.9590132457085171,"probability_error":0.0,"values":[0.03280261079457812,0.0967476088753323,0.02922781317583924,0.0788239454430767,0.05715801057726879,-0.0025888320881576163,0.13053971334482076,0.023966424702106595],"diagnostics":{"base":0.5123359508836521,"residual":5.551115123125783e-17},"individual":[{"id":"NetFractionRevolvingBurden","label":"Revolving balance ratio","weight":0.13053971334482076},{"id":"AverageMInFile","label":"Average account age","weight":0.0967476088753323},{"id":"PercentTradesNeverDelq","label":"Never-delinquent share","weight":0.0788239454430767},{"id":"MSinceMostRecentDelq","label":"Time since delinquency","weight":0.05715801057726879}],"grouped":[{"id":"Repayment history","label":"Repayment history","weight":0.16520976919618474},{"id":"Revolving balances","label":"Revolving balances","weight":0.13053971334482076},{"id":"Credit history","label":"Credit history","weight":0.12955021966991043},{"id":"Installment balances","label":"Installment balances","weight":0.023966424702106595}]}},"comparison":{"probability_difference":0.0,"maximum_original_probability_error":0.0,"individual":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"grouped":{"retention":1.0,"jaccard":1.0,"turnover":false,"first_changed":false,"empty":false},"same_order":true}}],"bands":[{"probability_band":0.005,"approximate_outputs_close":21,"both_close_to_original":12,"different_factor_sets_when_outputs_close":8,"different_factor_sets_when_both_close_to_original":4,"different_concept_sets_when_both_close_to_original":0},{"probability_band":0.01,"approximate_outputs_close":25,"both_close_to_original":12,"different_factor_sets_when_outputs_close":10,"different_factor_sets_when_both_close_to_original":4,"different_concept_sets_when_both_close_to_original":0},{"probability_band":0.02,"approximate_outputs_close":30,"both_close_to_original":21,"different_factor_sets_when_outputs_close":12,"different_factor_sets_when_both_close_to_original":6,"different_concept_sets_when_both_close_to_original":2},{"probability_band":0.05,"approximate_outputs_close":36,"both_close_to_original":28,"different_factor_sets_when_outputs_close":15,"different_factor_sets_when_both_close_to_original":9,"different_concept_sets_when_both_close_to_original":4}],"summary":{"audit_records":40,"different_factor_sets":17,"different_concept_sets":10,"test_probability_difference_mean":0.03108152314431477},"featured_case":8193,"featured_selection":"Record 8193 selected after inspection to illustrate detailed-factor changes with close original-prediction agreement and unchanged broader topics. See all forty records; the example is not representative.","background_record_ids":[4998,7409,4081,1002,6793,8931,2060,10124,3371,6905,6754,2185,4027,9004,9659,8726,7665,8317,3928,282,2174,9498,7589,812],"limitations":["Original external-score mechanism is unavailable.","Both explanations describe reconstructed systems, not authenticated original upstream factors.","References and feature dependence affect attributions.","Earlier test results and feasibility results were known; evaluation is exploratory.","Forty audited records are deliberately selected, not representative of lender decisions."]},"labels":{"MSinceOldestTradeOpen":"Oldest account age","AverageMInFile":"Average account age","NumSatisfactoryTrades":"Accounts with on-time payments","PercentTradesNeverDelq":"Accounts never delinquent","MSinceMostRecentDelq":"Time since most recent delinquency","NumInqLast6M":"Recent credit inquiries","NetFractionRevolvingBurden":"Revolving balance / credit limit","NetFractionInstallBurden":"Installment balance / original amount","ExternalRiskEstimate":"Externally supplied score"},"groups":{"MSinceOldestTradeOpen":"Credit history","AverageMInFile":"Credit history","NumSatisfactoryTrades":"Repayment history","PercentTradesNeverDelq":"Repayment history","MSinceMostRecentDelq":"Repayment history","NumInqLast6M":"Recent inquiries","NetFractionRevolvingBurden":"Revolving balances","NetFractionInstallBurden":"Installment balances","ExternalRiskEstimate":"External risk score"},"local":{"7240":[{"seed":0,"heldout_r2":0.2848522786582177,"heldout_mae":0.09017333490422284,"point_error":0.02836200906122771},{"seed":1,"heldout_r2":0.35999061775789154,"heldout_mae":0.08700560541154681,"point_error":0.027119778311529674},{"seed":2,"heldout_r2":0.3020536504526583,"heldout_mae":0.08856917360887447,"point_error":0.04274894471928181},{"seed":3,"heldout_r2":0.36790196839850886,"heldout_mae":0.08690984319937134,"point_error":0.03211914650865266},{"seed":4,"heldout_r2":0.36985232720593175,"heldout_mae":0.08668931826102555,"point_error":0.03564345688766335}],"9178":[{"seed":0,"heldout_r2":0.42534576617199205,"heldout_mae":0.07347012463929853,"point_error":0.03625718802448774},{"seed":1,"heldout_r2":0.3963087059115794,"heldout_mae":0.07224248659778475,"point_error":0.030647733155270585},{"seed":2,"heldout_r2":0.46454859883590105,"heldout_mae":0.07083920527474735,"point_error":0.036013023111766795},{"seed":3,"heldout_r2":0.43515873374548053,"heldout_mae":0.07144656491254008,"point_error":0.027133764955126427},{"seed":4,"heldout_r2":0.40002535132667916,"heldout_mae":0.07032387901370947,"point_error":0.024715562361395604}],"654":[{"seed":0,"heldout_r2":0.3451501935973291,"heldout_mae":0.07657121737814007,"point_error":0.012822331461080083},{"seed":1,"heldout_r2":0.35249074362291943,"heldout_mae":0.07497804907177602,"point_error":0.00861398705456845},{"seed":2,"heldout_r2":0.2844490921927396,"heldout_mae":0.07757359519771108,"point_error":0.021530785426675525},{"seed":3,"heldout_r2":0.3654785368864063,"heldout_mae":0.07330980640722368,"point_error":0.015148419073781894},{"seed":4,"heldout_r2":0.40956507765473626,"heldout_mae":0.07368260391390231,"point_error":0.01405716147780528}],"8745":[{"seed":0,"heldout_r2":0.3559240780341767,"heldout_mae":0.07955572175612646,"point_error":0.013736718122656422},{"seed":1,"heldout_r2":0.34073369211438154,"heldout_mae":0.07856132028755448,"point_error":0.007631614534246989},{"seed":2,"heldout_r2":0.287008458163595,"heldout_mae":0.08221277389532405,"point_error":0.020347617668774298},{"seed":3,"heldout_r2":0.36314253388349915,"heldout_mae":0.07607705146481224,"point_error":0.01322684766842952},{"seed":4,"heldout_r2":0.405910707942002,"heldout_mae":0.07816926232214876,"point_error":0.015059632947632129}],"7202":[{"seed":0,"heldout_r2":0.043175257423137325,"heldout_mae":0.08913388203797466,"point_error":0.017678006625970077},{"seed":1,"heldout_r2":0.04291623453457272,"heldout_mae":0.08728869049035429,"point_error":0.0073589349105105395},{"seed":2,"heldout_r2":0.03199688740428441,"heldout_mae":0.08893564497875257,"point_error":0.021016593907558667},{"seed":3,"heldout_r2":0.06929518507606258,"heldout_mae":0.08739202784634514,"point_error":0.02226205116550639},{"seed":4,"heldout_r2":0.0529663467041791,"heldout_mae":0.08474264808037621,"point_error":0.022962872203549978}],"4869":[{"seed":0,"heldout_r2":0.3249649577924395,"heldout_mae":0.0699376744747833,"point_error":0.022157895664937644},{"seed":1,"heldout_r2":0.3357603590777287,"heldout_mae":0.07096956808328418,"point_error":0.015392619255230222},{"seed":2,"heldout_r2":0.2820304459120544,"heldout_mae":0.07056315582892955,"point_error":0.012431875955580485},{"seed":3,"heldout_r2":0.3630565284560451,"heldout_mae":0.06802519074538643,"point_error":0.014093082712316463},{"seed":4,"heldout_r2":0.3720196014982614,"heldout_mae":0.06959269372518138,"point_error":0.011749697671678994}],"5103":[{"seed":0,"heldout_r2":0.06952299607822532,"heldout_mae":0.09428809504014159,"point_error":0.016177387551993938},{"seed":1,"heldout_r2":0.06289840148077697,"heldout_mae":0.08805850889090792,"point_error":0.010799884315281894},{"seed":2,"heldout_r2":0.0808882931166417,"heldout_mae":0.09299365053729937,"point_error":0.012032246702924243},{"seed":3,"heldout_r2":0.09861753439214704,"heldout_mae":0.0903894528542417,"point_error":0.01022827240574875},{"seed":4,"heldout_r2":0.07294107031768882,"heldout_mae":0.09277868612642294,"point_error":0.016281612441213067}],"5211":[{"seed":0,"heldout_r2":0.6129335542800498,"heldout_mae":0.06370825305891996,"point_error":0.008526241618485142},{"seed":1,"heldout_r2":0.6069967022156735,"heldout_mae":0.06391189474257077,"point_error":0.005076368091819883},{"seed":2,"heldout_r2":0.6548953777557868,"heldout_mae":0.06188663376688193,"point_error":0.014761407017046757},{"seed":3,"heldout_r2":0.6262283656782053,"heldout_mae":0.06245529574674211,"point_error":0.012276764268948082},{"seed":4,"heldout_r2":0.6135253316172924,"heldout_mae":0.05852539542483308,"point_error":0.009310537908089334}],"5318":[{"seed":0,"heldout_r2":0.16399155547760713,"heldout_mae":0.08029023269228962,"point_error":0.0030102693758356502},{"seed":1,"heldout_r2":0.19488709664474912,"heldout_mae":0.07920379603765662,"point_error":0.0070058588133233846},{"seed":2,"heldout_r2":0.1871457284217689,"heldout_mae":0.07524191148346633,"point_error":0.010752726387074585},{"seed":3,"heldout_r2":0.2198996422738293,"heldout_mae":0.07200789045749058,"point_error":0.012584312869038272},{"seed":4,"heldout_r2":0.24653714245682423,"heldout_mae":0.07524504559222117,"point_error":0.017302842044911726}],"10419":[{"seed":0,"heldout_r2":0.21307946002579492,"heldout_mae":0.08150792166903,"point_error":0.0024853551701741194},{"seed":1,"heldout_r2":0.28130195952221615,"heldout_mae":0.07633368222769188,"point_error":0.018161879290686023},{"seed":2,"heldout_r2":0.22692238260632236,"heldout_mae":0.07885845771834593,"point_error":0.0035828223489925426},{"seed":3,"heldout_r2":0.2688325555271476,"heldout_mae":0.07579987009882132,"point_error":0.0030982021979233387},{"seed":4,"heldout_r2":0.22365423970261178,"heldout_mae":0.07665596160977653,"point_error":0.005686697193972523}],"3886":[{"seed":0,"heldout_r2":0.2563530114482402,"heldout_mae":0.08868603364649612,"point_error":0.01715585094100147},{"seed":1,"heldout_r2":0.3166413147881628,"heldout_mae":0.08563401446628598,"point_error":0.02011370700383086},{"seed":2,"heldout_r2":0.2644119908857916,"heldout_mae":0.08699419705167646,"point_error":0.04219251440159938},{"seed":3,"heldout_r2":0.3504697182839911,"heldout_mae":0.08308829202185629,"point_error":0.03539137833671141},{"seed":4,"heldout_r2":0.3456562010856232,"heldout_mae":0.08448033034185762,"point_error":0.03494989721522074}],"9412":[{"seed":0,"heldout_r2":0.3066695018209059,"heldout_mae":0.08121416340707661,"point_error":0.013228473910272043},{"seed":1,"heldout_r2":0.30362778316978356,"heldout_mae":0.0810781477520996,"point_error":0.007324946937307453},{"seed":2,"heldout_r2":0.26823975885538476,"heldout_mae":0.08262416326797559,"point_error":0.01877883669492386},{"seed":3,"heldout_r2":0.33193248358405014,"heldout_mae":0.07791832009862527,"point_error":0.014089462286902998},{"seed":4,"heldout_r2":0.3687015652408616,"heldout_mae":0.07921498796136263,"point_error":0.016991474261688788}],"8792":[{"seed":0,"heldout_r2":0.2220854996140993,"heldout_mae":0.08827497467563489,"point_error":0.001293352571266393},{"seed":1,"heldout_r2":0.2557754311008381,"heldout_mae":0.08284810126346168,"point_error":0.008012986578665338},{"seed":2,"heldout_r2":0.20956092500011303,"heldout_mae":0.08571487935212875,"point_error":0.001679578457613684},{"seed":3,"heldout_r2":0.2653698853148062,"heldout_mae":0.08176731895489947,"point_error":0.0009346822297672253},{"seed":4,"heldout_r2":0.20547469797208828,"heldout_mae":0.08303335370722471,"point_error":0.006669242569959266}],"1568":[{"seed":0,"heldout_r2":0.28091265035147983,"heldout_mae":0.07474908321777189,"point_error":0.004780702335928311},{"seed":1,"heldout_r2":0.36527776313474736,"heldout_mae":0.06878806895533188,"point_error":0.0018287572360369264},{"seed":2,"heldout_r2":0.30052076784088433,"heldout_mae":0.07352665760504369,"point_error":0.017010798508348524},{"seed":3,"heldout_r2":0.3047839527187999,"heldout_mae":0.06876714610022348,"point_error":0.007418326323334723},{"seed":4,"heldout_r2":0.26279663222740823,"heldout_mae":0.06993090456952797,"point_error":0.005667830647310912}],"1707":[{"seed":0,"heldout_r2":0.1543439178441801,"heldout_mae":0.07887625703152316,"point_error":0.019550848273526378},{"seed":1,"heldout_r2":0.1615926970494529,"heldout_mae":0.07692387431512779,"point_error":0.01888045062620003},{"seed":2,"heldout_r2":0.1457052966590623,"heldout_mae":0.07898779715201062,"point_error":0.014280085448721391},{"seed":3,"heldout_r2":0.1599588507893075,"heldout_mae":0.07523339143484842,"point_error":0.021406927365611672},{"seed":4,"heldout_r2":0.13033839952668824,"heldout_mae":0.07670216506501065,"point_error":0.018715968039935138}],"7189":[{"seed":0,"heldout_r2":0.1643593022978348,"heldout_mae":0.06960573406719908,"point_error":0.008834536932905368},{"seed":1,"heldout_r2":0.11316417587913497,"heldout_mae":0.06674297916192462,"point_error":0.001017699469833877},{"seed":2,"heldout_r2":0.1330604318208114,"heldout_mae":0.06780883395500546,"point_error":0.007893201602064193},{"seed":3,"heldout_r2":0.1706452220916297,"heldout_mae":0.06810202360029778,"point_error":0.0034982428208194527},{"seed":4,"heldout_r2":0.11812646284253592,"heldout_mae":0.06923594355370619,"point_error":0.011592594354864372}],"10034":[{"seed":0,"heldout_r2":0.1659173046736877,"heldout_mae":0.06301340522340516,"point_error":0.012948634640492951},{"seed":1,"heldout_r2":0.13227605399873454,"heldout_mae":0.05990181156647934,"point_error":0.012772011636172254},{"seed":2,"heldout_r2":0.08532363531619347,"heldout_mae":0.06163191498420209,"point_error":0.01034127044699118},{"seed":3,"heldout_r2":0.15690567474508854,"heldout_mae":0.05997843696764252,"point_error":0.012684841401179925},{"seed":4,"heldout_r2":0.14488578612791725,"heldout_mae":0.05896390295453565,"point_error":0.013057592606521118}],"5420":[{"seed":0,"heldout_r2":0.04079912941050989,"heldout_mae":0.08956393055161906,"point_error":0.010452877877535627},{"seed":1,"heldout_r2":0.03530436009546045,"heldout_mae":0.0873505942934274,"point_error":0.005208174300303903},{"seed":2,"heldout_r2":0.03962741364990674,"heldout_mae":0.09258856999002198,"point_error":0.010343798990887776},{"seed":3,"heldout_r2":0.07939128735406309,"heldout_mae":0.08761036853370115,"point_error":0.002550589669626957},{"seed":4,"heldout_r2":0.06297215673484935,"heldout_mae":0.08892702505986076,"point_error":0.010153219670845537}],"8371":[{"seed":0,"heldout_r2":0.15196783329606778,"heldout_mae":0.09043756476328613,"point_error":0.024236796607166933},{"seed":1,"heldout_r2":0.20051339812508584,"heldout_mae":0.08392060099914699,"point_error":0.013118041705512784},{"seed":2,"heldout_r2":0.17843265792892482,"heldout_mae":0.08900837134327233,"point_error":0.022105604119753042},{"seed":3,"heldout_r2":0.2006738279197401,"heldout_mae":0.08511853652942282,"point_error":0.021158240519609017},{"seed":4,"heldout_r2":0.15461560782435424,"heldout_mae":0.08522693031441315,"point_error":0.027581024033202395}],"7928":[{"seed":0,"heldout_r2":0.2403003270919889,"heldout_mae":0.0724707888657536,"point_error":0.006554759782688269},{"seed":1,"heldout_r2":0.24837937932160736,"heldout_mae":0.07025252579947791,"point_error":0.0009489271519945719},{"seed":2,"heldout_r2":0.1744596636505653,"heldout_mae":0.0711647056685343,"point_error":0.010040273536442657},{"seed":3,"heldout_r2":0.26823082561642886,"heldout_mae":0.06989537104065022,"point_error":0.0032649897286206597},{"seed":4,"heldout_r2":0.23566416832461068,"heldout_mae":0.06974791286554897,"point_error":0.008467452762953354}],"5460":[{"seed":0,"heldout_r2":0.7095286361785071,"heldout_mae":0.03585680532215775,"point_error":0.005348026912112225},{"seed":1,"heldout_r2":0.6953294850364833,"heldout_mae":0.03801740050601359,"point_error":0.006720787663474059},{"seed":2,"heldout_r2":0.6957064466829043,"heldout_mae":0.038202334373629146,"point_error":0.00836288844619415},{"seed":3,"heldout_r2":0.6827279130333643,"heldout_mae":0.0383232805973275,"point_error":0.008498448734025166},{"seed":4,"heldout_r2":0.6756795022117547,"heldout_mae":0.04003384287701739,"point_error":0.0015909552920095882}],"6592":[{"seed":0,"heldout_r2":0.626879370640128,"heldout_mae":0.04206719631644244,"point_error":0.0005922509238593943},{"seed":1,"heldout_r2":0.6164012556199863,"heldout_mae":0.04288204619609072,"point_error":0.0007520943860185314},{"seed":2,"heldout_r2":0.6455876727362577,"heldout_mae":0.04170881366464011,"point_error":0.00015386992118215925},{"seed":3,"heldout_r2":0.6103419644133051,"heldout_mae":0.041701919836146795,"point_error":9.519663455559435e-05},{"seed":4,"heldout_r2":0.5937389000325677,"heldout_mae":0.04423935978302717,"point_error":0.00820626140019487}],"1661":[{"seed":0,"heldout_r2":0.35905075086671845,"heldout_mae":0.06137359332721028,"point_error":0.016276633689793374},{"seed":1,"heldout_r2":0.3324234006304947,"heldout_mae":0.06290826227131822,"point_error":0.023495796248850898},{"seed":2,"heldout_r2":0.34412394127234824,"heldout_mae":0.06282519987231999,"point_error":0.01622610765408561},{"seed":3,"heldout_r2":0.35777897886182153,"heldout_mae":0.05782689079779764,"point_error":0.01270728208922478},{"seed":4,"heldout_r2":0.32784663478814013,"heldout_mae":0.061141854866179,"point_error":0.01322934182234281}],"8561":[{"seed":0,"heldout_r2":0.6480406306016226,"heldout_mae":0.04212477125498776,"point_error":0.005282761992919216},{"seed":1,"heldout_r2":0.6650064397315814,"heldout_mae":0.040439825568836935,"point_error":0.0010111564929846661},{"seed":2,"heldout_r2":0.6631519452857562,"heldout_mae":0.041293909544660466,"point_error":0.004397271883694764},{"seed":3,"heldout_r2":0.6533820739213367,"heldout_mae":0.039702387212856516,"point_error":0.0019181712866522815},{"seed":4,"heldout_r2":0.6515435283338822,"heldout_mae":0.04068140472387049,"point_error":0.0022991983295009844}],"5473":[{"seed":0,"heldout_r2":0.722599273660171,"heldout_mae":0.04377298075114505,"point_error":0.016931556688034588},{"seed":1,"heldout_r2":0.7210127920337999,"heldout_mae":0.044183390906790854,"point_error":0.018446857302242825},{"seed":2,"heldout_r2":0.7135790534340027,"heldout_mae":0.04400962894750309,"point_error":0.017503498891322633},{"seed":3,"heldout_r2":0.7188411976045649,"heldout_mae":0.04304646337834549,"point_error":0.015642132324962588},{"seed":4,"heldout_r2":0.7234691421591926,"heldout_mae":0.043298179573660577,"point_error":0.01277263626964742}],"8193":[{"seed":0,"heldout_r2":0.67807348466593,"heldout_mae":0.04236905386695145,"point_error":0.007871730623413398},{"seed":1,"heldout_r2":0.6734922365316086,"heldout_mae":0.04289398575572273,"point_error":0.005601259048458207},{"seed":2,"heldout_r2":0.6885764833781891,"heldout_mae":0.04258760325474425,"point_error":0.009949932142146567},{"seed":3,"heldout_r2":0.6654455569722126,"heldout_mae":0.042606436266533085,"point_error":0.003501015694493237},{"seed":4,"heldout_r2":0.6668479026487752,"heldout_mae":0.042817136012117264,"point_error":0.0016993469669764893}],"5738":[{"seed":0,"heldout_r2":0.6979336006857362,"heldout_mae":0.04016684576299644,"point_error":0.0062732709210596704},{"seed":1,"heldout_r2":0.690713069380164,"heldout_mae":0.04130672256576826,"point_error":0.005034934926143708},{"seed":2,"heldout_r2":0.7045187810515807,"heldout_mae":0.04109802521566178,"point_error":0.007770796085882936},{"seed":3,"heldout_r2":0.6798800671710264,"heldout_mae":0.04092331487972646,"point_error":0.006413522531280247},{"seed":4,"heldout_r2":0.6817955296068872,"heldout_mae":0.04135483779952476,"point_error":0.0005860240569957043}],"7225":[{"seed":0,"heldout_r2":0.6745756308488084,"heldout_mae":0.033498319994683126,"point_error":0.004431998307844021},{"seed":1,"heldout_r2":0.6667126667350411,"heldout_mae":0.034849180209604244,"point_error":0.002594859205716671},{"seed":2,"heldout_r2":0.6728778767392383,"heldout_mae":0.034776907902238,"point_error":0.005770627246324822},{"seed":3,"heldout_r2":0.6710754526463334,"heldout_mae":0.0334786430323573,"point_error":0.004470328070690699},{"seed":4,"heldout_r2":0.65419148095338,"heldout_mae":0.034435491923237714,"point_error":0.0009110785277149924}],"7308":[{"seed":0,"heldout_r2":0.6516712898619692,"heldout_mae":0.04215923335790244,"point_error":0.004427295603127046},{"seed":1,"heldout_r2":0.6610475982386885,"heldout_mae":0.04110227685544864,"point_error":0.0043569093590259955},{"seed":2,"heldout_r2":0.658489757944684,"heldout_mae":0.04000424302777122,"point_error":0.009465272309971073},{"seed":3,"heldout_r2":0.6057947050205217,"heldout_mae":0.04018129941461909,"point_error":0.004110658023274594},{"seed":4,"heldout_r2":0.6396632732236904,"heldout_mae":0.04093588339735149,"point_error":0.0003016581187714751}],"8891":[{"seed":0,"heldout_r2":0.6940142463848222,"heldout_mae":0.03632349820489286,"point_error":0.005216325546881451},{"seed":1,"heldout_r2":0.6921853427854702,"heldout_mae":0.03750638174224809,"point_error":0.0009288898255617184},{"seed":2,"heldout_r2":0.6953629334715539,"heldout_mae":0.03773403897390956,"point_error":0.006225870418408941},{"seed":3,"heldout_r2":0.6845716863999003,"heldout_mae":0.036628590816703284,"point_error":0.002868426034019045},{"seed":4,"heldout_r2":0.6810535642505244,"heldout_mae":0.03684654508714294,"point_error":0.0015383571923058836}],"9811":[{"seed":0,"heldout_r2":0.6813151830566047,"heldout_mae":0.03373290252935008,"point_error":0.00432236913660089},{"seed":1,"heldout_r2":0.6787884213148246,"heldout_mae":0.03456215237403928,"point_error":0.0023155077633051357},{"seed":2,"heldout_r2":0.6590446301127115,"heldout_mae":0.03551534926049815,"point_error":0.0035487410948944476},{"seed":3,"heldout_r2":0.6544238384980243,"heldout_mae":0.03460072961904638,"point_error":0.003169293381156457},{"seed":4,"heldout_r2":0.6539304897463349,"heldout_mae":0.03550911649336392,"point_error":0.0010243015382828125}],"7112":[{"seed":0,"heldout_r2":0.6429687400853079,"heldout_mae":0.030835668866476946,"point_error":0.005200862652407268},{"seed":1,"heldout_r2":0.6318146599935599,"heldout_mae":0.03202448808576465,"point_error":0.003629448326430418},{"seed":2,"heldout_r2":0.6425206359297191,"heldout_mae":0.03158676546847768,"point_error":0.005604623734087544},{"seed":3,"heldout_r2":0.6389885537051796,"heldout_mae":0.032145142875871204,"point_error":0.006909808572514997},{"seed":4,"heldout_r2":0.6221702492807134,"heldout_mae":0.03271128830650347,"point_error":0.0007074841322507464}],"6611":[{"seed":0,"heldout_r2":0.6606754316433041,"heldout_mae":0.02983816445103,"point_error":0.006985297464095819},{"seed":1,"heldout_r2":0.6314120337185869,"heldout_mae":0.030972254200973874,"point_error":0.0030444086758103506},{"seed":2,"heldout_r2":0.6247271460567655,"heldout_mae":0.030822199614431344,"point_error":0.0024784808901158195},{"seed":3,"heldout_r2":0.6349580527375243,"heldout_mae":0.0303975311103697,"point_error":0.0035851271724003375},{"seed":4,"heldout_r2":0.6212514217823191,"heldout_mae":0.03132611686444936,"point_error":0.0016333273250909164}],"9808":[{"seed":0,"heldout_r2":0.36074213264627863,"heldout_mae":0.06614462263117765,"point_error":0.023667266721386304},{"seed":1,"heldout_r2":0.33702754100570087,"heldout_mae":0.0672407573364396,"point_error":0.03295314561284801},{"seed":2,"heldout_r2":0.3477135915239141,"heldout_mae":0.0683261731671877,"point_error":0.02590042016705074},{"seed":3,"heldout_r2":0.35235101268129243,"heldout_mae":0.06437090436323216,"point_error":0.019208536572931756},{"seed":4,"heldout_r2":0.33666636954315643,"heldout_mae":0.06587643207391884,"point_error":0.020448012140841243}],"9541":[{"seed":0,"heldout_r2":0.7146358470017382,"heldout_mae":0.03453078556512072,"point_error":0.009607311504981175},{"seed":1,"heldout_r2":0.7101115390710279,"heldout_mae":0.035135750422207526,"point_error":0.007065443936649873},{"seed":2,"heldout_r2":0.7068722677144726,"heldout_mae":0.036061995629512324,"point_error":0.010248536925309804},{"seed":3,"heldout_r2":0.706737111837805,"heldout_mae":0.034419432608678545,"point_error":0.009953735550549125},{"seed":4,"heldout_r2":0.7124373395280876,"heldout_mae":0.03445273868801264,"point_error":0.0009487769414388003}],"1897":[{"seed":0,"heldout_r2":0.6682557482371514,"heldout_mae":0.03728900270271039,"point_error":0.0014820182865965537},{"seed":1,"heldout_r2":0.6671204572938756,"heldout_mae":0.03773083323971453,"point_error":0.0009706840179066756},{"seed":2,"heldout_r2":0.6637218471343909,"heldout_mae":0.036854731363328695,"point_error":0.0028137489913925284},{"seed":3,"heldout_r2":0.648088878359455,"heldout_mae":0.03594668936214237,"point_error":0.002037186153616366},{"seed":4,"heldout_r2":0.6361638896019071,"heldout_mae":0.03932682390824058,"point_error":0.0026383592325113225}],"7124":[{"seed":0,"heldout_r2":0.6136949015785627,"heldout_mae":0.024073788553091977,"point_error":0.0054822362125959145},{"seed":1,"heldout_r2":0.6274320748027773,"heldout_mae":0.024288696084823487,"point_error":0.003394940234512145},{"seed":2,"heldout_r2":0.6036330352142558,"heldout_mae":0.025869852683890037,"point_error":0.0028391980357144186},{"seed":3,"heldout_r2":0.6141461026055064,"heldout_mae":0.02545163536770146,"point_error":0.0025546495383613754},{"seed":4,"heldout_r2":0.6049070505499556,"heldout_mae":0.02554633698121186,"point_error":0.0009160663057503626}],"2040":[{"seed":0,"heldout_r2":0.6767549880950738,"heldout_mae":0.031217011769942384,"point_error":0.004986831931233837},{"seed":1,"heldout_r2":0.6799153814618143,"heldout_mae":0.031565237518978294,"point_error":0.0028653921736392363},{"seed":2,"heldout_r2":0.6714646172189803,"heldout_mae":0.03165036505902986,"point_error":0.0030184912333403036},{"seed":3,"heldout_r2":0.6714269423392332,"heldout_mae":0.031238388566530256,"point_error":0.0031385604476749096},{"seed":4,"heldout_r2":0.6623864887835816,"heldout_mae":0.03208719277999132,"point_error":0.0007079998630517004}],"10057":[{"seed":0,"heldout_r2":0.7188689517433613,"heldout_mae":0.03138228627711573,"point_error":0.004555848975288712},{"seed":1,"heldout_r2":0.7252096481139125,"heldout_mae":0.03218488924055003,"point_error":0.00045532404294235107},{"seed":2,"heldout_r2":0.6879266362085821,"heldout_mae":0.035657238764401855,"point_error":0.005633391469909044},{"seed":3,"heldout_r2":0.7010078537599673,"heldout_mae":0.03264138912769723,"point_error":0.0036579469395995323},{"seed":4,"heldout_r2":0.7011994051624787,"heldout_mae":0.03358341523882742,"point_error":0.003922740054502993}],"8413":[{"seed":0,"heldout_r2":0.723581009134769,"heldout_mae":0.03183154728398739,"point_error":0.009508735560008863},{"seed":1,"heldout_r2":0.7200191836740271,"heldout_mae":0.03234664125919127,"point_error":0.007562486304362515},{"seed":2,"heldout_r2":0.7021758554887736,"heldout_mae":0.034528746859185645,"point_error":0.010518772766534568},{"seed":3,"heldout_r2":0.7166344289671417,"heldout_mae":0.032334508326043135,"point_error":0.0075521315824816915},{"seed":4,"heldout_r2":0.7161869951999189,"heldout_mae":0.03238124165580091,"point_error":0.0011296020001250184}]},"external_summary":{"paired_applications":40,"outcomes_changed":8,"external_in_top_adverse":33,"external_largest_absolute":28,"external_positive":33,"test_outcomes_disagree":134,"test_applications":1975,"test_auc_difference":0.011234574249280094,"test_auc_difference_ci":[0.004142437875468608,0.018218496341035832],"median_absolute_external_contribution":0.13141485165057304}};', 'site/evidence.js': "/* Numerical explanation-evidence utilities. No legal verdicts or wording dictionary. */\n(function(root){\n const finite=n=>typeof n==='number'&&Number.isFinite(n);\n function validate(pack){\n  if(!pack||pack.schema_version!=='credit-explanation-evidence/v1')throw Error('Use the credit-explanation-evidence/v1 JSON format. Download the example file to see the structure.');\n  if(!pack.model||typeof pack.model.id!=='string'||!pack.model.id.trim()||typeof pack.model.output!=='string')throw Error('Provide a model identifier and name the repayment outcome being explained.');\n  if(pack.model.score_scale!=='probability')throw Error('This version supports probability-scale scores only. Do not mix probability-scale predictions with log-odds or raw-margin contributions.');\n  if(!Array.isArray(pack.features)||pack.features.length<1||pack.features.length>300)throw Error('Provide between 1 and 300 feature definitions.');\n  const features=new Set();\n  for(const f of pack.features){if(typeof f.id!=='string'||!f.id.trim()||typeof f.label!=='string'||!f.label.trim()||(f.group!==undefined&&(typeof f.group!=='string'||!f.group.trim()))||f.id.length>150||f.label.length>250||features.has(f.id))throw Error('Feature identifiers must be unique, with readable labels.');features.add(f.id)}\n  if(!Array.isArray(pack.applications)||!pack.applications.length||pack.applications.length>500)throw Error('Provide between 1 and 500 application explanations.');\n  const ids=new Set();\n  for(const a of pack.applications){\n   if(typeof a.id!=='string'||!a.id.trim()||ids.has(a.id)||!finite(a.score)||a.score<0||a.score>1)throw Error('Each application needs a unique string ID and a finite model score between 0 and 1.');ids.add(a.id);\n   if(!Array.isArray(a.shapley)||a.shapley.length<2||a.shapley.length>20)throw Error('Each application needs 2–20 Shapley explanations of the same prediction.');\n   const explanations=new Set();\n   for(const ex of a.shapley){if(!ex||typeof ex.id!=='string'||!ex.id.trim()||explanations.has(ex.id)||typeof ex.reference!=='string'||!ex.reference.trim()||(!finite(ex.baseline)||ex.baseline<0||ex.baseline>1)||!ex.values)throw Error('Each explanation needs a unique ID, a comparison-group description, a baseline and all feature contributions.');explanations.add(ex.id);const keys=Object.keys(ex.values);if(keys.length!==features.size||keys.some(k=>!features.has(k)||!finite(ex.values[k])||Math.abs(ex.values[k])>1.000001))throw Error('Every Shapley explanation must supply one finite signed contribution for every declared feature.');}\n   if(a.local_approximations!==undefined){if(!Array.isArray(a.local_approximations)||a.local_approximations.length>20)throw Error('Local approximation attempts must be an array with at most 20 entries.');for(const run of a.local_approximations){if(!run.coefficients||Object.entries(run.coefficients).some(([k,v])=>!features.has(k)||!finite(v)))throw Error('Local coefficients must use declared feature IDs and finite numbers.');if(run.fidelity&&( !finite(run.fidelity.r2)||run.fidelity.r2>1||!finite(run.fidelity.mae)||run.fidelity.mae<0||!['independent_generated','fitting_only','other'].includes(run.fidelity.evaluation)))throw Error('Fidelity must state a finite R² (at most 1), nonnegative score error and evaluation type: independent_generated, fitting_only, or other.');}}\n  }\n  return pack;\n }\n function contributions(ex,features,grouped=false){let rows;if(grouped){const sums=new Map();for(const f of features){const group=f.group||f.label;sums.set(group,(sums.get(group)||0)+ex.values[f.id]);}rows=[...sums].map(([id,value])=>({id,label:id,value}));}else rows=features.map(f=>({id:f.id,label:f.label,value:ex.values[f.id]}));return rows.sort((a,b)=>b.value-a.value||a.id.localeCompare(b.id));}\n function leading(ex,features,grouped=false){return contributions(ex,features,grouped).filter(r=>r.value>1e-10).slice(0,4)}\n function compare(a,b,features,grouped=false){const A=leading(a,features,grouped),B=leading(b,features,grouped),as=new Set(A.map(r=>r.id)),bs=new Set(B.map(r=>r.id));return {left:A,right:B,shared:A.filter(r=>bs.has(r.id)).map(r=>r.id),added:B.filter(r=>!as.has(r.id)).map(r=>r.id),removed:A.filter(r=>!bs.has(r.id)).map(r=>r.id),changed:A.length!==B.length||A.some(r=>!bs.has(r.id)),first_changed:(A[0]?.id||null)!==(B[0]?.id||null)};}\n function residual(a,ex){return a.score-ex.baseline-Object.values(ex.values).reduce((a,b)=>a+b,0)}\n const api={validate,contributions,leading,compare,residual};if(typeof module!=='undefined')module.exports=api;else root.ExplanationEvidence=api;\n})(typeof globalThis!=='undefined'?globalThis:this);\n", 'site/examples/external-score-review-8193.html': '<!doctype html><html lang="en"><meta charset="utf-8"><title>Credit explanation review 8193</title><style>body{font:16px/1.65 system-ui;max-width:850px;margin:40px auto;padding:20px;color:#182c2d}pre{white-space:pre-wrap;font:inherit}</style><h1>Review of estimated credit-score explanations</h1><pre>Reason-integrity review for an estimated external credit score\nHistorical record: 8193\n\nPREDICTIONS AND EXPLANATIONS\nThe Random Forest predicts 95.5% serious-late-payment risk with the supplied score, 95.3% with the Ridge estimate and 95.5% with the gradient-boosting estimate. The leading individual-factor sets differ between the two estimates. The grouped-factor sets match.\n\nEVIDENCE FROM THE RANDOM FOREST\nThe external score contributes to the study’s Random Forest prediction relative to 24 reference records.\n\nEVIDENCE FROM THE SCORE ESTIMATORS\nThe reconstructed-system explanations describe the Random Forest using a score estimated by Ridge regression or gradient-boosted trees.\n\nWHAT REMAINS UNESTABLISHED\nThe inferred credit factors were used to calculate the original external credit-risk score.\n\nPOLICY RECOMMENDATION\nIf a consumer-facing reason depends on a material external score, separate what the lender model establishes from what can be supported about the score provider. A reconstruction is useful for testing, but provider-linked reasons or validation records are still needed for claims about the upstream score.\n\nCONSUMER-ADVOCATE TAKEAWAY\nAn external provider’s opacity should not leave the applicant with a specific reason that the creditor cannot connect to the decision path actually used.\n\nRECORDS NEEDED FOR FURTHER REVIEW\nObtain applicable provider reason codes, score/model version, input documentation and validation evidence linking proposed reasons to the original score. For a refusal, obtain the lender’s actual decision rules, reason-mapping version, human assessment and delivered notice.\n\nSTUDY LIMITATIONS\nThe original score mechanism and authenticated provider reasons are absent.\nThe study uses historical repayment data and does not evaluate actual notices.\nThe analysis is exploratory and the forty records are not representative of lending decisions.\nInterventional replacements can violate credit-feature relationships.\n\nSOURCES\nhttps://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/\nhttps://finreglab.org/research/explainability-and-fairness-in-machine-learning-for-credit-underwriting/\nhttps://business.cch.com/BFLD/CFPB-Supervisory-Highlights-Advanced-Technologies-Special-Edition-Issue-38-Winter-2025-01172025.pdf</pre></html>', 'site/examples/external-score-review-8193.json': '{\n  "title": "Reason-integrity review for an estimated external credit score",\n  "case_id": 8193,\n  "explanation_view": "factors",\n  "measurements": {\n    "case_id": 8193,\n    "inputs": {\n      "MSinceOldestTradeOpen": 38,\n      "AverageMInFile": 19,\n      "NumSatisfactoryTrades": 13,\n      "PercentTradesNeverDelq": 93,\n      "MSinceMostRecentDelq": 15,\n      "NumInqLast6M": 1,\n      "NetFractionRevolvingBurden": 82,\n      "NetFractionInstallBurden": 91\n    },\n    "observed_external_score": 58,\n    "original_probability": 0.954655663685682,\n    "original_shapley": {\n      "values": [\n        0.031571921996015447,\n        0.07864328841349487,\n        0.027876107074549978,\n        0.012975808319390084,\n        0.004843597668739871,\n        -0.0017070179420093953,\n        0.09387935960364248,\n        0.019514408570879743,\n        0.1742862253249367\n      ],\n      "diagnostics": {\n        "base": 0.5127719646560421,\n        "residual": -5.551115123125783e-17\n      },\n      "individual": [\n        {\n          "id": "ExternalRiskEstimate",\n          "label": "External risk estimate",\n          "weight": 0.1742862253249367\n        },\n        {\n          "id": "NetFractionRevolvingBurden",\n          "label": "Revolving balance ratio",\n          "weight": 0.09387935960364248\n        },\n        {\n          "id": "AverageMInFile",\n          "label": "Average account age",\n          "weight": 0.07864328841349487\n        },\n        {\n          "id": "MSinceOldestTradeOpen",\n          "label": "Oldest account age",\n          "weight": 0.031571921996015447\n        }\n      ],\n      "grouped": [\n        {\n          "id": "External risk score",\n          "label": "External risk score",\n          "weight": 0.1742862253249367\n        },\n        {\n          "id": "Credit history",\n          "label": "Credit history",\n          "weight": 0.11021521040951032\n        },\n        {\n          "id": "Revolving balances",\n          "label": "Revolving balances",\n          "weight": 0.09387935960364248\n        },\n        {\n          "id": "Repayment history",\n          "label": "Repayment history",\n          "weight": 0.04569551306267994\n        }\n      ]\n    },\n    "approximations": {\n      "additive": {\n        "estimated_external_score": 57.107905160592196,\n        "probability": 0.9534454484312339,\n        "probability_error": 0.0012102152544480882,\n        "values": [\n          0.03605035345522786,\n          0.1308715750198432,\n          0.0168554892484255,\n          0.008992587714686744,\n          0.041209010782488315,\n          -0.005051137284274818,\n          0.17181186411655266,\n          0.024738814617662235\n        ],\n        "diagnostics": {\n          "base": 0.5279668907606222,\n          "residual": 0\n        },\n        "individual": [\n          {\n            "id": "NetFractionRevolvingBurden",\n            "label": "Revolving balance ratio",\n            "weight": 0.17181186411655266\n          },\n          {\n            "id": "AverageMInFile",\n            "label": "Average account age",\n            "weight": 0.1308715750198432\n          },\n          {\n            "id": "MSinceMostRecentDelq",\n            "label": "Time since delinquency",\n            "weight": 0.041209010782488315\n          },\n          {\n            "id": "MSinceOldestTradeOpen",\n            "label": "Oldest account age",\n            "weight": 0.03605035345522786\n          }\n        ],\n        "grouped": [\n          {\n            "id": "Revolving balances",\n            "label": "Revolving balances",\n            "weight": 0.17181186411655266\n          },\n          {\n            "id": "Credit history",\n            "label": "Credit history",\n            "weight": 0.16692192847507106\n          },\n          {\n            "id": "Repayment history",\n            "label": "Repayment history",\n            "weight": 0.06705708774560055\n          },\n          {\n            "id": "Installment balances",\n            "label": "Installment balances",\n            "weight": 0.024738814617662235\n          }\n        ]\n      },\n      "nonlinear": {\n        "estimated_external_score": 58.853467052010565,\n        "probability": 0.954655663685682,\n        "probability_error": 0,\n        "values": [\n          0.03233236198628696,\n          0.18124818970672513,\n          0.017438485722681696,\n          0.012074297417847955,\n          -0.001323912531416938,\n          -0.002409453726910494,\n          0.18349796344173458,\n          0.01946178078508059\n        ],\n        "diagnostics": {\n          "base": 0.5123359508836521,\n          "residual": 2.220446049250313e-16\n        },\n        "individual": [\n          {\n            "id": "NetFractionRevolvingBurden",\n            "label": "Revolving balance ratio",\n            "weight": 0.18349796344173458\n          },\n          {\n            "id": "AverageMInFile",\n            "label": "Average account age",\n            "weight": 0.18124818970672513\n          },\n          {\n            "id": "MSinceOldestTradeOpen",\n            "label": "Oldest account age",\n            "weight": 0.03233236198628696\n          },\n          {\n            "id": "NetFractionInstallBurden",\n            "label": "Installment balance ratio",\n            "weight": 0.01946178078508059\n          }\n        ],\n        "grouped": [\n          {\n            "id": "Credit history",\n            "label": "Credit history",\n            "weight": 0.2135805516930121\n          },\n          {\n            "id": "Revolving balances",\n            "label": "Revolving balances",\n            "weight": 0.18349796344173458\n          },\n          {\n            "id": "Repayment history",\n            "label": "Repayment history",\n            "weight": 0.028188870609112713\n          },\n          {\n            "id": "Installment balances",\n            "label": "Installment balances",\n            "weight": 0.01946178078508059\n          }\n        ]\n      }\n    },\n    "comparison": {\n      "probability_difference": 0.0012102152544480882,\n      "maximum_original_probability_error": 0.0012102152544480882,\n      "individual": {\n        "retention": 0.75,\n        "jaccard": 0.6,\n        "turnover": true,\n        "first_changed": false,\n        "empty": false\n      },\n      "grouped": {\n        "retention": 1,\n        "jaccard": 1,\n        "turnover": false,\n        "first_changed": true,\n        "empty": false\n      },\n      "same_order": false\n    }\n  },\n  "study_metrics": {\n    "additive": {\n      "label": "Additive score approximation",\n      "test_records": 1973,\n      "external_score_mae": 3.848025717689414,\n      "external_score_r2": 0.7645333613478831,\n      "probability_mae": 0.050146878486452696,\n      "probability_error_quantiles": [\n        0.0277760686194648,\n        0.14276865478543,\n        0.18532556609494108,\n        0.2505555535612225\n      ],\n      "unclipped_out_of_training_range": 4\n    },\n    "nonlinear": {\n      "label": "Nonlinear score approximation",\n      "test_records": 1973,\n      "external_score_mae": 3.243338784287528,\n      "external_score_r2": 0.8183807334210851,\n      "probability_mae": 0.043288371094502384,\n      "probability_error_quantiles": [\n        0.021155569730752333,\n        0.12128815208297046,\n        0.16849930628847445,\n        0.2569754378784586\n      ],\n      "unclipped_out_of_training_range": 0\n    }\n  },\n  "local_approximation": [\n    {\n      "seed": 0,\n      "heldout_r2": 0.67807348466593,\n      "heldout_mae": 0.04236905386695145,\n      "point_error": 0.007871730623413398\n    },\n    {\n      "seed": 1,\n      "heldout_r2": 0.6734922365316086,\n      "heldout_mae": 0.04289398575572273,\n      "point_error": 0.005601259048458207\n    },\n    {\n      "seed": 2,\n      "heldout_r2": 0.6885764833781891,\n      "heldout_mae": 0.04258760325474425,\n      "point_error": 0.009949932142146567\n    },\n    {\n      "seed": 3,\n      "heldout_r2": 0.6654455569722126,\n      "heldout_mae": 0.042606436266533085,\n      "point_error": 0.003501015694493237\n    },\n    {\n      "seed": 4,\n      "heldout_r2": 0.6668479026487752,\n      "heldout_mae": 0.042817136012117264,\n      "point_error": 0.0016993469669764893\n    }\n  ],\n  "evidence_status": {\n    "downstream_model_contribution": "measured",\n    "reconstruction_behavior": "measured",\n    "original_provider_reason": "not established",\n    "production_notice_accuracy": "provider / lender records required"\n  },\n  "findings": {\n    "comparison": "The Random Forest predicts 95.5% serious-late-payment risk with the supplied score, 95.3% with the Ridge estimate and 95.5% with the gradient-boosting estimate. The leading individual-factor sets differ between the two estimates. The grouped-factor sets match.",\n    "supported_original_claim": "The external score contributes to the study’s Random Forest prediction relative to 24 reference records.",\n    "supported_approximation_claim": "The reconstructed-system explanations describe the Random Forest using a score estimated by Ridge regression or gradient-boosted trees.",\n    "unestablished_claim": "The inferred credit factors were used to calculate the original external credit-risk score."\n  },\n  "policy_recommendation": "If a consumer-facing reason depends on a material external score, separate what the lender model establishes from what can be supported about the score provider. A reconstruction is useful for testing, but provider-linked reasons or validation records are still needed for claims about the upstream score.",\n  "consumer_advocate_takeaway": "An external provider’s opacity should not leave the applicant with a specific reason that the creditor cannot connect to the decision path actually used.",\n  "request": "Obtain applicable provider reason codes, score/model version, input documentation and validation evidence linking proposed reasons to the original score. For a refusal, obtain the lender’s actual decision rules, reason-mapping version, human assessment and delivered notice.",\n  "limits": [\n    "The original score mechanism and authenticated provider reasons are absent.",\n    "The study uses historical repayment data and does not evaluate actual notices.",\n    "The analysis is exploratory and the forty records are not representative of lending decisions.",\n    "Interventional replacements can violate credit-feature relationships."\n  ],\n  "sources": {\n    "regulation_b": "https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/",\n    "finreglab": "https://finreglab.org/research/explainability-and-fairness-in-machine-learning-for-credit-underwriting/",\n    "cfpb_case": "https://business.cch.com/BFLD/CFPB-Supervisory-Highlights-Advanced-Technologies-Special-Edition-Issue-38-Winter-2025-01172025.pdf"\n  }\n}', 'site/index.html': '<!DOCTYPE html>\n\n<html lang="en"><head><meta charset="utf-8"/><meta content="width=device-width, initial-scale=1" name="viewport"/><title>Credit explanation review · External scores</title>\n<link href="https://fonts.googleapis.com" rel="preconnect"/><link crossorigin="" href="https://fonts.gstatic.com" rel="preconnect"/><link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&amp;family=Fraunces:opsz,wght@9..144,400;9..144,500&amp;family=IBM+Plex+Mono:wght@400;500&amp;display=swap" rel="stylesheet"/><style>\n:root{--paper:#f7f5f0;--ink:#182c2d;--sub:#506362;--line:#d8ddd7;--white:#fffefa;--teal:#14655d;--pale:#e5eee8;--rust:#a44125;--warm:#f8ebe2;--blue:#345fa2}*{box-sizing:border-box}body{margin:0;background:var(--paper);color:var(--ink);font:16px/1.65 system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif}button,select,textarea{font:inherit}button,a,select,summary{touch-action:manipulation}button{cursor:pointer}a{color:var(--teal);text-underline-offset:4px}button:focus-visible,a:focus-visible,select:focus-visible,summary:focus-visible{outline:3px solid var(--blue);outline-offset:4px}header{border-bottom:1px solid var(--line);padding:20px max(24px,calc((100vw - 1160px)/2));display:flex;align-items:center;justify-content:space-between;gap:24px}.brand{font-size:14px;font-weight:650;white-space:nowrap}.brand span{font-weight:400;color:var(--sub)}nav{display:flex;gap:4px;flex-wrap:wrap}nav button{background:none;border:0;padding:10px 12px;color:var(--sub);border-radius:4px}nav button[aria-current=page]{color:var(--ink);background:var(--pale);font-weight:650}main{max-width:1160px;margin:0 auto;padding:48px 28px 70px}h1,h2,h3,p{margin-top:0}h1,h2{font-family:Georgia,"Times New Roman",serif;font-weight:400;line-height:1.13;letter-spacing:-.03em}h1{font-size:clamp(38px,5vw,62px);max-width:870px;margin-bottom:24px}h2{font-size:30px;margin-bottom:18px}h3{font-size:18px;line-height:1.4;margin-bottom:12px}p{margin-bottom:20px}.intro{max-width:790px;font-size:19px;line-height:1.7;color:var(--sub)}.eyebrow{font-size:12px;letter-spacing:.08em;text-transform:uppercase;color:var(--teal);font-weight:650;margin-bottom:16px}.hero{max-width:850px;padding:16px 0 35px}.actions{display:flex;flex-wrap:wrap;gap:12px;margin:22px 0}.primary,.secondary{display:inline-block;border-radius:4px;border:1px solid var(--teal);padding:10px 18px;font-weight:600;text-decoration:none}.primary{background:var(--teal);color:var(--white)}.secondary{background:transparent;color:var(--teal)}.text-button{border:0;background:none;color:var(--teal);padding:6px 0;text-decoration:underline;text-underline-offset:4px;font-size:14px}.page[hidden],[hidden]{display:none!important}.reading{max-width:820px}.reading>section{margin:32px 0}.context{max-width:820px;padding:25px 0;border-top:1px solid var(--line)}.small{font-size:13px;line-height:1.6}.muted{color:var(--sub)}.grid{display:grid;grid-template-columns:1fr 1fr;gap:30px}.card{background:var(--white);border:1px solid var(--line);padding:24px;border-radius:5px;margin:20px 0}.metric{font:36px/1.2 Georgia,serif;margin-bottom:8px}.metric-row{display:grid;grid-template-columns:1fr 1.2fr;gap:35px;padding:22px 0;border-top:1px solid var(--line);border-bottom:1px solid var(--line);margin-bottom:28px}.metric-row p{margin:0}.metric-label{font-size:14px;font-weight:600;display:block;margin-bottom:8px}.case-toolbar{display:flex;justify-content:space-between;align-items:center;gap:20px;margin-bottom:28px;padding-bottom:20px;border-bottom:1px solid var(--line)}.case-toolbar p{margin:0;font-size:13px;color:var(--sub)}.field{font-size:13px;font-weight:600;display:flex;gap:12px;align-items:center;flex-wrap:wrap}.field select{max-width:100%;min-height:44px;padding:7px 30px 7px 10px;border:1px solid var(--line);background:var(--white);color:var(--ink);border-radius:4px}details{border-top:1px solid var(--line);padding:16px 0;margin-top:12px}summary{cursor:pointer;font-weight:600}details>div{padding-top:16px}.table-scroll{overflow:auto}table{width:100%;border-collapse:collapse;font-size:14px;text-align:left}th,td{padding:13px 10px;border-bottom:1px solid var(--line);vertical-align:top}th{font-weight:600;background:var(--pale)}caption{text-align:left;margin-bottom:12px;font-weight:600}.bar-chart{margin:20px 0}.bar-axis,.bar-row{display:grid;grid-template-columns:minmax(150px,1fr) minmax(90px,1.2fr) 72px;gap:10px;align-items:center}.bar-axis{font-size:12px;color:var(--sub);margin-bottom:9px}.bar-row{min-height:38px;font-size:13px;padding:5px 0}.bar-track{height:20px;position:relative;background:linear-gradient(to right,transparent 49.5%,var(--line) 49.5%,var(--line) 50.5%,transparent 50.5%)}.bar-fill{position:absolute;height:14px;top:3px}.bar-val{text-align:right;font-variant-numeric:tabular-nums;white-space:nowrap}.highlight-row{background:var(--pale);border-radius:3px}.chart-summary{display:flex;justify-content:space-between;gap:15px;flex-wrap:wrap;font-size:14px;border-top:1px solid var(--line);padding-top:14px}.compare-head{display:flex;justify-content:space-between;gap:18px;align-items:center;flex-wrap:wrap;margin:25px 0 10px}.compare-head h2{margin:0;font-size:25px}.segmented{display:flex;gap:4px;flex-wrap:wrap}.segmented button{border:1px solid var(--line);background:transparent;padding:8px 12px;font-size:13px;color:var(--sub);border-radius:3px}.segmented button[aria-pressed=true]{background:var(--teal);color:var(--white);border-color:var(--teal)}.stat-line{display:flex;justify-content:space-between;gap:15px;border-bottom:1px solid var(--line);padding:8px 0;font-size:14px}.stat-line strong{font-variant-numeric:tabular-nums;white-space:nowrap}.conclusion{max-width:820px;border-left:3px solid var(--teal);padding:4px 0 4px 20px;margin:24px 0}.conclusion p{margin:0}.download-links{display:flex;gap:18px;flex-wrap:wrap;margin:16px 0}#report-preview{width:100%;min-height:190px;background:var(--paper);border:1px solid var(--line);border-radius:4px;padding:14px;color:var(--ink)}.report-state{font-size:13px;color:var(--sub)}code{font-size:12px;overflow-wrap:anywhere}.legal-row{padding:22px 0;border-top:1px solid var(--line)}.legal-row h3{margin:8px 0}.pill{font-size:12px;font-weight:600;color:var(--teal)}.note{border-left:2px solid var(--line);padding-left:16px;color:var(--sub);font-size:14px}.result-lead{font-size:20px;line-height:1.65}.export{margin-top:35px;padding-top:22px;border-top:1px solid var(--line)}footer{max-width:1160px;margin:auto;border-top:1px solid var(--line);padding:20px 28px;display:flex;justify-content:space-between;gap:18px;flex-wrap:wrap;font-size:12px;color:var(--sub)}footer div{display:flex;gap:22px;flex-wrap:wrap}footer button{font-size:12px!important}ul{padding-left:20px}@media(max-width:750px){header{display:block;padding:18px 20px}nav{margin-top:15px;gap:4px}nav button{font-size:13px;padding:9px 10px;min-height:44px}main{padding:28px 20px 50px}.intro{font-size:17px}.grid{grid-template-columns:1fr;gap:0}.metric-row{gap:20px;grid-template-columns:1fr 1fr}.card{padding:18px}.case-toolbar{align-items:flex-start;flex-direction:column;gap:8px}.bar-axis,.bar-row{grid-template-columns:minmax(95px,1fr) minmax(70px,1.1fr) 62px;gap:6px}.bar-row{font-size:12px;min-height:44px}.bar-axis{font-size:11px}.bar-val{font-size:11px}.segmented button{min-height:44px}.reading table{min-width:460px}.result-lead{font-size:18px}.stat-line{font-size:13px}h2{font-size:27px}.metric{font-size:31px}}@media(max-width:420px){.metric-row{grid-template-columns:1fr;gap:18px}.brand span{display:none}}@media print{header,footer,.case-toolbar,.actions,.export,.segmented{display:none}.page{display:none!important}#research{display:block!important}body{background:white}main{padding:0}.card{break-inside:avoid}}\n\n/* Document excerpts provide context without adding dashboard controls. */\nheader{display:block;padding-top:22px;padding-bottom:0}.brand{margin-bottom:14px}nav{gap:12px}nav button{border-radius:0;padding:11px 0;margin-right:10px;border-bottom:2px solid transparent;font-size:14px}nav button[aria-current=page]{background:transparent;border-bottom-color:var(--teal)}.opening-layout{display:grid;grid-template-columns:minmax(0,1.55fr) minmax(260px,.85fr);gap:65px;align-items:start}.hero h1{font-size:clamp(38px,4.5vw,58px)}.hero .intro{font-size:19px}.hero p{max-width:640px}.document-example{padding:50px 5px 15px}.paper-notice{position:relative;transform:rotate(1deg);background:#fffdf6;color:#293333;padding:32px 28px 30px;box-shadow:0 5px 15px #182c2d15;border:1px solid #d9d4c7;border-radius:1px}.paper-notice:after{content:"";position:absolute;bottom:-6px;left:-1px;right:-1px;height:8px;background:#fffdf6;clip-path:polygon(0 0,100% 0,100% 55%,94% 80%,87% 55%,79% 95%,72% 63%,66% 80%,59% 45%,52% 85%,46% 60%,39% 95%,33% 65%,26% 80%,19% 40%,12% 80%,6% 55%,0 75%)}.document-label{font:12px/1.5 ui-monospace,monospace;letter-spacing:.03em;color:#6b695f;margin-bottom:26px}.paper-notice h2{font-size:29px;line-height:1.2;letter-spacing:-.025em}.document-rule{border-top:1px solid #ddd7c9;margin:20px 0}.notice-reason{font:17px/1.6 Georgia,serif;margin:18px 0;padding-left:15px;border-left:2px solid #bc8b38}.document-note{font-size:12px;line-height:1.6;color:#65655f}.document-source{font-size:12px;color:#315d59}.document-caption{font-size:12px;line-height:1.6;color:var(--sub);margin:20px 12px}.source-extract{max-width:820px;margin-top:8px;padding:24px 0 20px;border-top:1px solid var(--line)}.source-extract h2{font-size:27px;margin-bottom:12px}.source-extract h2 a{color:var(--ink);text-decoration:none}.source-extract h2 a:hover{text-decoration:underline}.source-extract p{max-width:760px}.source-date{font-size:12px;color:var(--sub);margin-bottom:12px}.case-quotation{font:23px/1.5 Georgia,serif;margin:16px 0 22px;padding-left:18px;border-left:2px solid #bc8b38}.case-quotation cite{display:block;font:12px/1.6 system-ui,sans-serif;color:var(--sub);margin-top:10px}@media(max-width:850px){.opening-layout{grid-template-columns:1fr;gap:0}.document-example{max-width:460px;padding-top:0;padding-bottom:20px}.paper-notice{transform:none}.hero{padding-bottom:15px}nav{gap:5px 14px}nav button{font-size:13px;margin-right:0;min-height:44px}}@media(prefers-reduced-motion:reduce){.paper-notice{transform:none}}\n\n.opening-layout{grid-template-columns:minmax(0,1.1fr) minmax(320px,1fr);gap:52px;align-items:start}.hero{padding-top:0}.hero h1{font-size:clamp(36px,4vw,53px)}.hero .intro{font-size:18px}.document-example{padding:0 0 20px}.paper-notice{transform:none;background:#fff;color:#20272b;padding:30px 32px;border:1px solid #ddd;box-shadow:0 3px 12px #182c2d10;border-radius:0;font:13px/1.65 Arial,Helvetica,sans-serif}.paper-notice:after{display:none}.paper-notice p{margin:0 0 17px}.document-label{font:12px/1.5 system-ui,sans-serif;margin:0 0 12px;color:var(--sub)}.letter-department{font-size:13px;font-weight:700;letter-spacing:.09em;padding-bottom:16px;border-bottom:1px solid #d7d7d7;margin-bottom:22px}.letter-date{font-size:12px;color:#50575a}.letter-subject{font-weight:700}.letter-reasons{padding-left:21px;margin:0 0 20px}.letter-reasons li{margin:9px 0}.letter-signature{margin-top:25px!important}.document-caption{margin:14px 2px 0}.landing-connection{max-width:none;display:grid;grid-template-columns:1fr 2fr;gap:10px 45px;padding-top:30px}.landing-connection h2{font-size:28px}.landing-connection p{max-width:710px}.landing-connection .small{grid-column:2}.case-entry{border-top:1px solid var(--line);padding:24px 0}.case-entry h2{font-size:26px}.case-entry p:last-child{margin-bottom:0}.evidence-table td:first-child{min-width:150px}.evidence-table td{line-height:1.65}.legal-standard{font-size:19px;line-height:1.7;padding-bottom:24px}.next-page{margin-top:28px}nav{column-gap:15px;row-gap:0}nav button{margin-right:0;font-size:13px}@media(max-width:850px){.opening-layout{grid-template-columns:1fr;gap:10px}.document-example{max-width:510px;margin-bottom:10px}.landing-connection{grid-template-columns:1fr;gap:0}.landing-connection .small{grid-column:auto}.paper-notice{padding:26px}.hero h1{font-size:40px}}@media(max-width:420px){.hero h1{font-size:35px}.paper-notice{padding:22px;font-size:12.5px}}\n\n/* Editorial palette, document textures and sliding controls. */\n:root{--paper:#f7f1e6;--ink:#32283d;--sub:#665b6e;--line:#ddd1d8;--white:#fffbf3;--teal:#6a315f;--pale:#eee1ef;--rust:#bf4935;--warm:#f7e0cd;--blue:#5f559d;--ochre:#e7b35a;--sea:#247972}body{font-family:\'DM Sans\',system-ui,sans-serif;background:radial-gradient(ellipse at 95% 5%,#e9dced55,transparent 45%),var(--paper)}h1,h2{font-family:\'Fraunces\',Georgia,serif;font-weight:400;letter-spacing:-.035em}header{background:#f7f1e6ee;border-top:5px solid var(--teal);border-bottom:1px solid #d8c8d5;padding-top:20px}.brand{font-family:\'IBM Plex Mono\',monospace;font-size:16px;letter-spacing:.09em;color:var(--teal)}.brand span{font:13px \'DM Sans\',system-ui,sans-serif;letter-spacing:0}.eyebrow{font-family:\'IBM Plex Mono\',monospace;font-size:11px;font-weight:500;letter-spacing:.06em}.hero .eyebrow{display:inline-block;background:var(--pale);padding:7px 10px;transform:rotate(-1deg)}.hero h1{color:var(--teal);font-size:clamp(39px,4.5vw,61px);line-height:1.07}.hero .intro{color:var(--ink);font-size:19px}.primary{background:var(--teal);border-color:var(--teal);box-shadow:3px 3px 0 #d8b2c5}.primary:hover{background:#502348}.secondary{border-color:#bba1b6;background:#f8f0f5}.primary,.secondary{border-radius:22px;font-size:14px;padding:11px 18px;transition:background .18s,box-shadow .18s}.primary:active{box-shadow:none}nav button{color:var(--sub)}nav button[aria-current=page]{color:var(--teal);border-bottom:3px solid var(--teal)}.metric-row{background:linear-gradient(110deg,#eee1ef,#fbedd9);padding:23px;border:0;border-radius:4px}.metric{font-family:\'Fraunces\',Georgia,serif;color:var(--teal);font-size:40px}.metric-label{font-weight:600}.card{border-color:#dfd0dd;background:#fffaf5;box-shadow:0 3px 0 #eadde7;border-radius:8px}.card:nth-child(2){background:#fff8ec}.card h3{color:var(--teal)}.highlight-row{background:#eee1ef}.bar-fill{border-radius:2px;transition:width .2s}.bar-row:not(.highlight-row) .bar-fill[style*="var(--teal)"]{background:var(--sea)!important}.chart-summary{font-family:\'IBM Plex Mono\',monospace;font-size:12px}.conclusion{border-color:var(--rust);background:#fae6d8;padding:16px 20px;border-radius:0 7px 7px 0}.case-entry{border-top:2px solid #dfbdc8}.case-entry .eyebrow{color:var(--rust)}.case-entry h2{color:var(--teal)}details{border-top-color:#d7c6d4}summary{padding-left:4px;color:var(--ink)}details[open]>summary{color:var(--teal)}th{background:var(--pale)}.note{border-left:3px solid var(--ochre);padding:13px 18px;background:#f5ead5}.paper-notice{border:0;background:linear-gradient(175deg,#fffef8 0%,#fffdf5 53%,#f2ead9 54%,#fffdf5 54.7%,#fffaf0 100%);padding:38px 34px 42px;transform:rotate(1.1deg);clip-path:polygon(.5% .5%,8% .2%,15% .7%,24% 0,36% .5%,48% .1%,60% .6%,73% 0,84% .5%,96% .1%,99.6% .6%,100% 98.2%,97% 98.9%,94% 98.3%,91% 99.5%,88% 98.8%,85% 99.3%,82% 98.4%,78% 99.5%,75% 98.6%,71% 99.3%,68% 98.3%,65% 99.7%,61% 98.9%,58% 99.4%,55% 98.5%,51% 99.6%,47% 98.8%,43% 99.5%,39% 98.2%,36% 99.4%,32% 98.9%,28% 99.6%,24% 98.3%,20% 99.4%,16% 98.8%,12% 99.6%,9% 98.5%,5% 99.3%,1% 98.5%,0 83%,.5% 66%,0 47%,.6% 28%);box-shadow:none}.document-example{filter:drop-shadow(3px 7px 5px #39273521);padding-top:10px}.paper-notice:after{display:none}.document-label{font:11px \'IBM Plex Mono\',monospace;color:var(--teal);margin-bottom:16px;letter-spacing:.02em}.letter-department{color:#453a49;border-bottom-color:#cebdbe;font-size:12px;letter-spacing:.12em;padding-right:70px}.letter-stamp{position:absolute;right:27px;top:27px;color:#9f5144;border:1px solid #bc8475;padding:4px 6px;font:10px \'IBM Plex Mono\',monospace;transform:rotate(-9deg)}.letter-reasons{background:linear-gradient(transparent 0,#f7e4b9 0);padding:8px 12px 8px 30px;margin-left:-8px;margin-right:-6px}.letter-reasons li{margin:6px 0}.document-caption{font-size:12px;padding-left:10px}.landing-connection{border-top:2px solid #d6b6c9}.landing-connection h2{color:var(--teal)}.sliding-options{position:relative;isolation:isolate;display:grid;grid-template-columns:1fr 1fr;background:#e9dbe9;border:1px solid #dac4d9;border-radius:26px;padding:4px;max-width:100%;min-height:43px}.sliding-options:before{content:"";position:absolute;z-index:-1;top:4px;bottom:4px;left:4px;width:calc((100% - 8px)/2);background:var(--teal);border-radius:22px;box-shadow:0 2px 3px #38213d22;transition:transform .22s ease}.sliding-options[data-mode="topics"]:before,.sliding-options[data-mode="methods"]:before{transform:translateX(100%)}.sliding-options button{border:0;background:none;position:relative;z-index:1;padding:8px 14px;color:#624e67;font-size:13px;font-weight:500;border-radius:22px;white-space:normal}.sliding-options button[aria-pressed="true"]{color:#fffaf4}.evidence-options{max-width:440px;margin:25px 0 30px}.evidence-panel details{padding:17px 0}.evidence-panel details[open]{border-left:3px solid #c7a0bb;padding-left:17px}.evidence-panel p{max-width:730px}.method-status{font-size:13px;background:#f1e4d2;padding:12px 15px;border-radius:4px}.threshold-control{padding:20px 24px 14px;background:#eee1ef;border-radius:9px;margin:25px 0 28px}.threshold-control label{display:block;font-size:14px;font-weight:500}.threshold-control output{display:block;font:25px \'Fraunces\',Georgia,serif;color:var(--teal);margin-top:8px}.threshold-control input{width:100%;accent-color:var(--teal);cursor:pointer;margin:20px 0 8px;min-height:24px}.range-labels{display:flex;justify-content:space-between;gap:12px;font-family:\'IBM Plex Mono\',monospace;font-size:11px;color:#6d546f}.threshold-control .small{margin:15px 0 0}.result-lead{color:var(--teal)}.reading>h1{color:var(--teal)}footer{background:#eee3e6;font-family:\'DM Sans\',system-ui,sans-serif;border-top:1px solid #d5c0d0}@media(max-width:850px){.hero h1{font-size:44px}.sliding-options{width:100%;max-width:420px}.opening-layout{gap:12px}.paper-notice{transform:rotate(.5deg)}.document-example{padding-right:7px}}@media(max-width:420px){.hero h1{font-size:38px}.paper-notice{padding:32px 25px 38px}.threshold-control{padding:18px}.sliding-options button{font-size:12px;padding:9px}.range-labels{font-size:11px;gap:5px}.letter-stamp{right:20px;top:24px}}@media(prefers-reduced-motion:reduce){.sliding-options:before,.bar-fill,.primary,.secondary{transition:none}.paper-notice{transform:none}}\n\n/* Retain aubergine; use warm stone for the palest supporting surfaces. */\nfooter{background:#eae7de;border-top-color:#d2cbbd}.highlight-row{background:#eae7de}.sliding-options{background:#eae7de;border-color:#d3cbbd}.secondary{background:#f6f2e8}.note{background:#f1eadb}.method-status{background:#f1eadb}\n/* Explicit chart text colors prevent light host-theme text on light cards. */\n.card{color:var(--ink)}#approximation-charts .card{color:#32283d}#approximation-charts .bar-row,#approximation-charts .bar-row span,#approximation-charts .bar-row strong,#approximation-charts .bar-val,#approximation-charts .stat-line,#approximation-charts .stat-line span,#approximation-charts .stat-line strong{color:#32283d}#approximation-charts .bar-axis,#approximation-charts .bar-axis span,#approximation-charts .small{color:#554c5b}#approximation-charts .card h3{color:#6a315f}.bar-row{color:var(--ink)}.bar-row{font-size:14px}.bar-val{font-size:13px}@media(max-width:750px){.bar-row{font-size:12px}.bar-val{font-size:12px}}\n\n/* Table surfaces and text are explicit so embedded dark themes cannot override contrast. */\n:root{--table-ink:#32283d;--table-sub:#554c5b;--table-head:#f6e8c8;--table-cell:#fffaf0;--table-rule:#d9cfb9}\n#score-investigation-render table,table{color:var(--table-ink);background:var(--table-cell)}\n#score-investigation-render table thead,#score-investigation-render table th,table thead,table th{background:var(--table-head);color:var(--table-ink)}\n#score-investigation-render table tbody,#score-investigation-render table td,table tbody,table td{background:var(--table-cell);color:var(--table-ink)}\n#score-investigation-render table :is(caption,th,td,p,span,strong,code),table :is(caption,th,td,p,span,strong,code){color:var(--table-ink)}\n#score-investigation-render table caption,table caption{background:transparent;color:var(--table-ink)}\n#score-investigation-render table :is(th,td),table :is(th,td){border-bottom-color:var(--table-rule)}\n#score-investigation-render table a,table a{color:#6a315f}\n.threshold-control{background:#f7edd6;color:#32283d;border:1px solid #e7d7b6}.threshold-control label,.threshold-control p,.threshold-control .small,.threshold-control .muted,.threshold-control .range-labels,.threshold-control .range-labels span{color:#554c5b}.threshold-control output{color:#6a315f}.highlight-row{background:#f7edd6}.sliding-options{background:#f5eddd;border-color:#d9ccb4}footer{background:#f2ecdf;border-top-color:#ded3bd}\n\n/* Measured findings: direct labels and symbols preserve meaning without color. */\n.cohort-figure{margin:28px 0 34px;padding:28px;background:#fffaf0;border:1px solid #e3d8c2;border-radius:12px;color:#32283d}.cohort-figure h2{font-size:28px}.cohort-row{padding:22px 0;border-top:1px solid #ded3bd;display:grid;grid-template-columns:1fr 1fr;gap:20px;align-items:center}.cohort-row p{margin:6px 0 0;font-size:15px;color:#554c5b}.cohort-marks{display:grid;grid-template-columns:repeat(6,30px);gap:7px;justify-content:end}.cohort-dot{height:30px;width:30px;display:grid;place-items:center;border-radius:5px;font:600 19px/1 system-ui}.cohort-dot.changed{background:#6a315f;color:#fffaf0;border:1px solid #6a315f}.cohort-dot.unchanged{background:#f7edd6;color:#493747;border:1px solid #bdac94}.cohort-key{margin:12px 0;color:#554c5b}.evidence-outcomes{margin:22px 0}.evidence-outcomes>div{padding:17px 0;border-top:1px solid var(--line)}.evidence-outcomes dt{font-weight:700;color:#6a315f;margin-bottom:5px}.evidence-outcomes dd{margin:0;color:#32283d}.review-implication{margin-top:30px}@media(max-width:600px){.cohort-figure{padding:20px}.cohort-row{grid-template-columns:1fr;gap:12px}.cohort-marks{justify-content:start}}\n\n/* Final policy-facing framing */\n.policy-chain{display:grid;grid-template-columns:repeat(5,minmax(0,1fr));gap:8px;align-items:stretch;margin:24px 0 10px}.policy-chain .node{background:#fffaf0;border:1px solid #d9ccb4;border-radius:8px;padding:14px 12px;text-align:center;font-size:13px;line-height:1.4;position:relative}.policy-chain .node:not(:last-child):after{content:"→";position:absolute;right:-10px;top:50%;transform:translateY(-50%);z-index:2;color:#6a315f;font-weight:700}.handoff-line{font:12px \'IBM Plex Mono\',monospace;color:#6a315f;margin:8px 0 24px}.audience-strip{display:flex;gap:8px;flex-wrap:wrap;margin:14px 0 24px}.audience-strip span{border:1px solid #d9ccb4;background:#fffaf0;border-radius:18px;padding:7px 11px;font-size:12px;font-weight:600}.policy-box{background:#f7edd6;border-left:4px solid #bf4935;padding:22px 24px;margin:28px 0}.policy-box h2{margin-bottom:10px}.policy-box p:last-child{margin-bottom:0}.finding-pair{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin:22px 0}.finding-pair>div{background:#fffaf0;border:1px solid #ded3bd;border-radius:10px;padding:20px}.finding-number{font:42px/1 \'Fraunces\',Georgia,serif;color:#6a315f;display:block;margin-bottom:7px}.claim-matrix{width:100%;border-collapse:separate;border-spacing:0;margin:20px 0}.claim-matrix th,.claim-matrix td{padding:12px;border-bottom:1px solid #ded3bd;vertical-align:top}.status-tag{display:inline-block;border-radius:14px;padding:3px 8px;font:11px \'IBM Plex Mono\',monospace;font-weight:600}.status-measured{background:#e1efe9;color:#1d5e51}.status-limited{background:#f7edd6;color:#744f1a}.status-gap{background:#f6dfda;color:#8a3427}.policy-actions{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:14px;margin:20px 0}.policy-actions>div{background:#fffaf0;border:1px solid #ded3bd;border-radius:8px;padding:18px}.consumer-takeaway{border:1px solid #d7c6d4;background:#eee1ef;padding:20px 22px;border-radius:8px;margin:24px 0}@media(max-width:760px){.policy-chain{grid-template-columns:1fr}.policy-chain .node:not(:last-child):after{content:"↓";right:50%;top:auto;bottom:-15px;transform:translateX(50%)}.finding-pair,.policy-actions{grid-template-columns:1fr}}\n\n/* Rules-to-tests and validation protocol */\n.rules-table{min-width:900px}.rules-table th{font-size:12px;letter-spacing:.02em}.rules-table td{line-height:1.5}.source-row{margin-top:14px}.protocol-tests{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin:28px 0}.protocol-card{background:#fffaf0;border:1px solid #ded3bd;border-radius:10px;padding:20px}.protocol-card h3{font-size:21px;color:#6a315f;margin:8px 0 10px}.protocol-card p{margin-bottom:10px}.protocol-card .reviewer-action{padding-top:10px;border-top:1px solid #ded3bd;font-size:13px;color:#554c5b}.protocol-card.wide{grid-column:1/-1}.protocol-matrix-section{margin-top:34px}.protocol-matrix-section h2{font-size:28px}.landing-links{display:grid;grid-template-columns:repeat(3,1fr);gap:14px}.landing-links button{text-align:left;text-decoration:none;border:1px solid #ded3bd;background:#fffaf0;border-radius:8px;padding:16px;color:#32283d}.landing-links button strong{display:block;color:#6a315f;font-size:15px;margin-bottom:6px}.landing-links button span{font-size:13px;line-height:1.5;color:#554c5b}.landing-next{max-width:none}.stress-test{margin-top:38px}.status-partial{background:#eee1ef;color:#60365a}@media(max-width:760px){.protocol-tests,.landing-links{grid-template-columns:1fr}.protocol-card.wide{grid-column:auto}}\n</style></head><body>\n<header><div class="brand">REASON <span>/ Explanation assurance for credit</span></div><nav aria-label="Main navigation"><button aria-current="page" data-page="about">Overview</button><button data-page="legal">Rules → tests</button><button data-page="investigate">Review a record</button><button data-page="compare">Compare reconstructions</button><button data-page="research">Validation protocol</button><button data-page="evidence">Evidence &amp; limits</button><button data-page="methods">Methods</button></nav></header>\n<main>\n<div class="case-toolbar" hidden="" id="case-toolbar"><label class="field">Historical credit record<select aria-label="Choose a historical credit record" id="case-select"></select></label><p>The same record is used on both comparison pages.</p></div>\n<section class="page" id="about">\n<div class="opening-layout">\n<div class="hero">\n<p class="eyebrow">Independent research · Consumer credit</p>\n<h1>From risk scores to reasons</h1>\n<p class="intro">When a credit application is declined, the applicant may receive only a short list of reasons. Those reasons can sit at the end of a much longer process: a lender model, a score supplied by another company, an explanation method, and a rule that turns technical evidence into notice language.</p>\n<p>This project looks at a basic governance problem inside that chain: <strong>what can the creditor actually substantiate about the reason it gives the applicant?</strong></p>\n<p>I use the public FICO HELOC data because they contain an external credit-risk score but not the formula or provider reasoning behind it. That makes it possible to test what a lender can learn by reconstructing an unavailable score, and where that inference has to stop.</p>\n<div aria-label="Intended users" class="audience-strip"><span>Consumer compliance</span><span>Model validation</span><span>Model risk</span><span>Counsel</span><span>Supervisory review</span></div>\n<div class="actions"><button class="primary" data-page="legal">Start with the rules and tests</button><button class="secondary" data-page="investigate">Inspect a record</button></div>\n</div>\n<aside aria-label="Illustrative credit denial letter excerpt" class="document-example">\n<p class="document-label">Illustrative letter · Decision and reasons excerpt</p>\n<div class="paper-notice"><span class="letter-stamp">EXCERPT</span><div class="letter-department">APPLICATION SERVICES</div><p class="letter-date">18 October 2026</p><p class="letter-subject">Re: Home equity line of credit application</p><p>Dear Applicant,</p><p>Following our review, we are unable to approve the home equity line of credit you requested.</p><p>The principal reasons for this decision are:</p><ul class="letter-reasons"><li>Limited credit experience</li><li>Number of recent inquiries on credit bureau report</li></ul><p>Your credit report was part of this review. The reporting agency did not make the credit decision.</p><p class="letter-signature">Sincerely,<br/>Credit Application Team</p></div>\n<p class="document-caption">Fictional illustration using reason wording from Regulation B sample forms. These reasons have not been assigned to a study record.</p>\n</aside>\n</div>\n<div class="context">\n<p class="eyebrow">Where the evidence can break</p>\n<div aria-label="Consumer-facing reason traces through notice mapping, lender model, external score, and score provider" class="policy-chain" role="img"><div class="node"><strong>Consumer-facing reason</strong><br/>What the applicant is told</div><div class="node"><strong>Reason mapping</strong><br/>How model evidence becomes notice language</div><div class="node"><strong>Lender model</strong><br/>The model the creditor can inspect</div><div class="node"><strong>External score</strong><br/>A supplied model input</div><div class="node"><strong>Provider model</strong><br/>The unavailable upstream mechanism</div></div>\n<p>Each step can require a different record or source of evidence. The experiment focuses on the last two steps: the lender observes the external score, but the HELOC data do not reveal how that score was calculated.</p>\n</div>\n<div class="context landing-connection">\n<h2>What the experiment found</h2>\n<div><div class="finding-pair"><div><span class="finding-number">4 / 12</span><strong>close-prediction cases had different leading detailed-factor sets</strong></div><div><span class="finding-number">0 / 12</span><strong>had different leading broader-topic sets</strong></div></div><p>The reconstructions can agree on the downstream prediction and on the broad topic while disagreeing about the detailed factors. That is why a close prediction is not enough to establish the original provider\'s reasons.</p><p class="small muted">The 12 records are an exploratory subset of 40 selected historical HELOC records. The counts are descriptive, not population estimates or legal thresholds.</p><button class="text-button" data-page="research">See the validation protocol and study-wide stress test →</button></div>\n</div>\n<div class="context landing-next">\n<h2>How to read the project</h2>\n<div class="landing-links"><button class="text-button" data-page="legal"><strong>Rules → tests</strong><br/><span>How Regulation B, CFPB supervision, model-risk practice and other sources translate into review questions.</span></button><button class="text-button" data-page="investigate"><strong>Review a record</strong><br/><span>What the supplied score contributes to the research model, and what the data leave unknown.</span></button><button class="text-button" data-page="research"><strong>Validation protocol</strong><br/><span>Five checks that separate measured evidence from claims that need lender or provider records.</span></button></div>\n</div>\n</section>\n<section class="page" hidden="" id="investigate"><p class="eyebrow">The original research model</p><h1>A repayment prediction using an externally supplied score</h1><p class="intro">The Random Forest uses account history, balances and inquiries alongside the score recorded as <code>"ExternalRiskEstimate"</code>. It predicts the dataset’s serious-late-payment outcome: at least 90 days overdue, or worse, within 24 months of opening. The data contain repayment outcomes rather than actual lending decisions.</p><div aria-live="polite" id="case-content"></div><details><summary>Check the LIME-style approximation of this prediction</summary><div id="local-fit"></div></details><details><summary>View the credit information for this record</summary><div class="table-scroll" id="record-inputs"></div></details><div class="actions"><button class="primary" data-page="compare">Compare the two score estimates</button></div></section>\n<section class="page" hidden="" id="compare"><p class="eyebrow">The external-score experiment</p><h1>Explanations depend on the model used to estimate the score</h1><p class="intro">I trained two models to estimate the supplied score from eight other credit attributes: Ridge regression adds weighted inputs, while gradient-boosted trees can learn interactions between them. Each estimate replaces the score in the unchanged Random Forest. The charts explain the resulting repayment predictions, including the contribution through the estimated score. Different models can legitimately have different contributions; the question is how those differences relate to prediction agreement and to the broader topics a notice might use.</p><div class="compare-head"><h2>Credit factors in each explanation</h2><div aria-label="Choose explanation detail" class="sliding-options" data-mode="factors" id="explanation-mode" role="group"><button aria-pressed="true" id="factors-view">Individual factors</button><button aria-pressed="false" id="topics-view">Grouped factors</button></div></div><div aria-live="polite" id="approximation-charts"></div><div aria-live="polite" class="conclusion" id="comparison-result"></div><p class="small muted">Grouping adds the signed contributions of related measures, such as account ages, into broader credit topics. Lenders sometimes group features when producing notice reasons; these research groupings allow the comparison to show whether detailed changes also affect the broader topics.</p><div class="actions"><button class="primary" data-page="research">Read the findings</button></div></section>\n<section class="page reading" hidden="" id="research">\n<p class="eyebrow">Validation protocol</p>\n<h1>Five tests connect XAI evidence to a reviewable claim</h1>\n<p class="intro">The protocol does not produce a single explanation score. Each test asks a different question and says whether this project measures it, only partly addresses it, or lacks the evidence needed to answer it.</p>\n<div aria-live="polite" class="protocol-tests" id="protocol-tests"></div>\n<section class="protocol-matrix-section"><div aria-live="polite" id="protocol-case-matrix"></div></section>\n<section class="stress-test"><p class="eyebrow">Study-wide stress test</p><h2>Prediction agreement and explanation agreement</h2><p>Change the prediction-agreement band to see how the result changes. These are research comparison bands, not legal acceptance thresholds.</p><div class="threshold-control"><label for="agreement-band">Maximum difference between the three repayment predictions <output for="agreement-band" id="agreement-band-label">1.0 percentage points</output></label><input aria-valuetext="1.0 percentage points" id="agreement-band" max="3" min="0" step="1" type="range" value="1"/><div aria-hidden="true" class="range-labels"><span>0.5</span><span>1.0</span><span>2.0</span><span>5.0</span></div><p class="small muted">Only the records inside the chosen band are counted below.</p></div><div id="findings-summary"></div></section>\n<section class="export"><h2>Save the evidence for a record</h2><p class="small">The note uses the record selected above and separates what is measured from what still needs lender or provider records.</p><button class="text-button" data-page="compare">Inspect or change the selected record →</button><p class="small muted" id="report-selection"></p><div class="actions"><button class="secondary" id="prepare-report">Prepare review note</button></div><div hidden="" id="report-output"><div class="download-links"><a id="open-report" rel="noopener" target="_blank">Open note</a><a download="score-review.html" id="download-report">Download note (.html)</a><a download="score-review.json" id="download-json">Download measurements (.json)</a></div><p class="report-state" id="report-state" role="status"></p><details><summary>Copy the note text</summary><div><textarea aria-label="Review note text" id="report-preview" readonly=""></textarea></div></details></div></section>\n</section>\n<section class="page reading" hidden="" id="methods"><p class="eyebrow">Research documentation</p><h1>Methods and numerical results</h1><section><h2>Dataset and models</h2><p>The public FICO HELOC data contain credit attributes and an account repayment outcome. After cleaning, the study fitted a nine-input Random Forest to predict the “Bad” repayment category. Eight inputs describe credit history; the ninth is the externally supplied score. The data do not identify the score’s original formula or provider.</p><p>Ridge regression and histogram gradient boosting were trained to estimate that score using the eight credit attributes. Training used only the original training partition, excluding negative external-score status codes. Each estimate was clipped to the observed training range before substitution into the frozen Random Forest.</p><div class="table-scroll" id="performance-table"></div></section><section><h2>Shapley and LIME-style calculations</h2><p>The displayed Shapley values were calculated by exact empirical interventional enumeration against 24 training records. Each value allocates part of the difference between the reference average and the prediction. All three systems use the same reference-record identities, with estimated scores substituted in the reconstructed systems. Correlated-input replacements can create implausible combinations.</p><p>A custom LIME-style weighted regression was fitted five times near each record. Separate generated records evaluated its approximation error and weighted R². Generated records mix the applicant’s values with training values and may violate credit-feature dependencies. The displayed results use these custom implementations; the official SHAP and LIME packages were not executed.</p><p>For each explanation, the leading list contains up to four positive contributions. Grouped comparisons first add signed contributions within research-defined credit topics. Neither list is treated as a validated consumer notice.</p></section><section><h2>Sample and comparison limits</h2><p>Prediction accuracy was evaluated on 1,973 test records with observed external scores. Forty previously selected records received detailed explanations. They were originally selected to contrast predictions near 0.50 and high predictions in an earlier eight-input model; they are not a representative sample of credit refusals. The default example, record 8193, was selected after inspection.</p><p>The table counts records where both reconstructed probabilities are within the stated distance of the original probability and of each other. These are research comparison limits, not regulatory thresholds. Test results had already been inspected during earlier work and a feasibility run, so this study is exploratory.</p><div class="table-scroll" id="agreement-table"></div><p class="small muted">The study does not evaluate actual notices, protected-group fairness, consumer understanding or reviewer effectiveness.</p></section><section><h2>Data and method references</h2><p><a href="https://github.com/Trusted-AI/AIX360/blob/master/aix360/datasets/heloc_dataset.py" rel="noopener" target="_blank">HELOC dataset documentation</a> · <a href="https://arxiv.org/abs/1705.07874" rel="noopener" target="_blank">SHAP</a> · <a href="https://arxiv.org/abs/1602.04938" rel="noopener" target="_blank">LIME</a> · <a href="https://arxiv.org/abs/1908.08474" rel="noopener" target="_blank">Alternative Shapley definitions</a></p><p>FinRegLab’s consumer-lending research evaluates explanation fidelity and consistency across model families. Its tests offer a basis for evaluating explanations, without establishing the adequacy of actual notices. <a href="https://finreglab.org/wp-content/uploads/2023/12/FinRegLab_2023-07-13_Empirical-White-Paper_Explainability-and-Fairness_Insights-from-Consumer-Lending.pdf" rel="noopener" target="_blank">Empirical report</a></p></section></section>\n<section class="page reading" hidden="" id="legal">\n<p class="eyebrow">Rules → claims → tests</p>\n<h1>The governance material becomes the specification</h1>\n<p class="intro">The legal and governance sources do not all ask the same question. I use them to define what a reviewer would need to establish, then map each claim to the test this project can run and the records it still cannot supply.</p>\n<div class="table-scroll"><table class="rules-table"><thead><tr><th>Source / practice</th><th>Review claim</th><th>Protocol test</th><th>Evidence still needed</th></tr></thead><tbody>\n<tr><td><strong>ECOA / Regulation B</strong><br/><span class="small">Specific reasons must relate to factors actually considered or scored.</span></td><td>Does the proposed reason trace to the decision evidence?</td><td>Decision fidelity + mapping integrity</td><td>Actual decision path, scored factors, mapping/version and notice wording</td></tr>\n<tr><td><strong>CFPB supervisory finding</strong><br/><span class="small">Reason-selection methods need validation.</span></td><td>Has the process for selecting reasons been tested rather than assumed?</td><td>Fidelity + reason consistency</td><td>Validation design, cases, exceptions and retained results</td></tr>\n<tr><td><strong>Industry example: Upstart</strong><br/><span class="small">Feature-level SHAP contributions may be grouped into concept-level reasons.</span></td><td>What changes when detailed features are translated into notice categories?</td><td>Reason consistency + mapping integrity</td><td>Grouping rules, directionality checks and production mapping</td></tr>\n<tr><td><strong>Third-party model risk</strong><br/><span class="small">Proprietary vendor components complicate validation.</span></td><td>Is the reason evidence from the original provider or from a reconstruction?</td><td>Provenance</td><td>Provider version, inputs, reason codes and validation access</td></tr>\n<tr><td><strong>EU AI Act comparison</strong><br/><span class="small">High-risk systems emphasize records, information, oversight and traceability.</span></td><td>Could the explanation process be reconstructed and reviewed?</td><td>Auditability</td><td>Versioned records, system information and oversight trail</td></tr>\n<tr><td><strong>State / product rules</strong></td><td>Which lending and notice requirements apply to this transaction?</td><td>Applicability overlay, not an XAI metric</td><td>Charter, product, collateral, terms, jurisdiction and consumer-report use</td></tr>\n</tbody></table></div>\n<p class="small source-row"><a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/9/" rel="noopener" target="_blank">Regulation B §1002.9</a> · <a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/" rel="noopener" target="_blank">Official interpretation</a> · <a href="https://www.upstart.com/lenders/regulatory-compliance/adverse-action-notices/" rel="noopener" target="_blank">Upstart adverse-action methodology</a> · <a href="https://eur-lex.europa.eu/eli/reg/2024/1689/eng" rel="noopener" target="_blank">EU AI Act</a> · <a href="https://www.federalreserve.gov/supervisionreg/srletters/SR2602.pdf" rel="noopener" target="_blank">SR 26-2 model-risk guidance</a></p>\n<section class="policy-box"><p class="eyebrow">Policy recommendation</p><h2>Treat an external score as a separate explanation dependency</h2><p>If a consumer-facing reason depends on a material external score, the creditor should be able to show which part of the explanation comes from its own model and which part is supported by the score provider. A reconstruction can be useful for testing sensitivity; it is not a substitute for provider-linked reasons or validation records.</p></section>\n<section class="case-entry"><p class="eyebrow">CFPB · Auto lending · 2025</p><h2>Lenders had not validated the accuracy of their denial reasons</h2><p>CFPB examiners reviewed AI/ML credit-scoring models, some with more than 1,000 variables. They found that institutions had not validated whether their reason-selection processes produced accurate results and directed the institutions to test those methods.</p><p>This project examines a related evidence problem: producing a prediction and an explanation does not by itself establish that the selected denial reason is accurate. The CFPB report does not identify SHAP or LIME as the methods used in the cited examinations.</p><a href="https://business.cch.com/BFLD/CFPB-Supervisory-Highlights-Advanced-Technologies-Special-Edition-Issue-38-Winter-2025-01172025.pdf" rel="noopener" target="_blank">Supervisory Highlights, Winter 2025, §2.1.3, pp. 7–8</a></section>\n<section class="case-entry"><p class="eyebrow">NYDFS · Apple Card · 2021</p><h2>A regulator could inspect decisions that customers struggled to understand</h2><p>Goldman could explain individual credit decisions to NYDFS, while customers reported poor transparency and difficult appeals over credit terms. The investigation did not establish unlawful discrimination. The case is useful here because regulator access to decision evidence and consumer access to a usable explanation are different questions. This project does not test applicant comprehension.</p><a href="https://www.dfs.ny.gov/system/files/documents/2021/03/rpt_202103_apple_card_investigation.pdf" rel="noopener" target="_blank">NYDFS investigation report, pp. 7–8</a></section>\n<details><summary>Further examples: incorrect notices and an external credit assessment</summary><div><p><strong>CFPB, 2020:</strong> payday lenders sent incorrect principal reasons, including notices affected by a coding error. The lenders corrected the notices and changed their systems. No AI explanation method was identified. <a href="https://files.consumerfinance.gov/f/documents/cfpb_supervisory-highlights_issue-21_2020-02.pdf" rel="noopener" target="_blank">Official finding, §2.3.5</a></p><p><strong>Dun &amp; Bradstreet Austria, 2025:</strong> after a mobile-contract refusal based on an automated credit assessment, the CJEU required an intelligible account of how the person\'s data were used. This case arises under GDPR, not U.S. adverse-action law. <a href="https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:62022CJ0203" rel="noopener" target="_blank">C-203/22 judgment</a></p><p><strong>CFPB Adverse Action Tech Sprint, 2020:</strong> participants explored digital notices and interactive explanations. It was a design event rather than an enforcement finding. <a href="https://www.consumerfinance.gov/rules-policy/competition-innovation/cfpb-tech-sprints/electronic-disclosures-tech-sprint/" rel="noopener" target="_blank">Official account</a></p></div></details>\n<p class="small muted">The CFPB withdrew Circulars 2022-03 and 2023-03 in May 2025. The notice standard used here rests on Regulation B and its official interpretation. <a href="https://www.consumerfinance.gov/compliance/guidance/withdrawn-guidance/" rel="noopener" target="_blank">Source status</a></p>\n<div class="actions next-page"><button class="primary" data-page="research">Apply the validation protocol</button></div>\n</section>\n<section class="page reading" hidden="" id="evidence"><p class="eyebrow">Evidence &amp; limits</p><h1>What the project can support, and what it cannot</h1><p class="intro">An attribution is evidence about the model it explains. The harder question is whether that evidence is enough for the claim a reviewer wants to make about the full decision process.</p><p class="intro">The sections below separate records needed for review from the explanation methods that can help examine them.</p>\n<section class="policy-box"><p class="eyebrow">Core claim</p><h2>A downstream explanation can establish what happened in a model you can inspect. It cannot, by itself, authenticate an unavailable upstream scoring mechanism.</h2><p>For an external score, the review should end in one of four states: <span class="status-tag status-measured">MEASURED</span> <span class="status-tag status-limited">SUPPORTED WITH LIMITATIONS</span> <span class="status-tag status-gap">NOT ESTABLISHED</span> or <span class="status-tag status-gap">PROVIDER / LENDER RECORDS REQUIRED</span>.</p></section>\n<table class="claim-matrix"><thead><tr><th>Claim a reviewer may want to make</th><th>Evidence in this project</th><th>Status</th></tr></thead><tbody><tr><td>The supplied external score affected the lender-model prediction.</td><td>Exact Shapley contribution in the research Random Forest.</td><td><span class="status-tag status-measured">MEASURED</span></td></tr><tr><td>A reconstructed score can reproduce a similar downstream prediction.</td><td>Ridge and boosted-tree substitution tests.</td><td><span class="status-tag status-measured">MEASURED</span></td></tr><tr><td>The inferred detailed factors were the original provider’s reasons.</td><td>No authenticated upstream formula or provider reasons are present.</td><td><span class="status-tag status-gap">NOT ESTABLISHED</span></td></tr><tr><td>A real consumer notice accurately reflects the production decision path.</td><td>No actual lender decision, mapping version or delivered notice is in the dataset.</td><td><span class="status-tag status-gap">RECORDS REQUIRED</span></td></tr></tbody></table>\n<div aria-label="Choose evidence reference" class="sliding-options evidence-options" data-mode="requirements" id="evidence-switch" role="group"><button aria-pressed="true" id="requirements-view">Requirements &amp; records</button><button aria-pressed="false" id="other-methods-view">Explanation methods</button></div>\n<div class="evidence-panel" id="requirements-panel">\n<details open=""><summary>Specific principal reasons for the adverse action</summary><div><p><strong>Records needed.</strong> The actual decision path, scored factors, policy overrides, reason-selection method and notice wording.</p><p><strong>Role of XAI.</strong> Signed Shapley/SHAP contributions explain a prediction relative to a reference. A reviewer must validate the reference and how selected factors become the stated reasons. An actual scorecard or rule-based model may expose its contributions directly. Estimating a supplied score does not identify its original factors.</p><p class="small"><a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/9/" rel="noopener" target="_blank">§1002.9(b)(2)</a> · <a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/" rel="noopener" target="_blank">Comments 9(b)(2)-2, -4 and -5</a></p></div></details>\n<details><summary>Choice of comparison population</summary><div><p><strong>Records needed.</strong> The reference-group definition, selection process, sample sizes and results under alternative choices.</p><p><strong>Role of XAI.</strong> A reference-sensitivity experiment can show how Shapley reason selection changes when the comparison group changes. The official interpretation describes scoring comparisons with applicants near a passing score or with all applicants. A generic SHAP background is not automatically equivalent. This external-score study uses a fixed set of 24 reference records.</p><a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/" rel="noopener" target="_blank">Comment 9(b)(2)-5</a></div></details>\n<details><summary>Translation from detailed factors to notice categories</summary><div><p><strong>Records needed.</strong> Versioned mapping rules, the factors in each category, signed effects and paired examples before and after grouping.</p><p><strong>Role of XAI.</strong> Compare the retention and turnover of leading factors with the retention and turnover of grouped categories. The site provides both views. Independent evaluation of whether the wording accurately describes those factors remains additional work; grouping alone does not validate a notice.</p><a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/" rel="noopener" target="_blank">Comments 9(b)(2)-2 through -4</a></div></details>\n<details><summary>Notification, request procedure and retained records</summary><div><p><strong>Records needed.</strong> Application-completion and action dates, delivery records, creditor and agency identity, the ECOA statement, and reasons or the applicable request procedure.</p><p><strong>Role of XAI.</strong> Attribution methods do not produce these records. Workflow and notice-template checks need actual operational data. Consumer-application records are generally retained for 25 months, with longer periods in specified proceedings. Versioned models and mappings can help reconstruct the explanation used at the time.</p><a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/9/" rel="noopener" target="_blank">§1002.9(a)</a> · <a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/12/" rel="noopener" target="_blank">§1002.12</a></div></details>\n<details><summary>Consumer-report and credit-score disclosures</summary><div><p><strong>Records needed.</strong> Whether a consumer report or qualifying credit score was used, the supplier, actual score and applicable disclosure fields.</p><p><strong>Role of XAI.</strong> Track creditor-decision reasons separately from the score’s adverse factors. FCRA disclosures do not automatically satisfy Regulation B’s specific-reasons duty. The dataset’s external-score field does not identify the original supplier or applicable notice contents.</p><a href="https://www.ftc.gov/business-guidance/resources/using-consumer-reports-credit-decisions-what-know-about-adverse-action-risk-based-pricing-notices" rel="noopener" target="_blank">FTC notice guidance</a> · <a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/" rel="noopener" target="_blank">Comment 9(b)(2)-9</a></div></details>\n<details><summary>Group-level discrimination and the applicable legal framework</summary><div><p><strong>Records needed.</strong> Protected-group information, lending outcomes, the challenged practice, appropriate comparisons and evidence about alternatives.</p><p><strong>Role of XAI.</strong> Subgroup evaluation can locate differences, and explanations can help inspect features. Neither supplies a complete discrimination analysis. For housing-related lending, the Fair Housing Act’s discriminatory-effects framework requires separate analysis. This study does not evaluate protected-group outcomes.</p><a href="https://www.ecfr.gov/current/title-24/subtitle-B/chapter-I/part-100/subpart-G/section-100.500" rel="noopener" target="_blank">24 CFR 100.500</a> · <a href="https://www.consumerfinance.gov/rules-policy/regulations/1002/6/" rel="noopener" target="_blank">Current Regulation B evaluation rules</a></div></details>\n<details><summary>North Carolina product requirements and third-party governance</summary><div><p><strong>Records needed.</strong> Lender charter, collateral, lien position, transaction terms, consumer-report use and state/federal applicability analysis. For an outside score, obtain the applicable version, input documentation, provider reasons and relevant validation evidence.</p><p><strong>Role of XAI.</strong> Trace a displayed explanation to its evidence to investigate its accuracy. The explanation cannot establish the loan’s terms or legal scope. Current interagency model-risk guidance discusses proprietary vendor products; it does not prescribe a single explanation method.</p><a href="https://www.ncleg.gov/EnactedLegislation/Statutes/HTML/BySection/Chapter_75/GS_75-1.1.html" rel="noopener" target="_blank">NC G.S. 75-1.1</a> · <a href="https://www.ncleg.gov/EnactedLegislation/Statutes/HTML/ByChapter/Chapter_24.html" rel="noopener" target="_blank">NC Chapter 24</a> · <a href="https://www.federalreserve.gov/supervisionreg/srletters/SR2602a1.pdf" rel="noopener" target="_blank">Model-risk guidance, §VII</a></div></details>\n</div>\n<div class="evidence-panel" hidden="" id="other-methods-panel">\n<p class="method-status">Shapley and custom LIME-style calculations were run in this study. The other methods below are possible additions, not reported experiments.</p>\n<details open=""><summary>Shapley / SHAP: contributions relative to a comparison</summary><div><p>Allocates a prediction’s difference from a reference across features. Reference sensitivity and individual-versus-grouped contributions can investigate reason-selection choices.</p><p><strong>Evaluate:</strong> reference justification, feature dependence, signed aggregation and the connection to actual scored factors. Attribution of a supplied score leaves that score’s original calculation unexplained.</p><a href="https://arxiv.org/abs/2204.12365" rel="noopener" target="_blank">Nair and colleagues: adverse-action Shapley decomposition</a></div></details>\n<details><summary>LIME: a simple approximation near one record</summary><div><p>Fits an interpretable local model around a prediction. Separate generated records and repeated random seeds can assess local fidelity and repeatability.</p><p><strong>Evaluate:</strong> approximation error at the record, held-out local fit, sampling variability and whether generated combinations respect credit-feature relationships. A good local fit cannot establish the original reasons of an unknown upstream scorer.</p><a href="https://arxiv.org/abs/1602.04938" rel="noopener" target="_blank">Ribeiro, Singh and Guestrin: LIME</a></div></details>\n<details><summary>Interpretable scorecards, additive models and decision rules</summary><div><p>Expose the contributions or rules of the model actually used. A separately evaluated interpretable alternative could establish whether comparable predictive performance is achievable with a more inspectable design.</p><p><strong>Evaluate:</strong> predictive quality, actual rule execution and the wording of the resulting reasons. A transparent substitute explains the substitute, not the original complex model.</p><a href="https://docs.interpretable.ai/stable/examples/fico/" rel="noopener" target="_blank">HELOC interpretable-model example</a></div></details>\n<details><summary>Counterfactual explanations / DiCE: changes that reverse a decision</summary><div><p>Search for alternative inputs that change the specified model’s output. They can support a separate investigation of feasible recourse.</p><p><strong>Evaluate:</strong> decision reversal, immutable fields, costs, feasible changes, stability and the full lending policy. A path to approval answers a different question from the principal reasons for the original refusal.</p><a href="https://interpret.ml/DiCE/readme.html" rel="noopener" target="_blank">DiCE documentation</a></div></details>\n<details><summary>Anchors: local conditions that preserve a prediction</summary><div><p>Finds rule-style conditions under which a prediction remains the same with high precision under a defined sampling distribution. It can supplement ranked feature explanations.</p><p><strong>Evaluate:</strong> precision, coverage and the relevance of the perturbation distribution. An anchor is a local behavioral rule; it does not recover an unknown provider’s internal mechanism.</p><a href="https://ojs.aaai.org/index.php/AAAI/article/view/11491" rel="noopener" target="_blank">Ribeiro and colleagues: Anchors</a></div></details>\n</div>\n<p class="note">There is no general legal pass mark for AUC, explanation fit or reason retention. The conclusion must address a specific claim and the records supporting it.</p>\n</section>\n</main><footer><span>Independent project · ms@duke.edu · Public FICO HELOC data</span><span>Dataset, experiment and sources documented in the tabs above</span></footer>\n<script src="data.js"></script><script src="app.js"></script>\n</body></html>\n', 'docs/CHALLENGE_ADDENDUM.md': '# Supplementary challenge examples\n\nWritten after the main benchmark results were inspected; not part of the frozen primary evaluation. No main benchmark thresholds or review rules will change in response.\n\nPurpose: expose limits of exact catalog checks and trusted source evidence, so favorable controlled results are not mistaken for general semantic or legal validation.\n\nOn each of the same 20 evaluation records, create two additional examples:\n1. An accurate neutral paraphrase not in the approved wording catalog. The expected audit behavior is a wording flag, although the phrase is stipulated correct in this constructed example. This measures a narrow-catalog false-alarm challenge, not real consumer readability.\n2. A wrong reason description entered consistently in both the notice and retained catalog, preserving matching source-feature IDs. The expected audit behavior is no discrepancy: the supplied catalog is the source of truth for the checker and it has no independent semantic oracle. This exposes a trusted-evidence blind spot.\n\nReport these 40 challenge cases separately. These examples are same-author constructions deliberately targeting known checker limitations. They do not estimate prevalence or constitute a second held-out generalization study.\n', 'docs/DECISION_RECONSTRUCTION_PROTOCOL.md': "# Can the institution reconstruct the denial?\n\n## Question and contribution\n\nCan an institution reconstruct a credit decision from an explanation report alone? Which additional retained records make specified investigative tasks possible, and which questions remain unanswered?\n\nThis is a controlled evidence-removal experiment, specified on 18 October 2026 after the earlier explanation experiments. It is not preregistered. The contribution is a source-linked evidence specification, executable reconstruction procedure, and measured demonstration of the consequences of missing or mismatched records. It is not a new explanation algorithm.\n\n## What is measured and what is constructed\n\nMeasured: the existing frozen eight-input Random Forest, forty selected historical HELOC records, its predictions, exact empirical interventional Shapley decompositions, custom LIME-style fits and independent-neighborhood fidelity. Original model outputs remain unchanged.\n\nConstructed: an original lending policy that declines when the model's estimated probability of the dataset's `Bad` category is at least 0.50; a later policy using 0.65; decision timestamps, policy versions, and event records. These thresholds have no regulatory status and are not observed lender practices. Selection of twenty records just above 0.50 and twenty high estimates predates this experiment. Results describe this deliberately contrasting sample, not an estimate of lender error frequency.\n\nHELOC means home equity line of credit, borrowing secured against equity in a home. Repayment labels are not lending decisions. No actual notices, appeals, human decisions, or protected-group attributes are supplied by this dataset.\n\n## Procedure\n\nFor each of the forty records, evaluate seven retained-evidence packages:\n\n1. Explanation report only: prediction and Shapley contributions, without executable model, original input snapshot or policy.\n2. Original records: executable model and preprocessing, original input snapshot, original policy, logged prediction and decision, and explanation.\n3. Original policy missing: original inputs and model remain.\n4. Newer policy substituted: the original rule is replaced with the 0.65 rule.\n5. Different model supplied: the retained artifact is substituted; its digest no longer matches the logged model identity.\n6. Input snapshot altered: an account-age value is changed without changing the logged input identity.\n7. Explanation altered: a contribution is increased by 0.10 without changing the recorded prediction or reference value.\n\nThe evaluator checks content digests against the original event record, actually reruns the frozen Python predictor for valid input/model combinations, applies a verified original policy, and checks whether reference plus contributions reconstructs the logged prediction. It abstains from original-decision reconstruction when identity cannot be verified. Hashes are assumed to have been retained in an independent, trustworthy event record; an attacker changing both records and digests defeats this check.\n\nSeparately, the experiment applies the later policy to the same measured predictions. This demonstrates how a changed institutional rule can alter outcomes without changing a prediction or its explanation. It is not an experiment in model fairness or explanation accuracy.\n\n## Outcomes\n\nReport counts for (a) arithmetic agreement of the explanation, (b) successful original-prediction reproduction, (c) successful original-decision reconstruction, and (d) detected identity or arithmetic inconsistencies. Keep missing evidence separate from contradictory evidence. No aggregate legal pass/fail score is produced.\n\nThe expected results are constructed from the manipulations and independently asserted in tests. Agreement with those expectations is an implementation check, not an external validation of the framework. No clinician, lender, regulator or user-study participant has validated the process. A subsequent reviewer study should measure task accuracy, unsupported conclusions, evidence requests and completion time against a chart-only baseline.\n\n## What reconstruction cannot establish\n\nArithmetic agreement does not prove a Shapley attribution identifies legally sufficient principal reasons, causal drivers, valid counterfactuals or fair treatment. Successful replay does not establish data accuracy at source, lender-policy lawfulness, effective human oversight or notice delivery. Upstream score calculations remain unavailable even in the strongest package.\n\n## Legal interpretation versus engineering proposal\n\nRegulation B commentary 9(b)(2)-2, -4 and -7 connects reasons to actual scored/considered factors and the component responsible for denial. EU AI Act Articles 12, 13, 14, 26, 27 and 86 motivate distinct tasks concerning records, output interpretation, oversight, impact assessment and explanation. Our particular JSON fields and digest checks are proposed implementation choices, not statutory field specifications.\n\nUS and EU regimes are separate scope analyses. The US HELOC experiment does not become an EU-regulated deployment. New York S8115C is proposed lending legislation; NYDFS Circular Letter 7 (2024) concerns insurance. Neither is an additional applicable HELOC duty by default. Source status and timing are presented in the site.\n", 'docs/EXTERNAL_SCORE_ADDENDUM.md': "# External-score experiment and review redesign\n\nRecorded October 16, 2026 before generating the new paired experiment. The earlier with-score model's AUC and 32 common rejected cases were already known; this is an exploratory extension, not an independent preregistration or confirmation.\n\nQuestion: Does explaining the lender-side model leave reliance on the supplied external risk score unexplained?\n\nUse the existing train/validation/test partition and eight-feature Random Forest settings. Train a second model adding only ExternalRiskEstimate. Explain all 40 original audited applications in both models, including applications whose simulated outcome changes; do not filter for rejection by both. Use the same 24 training record IDs as the original seed-2026 background. Original inputs and status indicators move together. The original eight-feature results must reproduce numerically.\n\nCompute exact empirical interventional Shapley values for both models (256 and 512 coalitions). Report per-application scores, full signed contributions, grouped contributions, whether the external score is among at most four positive contributors, and outcome changes at the fixed research threshold 0.50. Report AUC on the entire original test partition, paired bootstrap AUC differences (1,000 resamples, seed 20261016), and disagreement at the threshold. Test-set permutation importance uses ten independent column permutations per input, on raw input blocks before encoding. Correlations can redistribute importance; permutation is model reliance under a defined intervention, not causal effect or upstream-score validation.\n\nFor each paired application and both models, compare five custom LIME-style fits with 1,200 independently sampled evaluation points. Reuse the original eight-feature fidelity results; evaluate nine-feature fits using the same neighborhood generator. Retain all seeds. Rank and grouping changes are descriptive across different fitted models, not interventions on a fixed lending decision and not explanations of the original model.\n\nThe HELOC data contain values of ExternalRiskEstimate but no authenticated upstream scoring model, calculation, vendor specification or upstream reason records. Do not infer these from correlated bureau features. Do not label the supplied score proprietary without evidence. Treat this as an external-score dependency whose upstream explanation is absent from this snapshot.\n\nThe workbench must distinguish model diagnosis, the recorded research decision, applicant-facing wording, and reviewer judgment. Add an external-score evidence request to downloadable reviews only when the reviewer chooses it. Explain technical terms at point of use, preserve precise methods in expandable details, and keep the primary protocol unchanged.\n", 'docs/EXTERNAL_SCORE_RECONSTRUCTION.md': "# Can an inferred scoring model substantiate the original reasons?\n\n## Design and provenance\n\nExploratory extension specified 18 October 2026 after a feasibility run. The pilot had already shown external-score test MAE about 3.85 points for standardized Ridge and 3.24 for histogram gradient boosting using the eight existing credit inputs. No claim of preregistration or independent confirmation is made. No reviewer study is planned for this submission. Record 8193 was selected after inspection as a detailed-factor-change example with close prediction agreement; all forty audited records remain selectable.\n\nThe existing nine-input Random Forest is held fixed. Its observed `ExternalRiskEstimate` input is replaced with a prediction from either of two approximations trained only on the original training partition: standardized Ridge(alpha=10) or HistGradientBoostingRegressor(max_iter=150,max_leaf_nodes=15,l2_regularization=5,random_state=2026). Approximation fitting excludes records whose external score is a negative status code. Neither approximation is an authenticated reconstruction of the original scoring system.\n\nThe raw external score is the regression target; repayment class is not the approximation target. The approximation encoder is fitted on eligible training records only. Estimates are clipped to the eligible training target's observed min/max; the clipping rule is reported. Both the lender-side predictor and the two approximation models are deterministic for this run. The test partition had already been used in prior studies and in the feasibility run: estimates are exploratory and must not be described as pristine confirmatory performance.\n\n## Questions and measurements\n\n1. How closely can available inputs predict the observed external score? Report test MAE, R-squared and error quantiles for both approximations.\n2. How closely do the reconstructed combined systems match the original nine-input model? Report mean absolute probability difference on all eligible test records and per audited record. No observed approval/denial or lender policy is inferred.\n3. Do reconstructed combined systems assign the same leading adverse contributions? For the forty previously selected records with observed external scores, compute exact empirical interventional Shapley attributions for both eight-input reconstructed pipelines against the identical 24 eligible training reference records. Recalculate the original nine-input model’s attributions against those same record IDs, using their observed external scores. Its model and predictions remain unchanged; these new reference-conditioned contributions are distinct from the earlier experiment’s configuration.\n4. Compare top-four positive-contribution sets, their order and credit-concept grouping. Report different memberships separately from changed ordering and differences between credit concepts. Do not label any of these legal violations or consumer harm.\n5. Report joint output-agreement and explanation-agreement results at several descriptive probability-difference bands (0.5, 1, 2 and 5 percentage points). These are sensitivity bands, not adequacy thresholds. Also report differences from the original model so two equally inaccurate reconstructions are not mistaken for good approximations.\n\nThe constructed combined systems use the original eight raw inputs, each estimated score and the frozen original nine-input model. Other raw inputs remain directly available to that model. Therefore direct and score-mediated paths can overlap. Their explanations are of these combined systems, not estimates of the provider's true internal feature contributions. Correlated-feature replacements and special missingness states can create implausible Shapley combinations.\n\n## Governance response\n\nAssess four claims separately: the original lender-side model's reliance on an observed score; predictive accuracy of an inferred score model; the inferred model's attributions; and identification of factors actually used inside the original score. Empirical predictive fit supports the second claim, not automatically the fourth. Agreement between inferred models does not authenticate the original process; disagreement can show that this reconstruction strategy has not resolved explanatory uncertainty.\n\nThe outcome is a documented assessment of what each evidence source supports and a specific request for score-provider documentation, decision-linked reason codes or validation access where the proposed use requires those facts. It is not an automated notice certification or a blanket restriction on using external scores. US Regulation B reason duties, FCRA score disclosures, EU AI Act institutional transparency and New York materials are separate scope analyses.\n\n## Evaluation and boundaries\n\nEvaluate actual numerical accuracy, decomposition reconstruction and cross-approximation differences. Do not inflate software checks into evidence of reviewer effectiveness. The study cannot establish what the original external-score system actually used, real notice accuracy, group fairness, legal compliance or applicant comprehension. A poor reconstruction is still a reportable result. No result is required to show high predictive agreement with explanation disagreement.\n", 'docs/IMPORT_FORMAT.md': '# Import your own explanation evidence\n\nUse case: a lender\'s consumer-compliance or model-risk reviewer has received explanation outputs from a model developer or vendor and wants to compare how a fixed prediction is attributed under different comparison populations or samples.\n\nThe browser compares the supplied attributions and generates a source-linked evidence brief. It does not execute an unknown model or infer that its outputs are authentic. It does not compare wording against a supposed approved list and does not generate an overall compliance verdict.\n\nOpen **Import explanation results** in the site. Download its example JSON, or export using `src/evidence_export.py`. Files are processed in memory in the browser; no backend or network request processes them. Maximum file size 5 MB, 500 application examples, 300 inputs, and 20 explanations per application.\n\nRequired structure (schematic—not empirical research results):\n\n```json\n{\n  "schema_version": "credit-explanation-evidence/v1",\n  "title": "Example vendor evidence export",\n  "model": {\n    "id": "model-version-identifier",\n    "output": "the defined adverse repayment outcome",\n    "score_scale": "probability"\n  },\n  "features": [\n    {"id": "account_age", "label": "Average account age", "group": "Credit history"},\n    {"id": "balance", "label": "Revolving balance", "group": "Balances"}\n  ],\n  "applications": [{\n    "id": "example-1",\n    "score": 0.60,\n    "shapley": [\n      {"id": "A", "reference": "Comparison A", "baseline": 0.40,\n       "values": {"account_age": 0.15, "balance": 0.05}},\n      {"id": "B", "reference": "Comparison B", "baseline": 0.45,\n       "values": {"account_age": 0.05, "balance": 0.10}}\n    ]\n  }]\n}\n```\n\nEvery explanation must refer to the same input record, model version, output class and prediction scale. The format holds one prediction per application; the reviewer must verify that this identity is genuine. Supply **all** input contributions, including negative and zero contributions, not only a top-four list. Baseline plus contributions should reconstruct the supplied score. Residual discrepancies are surfaced in the finding rather than hidden.\n\nOnly probability-scale model outputs are supported. Do not use log-odds or raw-margin SHAP values against a probability prediction; export a probability-output explanation or adapt the analysis outside this tool. A probability-scale score is a model estimate: interpretation as an empirically reliable default rate requires calibration and population evidence.\n\nOptional explanation fields: `detail` describes the sample and `method` identifies the implementation. Optional feature field `group` specifies a research or institution-defined grouping. Signed contributions are summed before selecting positive grouped contributions. The grouping is not assumed to be approved for notice use.\n\nAn application may also contain `local_approximations`, an array of:\n\n```json\n{\n  "id": "attempt-1",\n  "method": "Official LIME or accurately named alternative",\n  "description": "Representation, neighborhood, weighting and evaluation design",\n  "coefficients": {"account_age": 0.1, "balance": 0.05},\n  "prediction_at_application": 0.59,\n  "fidelity": {"r2": 0.7, "mae": 0.03, "evaluation": "independent_generated"}\n}\n```\n\nThis is a schematic example. `evaluation` must be `independent_generated`, `fitting_only`, or `other`. R² and MAE are supplied measurements; the browser does not recompute them from evaluation data or authenticate their origin. The original report and separately held evaluation samples should be requested when needed. Omit local approximations if none were measured; the product explicitly identifies the evidence gap.\n\nThe paired external-score experiment is available only for the measured HELOC demonstration. An imported Shapley export is not silently treated as a paired model experiment.\n', 'docs/LEGAL_REQUIREMENTS_AND_CASES.md': '# Credit explanations: requirements, evidence and documented failures\n\nSource status checked 18 October 2026. The legal analysis concerns a proposed US consumer-credit use, with a North Carolina home equity line of credit (HELOC) as the jurisdiction/product example. The dataset does not identify a real creditor, application decision, notice or external-score provider. European cases are comparisons under different law.\n\n## The review question\n\nCan the creditor substantiate that the disclosed principal reasons describe the factors used in the applicant\'s actual adverse action? This requires the decision process, the reason-selection process and the delivered notice. Explaining a repayment-risk prediction supplies only part of that evidence.\n\nThere is no general regulatory SHAP score, LIME fidelity score, prediction AUC or reason-retention percentage that makes a credit system compliant. Numerical tolerances in this study are research comparisons. A reason can be stable and wrong. Different reasonable explanations do not, by themselves, prove that a notice violates the law.\n\n## Definitions and scope\n\n- **Adverse action, Regulation B:** includes refusing requested credit, specified unfavorable account changes or terminations, and refusing an increase. Accepted counteroffers and the exclusions in 12 CFR 1002.2(c) matter. A high risk prediction alone is not an adverse action.\n- **Principal reasons:** the main reasons for the action, not every important input across the model\'s population. The official interpretation requires disclosed reasons to correspond to factors actually considered or scored. It addresses scoring, judgmental review, combined processes and automatic-denial rules separately.\n- **Credit score, FCRA:** the statutory definition is in 15 USC 1681g(f)(2)(A). It excludes certain mortgage automated-underwriting ratings and other underwriting elements. A column named "ExternalRiskEstimate" does not establish the supplier\'s identity, consumer-report status or every statutory trigger.\n- **Fair lending:** whether a practice unlawfully discriminates is a separate question from whether a particular notice gives accurate reasons. Explaining a decision does not establish nondiscrimination. This study does not measure protected-group outcomes.\n\nSources: [Regulation B definitions](https://www.consumerfinance.gov/rules-policy/regulations/1002/2/); [official notice interpretation](https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/); [FTC compilation of FCRA, sections 609 and 615](https://www.ftc.gov/system/files/ftc_gov/pdf/fcra-may2023-508.pdf).\n\n## Binding US duties and the evidence a reviewer would need\n\n| Duty and trigger | What the reviewer needs | What this project contributes; what remains missing |\n| --- | --- | --- |\n| Specific principal reasons under 12 CFR 1002.9(b)(2), when reasons are provided for covered adverse action | The actual inputs, scoring and policy rules, overrides and human review; a validated way of selecting principal factors; wording that describes those factors | Shapley attribution explains our research Random Forest relative to a defined reference. Score-estimation accuracy does not authenticate the original score\'s factors. Actual lender decision rules and delivered reasons are missing. |\n| Reasons from the relevant decision component, official comments 9(b)(2)-6 through -8 | Which scoring stage, judgmental assessment or automatic-denial rule led to refusal | A credit-risk attribution cannot identify a separate affordability, collateral or policy rule that actually caused denial. Request the complete decision record rather than explain the risk model alone. |\n| Timely notification and required content, 1002.9(a) | Completed-application and action dates, creditor identity and address, applicable federal enforcement agency, ECOA notice, reasons or the required request procedure; delivery records | Not measured by the public credit data. For a standard completed consumer application, notification is generally due within 30 days. If the request route is used, an applicant has 60 days to request reasons and the creditor must provide the statement within 30 days of that request. Exceptions and other application/account situations require separate analysis. |\n| Retention, 1002.12(b) | Evaluated application information, action notice and reasons, and relevant written applicant complaints | Generally 25 months for consumer applications, with extensions for investigations/proceedings. Retaining model, preprocessing, reason-mapping and explanation versions is a proposed engineering control supporting reconstruction; the rule does not specify a universal ML log schema. Our reproducible research files are not a lender\'s retention system. |\n| FCRA 1681m(a), adverse action based wholly or partly on a consumer report | Report supplier contact information; notice that the reporting agency did not make the decision; free-report and dispute rights; evidence of actual report use | Identify the source and role of each input. The dataset supplies neither the reporting agency nor the required notice/delivery information. |\n| Additional FCRA disclosures when a numerical credit score is used in the covered action | Actual score, possible range, date, source and key adverse score factors, subject to statutory limits | Score factors and the creditor\'s principal denial reasons must be kept distinct. One disclosure does not automatically satisfy the other. Do not substitute reasons from our estimated score for the original provider\'s factors. |\n| HELOC-specific and property-valuation duties, where triggered | Product terms, lien position, collateral and valuations, application/disclosure records | Regulation Z 1026.40 governs home-equity-plan disclosures and restrictions. Regulation B 1002.14 concerns copies of appraisals and other written valuations for first-lien dwelling applications. None can be established from a SHAP plot. |\n\nSources: [1002.9](https://www.consumerfinance.gov/rules-policy/regulations/1002/9/), [interpretation](https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/), [1002.12](https://www.consumerfinance.gov/rules-policy/regulations/1002/12/), [FCRA text](https://www.ftc.gov/system/files/ftc_gov/pdf/fcra-may2023-508.pdf), [FTC notice explanation](https://www.ftc.gov/business-guidance/resources/using-consumer-reports-credit-decisions-what-know-about-adverse-action-risk-based-pricing-notices), [Regulation Z 1026.40](https://www.consumerfinance.gov/rules-policy/regulations/1026/40/), [1002.14](https://www.consumerfinance.gov/rules-policy/regulations/1002/14/).\n\n### What should not become an invented legal test\n\nThe official interpretation does not require precisely four ECOA reasons, a directional adjective, a guaranteed path to approval, a causal explanation or one specified XAI method. The FCRA score-factor limits are a separate matter. SHAP\'s mathematical consistency property is not a promise that rankings will remain unchanged when the applicant, comparison population or explanation definition changes. No primary source reviewed establishes a general regulator preference for SHAP over LIME. Do not hide an actually used suspect proxy by renaming it as a more acceptable feature: investigate its use and describe actual factors accurately.\n\n### Current source status\n\nCFPB Circulars 2022-03 and 2023-03 were withdrawn on 12 May 2025. They remain relevant historical explanations of the Bureau\'s concern about complex models and notices, but should be labeled withdrawn. Current legal conclusions here rely on Regulation B and its official interpretation. [CFPB withdrawal list](https://www.consumerfinance.gov/compliance/guidance/withdrawn-guidance/).\n\n### Accountability and liability\n\nRegulation B 1002.16 identifies administrative enforcement and potential civil remedies, including actual damages, specified punitive damages, equitable relief and attorney fees under ECOA. The relevant regulator depends on the creditor. An explanation diagnostic is evidence for review, not immunity from those duties. Liability depends on the claim, facts and applicable defenses; vendor involvement does not turn a prediction-quality metric into a defense. The commentary also contains a specific defense for certain third-party notice failures when the creditor supplied timely accurate information and maintained reasonable preventive procedures. [1002.16](https://www.consumerfinance.gov/rules-policy/regulations/1002/16/), [comment 9(g)-3](https://www.consumerfinance.gov/rules-policy/regulations/1002/interp-9/).\n\n## Supervisory model governance and third-party opacity\n\nThe April 2026 interagency revised model-risk guidance, issued by the Federal Reserve as **SR 26-2**, supersedes SR 11-7. The Fed identifies it as most relevant to its regulated banking organizations above $30 billion in assets. It is supervisory guidance, distinct from the binding notice duties. Its risk-based principles address model development and use, validation, monitoring and governance. Section VII explicitly addresses vendor data, parameters and models: proprietary components can restrict access, but model-risk principles still apply.\n\nUseful evidence includes development and performance documentation, intended-use limits, testing on the institution\'s population, independent challenge proportionate to risk, change controls, monitoring and a responsible owner. For an external score, request the applicable score version, input definitions and provenance, provider reason codes and their meanings, validation information or access, and change notifications. This is a proposed evidence request informed by the guidance; it is not a statutory demand to publish all vendor weights.\n\nSources: [SR 26-2 letter and applicability](https://www.federalreserve.gov/supervisionreg/srletters/SR2602.htm), [guidance including vendor products, section VII](https://www.federalreserve.gov/supervisionreg/srletters/SR2602a1.pdf).\n\n## North Carolina: scope before conclusions\n\nFor a North Carolina HELOC review, establish the creditor\'s charter/regulator, borrower and property location, consumer purpose, first or junior lien, line limit, rates and fees, and whether this is an initial refusal or a later restriction. NC G.S. 41A-4(b1) addresses discrimination in residential-real-estate-related lending. G.S. 75-1.1 concerns unfair or deceptive practices in commerce; a particular claim requires its elements and applicable defenses. G.S. 24-1.1E restricts defined high-cost home loans, including qualifying open-end plans. Its amount, security and pricing conditions cannot be evaluated from these credit attributes. These laws do not supply a SHAP acceptance threshold.\n\nSources: [41A-4](https://www.ncleg.gov/EnactedLegislation/Statutes/HTML/BySection/Chapter_41A/GS_41A-4.html), [75-1.1](https://www.ncleg.gov/EnactedLegislation/Statutes/HTML/BySection/Chapter_75/GS_75-1.1.html), [24-1.1E](https://www.ncleg.gov/EnactedLegislation/Statutes/HTML/BySection/Chapter_24/GS_24-1.1E.html).\n\n## Documented cases: distinguish findings, governance disputes and comparisons\n\n1. **CFPB, Winter 2025, Advanced Technologies Special Edition, issue 38, section 2.1.3, pp. 7–8.** Examiners reviewed auto lenders using AI/ML scoring, in some cases with over 1,000 variables. They found insufficient assurance of notice compliance and no validation that reason-selection processes produced accurate results. Institutions were directed to test and validate their methods. The report does not identify the lenders or name SHAP or LIME. This is direct evidence of a real explanation-validation problem, not a finding that one XAI method is unlawful. The Bureau\'s original PDF link currently returns 404; the linked copy reproduces the Bureau\'s report. [CFPB report, preserved copy](https://business.cch.com/BFLD/CFPB-Supervisory-Highlights-Advanced-Technologies-Special-Edition-Issue-38-Winter-2025-01172025.pdf).\n2. **CFPB, Winter 2020, issue 21, section 2.3.5, p. 9.** Incorrect principal reasons in payday-lending notices violated Regulation B. A coding error caused numerous incorrect notices; lenders corrected notices and changed generation systems. The report does not identify an AI method. Implication: validate the delivered reason-generation process, not just numerical attributions. [Official report](https://files.consumerfinance.gov/f/documents/cfpb_supervisory-highlights_issue-21_2020-02.pdf).\n3. **NYDFS, Apple Card investigation, March 2021, pp. 7–8.** Goldman could explain individual decisions to the regulator, but applicants experienced poor transparency and frustrating appeals over credit terms. The investigation did not establish unlawful discrimination. Distinguish these communication findings from an adverse-action-notice violation; not every granted-credit term triggers denial-notice duties. [Official report](https://www.dfs.ny.gov/system/files/documents/2021/03/rpt_202103_apple_card_investigation.pdf).\n4. **CFPB/Upstart, June 2022.** The no-action letter ended at Upstart\'s request after proposed model changes required agency review time. This documents the governance significance of changing proprietary models. It is not a finding that Upstart\'s explanation method failed or that the platform was banned. [Official account](https://www.consumerfinance.gov/archive/newsroom/cfpb-issues-order-to-terminate-upstart-no-action-letter/).\n5. **CJEU, Dun & Bradstreet Austria, C-203/22, 27 February 2025.** A mobile-contract refusal relied on an automated credit assessment. Applying GDPR, the Court required an intelligible account of how the person\'s data were actually used; an algorithm alone is insufficient. Input-variation examples may help. Claims of trade secrecy require balancing through the competent authority/court, rather than categorical refusal. This is a separate EU standard, especially relevant to the problem of an outside scoring provider. [Judgment, paragraphs 58–66 and operative part](https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:62022CJ0203).\n6. **CJEU, SCHUFA, C-634/21, 7 December 2023.** An external score can itself fall within GDPR Article 22\'s automated-decision rules when the lender gives it a determining role. The case does not ban all scoring. [Court account](https://curia.europa.eu/jcms/upload/docs/application/pdf/2023-12/cp230186en.pdf).\n\n## Actual practice and empirical evidence\n\nUpstart publicly reports using SHAP to generate reasons for AI-model declines, aggregating related features and checking directional wording and missing inputs. This is provider-reported practice, not an independent regulatory finding that every notice complies. Its page also discusses planned semantic clustering and LLM-assisted drafting; do not treat the stated rollout plan as independently verified deployment. [Upstart](https://www.upstart.com/lenders/regulatory-compliance/adverse-action-notices/).\n\nFinRegLab\'s 2023 empirical research tested industry and open-source tools using consumer credit data. Models included logistic regression, neural networks, gradient boosting and a company-submitted Random Forest. These are research implementations of model families relevant to lending, not proof that our nine-input forest matches a production lender. Signed adverse SHAP rankings performed well on its fidelity tests for complex models, but the project did not validate actual consumer-facing notices or certify compliance. Features were masked, limiting semantic review. [Empirical report, sections 3.4 and 4](https://finreglab.org/wp-content/uploads/2023/12/FinRegLab_2023-07-13_Empirical-White-Paper_Explainability-and-Fairness_Insights-from-Consumer-Lending.pdf).\n\n## Which method supplies which missing evidence?\n\n| Question | Useful method/control | Remaining limit |\n| --- | --- | --- |\n| Which inputs contributed to this model\'s prediction relative to a stated comparison? | Signed Shapley/SHAP, with justified comparison records and dependency/grouping checks | An additive decomposition is not causal proof, a complete decision trace or an explanation of an unknown provider\'s score formula. |\n| Does a simpler explanation track the model near this applicant? | LIME with independently evaluated local error, plausible generated records and repeatability testing | Fidelity depends on the sampled neighborhood. A good average fit can hide poor fit for the specific record or a subgroup. |\n| What did the actual transparent model do? | Direct decomposition of a deployed scorecard/additive model or execution of its actual rules | Training a new transparent substitute explains the substitute, not the original complex model. |\n| What feasible changes would reverse the decision? | Constrained counterfactuals, checked against the complete current decision process | Recourse differs from original principal reasons; correlated fields, time, costs and changes in policy affect feasibility. |\n| How was the outside score actually produced? | Provider\'s applicable input/reason documentation, validation access and records tied to the score version | Better predictive reconstruction cannot authenticate the original inputs or logic. Additional XAI on our approximation explains our approximation. |\n| Was the right notice produced and retained? | Decision replay, versioned reason mappings, notice/delivery records and retention controls | These are operational evidence, not outputs obtainable from SHAP/LIME. |\n\nMethod sources: [SHAP](https://arxiv.org/abs/1705.07874), [LIME](https://arxiv.org/abs/1602.04938), [alternative Shapley definitions](https://arxiv.org/abs/1908.08474), [DiCE](https://interpret.ml/DiCE/readme.html). Controls and the proposed use of diagnostics above are this project\'s analysis, not a regulatory prescription.\n\n## Comparative law without importing it into the US assessment\n\nFor a separately scoped EU deployment, AI Act Articles 12–14 address records, deployer information and oversight; Article 86 concerns explanations for specified consequential decisions. Roles, high-risk classification, exceptions and transition dates must be established. The Commission\'s July 2026 timetable states 2 December 2027 for Annex III high-risk rules. GDPR obligations already require their own analysis. [Article 13](https://ai-act-service-desk.ec.europa.eu/en/ai-act/article-13), [Article 86](https://ai-act-service-desk.ec.europa.eu/en/ai-act/article-86), [amended timetable](https://digital-strategy.ec.europa.eu/en/news/ai-omnibus-enters-force).\n\nNew York S8115C is a proposed automated-lending assessment bill, still in Senate committee when checked, not an enacted obligation for this experiment. [Bill status](https://www.nysenate.gov/legislation/bills/2025/S8115/amendment/C). Do not import employment bias-audit law or insurance circulars as credit-notice requirements.\n\n## What conclusion can the workbench support?\n\nIt can document measured fit and explanation behavior, identify the precise claim an estimated score does not substantiate, and specify the evidence needed to review that claim. It cannot resolve the original score\'s actual reasons, a real lender\'s complete decision or the adequacy of a delivered notice without those records. The useful governance output is a reasoned, source-linked conclusion about evidence sufficiency for a particular claim—not a percentage labeled legal compliance.\n', 'docs/POLICY_RECOMMENDATIONS.md': "# Policy implications\n\n## Recommendation\n\nWhen a consumer-facing reason depends on a material external score, the creditor should be able to distinguish two things: what its own model shows, and what it can actually support about the score provider. A downstream explanation or reconstructed score does not, by itself, establish the provider's reasons.\n\nThis does not imply that external scores should be banned or that one explanation method should be required. The issue is narrower: the record supporting the notice can cross an organizational boundary, and the evidence has to cross that boundary too.\n\n## What should change\n\n1. **Keep a provenance record for external-score decisions.** Retain the external score and version, lender model/version, reason-selection logic, explanation settings/reference, reason-mapping version, and notice output associated with the decision under review.\n2. **Mark inference as inference.** A downstream feature contribution can be measured for the model being explained. Factors inferred from a reconstruction of the external score should not be presented as the provider's factors unless provider-linked records support that claim.\n3. **Validate the basis of the notice category, not just the category label.** In this experiment, broad topic sets can stay the same even when the leading detailed factors change. A stable label does not show that the detailed basis for the label is stable.\n4. **Allow the review to stop at an evidence gap.** If a proprietary component prevents the reviewer from tracing the reason any further, the next step is to request records from the lender or provider. The workflow should not convert that gap into a more confident explanation.\n\n## Why the external-score result matters\n\nAmong 12 selected historical records where all three downstream predictions were within one percentage point, 4 had different leading individual-factor sets and 0 had different leading broader-topic sets. The counts are descriptive and come from an exploratory sample.\n\nThe result shows why output agreement is not enough. Two substitute mechanisms can land on almost the same downstream prediction and the same broad topic while attributing that result to different detailed factors. That leaves the original provider's mechanism unresolved.\n\n## Consumer-advocate questions\n\nAn external provider's opacity should not leave the applicant with a specific reason that the creditor cannot connect to the decision path actually used. For a disputed reason, useful questions include:\n\n- Did the reason come from the lender's own model, the external score, or a later mapping step?\n- If the external score mattered, did the lender receive provider-linked reasons or infer them internally?\n- Which score/model and mapping versions were used for this decision?\n- Can the institution reproduce the path from model evidence to the wording given to the applicant?\n\nThis project does not establish that any named lender failed these expectations, and it does not determine legal compliance for an actual transaction.\n\n## Supervisory and counsel review\n\nFor a sampled adverse action, ask for the evidence behind the reason, not just the name of the XAI method. Relevant records may include the lender model/version, external score/version, feature or score contribution used in reason selection, the mapping into consumer-facing wording, validation of that mapping, and provider-linked evidence for any reason attributed to the external score.\n\n## Next evaluation\n\nA stronger follow-up study would use a controlled or real adverse-action pipeline where the decision path and reason mapping are known. That would make it possible to test whether reviewers using this protocol identify unsupported reasons more accurately and consistently.\n", 'docs/PRESENTATION.md': "# Five-minute demonstration\n\n## 0:00–0:45 · Problem and research question\n\nOpen **About**, then briefly show **Rules & real cases**.\n\nA credit denial gives an applicant reasons for the decision. A compliance reviewer needs evidence that those reasons describe the factors actually considered. The CFPB's Winter 2025 findings give a concrete motivation: examiners identified AI/ML lending reason-selection methods that had not been adequately validated.\n\nMy project examines one complication: a lending model can use a credit-risk score calculated by another system. Explaining how the lending model uses that number does not tell us how the number itself was calculated. I tested whether estimating the score from other credit information helps resolve that uncertainty.\n\n## 0:45–1:35 · What the model predicts\n\nOpen **Inspect a prediction**, using the default record.\n\nHELOC means home equity line of credit: borrowing against equity in a home. FICO's historical dataset contains credit information and repayment outcomes. It does not contain actual denial notices or the formula behind its external score.\n\nI trained a Random Forest to predict the dataset's serious-late-payment outcome: at least 90 days overdue or worse within 24 months. The supplied score is 58 for this record. That is a score in the dataset, not a 58% probability. The Random Forest separately estimates a 95.47% probability for this historical record. This is a research-model output, not an actual lender's refusal or a verified forecast for a current applicant.\n\nThe Shapley chart shows the external score's contribution relative to 24 reference records. It explains a dependency in my model; it does not explain the provider's scoring process.\n\n## 1:35–2:40 · Compare two estimated-score systems\n\nOpen **Compare explanations**. Show individual factors, then grouped factors.\n\nI trained Ridge regression and gradient-boosted trees to estimate the supplied score using eight other credit attributes. Each estimate replaces the score in the same unchanged Random Forest. The charts explain those two combined research systems.\n\nFor the selected record, the repayment predictions differ by about 0.12 percentage points. Their leading individual-factor sets differ, but their broader topic sets match. Related measures can be grouped into topics when producing explanations; grouping here is explicitly a research choice, not a validated notice taxonomy.\n\nDifferent models can legitimately have different explanations. I am examining how much information prediction agreement and topic agreement actually give a reviewer.\n\n## 2:40–3:35 · Measured finding and its limits\n\nOpen **Findings** and demonstrate one alternative slider position.\n\nOf 40 examined records, 12 have all three predictions within one percentage point of each other. Four of those 12 have different leading individual-factor sets. None have different leading broader-topic sets. The figure preserves both results. It does not imply that four applicants received inaccurate notices.\n\nThe approximations are also imperfect across the larger test sample. Their average repayment-prediction errors are 5.01 and 4.33 percentage points. The close default example is not typical of every record. The forty explanation records were selected earlier, and the analysis is exploratory.\n\n## 3:35–4:30 · Governance contribution\n\nShow the evidence assessment beneath the figure.\n\nThe workbench distinguishes what the evidence establishes from the proposed claim it cannot establish. It measures prediction error and contributions in the research systems. It cannot identify the factors actually used by the original provider.\n\nA reviewer considering inferred factors as reasons for the original score should obtain provider reasons tied to that score, input definitions and validation evidence connecting the reasons to the scoring process. Reviewing an actual denial also requires the lender's decision rules and delivered notice. Agreement between approximations does not remove that evidence need.\n\n## 4:30–5:00 · Supporting method, artifact and future work\n\nBriefly show the LIME-style disclosure or the exported note; do not attempt both if time is short.\n\nThe supporting LIME-style check evaluates whether a simpler local explanation approximates the Random Forest. Even high local fidelity would leave the upstream scoring question unresolved.\n\nThe contribution is a reproducible experiment and interactive assessment of explanation evidence. Future work would obtain a scoring system with known internal factors and test which reconstruction or provider-evidence strategies resolve the uncertainty. This study has not evaluated actual notices or professional reviewer effectiveness.\n", 'docs/PRODUCT_AND_RUBRIC.md': '# Current contribution and rubric alignment\n\nThe active project is **The score inside the model**. README.md and EXTERNAL_SCORE_RECONSTRUCTION.md define its research question and interpretation. Earlier replay/checker products are historical.\n\n| Criterion | Current evidence | Boundary |\n| --- | --- | --- |\n| Course concepts | Exact empirical Shapley; independently evaluated custom LIME-style fidelity; approximation and uncertainty | Official SHAP/LIME packages are not claimed as executed |\n| Contribution | Tested external-score reconstruction strategy and a source-linked assessment distinguishing four claims | No first-of-kind or new-algorithm claim |\n| Developed artifact | Static case explorer; all forty records; factor/topic comparison; directly labeled cohort figure at all four prediction bands; evidence/unresolved claim/next-request assessment; readable and JSON exports | Browser download verification remains separate from simulated tests |\n| Evidence | Predictive accuracy on 1,973 eligible existing test records; 120 system-specific decompositions for forty cases; descriptive agreement bands | Exploratory test reuse, selected sample and unavailable upstream mechanism |\n| Governance | Distinguishes predictive fit from substantiation of original reasons; identifies provider evidence to request | Does not establish notice legality, consumer harm or effective institutional use |\n| Reproduction | Protocol, pinned dependencies, executable scripts, saved results, portable notebook and numerical tests | Requires authorized raw HELOC file |\n| Ambition | Investigates a score-within-model dependency and evaluates a plausible reconstruction approach | No reviewer exercise or professional-effectiveness validation |\n| Presentation | Five-minute demonstration script and updated 24 x 24 inch poster | Poster includes lending practice, US/EU requirements, Maitreyee Singh byline, a Shapley waterfall, paired explanations and LIME-style fidelity; supplied Duke wordmark is included with a navy/white palette and a shorter visual workflow |\n\nPreserve the negative finding: among twelve tightly matching cases, four detailed-factor sets differ but no credit-topic sets differ. Do not relabel all detailed differences as different principal reasons or consumer harm.\n', 'docs/PROTOCOL_v2.md': '# Protocol v2: From Risk Scores to Reasons\n\nFrozen before generating and inspecting the new experiment outputs. This is a prospective protocol for the added benchmark and reference experiment, not a preregistration of the earlier pilot. Earlier HELOC audit results and model benchmark were already known.\n\n## Research question\nWhich decision-to-notice defects escape explanation-repeatability screening, and which can a structured evidence review identify?\n\n## Two evidence tracks\n1. HELOC attribution experiment: retain the original eight-feature Random Forest so its exact empirical interventional Shapley calculation remains tractable. Repeat the fixed training/validation/test partition, original seeds and original audited records. Do not substitute the expanded-model AUC into these results.\n2. Controlled decision-to-notice benchmark: use these HELOC records and measured attribution outputs inside an explicitly constructed research decision policy. Institutional policies, notices and errors in this track are simulated. Regulatory cases motivate failure patterns; they are not reconstructed from confidential lender records.\n\n## Reference and fidelity experiment\nFor the same 40 audited records, calculate exact empirical interventional Shapley values using (a) 24 training records drawn from the training population and (b) 24 records with Bad-class scores in [0.45, 0.50), immediately on the research acceptance side of the 0.50 cutoff. Use background seeds 101, 202 and 303 for both definitions. Report pool counts; if fewer than 24 records satisfy the cutoff band, stop rather than silently widen it. No test/validation records enter the reference pools.\nCompare population vs near-cutoff reference using matched seeds; compare resampling within each reference definition. Measure reason-set turnover, retention and first-reason change using the actual list length, signed contributions, and at most four positive feature reasons. Aggregate comparisons within records before averaging across records. Bootstrap whole records 1,000 times with seed 2026. This selected rejection sample is not representative of US borrowers.\nFor the custom LIME-style surrogate, fit with 1,200 samples at seeds 0–4. Evaluate weighted R² and weighted mean absolute prediction error on an independently sampled 1,200-record neighborhood at seed+10,000, using the same interpretable representation and kernel. Also report absolute prediction error at the original applicant. These neighborhoods may break feature dependence. R² is approximation fit under this sampling scheme, not a fraction of correct reasons. Do not call custom output official LIME.\n\n## Benchmark partition and scenarios\nPartition the 40 original audited records into 20 development and 20 evaluation records using seed 20261016. All scenarios derived from a record remain in its partition. Store the partition before generating results. Thresholds and review logic are frozen before evaluation. The evaluator must not expose scenario labels or split membership to the audit function.\n\nThree defect families:\n- Mapping: a principal feature is translated to the wrong concept or unsupported directional wording. Development and evaluation use different replacement descriptions.\n- Decision coverage: the research policy first applies an eligibility gate and stops if it fails; a notice explains an unused model score instead of the gate. A failed gate is constructed as an independent research flag, not inferred from HELOC.\n- Provenance: explanation/model or mapping versions do not match the decision record being explained.\n\nFour correct controls: correct model-driven notice; approved paraphrase of a correct reason; correct eligibility-gate notice; archived historical decision using its matching historical artifacts even when a newer model exists. A ninth scenario intentionally withholds the decision trace; the correct review disposition is insufficient evidence, not a confirmed wrong notice.\n\nNine scenarios per record: four controls; two mapping defects; one decision-coverage defect; one provenance defect; one missing-evidence case. Evaluation therefore contains 80 known-defective notices, 80 correct notices and 20 insufficient-evidence cases. These are constructed equal-frequency scenarios, not deployment prevalence. Do not treat 180 scenarios as independent borrowers.\n\n## Comparators\nStability-only: flag if the original five custom-surrogate runs produce a differing top-four feature reason set in more than 20% of their ten pairwise comparisons. This illustrative screening threshold is not a legal standard. Apply the same stability result to every downstream notice for that record, since the model explanations are held fixed. Report a supplementary 0–100% threshold sweep; do not tune the primary threshold to evaluation outcomes.\nEvidence review: inspect the candidate notice against independently retained decision trace, reason-mapping catalog and version identifiers. Check declared source features and category wording, signed support for model reasons, completeness of active policy reasons and decision/artifact version agreement. Return findings with evidence locations, or insufficient evidence when the trace/catalog is absent. Never read injected-defect labels.\n\n## Outcomes and uncertainty\nFor evaluation records only, report confirmed-defect detection rate per family, overall sensitivity, false-alarm rate on correct controls, and insufficient-evidence disposition. A flagged correct notice is a false alarm. Report all method abstentions. Provide record-cluster bootstrap intervals and per-record paired comparisons, not binomial intervals assuming independent scenario rows. Include counts and original case IDs.\n\n## Hypotheses and limitations\nH1: downstream mapping, coverage and version errors can remain unchanged despite repeatable attribution outputs.\nH2: structured evidence checks detect more constructed downstream errors than the repeatability-only screen while avoiding some unnecessary flags.\nH3: reference definition affects feature reasons even when model and applicant are unchanged. No direction or effect size is assumed.\nThe structured audit has access to more evidence than the stability screen: this is an incremental-evidence comparison, not an equal-information algorithm contest. Same-author defect construction and deterministic review rules create favorable conditions; held-out records and variants do not establish robustness to new defect families. A successful result validates consistency with the research contracts, not legal principal-reason ground truth. Missing, false or collusively altered source evidence can defeat reconstruction checks. No actual lender notices, consumer comprehension, protected-group effects or feasible recourse are evaluated.\n\n## Artifact and communication\nPublish protocol hash, partition, code, model settings, explanation methods, scenario truth labels separately from audit inputs, findings and reproducible summary. The landing page contains title, why this matters and proposed use. Technical counts and method identities belong with results and research design. Poster reports completed results only and identifies the controlled nature of the benchmark beside its principal chart.\n', 'docs/RESTORED_PROJECT_STRUCTURE.md': '> Historical version: this document describes an earlier interface or verification run. See README.md and DECISION_RECONSTRUCTION_PROTOCOL.md for the current project.\n\n# Restored project structure\n\nThe preferred project identity and legal/evidence context are restored. This is an integration of the earlier research workbench and the later practical evidence-comparison tool, not a replacement by a different product.\n\n- About the project: Same Decision, Different Reasons; why this matters; proposed use.\n- Inspect a decision: measured comparison-group, local-approximation and external-score evidence; optional local import. The research model predicts a repayment label; it does not reconstruct an actual lender decision.\n- Compare results: detailed versus grouped reason changes, sampling sensitivity, approximation fidelity and upstream-score dependence. The masked-change example links to a retained measured record and matched sampling seed.\n- Review findings: an actual session record combining numerical evidence with reviewer interpretation, evidence request, responsible owner and next action. Readable and JSON exports preserve those distinctions. No outbound request is sent.\n- Legal requirements: adverse action versus discriminatory effects; current framework separation; requirements/evidence/method table; alternative-method comparison; CFPB, NYDFS and Tech Sprint examples; responsibility and source status.\n- Research design: methods, data, protocol history, limitations, implementation identity, literature and reproduction.\n\nNo artificial wording dictionary is used by the main interface. Earlier constructed-notice checks remain supplementary reproducible work.\n\nThe added grouping summary is an exploratory descriptive analysis of already generated reference explanations. It measures up to four positive signed factors/categories after aggregation. It is not a new preregistered confirmation, an independent user study, or a rate of incorrect actual notices. The original model and explanation outputs remain unchanged.\n', 'docs/RESULTS_external_score.md': "# External-score extension: measured results\n\nThe same Random Forest settings were applied to the same train/test partitions, with eight original credit-history inputs and then with ExternalRiskEstimate added. All 40 original audit applications were retained, including changed research outcomes. This exploratory design was recorded after overlapping earlier pilot results were known. See EXTERNAL_SCORE_ADDENDUM.md.\n\n- Full-test ROC AUC without external score: 0.7708787536.\n- Full-test ROC AUC with external score: 0.7821133278.\n- Paired AUC difference: 0.0112345742; application-bootstrap 95% interval [0.0041424379, 0.0182184963], 1,000 resamples.\n- Outcomes at the research threshold of 0.50 differ for 134/1,975 test applications.\n- Among all 40 original audit applications, 8 research outcomes change.\n- The external score is among up to four positive Shapley contributors for 33/40 applications; it has the largest absolute contribution for 28/40. These are different measures, not interchangeable definitions of dominance.\n- Exact empirical interventional Shapley explanations reconstruct both models' scores for all 80 application-model pairs. All eight-input scores and contributions reproduce the earlier audit numerically.\n- Ten test-set permutations per raw input assess model reliance through mean AUC loss. Dependence among features limits causal interpretation.\n- Five custom LIME-style local fits per application-model pair have independent-neighborhood fidelity measurements. The original eight-input fidelity is reused; nine-input fits are newly measured.\n\nThe available snapshot contains external-score values, but no authenticated upstream model, score calculation or upstream reasons. Reconstructing the lender-side model does not reconstruct that scoring system. Removing the score trains a different model; neither explanation is an explanation of the other model's decision.\n\nThe selected 40 applications do not estimate prevalence across US borrowers. No consumer comprehension or institutional liability was measured.\n", 'docs/RESULTS_v2.md': '# Measured results and interpretation\n\nThe new protocol was frozen before these runs, after the earlier pilot and classifier comparison were known. It is an internal prospective protocol, not an external preregistration. All scenario policies, notices and defects are constructed.\n\n## Primary notice benchmark\n\n| Method | Scenario family | Flagged / scenarios | Rate | 95% record-bootstrap interval |\n|---|---|---:|---:|---:|\n| Repeatability screen | mapping | 24 / 40 | 0.600 | 0.400–0.800 |\n| Repeatability screen | coverage | 12 / 20 | 0.600 | 0.400–0.800 |\n| Repeatability screen | provenance | 12 / 20 | 0.600 | 0.400–0.800 |\n| Repeatability screen | all_defects | 48 / 80 | 0.600 | 0.400–0.800 |\n| Repeatability screen | correct | 48 / 80 | 0.600 | 0.400–0.800 |\n| Structured evidence review | mapping | 40 / 40 | 1.000 | 1.000–1.000 |\n| Structured evidence review | coverage | 20 / 20 | 1.000 | 1.000–1.000 |\n| Structured evidence review | provenance | 20 / 20 | 1.000 | 1.000–1.000 |\n| Structured evidence review | all_defects | 80 / 80 | 1.000 | 1.000–1.000 |\n| Structured evidence review | correct | 0 / 80 | 0.000 | 0.000–0.000 |\n\nEvaluation contains 20 held-out records; each contributes four defect variants, four correct controls and one missing-trace case. The additional 20 development records are excluded. All 20 missing traces produce an insufficient-evidence disposition. Bootstrap resamples the 20 records rather than treating 180 scenarios as independent. Degenerate zero/one intervals reflect identical observed outcomes; they do not prove future certainty.\n\nThe repeatability flag is identical across notice variants for a record by construction. Equal detection and false-alarm rates are therefore a property of the design, not a surprising statistical discovery. The evidence checker receives additional decision traces and catalogs; its favorable primary detection tests consistency against these supplied contracts. It is not a superior equal-information estimator of legal correctness.\n\n## Deliberate post-primary challenges\n\nAll 20 stipulated-correct unlisted paraphrases are flagged. None of 20 wrong reasons entered consistently in the trusted mapping catalog are detected. These same-author challenge examples were added after primary results and targeted known checker limitations; no primary thresholds or rules changed. They show exact-catalog false alarms and the dependence on accurate source evidence.\n\n## Reference sensitivity\n\n| Comparison | Reason-set turnover | 95% record-bootstrap interval | Retention | First reason changed |\n|---|---:|---:|---:|---:|\n| population_vs_near_cutoff | 0.5333 | 0.4250–0.6500 | 0.8597 | 0.1750 |\n| population | 0.5750 | 0.4667–0.6752 | 0.8306 | 0.0500 |\n| near_cutoff | 0.4583 | 0.3417–0.5750 | 0.8993 | 0.2667 |\n\nThe model and applicant are held fixed. Each of 40 selected records contributes three comparisons per condition; comparisons are averaged within records. Each reference contains 24 rows. The population reference pool has 5,922 training records; the near-cutoff pool has 273 records with model scores in [0.45, 0.50). These are research acceptance-side records, not observed approvals.\n\nResampling variability is comparable to differences between reference definitions. With only three seeds and small background samples, the experiment does not establish that the policy choice dominates sampling effects or identify a legally preferred reference. Exact coalition enumeration removes coalition-estimation error, not uncertainty from the empirical reference distribution.\n\n## Independent-neighborhood local fit\n\n| Statistic | Median | Minimum | Maximum |\n|---|---:|---:|---:|\n| fit_r2 | 0.565260 | 0.074959 | 0.781446 |\n| heldout_r2 | 0.552867 | 0.068999 | 0.769229 |\n| heldout_mae | 0.051714 | 0.020510 | 0.108963 |\n| point_error | 0.008499 | 0.000050 | 0.044552 |\n\nThe 200 fits are five seeds for each of 40 records. Fit and evaluation neighborhoods each contain 1,200 points, generated with different seeds using the same sampling process. They may contain implausible feature combinations. R² is not the fraction of correct reasons and these custom-surrogate results do not characterize the official LIME package.\n\n## Models and reconstruction\n\nThe audited eight-feature Random Forest has test AUC 0.770879. The separate validation-selected 22-feature model has test AUC 0.784694; its explanations were not substituted. The browser evaluates the retained 100 trees, with the original medians and separate special-state indicators. All 40 original scores reconstruct within numerical precision. This is a numerical consistency check, not authentication of historical institutional artifacts.\n\n## Contribution and boundary\n\nThe contribution is a reproducible decision-to-notice workbench and controlled comparison of the evidence supplied to a review, together with an empirical reference/fidelity experiment. Known-defect evaluation, grouping and explanation sensitivity are established ideas. No actual lender notices, new defect-family generalization, consumer comprehension, fair-lending outcomes or feasible recourse were evaluated. Official SHAP/LIME runs remain pending; their portable notebook and plots are supplied.\n', 'docs/REVIEW_LANGUAGE.md': "> Historical version: this document describes an earlier interface or verification run. See README.md and DECISION_RECONSTRUCTION_PROTOCOL.md for the current project.\n\n# Accessible review language\n\nPrimary audience: consumer-compliance teams reviewing automated credit decisions. Secondary audiences: model-risk reviewers, students and hiring reviewers exploring a guided example.\n\nThe four questions are: What was the applicant told? What determined the decision? How much confidence should we place in the explanation? What should the reviewer do next?\n\nTechnical labels belong with definitions and detailed evidence. Main labels describe the user's question.\n\n| Technical term | Main-interface explanation |\n|---|---|\n| Dataset record | Example application: credit-history information and a repayment outcome from HELOC |\n| Classifier | Credit-risk prediction model |\n| Cutoff | Research score threshold that separates acceptance and denial |\n| Reference/background sample | Applications used as the comparison group |\n| Random seed | Technical setting controlling a reproducible sample or repeated attempt |\n| Signed Shapley contribution | How much an input raises or lowers predicted risk relative to the comparison group |\n| Grouping | Which related credit factors are combined into a broader reason category |\n| Local surrogate | Simpler explanation fitted around one application |\n| Fidelity / local R² | How closely that explanation follows the prediction model on a defined nearby test |\n| Reason-set turnover | How often at least one factor enters or leaves a reason list |\n| Controlled benchmark | Test using deliberately introduced notice or record errors |\n| Held-out records | Applications reserved for final evaluation |\n| Controls | Examples without the errors being tested |\n| Decision trace | Recorded conditions checked and what determined the outcome |\n| Version provenance | Which model and explanation settings were used at the decision date |\n| Unlisted paraphrase | Accurate wording absent from the approved list |\n| Corrupted catalog | Incorrect wording also entered into the approved list |\n\nMethod names remain explicit. The shipped calculations use exact empirical interventional Shapley values and a custom LIME-style approximation. Optional official SHAP/LIME outputs acquire their own labels after execution. The site never presents the custom approximation as the official package.\n", 'docs/VERIFICATION_v2.md': '> Historical version: this document describes an earlier interface or verification run. See README.md and DECISION_RECONSTRUCTION_PROTOCOL.md for the current project.\n\n# Verification and release status\n\n## Completed\n- Thirteen analytical tests: status encoding, exact Shapley efficiency, signed grouping, empty-list comparisons, record partitions, matching historical artifacts, missing-evidence abstention, terminating-rule coverage, reference reconstruction and independent-neighborhood anchor handling.\n- Browser/Python checker parity on all 360 primary scenarios and 40 supplementary challenges. Truth labels are not passed to the review function.\n- Interaction smoke checks for all nine notice variants, six sections, editable-wording handlers, reference/method selection, challenge cases, saved reviewer notes and the threshold sweep. These use a simulated DOM, not a real browser.\n- Numerical browser reconstruction of all 40 retained predictions from 100-tree model arrays and original encoding; maximum observed error zero.\n- Every one of 240 reference explanations reconstructs its unchanged prediction to numerical precision.\n- The refitted surrogate coefficients match the earlier pilot. Independent evaluation does not force the original fitting anchor into its sample; the original-point error is reported separately.\n- Portable notebook Python cells parse. The notebook stages its own sources and protocol and excludes raw CSV from exports.\n- Poster is a one-page 36 × 24 inch PDF, rendered to PNG and visually inspected. Text, sources and measured values are embedded in the PDF.\n\n## Outstanding\n- Official SHAP/LIME library execution: dependencies unavailable in this session. Colab reproduction and optional official figure generation are supplied. The site automatically exposes official views for matching cases if an official output file is added and its scores agree.\n- Actual browser visual and mobile-device testing: no browser executable was available. Desktop/mobile design concepts were produced; the implemented site uses responsive layout and horizontally scrollable detailed plots, but simulated-DOM tests cannot establish rendered visual quality.\n- GitHub merge/deployment: remote publishing unavailable. The source includes a Pages workflow; no push or deployment is claimed.\n- Original FICO data dictionary and redistribution terms: source-access verification remains incomplete. A supplied-file checksum is recorded; no raw CSV is redistributed. Public-record explanations/model artifacts are included for reproducibility.\n- Independent domain review, independently authored challenge cases, consumer comprehension and actual lender validation remain future evaluation.\n\n## Interpretation\nPrimary contract-detection results reflect favorable constructed conditions and access to extra evidence. Held-out records do not establish performance on unknown failure families. The separate challenges deliberately expose wording and trusted-catalog failure modes; they were introduced after primary results without modifying the checker.\n', 'docs/VERIFICATION_v3.md': '> Historical version: this document describes an earlier interface or verification run. See README.md and DECISION_RECONSTRUCTION_PROTOCOL.md for the current project.\n\n# Verification: accessible review and external-score extension\n\n- 17 Python analytical tests pass, including all 80 Shapley reconstructions, 40 unchanged original-model contributions, unfiltered pairing, and positive-contribution summary counts.\n- Python and browser JavaScript evidence checkers agree on 360 primary examples and 40 follow-up challenges.\n- JavaScript minimal-DOM checks exercise guided navigation through four questions, all nine notice types, selectable reasons, five local-approximation attempts, comparison-group/grouping controls, challenge failures, saved requests, all 40 external-score comparisons and primary-model numerical reconstruction.\n- Site JavaScript passes syntax checking. All Colab code cells and the extracted Python script parse. The notebook stages the external-score source and runs it after the original studies and before the site build.\n- The complete new external-score experiment ran locally. Baseline eight-input predictions and Shapley values match retained original results within 1e-10. Sampling, model settings and measured per-record outputs are retained.\n- Raw CSV and cache/archive directories are excluded from distribution ZIPs. The site-only ZIP has index.html, app.js, review.js and data.js at its root.\n- A browser executable is not installed; desktop/mobile visual rendering and real-browser keyboard QA were not performed. Minimal-DOM tests do not replace them. The site uses responsive columns, native controls, visible focus, direct labels and stacked mobile evidence views.\n- Official SHAP/LIME packages were not available locally. The optional Colab path remains available and clearly distinguished. The shipped exact Shapley and custom LIME-style outputs are measured.\n- The previously supplied poster represents the earlier study. It is not silently updated to claim the new experiment was included.\n', 'docs/VERIFICATION_v4.md': '> Historical version: this document describes an earlier interface or verification run. See README.md and DECISION_RECONSTRUCTION_PROTOCOL.md for the current project.\n\n# Verification of the focused explanation-evidence product\n\n- The original 17 Python analytical tests still pass; the underlying experiments were not changed by the UI simplification.\n- Browser-side numerical tests cover imported schema, probability-scale restrictions, positive-factor comparison, signed group sums, additive residuals, invalid/duplicate/nonfinite inputs, and empty lists.\n- Simulated DOM tests cover all 40 HELOC applications, selectable input definitions, comparison/grouping controls, five local-approximation attempts, paired external-score displays, successful and invalid imports, preservation of the previous evidence after invalid JSON, example-file export, and readable source-linked brief export.\n- The mathematical product does not load the earlier notice dictionary or reviewer. Their supplementary Python/JavaScript parity checks still agree across 400 examples.\n- The product makes no network requests to process imported data and does not persist those files in localStorage. Imported fidelity statistics and identities are supplied evidence, not authenticated or independently recomputed by the browser.\n- No browser executable is installed, so rendered desktop/mobile visual QA remains outstanding. The displayed design concept is not evidence of rendered implementation QA.\n- Official SHAP/LIME runs remain optional Colab execution steps. The site explicitly labels the measured exact Shapley and custom LIME-style results.\n', 'tests/test_audit.py': "import unittest\nimport numpy as np\nfrom src.audit import exact_shapley, Encoding, reason_list, compare\nclass AuditChecks(unittest.TestCase):\n def test_shapley_known_additive_model(self):\n  phi,m=exact_shapley(np.array([3.,4.]),np.array([[0.,0.],[2.,2.]]),lambda a:2*a[:,0]+3*a[:,1])\n  np.testing.assert_allclose(phi,[4,9],atol=1e-10);self.assertLess(abs(m['residual']),1e-10)\n def test_special_states_distinct(self):\n  e=Encoding().fit(np.array([[2.],[6.]]));z=e.transform(np.array([[-7.],[-8.],[-9.],[4.]]))\n  self.assertEqual(len(np.unique(z,axis=0)),4);np.testing.assert_array_equal(z[:,0],[4,4,4,4])\n def test_group_signed_cancellation(self):\n  rs=reason_list([.3,-.4,.1],['MSinceOldestTradeOpen','AverageMInFile','NumInqLast6M'],True)\n  self.assertEqual([r['id'] for r in rs],['Recent inquiries'])\n def test_empty_list_is_unresolved(self):self.assertIsNone(compare([],[])['retention'])\n def test_distinct_overlap_denominators(self):\n  c=compare([{'id':i} for i in ['a','b','c','d']],[{'id':i} for i in ['a','b','e','f']])\n  self.assertEqual(c['retention'],.5);self.assertAlmostEqual(c['jaccard'],1/3)\nif __name__=='__main__':unittest.main()\n", 'tests/test_decision_reconstruction.py': '"""Meaningful replay/abstention checks for missing and contradictory evidence."""\nimport copy\nimport json\nfrom pathlib import Path\nimport sys\nimport unittest\nimport numpy as np\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \'src\'))\nfrom decision_reconstruction import assess, digest, decide\nR=Path(__file__).resolve().parents[1]\n\nclass ReconstructionTests(unittest.TestCase):\n    def setUp(self):\n        self.inputs={\'a\':.4,\'b\':.2}; self.policy={\'decline_at_or_above\':.5}\n        self.p={\'inputs\':self.inputs,\'policy\':self.policy,\'model_sha256\':\'original\',\n                \'explanation\':{\'base\':.2,\'values\':[.2,.2]},\n                \'event\':{\'prediction\':.6,\'decision\':\'Declined\',\'inputs_sha256\':digest(self.inputs),\'policy_sha256\':digest(self.policy),\'model_sha256\':\'original\'}}\n        self.calls=0\n    def predict(self,a):\n        self.calls+=1\n        return np.array([sum(a[0])])\n    def evaluate(self,p):return assess(p,self.predict,\'original\',[\'a\',\'b\'])\n    def test_original_replays_actual_predictor(self):\n        r=self.evaluate(self.p)\n        self.assertTrue(r[\'original_decision_reconstructed\']);self.assertEqual(self.calls,1)\n    def test_missing_policy_preserves_prediction_only(self):\n        self.p.pop(\'policy\');r=self.evaluate(self.p)\n        self.assertTrue(r[\'prediction_reproduced\']);self.assertFalse(r[\'original_decision_reconstructed\'])\n        self.assertEqual(r[\'inconsistencies\'],[])\n    def test_new_policy_does_not_establish_old_decision_even_if_same_outcome(self):\n        self.p[\'policy\']={\'decline_at_or_above\':.55};r=self.evaluate(self.p)\n        self.assertEqual(decide(.6,self.p[\'policy\']),\'Declined\')\n        self.assertFalse(r[\'original_decision_reconstructed\']);self.assertIn(\'policy\',r[\'inconsistencies\'])\n    def test_changed_input_is_not_silently_replayed(self):\n        self.p[\'inputs\'][\'a\']=.8;r=self.evaluate(self.p)\n        self.assertEqual(self.calls,0);self.assertFalse(r[\'prediction_reproduced\'])\n    def test_different_model_identity_is_not_silently_replayed(self):\n        self.p[\'model_sha256\']=\'other\';r=self.evaluate(self.p)\n        self.assertEqual(self.calls,0);self.assertFalse(r[\'prediction_reproduced\'])\n    def test_arithmetic_failure_does_not_erase_valid_decision_replay(self):\n        self.p[\'explanation\'][\'values\'][0]+=.1;r=self.evaluate(self.p)\n        self.assertFalse(r[\'explanation_sum_matches\']);self.assertTrue(r[\'original_decision_reconstructed\'])\n    def test_matching_explanation_does_not_authenticate_prediction(self):\n        self.p.pop(\'inputs\');self.p.pop(\'model_sha256\');self.p.pop(\'policy\');r=self.evaluate(self.p)\n        self.assertTrue(r[\'explanation_sum_matches\']);self.assertFalse(r[\'prediction_reproduced\'])\n    def test_measured_case_coverage_and_expected_manipulations(self):\n        d=json.loads((R/\'results/decision_reconstruction/experiment.json\').read_text())\n        self.assertEqual(d[\'package_count\'],280);self.assertEqual(len(d[\'cases\']),40)\n        self.assertEqual(d[\'changed_outcomes_under_later_policy\'],20)\n        for c in d[\'cases\']:\n            self.assertTrue(c[\'packages\'][\'original\'][\'assessment\'][\'original_decision_reconstructed\'])\n            self.assertFalse(c[\'packages\'][\'chart_only\'][\'assessment\'][\'original_decision_reconstructed\'])\n            self.assertFalse(c[\'packages\'][\'new_policy\'][\'assessment\'][\'original_decision_reconstructed\'])\n        self.assertEqual({r[\'records\'] for r in d[\'summary\']},{40})\n\nif __name__==\'__main__\':unittest.main()\n', 'tests/test_evidence_engine.cjs': "const assert=require('assert'),E=require('../site/evidence.js');\nconst pack={schema_version:'credit-explanation-evidence/v1',model:{id:'example-model',output:'Bad repayment outcome',score_scale:'probability'},features:[{id:'a',label:'Account age',group:'History'},{id:'b',label:'Payment history',group:'History'},{id:'c',label:'Balances'}],applications:[{id:'example',score:.6,shapley:[{id:'A',reference:'Comparison A',baseline:.3,values:{a:.2,b:.1,c:0}},{id:'B',reference:'Comparison B',baseline:.3,values:{a:0,b:.1,c:.2}}]}]};\nassert.equal(E.validate(pack),pack);let a=pack.applications[0],r=E.compare(a.shapley[0],a.shapley[1],pack.features);assert.deepEqual(r.shared,['b']);assert.deepEqual(r.removed,['a']);assert.deepEqual(r.added,['c']);assert(r.changed);assert(Math.abs(E.residual(a,a.shapley[0]))<1e-12);\nlet g=E.contributions(a.shapley[0],pack.features,true);assert(Math.abs(g[0].value-.3)<1e-12);assert.equal(g[0].id,'History');\nfor(const mutate of [p=>p.model.score_scale='logodds',p=>p.features[0].group={},p=>p.applications[0].shapley[0].values.a=10,p=>p.applications[0].score=2,p=>delete p.applications[0].shapley[0].values.a,p=>p.applications[0].shapley[0].values.a=NaN,p=>p.features.push(p.features[0]),p=>p.applications.push(p.applications[0]),p=>p.applications[0].shapley[1].id='A',p=>p.applications[0].shapley[0].baseline=-.1,p=>p.applications[0].local_approximations=[{coefficients:{a:.1},fidelity:{r2:2,mae:0,evaluation:'independent_generated'}}]]){let bad=structuredClone(pack);mutate(bad);assert.throws(()=>E.validate(bad));}\n// A rounding/incomplete-attribution discrepancy remains inspectable, not a legal verdict.\nlet discrepancy=structuredClone(pack);discrepancy.applications[0].shapley[0].values.a=.1;E.validate(discrepancy);assert(Math.abs(E.residual(discrepancy.applications[0],discrepancy.applications[0].shapley[0]))>.09);\nlet empty={baseline:.6,values:{a:0,b:0,c:0}};assert.equal(E.leading(empty,pack.features).length,0);assert.equal(E.compare(empty,empty,pack.features).changed,false);\nconsole.log('PASS: evidence schema, probability-scale restriction, changed-factor lists, signed group sums, decomposition residuals, duplicate/missing/nonfinite rejection and empty-list behavior.');\n", 'tests/test_evidence_review.py': "import json,unittest\nfrom pathlib import Path\nfrom copy import deepcopy\nfrom src.evidence_review import review\nR=Path(__file__).resolve().parents[1]\n\nclass EvidenceReviewTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        cls.inputs=json.loads((R/'results/notice_audit/inputs.json').read_text())\n        cls.truth={r['id']:r for r in json.loads((R/'results/notice_audit/truth.json').read_text())}\n    def test_borrower_partitions_do_not_overlap(self):\n        p=json.loads((R/'results/notice_audit/partition.json').read_text())\n        self.assertEqual(sum(v=='evaluation' for v in p.values()),20)\n        for s in self.inputs:self.assertEqual(self.truth[s['id']]['partition'],p[str(s['case_id'])])\n    def test_matching_historical_artifact_is_not_flagged_for_being_old(self):\n        s=next(s for s in self.inputs if self.truth[s['id']]['scenario']=='historical')\n        self.assertEqual(review(s['bundle'])['disposition'],'No discrepancy detected')\n    def test_missing_evidence_is_not_a_confirmed_defect(self):\n        self.assertEqual(review({'notice':{}})['disposition'],'Insufficient evidence')\n    def test_checker_cannot_see_truth_and_same_input_has_same_review(self):\n        for s in self.inputs[:20]:\n            self.assertNotIn('family',s['bundle']);self.assertNotIn('scenario',s['bundle'])\n            self.assertEqual(review(s['bundle']),review(deepcopy(s['bundle'])))\n    def test_termination_rule_distinguished_from_diagnostic_score(self):\n        s=next(s for s in self.inputs if self.truth[s['id']]['scenario']=='omitted_policy')\n        self.assertIn('decision_coverage',{f['code'] for f in review(s['bundle'])['findings']})\n\nif __name__=='__main__':unittest.main()\n", 'tests/test_external_score.py': '"""Validate the reported paired experiment against its retained per-application results."""\nimport json,unittest\nfrom pathlib import Path\nimport numpy as np\nR=Path(__file__).resolve().parents[1]\nclass ExternalScoreTests(unittest.TestCase):\n @classmethod\n def setUpClass(cls):\n  cls.d=json.loads((R/\'results/external_score/experiment.json\').read_text())\n  cls.old=json.loads((R/\'results/reference/audit_results.json\').read_text())\n def test_pairing_keeps_changed_outcomes(self):\n  expected={c[\'case_id\'] for c in self.old[\'cases\'] if c[\'model\']==\'without_score\'}\n  self.assertEqual(expected,{p[\'case_id\'] for p in self.d[\'pairs\']})\n  for variant in [\'with_score\',\'without_score\']:self.assertEqual(expected,{r[\'case_id\'] for r in self.d[\'records\'] if r[\'model\']==variant})\n  self.assertEqual(sum(p[\'outcome_changed\'] for p in self.d[\'pairs\']),self.d[\'summary\'][\'outcomes_changed\'])\n def test_all_shapley_explanations_reconstruct_predictions(self):\n  for r in self.d[\'records\']:\n   self.assertAlmostEqual(r[\'score\'],r[\'shapley\'][\'diagnostics\'][\'base\']+sum(r[\'shapley\'][\'values\']),places=10)\n   self.assertEqual(len(r[\'shapley\'][\'values\']),len(r[\'features\']))\n def test_original_scores_and_contributions_preserved(self):\n  for c in self.old[\'cases\']:\n   if c[\'model\']!=\'without_score\':continue\n   r=next(r for r in self.d[\'records\'] if r[\'case_id\']==c[\'case_id\'] and r[\'model\']==\'without_score\')\n   self.assertAlmostEqual(c[\'score\'],r[\'score\'],places=10)\n   np.testing.assert_allclose(c[\'explanations\'][\'Exact interventional Shapley\'][0][\'values\'],r[\'shapley\'][\'values\'],atol=1e-10)\n def test_external_summary_uses_positive_contributions_and_correct_denominator(self):\n  records=[r for r in self.d[\'records\'] if r[\'model\']==\'with_score\']\n  n=sum(any(v[\'id\']==\'ExternalRiskEstimate\' for v in r[\'shapley\'][\'individual\']) for r in records)\n  self.assertEqual(n,self.d[\'summary\'][\'external_in_top_adverse\'])\n  self.assertEqual(len(records),self.d[\'summary\'][\'paired_applications\'])\n  for r in records:\n   self.assertEqual(5,len(r[\'surrogate\']))\n   self.assertTrue(all(\'heldout_r2\' in run for run in r[\'surrogate\']))\nif __name__==\'__main__\':unittest.main()\n', 'tests/test_grouping_analysis.py': '"""Verify the descriptive masking count against retained attribution lists."""\nimport json,unittest\nfrom pathlib import Path\nR=Path(__file__).resolve().parents[1]\nclass GroupingAnalysisTests(unittest.TestCase):\n def test_hidden_changes_have_changed_factors_and_unchanged_categories(self):\n  data=json.loads((R/\'results/reference_sensitivity/grouping_analysis.json\').read_text())\n  ref=json.loads((R/\'results/reference_sensitivity/experiment.json\').read_text())\n  hidden=0;apps=set()\n  for r in data[\'records\']:\n   a=next(x for x in ref[\'records\'] if x[\'case_id\']==r[\'case_id\'] and x[\'reference\']==\'population\' and x[\'seed\']==r[\'seed\'])\n   b=next(x for x in ref[\'records\'] if x[\'case_id\']==r[\'case_id\'] and x[\'reference\']==\'near_cutoff\' and x[\'seed\']==r[\'seed\'])\n   detailed={x[\'id\'] for x in a[\'individual\']}!={x[\'id\'] for x in b[\'individual\']}\n   grouped={x[\'id\'] for x in a[\'grouped\']}!={x[\'id\'] for x in b[\'grouped\']}\n   expected=detailed and not grouped\n   self.assertEqual(expected,r[\'hidden_change\'])\n   if expected:hidden+=1;apps.add(r[\'case_id\'])\n  self.assertEqual(hidden,data[\'hidden_change_comparisons\'])\n  self.assertEqual(len(apps),data[\'applications_with_hidden_change\'])\n  self.assertEqual(len(data[\'records\']),data[\'paired_comparisons\'])\nif __name__==\'__main__\':unittest.main()\n', 'tests/test_reference_outputs.py': "import json,unittest\nfrom pathlib import Path\nimport numpy as np\nfrom src.reference_experiment import neighborhood\nR=Path(__file__).resolve().parents[1]\n\nclass ReferenceOutputTests(unittest.TestCase):\n    def test_every_reference_explanation_reconstructs_unchanged_prediction(self):\n        d=json.loads((R/'results/reference_sensitivity/experiment.json').read_text())\n        self.assertEqual(len(d['records']),240)\n        for r in d['records']:\n            self.assertAlmostEqual(r['diagnostics']['base']+sum(r['values']),r['score'],places=10)\n    def test_evaluation_neighborhood_does_not_force_the_fitting_anchor(self):\n        train=np.arange(800,dtype=float).reshape(100,8);x=train[50]\n        z,_,_=neighborhood(x,train,10000,include_anchor=False)\n        self.assertFalse(np.array_equal(z[0],x))\n        fit,_,_=neighborhood(x,train,0)\n        self.assertTrue(np.array_equal(fit[0],x))\n    def test_record_cluster_counts_and_reference_cardinality(self):\n        d=json.loads((R/'results/reference_sensitivity/experiment.json').read_text())\n        for row in d['summary']:\n            self.assertEqual(row['turnover']['records'],40)\n            self.assertEqual(row['turnover']['scenarios'],120)\n        for m in d['reference_manifests']:self.assertEqual(len(m['values']),24)\n\nif __name__=='__main__':unittest.main()\n", 'tests/test_review_parity.cjs': "const fs=require('fs'),path=require('path');\nconst {reviewEvidence}=require('../src/notice_review_legacy.js');\nconst dir=path.join(__dirname,'../results/notice_audit');\nconst inputs=JSON.parse(fs.readFileSync(path.join(dir,'inputs.json')));\nconst expected=new Map(JSON.parse(fs.readFileSync(path.join(dir,'evaluation.json'))).findings.map(x=>[x.id,x]));\nfor(const row of inputs){const a=reviewEvidence(row.bundle),b=expected.get(row.id);if(JSON.stringify(a)!==JSON.stringify({disposition:b.disposition,findings:b.findings}))throw Error('Browser/Python mismatch: '+row.id)}\nfor(const row of JSON.parse(fs.readFileSync(path.join(dir,'challenges.json'))).cases){if(JSON.stringify(reviewEvidence(row.bundle))!==JSON.stringify(row.review))throw Error('Challenge review mismatch')}\nconsole.log('PASS: browser and Python evidence reviewers agree on 360 primary scenarios and 40 challenge examples.');\n", 'tests/test_score_reconstruction.py': '"""Validate measured decompositions, partitions and executable reconstructions."""\nfrom pathlib import Path\nimport sys,json,hashlib,unittest\nimport numpy as np,joblib\nR=Path(__file__).resolve().parents[1];sys.path.insert(0,str(R/\'src\'))\nfrom audit import Predictor\nfrom score_reconstruction import Combined\nclass ScoreReconstructionTests(unittest.TestCase):\n @classmethod\n def setUpClass(cls):\n  cls.d=json.loads((R/\'results/score_reconstruction/experiment.json\').read_text())\n  cls.m=joblib.load(R/\'results/score_reconstruction/models.joblib\')\n def test_split_and_reference_integrity(self):\n  splits=json.loads((R/\'results/reference/split.json\').read_text())\n  self.assertTrue(set(self.d[\'background_record_ids\'])<=set(splits[\'train\']))\n  self.assertTrue(set(c[\'case_id\'] for c in self.d[\'cases\'])<=set(splits[\'test\']))\n  self.assertEqual(len(set(self.d[\'background_record_ids\'])),24)\n def test_all_decompositions_reconstruct_their_own_system(self):\n  for c in self.d[\'cases\']:\n   original=c[\'original_shapley\'];self.assertAlmostEqual(original[\'diagnostics\'][\'base\']+sum(original[\'values\']),c[\'original_probability\'],places=9)\n   for a in c[\'approximations\'].values():\n    self.assertAlmostEqual(a[\'diagnostics\'][\'base\']+sum(a[\'values\']),a[\'probability\'],places=9)\n    self.assertTrue(np.isfinite(a[\'values\']).all())\n def test_executable_models_reproduce_saved_record_predictions(self):\n  m=self.m;predict=Predictor(m[\'original_model\'],m[\'original_encoding\'])\n  for c in self.d[\'cases\']:\n   x=np.array([[c[\'inputs\'][f] for f in self.d[\'features\']]])\n   true=predict(np.column_stack([x,[c[\'observed_external_score\']]]))[0]\n   self.assertAlmostEqual(true,c[\'original_probability\'],places=10)\n   for k,proxy in m[\'approximations\'].items():\n    combined=Combined(proxy,m[\'encoding\'],predict,m[\'score_clipping_bounds\'])\n    self.assertAlmostEqual(combined(x)[0],c[\'approximations\'][k][\'probability\'],places=10)\n def test_agreement_bands_require_original_and_pair_agreement(self):\n  for b in self.d[\'bands\']:\n   t=b[\'probability_band\'];rows=[c for c in self.d[\'cases\'] if c[\'comparison\'][\'probability_difference\']<=t and c[\'comparison\'][\'maximum_original_probability_error\']<=t]\n   self.assertEqual(len(rows),b[\'both_close_to_original\'])\n   self.assertEqual(sum(bool(c[\'comparison\'][\'grouped\'][\'turnover\']) for c in rows),b[\'different_concept_sets_when_both_close_to_original\'])\n def test_protocol_fingerprint_matches_delivered_design(self):\n  self.assertEqual(self.d[\'protocol_sha256\'],hashlib.sha256((R/\'docs/EXTERNAL_SCORE_RECONSTRUCTION.md\').read_bytes()).hexdigest())\nif __name__==\'__main__\':unittest.main()\n', 'tests/test_site_smoke.cjs': '/* Simulated DOM and Blob tests. Not a replacement for browser/visual QA. */\nconst fs=require(\'fs\'),vm=require(\'vm\'),path=require(\'path\'),assert=require(\'assert\');\nconst root=path.resolve(process.argv[2]||\'.\'),html=fs.readFileSync(path.join(root,\'site/index.html\'),\'utf8\');\nlet els=[],ids={},blobs=new Map(),serial=0;\nclass El{\n constructor(tag,attrs){this.tag=tag;this.attrs=attrs;this.id=attrs.id;this.dataset={};for(const[k,v]of Object.entries(attrs))if(k.startsWith(\'data-\'))this.dataset[k.slice(5)]=v;this.hidden=\'hidden\'in attrs;this.value=attrs.value||\'\';this.listeners={};this._html=\'\';}\n set innerHTML(s){this._html=s;parse(s);if(this.tag===\'select\')this.value=(s.match(/<option value="([^"]*)"/)||[])[1]||\'\';}\n get innerHTML(){return this._html}\n setAttribute(k,v){this.attrs[k]=v}\n removeAttribute(k){delete this.attrs[k]}\n addEventListener(k,cb){this.listeners[k]=cb}\n fire(k){return this.listeners[k]?.({target:this})}\n}\nfunction parse(h){for(const m of h.matchAll(/<(\\w+)([^>]*)>/g)){const a={};for(const x of m[2].matchAll(/([\\w-]+)(?:="([^"]*)")?/g))a[x[1]]=x[2]||\'\';if(a.id||a.class||Object.keys(a).some(k=>k.startsWith(\'data-\'))){const e=new El(m[1],a);els.push(e);if(a.id)ids[a.id]=e;}}}\nparse(html);\nconst doc={getElementById:id=>{if(id===\'score-investigation-render\')return null;assert(ids[id],\'Missing \'+id);return ids[id]},querySelector:sel=>{assert(sel.startsWith(\'#\'));assert(ids[sel.slice(1)],\'Missing \'+sel);return ids[sel.slice(1)]},querySelectorAll:sel=>{\n if(sel===\'.page\')return els.filter(e=>e.attrs.class===\'page\'||e.attrs.class?.split(\' \').includes(\'page\'));\n if(sel===\'nav [data-page]\')return els.filter(e=>e.tag===\'button\'&&e.dataset.page&&[\'The project\',\'\'].includes(\'\')).filter(e=>e.dataset.page&& !e.attrs.class);\n if(sel.startsWith(\'[data-\'))return els.filter(e=>e.dataset[sel.slice(6,-1)]!==undefined);\n throw new Error(\'Unexpected selector \'+sel);\n}};\nconst ctx={document:doc,window:{scrollTo:()=>{},print:()=>{}},console,Blob,URL:{createObjectURL:b=>{const u=\'blob:mock-\'+(++serial);blobs.set(u,b);return u},revokeObjectURL:u=>blobs.delete(u)}};\nvm.createContext(ctx);for(const f of [\'data.js\',\'evidence.js\',\'app.js\'])vm.runInContext(fs.readFileSync(path.join(root,\'site\',f),\'utf8\'),ctx);\n(async()=>{\n assert(!ids.about.hidden);vm.runInContext("page(\'investigate\')",ctx);assert(!ids.investigate.hidden);assert(ids.about.hidden);\n const all=vm.runInContext(\'S.cases.map(c=>c.case_id)\',ctx);\n for(const id of all){\n  ids[\'case-select\'].value=String(id);ids[\'case-select\'].fire(\'change\');\n  assert.equal(vm.runInContext(\'caseId\',ctx),id);\n  assert(ids[\'case-content\'].innerHTML.includes(\'Shapley contributions to the repayment prediction\'));\n  vm.runInContext("page(\'investigate\')",ctx);assert(!ids[\'case-toolbar\'].hidden);\n  vm.runInContext("page(\'compare\')",ctx);assert(!ids.compare.hidden);assert(ids.investigate.hidden);\n  ids[\'topics-view\'].fire(\'click\');assert.equal(vm.runInContext(\'detail\',ctx),\'topics\');\n  assert(ids[\'approximation-charts\'].innerHTML.includes(\'Revolving balances\'));\n  assert.equal(ids[\'topics-view\'].attrs[\'aria-pressed\'],\'true\');\n  ids[\'factors-view\'].fire(\'click\');assert.equal(vm.runInContext(\'detail\',ctx),\'factors\');\n  assert(ids[\'approximation-charts\'].innerHTML.includes(\'Average account age\'));\n  assert(ids[\'local-fit\'].innerHTML.includes(\'median weighted R²\'));\n }\n ids[\'case-select\'].value=\'8193\';ids[\'case-select\'].fire(\'change\');\n ids[\'prepare-report\'].fire(\'click\');assert(!ids[\'report-output\'].hidden);\n const note=await blobs.get(ids[\'download-report\'].href).text();assert(note.includes(\'WHAT REMAINS UNESTABLISHED\'));assert(note.includes(\'Historical record: 8193\'));assert(note.startsWith(\'<!doctype html>\'));\n const data=JSON.parse(await blobs.get(ids[\'download-json\'].href).text());assert.equal(data.case_id,8193);assert.equal(data.measurements.case_id,8193);assert(data.findings.unestablished_claim.includes(\'original external credit-risk score\'));\n fs.mkdirSync(path.join(root,\'site/examples\'),{recursive:true});fs.writeFileSync(path.join(root,\'site/examples/external-score-review-8193.html\'),note);fs.writeFileSync(path.join(root,\'site/examples/external-score-review-8193.json\'),JSON.stringify(data,null,2));\n ids[\'topics-view\'].fire(\'click\');assert(ids[\'report-output\'].hidden);assert.equal(blobs.size,0);\n vm.runInContext("page(\'research\')",ctx);assert(!ids.research.hidden);assert(!ids[\'case-toolbar\'].hidden);assert(ids[\'protocol-tests\'].innerHTML.includes(\'Decision fidelity\'));assert(ids[\'protocol-case-matrix\'].innerHTML.includes(\'record 8193\'));vm.runInContext("page(\'methods\')",ctx);assert(!ids.methods.hidden);vm.runInContext("page(\'legal\')",ctx);assert(!ids.legal.hidden);vm.runInContext("page(\'evidence\')",ctx);assert(!ids.evidence.hidden);assert(ids.legal.hidden);assert(ids[\'findings-summary\'].innerHTML.includes(\'4 of 12\'));assert(ids[\'agreement-table\'].innerHTML.includes(\'12\'));\n for(let index=0;index<4;index++){\n  ids[\'agreement-band\'].value=String(index);ids[\'agreement-band\'].fire(\'input\');\n  const band=vm.runInContext(\'S.bands[\'+index+\']\',ctx);\n  assert(ids[\'findings-summary\'].innerHTML.includes(band.different_factor_sets_when_both_close_to_original+\' of \'+band.both_close_to_original));\n  assert.equal(ids[\'agreement-band\'].attrs[\'aria-valuetext\'],(band.probability_band*100).toFixed(1)+\' percentage points\');\n }\n ids[\'other-methods-view\'].fire(\'click\');assert(ids[\'requirements-panel\'].hidden);assert(!ids[\'other-methods-panel\'].hidden);assert.equal(ids[\'evidence-switch\'].dataset.mode,\'methods\');\n ids[\'requirements-view\'].fire(\'click\');assert(!ids[\'requirements-panel\'].hidden);assert(ids[\'other-methods-panel\'].hidden);\n assert(!html.includes(\'Newer policy supplied\'));assert(!html.includes(\'Review findings\'));\n console.log(\'PASS: all forty records, both factor/topic views, all seven navigation pages, local-fidelity evidence, all four measured slider positions, evidence/method selector, readable/JSON note contents and stale-export invalidation. Browser visual and download behavior remain outside this simulation.\');\n})().catch(e=>{console.error(e);process.exitCode=1});\n', 'results/model_benchmark.json': '{\n  "selection": "Highest validation AUC among six predeclared candidates. Only selected candidate evaluated on test. Existing split reused; original eight-feature test score was already known.",\n  "candidates": [\n    {\n      "scope": "all_23",\n      "model": "Logistic regression",\n      "features": 23,\n      "validation_auc": 0.8002866939434103,\n      "validation_brier": 0.1814925221130811\n    },\n    {\n      "scope": "all_23",\n      "model": "Random Forest",\n      "features": 23,\n      "validation_auc": 0.7958617694758421,\n      "validation_brier": 0.18325147185457327\n    },\n    {\n      "scope": "all_23",\n      "model": "Histogram gradient boosting",\n      "features": 23,\n      "validation_auc": 0.7996877855619859,\n      "validation_brier": 0.18175961725094913\n    },\n    {\n      "scope": "all_22_without_external_score",\n      "model": "Logistic regression",\n      "features": 22,\n      "validation_auc": 0.8003638728585423,\n      "validation_brier": 0.18154901586375666\n    },\n    {\n      "scope": "all_22_without_external_score",\n      "model": "Random Forest",\n      "features": 22,\n      "validation_auc": 0.8022233701871229,\n      "validation_brier": 0.18073277856181175\n    },\n    {\n      "scope": "all_22_without_external_score",\n      "model": "Histogram gradient boosting",\n      "features": 22,\n      "validation_auc": 0.8012684097438894,\n      "validation_brier": 0.1805879011073478\n    }\n  ],\n  "selected": {\n    "scope": "all_22_without_external_score",\n    "model": "Random Forest",\n    "features": 22,\n    "validation_auc": 0.8022233701871229,\n    "validation_brier": 0.18073277856181175,\n    "test_auc": 0.7846935417523653,\n    "test_brier": 0.18856521599401835\n  },\n  "audit_status": "Benchmark only. Existing explanation audit still describes the eight/nine-feature Random Forests; explanations have not been rerun for this candidate."\n}', 'results/protocol_v2_lock.json': '{\n  "protocol_sha256": "0c209583892ed35cc38608f89dd9c8d0429b80539b32de72496949c2088a12fc",\n  "frozen_at": "2026-10-16T03:50:50.374604+00:00",\n  "note": "Local system clock; protocol predates new result generation, not an external preregistration."\n}'}
for name,text in FILES.items():
    path=ROOT/name
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(text)
os.chdir(ROOT)
print(f'Staged {len(FILES)} project files in {ROOT}')


## Install pinned dependencies


In [ ]:
%pip install -q -r requirements-reference.txt


## Supply the dataset

The project uses the public FICO HELOC dataset. The data include `ExternalRiskEstimate`, but not the formula or provider reasoning behind that field. Upload the CSV only if you are authorized to use it.


In [ ]:

import hashlib
from pathlib import Path
Path('data').mkdir(exist_ok=True)
try:
    from google.colab import files as colab_files
    uploaded=colab_files.upload()
    if 'heloc_dataset_v1.csv' not in uploaded:
        raise FileNotFoundError('Upload a file named heloc_dataset_v1.csv')
    Path('data/heloc_dataset_v1.csv').write_bytes(uploaded['heloc_dataset_v1.csv'])
except ImportError:
    print('Jupyter: place heloc_dataset_v1.csv in', Path.cwd()/'data')
DATA=Path('data/heloc_dataset_v1.csv')
assert DATA.exists(), 'heloc_dataset_v1.csv is required'
print('SHA-256:', hashlib.sha256(DATA.read_bytes()).hexdigest())


## Run the analysis

The commands below fit the audit models, run the external-score and grouping comparisons, rebuild the workbench, and execute the checks used by the site.


In [ ]:

import subprocess, sys, os
commands=[
 ['src/audit.py','--data',str(DATA),'--out','results/reference','--cases','40','--repeats','5'],
 ['src/notice_benchmark.py'],
 ['src/reference_experiment.py'],
 ['src/challenge_examples.py'],
 ['src/external_score_experiment.py'],
 ['src/grouping_analysis.py'],
 ['src/decision_reconstruction.py'],
 ['src/score_reconstruction.py'],
 ['src/build_workbench.py'],
]
env=os.environ.copy(); env['OPENBLAS_NUM_THREADS']='2'; env['OMP_NUM_THREADS']='2'
for args in commands:
    print('Running', args[0], flush=True)
    subprocess.run([sys.executable,*args], check=True, env=env)


## Verify the headline result


In [ ]:

import json
from pathlib import Path
study=json.loads(Path('results/score_reconstruction/experiment.json').read_text())
b=next(x for x in study['bands'] if abs(x['probability_band']-0.01)<1e-12)
print('1.0 pp agreement band:', b['both_close_to_original'], 'records')
print('different detailed-factor sets:', b['different_factor_sets_when_both_close_to_original'])
print('different grouped-topic sets:', b['different_concept_sets_when_both_close_to_original'])
assert b['both_close_to_original']==12
assert b['different_factor_sets_when_both_close_to_original']==4
assert b['different_concept_sets_when_both_close_to_original']==0
print('PASS: headline result reproduced')


## Interpret the result without overclaiming

- Close downstream predictions can coexist with different detailed factor attributions.
- Stable broad topic sets do not authenticate the original external-score mechanism.
- The reconstruction is useful as a **stress test and diagnostic**, not as provider-linked reason evidence.
- An actual adverse-action review would still need the lender's decision path, mapping/version, delivered notice, and provider-linked score reasons or validation evidence where relevant.

**Prediction agreement is not explanation provenance.**

## Run static tests


In [ ]:

subprocess.run(['node','tests/test_site_smoke.cjs','.'],check=True)
subprocess.run(['node','tests/test_evidence_engine.cjs'],check=True)


## Package reproduced outputs


In [ ]:

import zipfile
with zipfile.ZipFile('reproduced_outputs.zip','w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['results','site','docs']:
        for p in Path(folder).rglob('*'):
            if p.is_file(): z.write(p,str(p))
print(Path('reproduced_outputs.zip').resolve())
try:
    from google.colab import files as colab_files
    colab_files.download('reproduced_outputs.zip')
except ImportError:
    pass
